# Proactive Churn - v1 feature selection

Reads the encoded model matrix `data/dataset_creation.ipynb` wrote into `data/raw/`, runs BorutaSHAP
on the `train` cell, and scores the full candidate set against the BorutaSHAP-accepted subset on the
two readable holdout cells. No Redshift connection - this notebook is offline.

**What it reads**, from `data/raw/` (gitignored), located by glob rather than by literal name because
the stem carries the extract's date range:

| glob | contents |
| --- | --- |
| `churn_v1_*_h*_encoded.parquet` | the model matrix - keys, label, `split`, passthroughs, raw categoricals, all of `model_cols` |
| `churn_v1_*_h*_encoding.json` | the encoding schema - `model_cols` in exact order, `categorical_features`, keys, passthroughs |

`model_cols` from the JSON **is** the feature list. This notebook does not re-derive it.

**What it produces**, in `data/raw/`:

| file | contents |
| --- | --- |
| `..._boruta_selection.json` | accepted / tentative / rejected lists + the run config (trials, sample size, seeds, source parquet) |
| `..._boruta_importance.csv` | the SHAP importance table - per feature mean / sd importance, decision, rank |
| `..._selection_eval.csv` | precision@k / recall@k / PR-AUC, full vs. selected, per holdout cell |
| `..._k_sweep.csv` | precision / recall / at-risk found per month / lift across `k` = 40 -> the whole month |
| `..._baseline_comparison.csv` | the model vs. the Retention Health Score at matched list sizes, plus the lead-list trigger rule |

**Split discipline.** Fit and select on `train` only. Read `holdout_school` and `holdout_time`.
`test` is the ship gate: it is touched once, later, and no cell here loads it. `embargo` and
`retrain_only` are excluded from everything. §6 asserts all of this rather than documenting it.

**Sections**

1. Imports
2. The `scipy.stats.binom_test` shim - runs BEFORE §3
3. Import BorutaShap
4. Locate the inputs
5. Load the matrix and the schema
6. Split discipline
7. `X` / `y` and the feature guardrails
8. Model configuration
9. BorutaSHAP
10. Persist the selection
11. Evaluation harness (11a. helpers, 11b. full vs. selected)
12. The `k` sweep and the operating point
13. Leakage / business-sense surface
14. Baseline comparison: model vs the Retention Health Score (14a. the rebuilt trend, 14b. three rankers, 14c. the trigger as used)
15. Audit, and what is deliberately not here

**Before you run anything:**

1. **Run `data/dataset_creation.ipynb` first** - §11 of it writes both input files. No VPN needed here.
2. **Kernel = "Python (Proactive Churn)"** - created by `setup.sh`. The pins in `requirements.txt`
   are load-bearing; §1 and §2 assert the two that this notebook can check.

Design decisions and the rationale behind them live in the Obsidian vault at
`Proactive Churn/Design Decisions.md`.

## 1. Imports

Modelling stack only - no `redshift_connector`, no `.env`. BorutaShap is **not** imported here; it
imports `scipy.stats.binom_test` at module level, so §2 has to run first.

The two pins this cell can check are asserted: `numpy<2` (BorutaShap uses `np.NaN`) and `xgboost<3`
(`shap`'s XGBoost tree loader cannot parse the 3.x `base_score` serialization). The third,
`scipy<1.14`, is §2's subject.

In [ ]:
# --- stdlib ---
import ast          # §6 statically scans this notebook's own code cells
import json
import re
import time
from pathlib import Path

# --- data ---
import numpy as np
import pandas as pd

# --- modelling ---
import shap
import sklearn
import xgboost as xgb
from sklearn.metrics import average_precision_score
from xgboost import XGBClassifier

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# --- the two pins that are checkable from here -----------------------------
# BorutaShap 1.0.17 (last released Dec 2023) is unmaintained. requirements.txt
# pins around three incompatibilities; these two cannot be shimmed at runtime, so
# fail here rather than several minutes into §9.
assert int(np.__version__.split(".")[0]) < 2, (
    f"numpy {np.__version__} - BorutaShap 1.0.17 calls np.NaN in "
    "update_importance_history(), removed in numpy 2.0, so §9 dies partway "
    "through the trial loop. requirements.txt pins numpy>=1.26,<2. Fix the "
    "environment, then RESTART THE KERNEL."
)
assert int(xgb.__version__.split(".")[0]) < 3, (
    f"xgboost {xgb.__version__} - shap's XGBTreeModelLoader cannot parse the 3.x "
    "base_score serialization, and BorutaSHAP's importance measure is SHAP, so "
    "every trial fails. requirements.txt pins xgboost>=2.1,<3."
)

print(f"numpy        {np.__version__}")
print(f"pandas       {pd.__version__}")
print(f"scikit-learn {sklearn.__version__}")
print(f"xgboost      {xgb.__version__}")
print(f"shap         {shap.__version__}")
print("\nBorutaShap is imported in §3, AFTER §2's shim.")

## 2. The `scipy.stats.binom_test` shim

**Run this before §3.** BorutaShap 1.0.17 does `from scipy.stats import binom_test, ks_2samp` at
module level. `binom_test` was removed in scipy 1.12, and the project pins `scipy>=1.11,<1.14`, so
whether the name exists depends on which end of that range is installed. The shim defines it in terms
of `scipy.stats.binomtest` - returning the `.pvalue` - **only if it is absent**, and prints which of
the two states it found.

`BorutaShap.binomial_H0_test` passes the hit counts straight through, and those are floats from
`np.zeros`, so the shim casts `x` and `n` to `int` - `binomtest` rejects a float `k`. A known value
is asserted against the closed form before §3 imports anything.

This is documented in `README.md` alongside the pins.

In [ ]:
import scipy
from scipy import stats as _st

BINOM_TEST_SHIM_NEEDED = not hasattr(_st, "binom_test")

if BINOM_TEST_SHIM_NEEDED:

    def binom_test(x, n=None, p=0.5, alternative="two-sided"):
        """scipy.stats.binom_test (removed in scipy 1.12) via binomtest.

        Signature-compatible with what BorutaShap 1.0.17 calls:
        binom_test(x, n=n, p=0.5, alternative='greater' | 'less').

        x and n are cast to int because BorutaShap passes the hit counters, which
        are floats out of np.zeros / np.vstack, and binomtest rejects a float k.
        """
        return float(_st.binomtest(int(x), n=int(n), p=p, alternative=alternative).pvalue)

    _st.binom_test = binom_test

# Whichever branch ran, the name has to be importable from scipy.stats now -
# BorutaShap's module-level import is what consumes it.
assert hasattr(scipy.stats, "binom_test"), (
    "scipy.stats.binom_test is still missing, so §3's `from BorutaShap import "
    "BorutaShap` will raise ImportError. The shim assigns onto the module object "
    "`scipy.stats`; if that is not the module BorutaShap imports from, something "
    "has shadowed it in this kernel - restart and re-run §1 -> §2 -> §3 in order."
)

# 8 or more heads in 10 fair tosses: sum(C(10,k)/2^10 for k in 8..10) = 56/1024.
_p = scipy.stats.binom_test(8, n=10, p=0.5, alternative="greater")
assert np.isclose(_p, 56 / 1024), (
    f"binom_test(8, n=10, p=0.5, alternative='greater') returned {_p!r}, expected "
    f"{56 / 1024}. A wrong one-sided p-value would not raise in §9 - it would "
    "silently change which features BorutaSHAP accepts."
)

print(f"scipy {scipy.__version__}")
print(f"shim  {'INSTALLED - scipy.stats had no binom_test' if BINOM_TEST_SHIM_NEEDED else 'NOT NEEDED - scipy.stats.binom_test exists natively'}")
print(f"check binom_test(8, n=10, p=.5, 'greater') = {_p:.10f} == 56/1024  OK")

## 3. Import BorutaShap

Imports the package and pins its API by assertion, because the signature is unusual in two ways that
matter and neither raises anything readable if it drifts:

- `importance_measure` must be the **lowercase** `"shap"`. The constructor's own default is
  `"Shap"`, and `feature_importance()` compares `== "shap"`, so the default value raises
  `ValueError: No Importance_measure was specified` on the first trial.
- `fit()` takes `n_trials`, `random_state`, `sample`, `train_or_test`, `normalize`, `verbose` and
  `stratify` - not the scikit-learn-shaped signature the class's `get_params` / `set_params` imply.

`train_or_test` is BorutaShap's own in-sample / out-of-sample switch for the SHAP computation and has
**nothing to do with this project's `test` split cell**; §9 passes `"train"`.

In [ ]:
import inspect

from BorutaShap import BorutaShap

# LOWERCASE. The constructor default is "Shap"; feature_importance() compares
# `self.importance_measure == "shap"` and raises ValueError on anything else, so
# the default breaks the class.
IMPORTANCE_MEASURE = "shap"
assert IMPORTANCE_MEASURE == "shap", (
    "IMPORTANCE_MEASURE must be the lowercase 'shap'. BorutaShap's constructor "
    "defaults to 'Shap' and its feature_importance() does an exact "
    "`== 'shap'` comparison, so the capitalised default raises "
    "ValueError('No Importance_measure was specified select one of (shap, perm, "
    "gini)') on the first trial - after the first model fit, several minutes in."
)

_init_params = set(inspect.signature(BorutaShap.__init__).parameters)
_fit_params = set(inspect.signature(BorutaShap.fit).parameters)

_expected_init = {"model", "importance_measure", "classification", "percentile", "pvalue"}
_expected_fit = {"X", "y", "sample_weight", "n_trials", "random_state", "sample",
                 "train_or_test", "normalize", "verbose", "stratify"}

assert _expected_init <= _init_params, (
    f"BorutaShap.__init__ is missing {sorted(_expected_init - _init_params)}. The "
    "installed version is not the pinned 1.0.17 and §9 passes keywords it does "
    "not accept."
)
assert _expected_fit <= _fit_params, (
    f"BorutaShap.fit is missing {sorted(_expected_fit - _fit_params)}. The "
    "installed version is not the pinned 1.0.17."
)

print(f"BorutaShap.__init__{inspect.signature(BorutaShap.__init__)}")
print(f"BorutaShap.fit{inspect.signature(BorutaShap.fit)}")
print(f"\nimportance_measure = {IMPORTANCE_MEASURE!r} (lowercase - the 'Shap' default raises)")

## 4. Locate the inputs

Globs `data/raw/` for the encoded parquet and the encoding JSON and demands **exactly one** of each.
The stem embeds the extract's observation window (`churn_v1_is_reportable_request_202401_202605_h3`),
so it changes whenever `AS_OF_DATE` or `OBS_START` moves and must not be hard-coded here.

Zero matches and more than one match both fail with a message naming the directory and what it found:
zero means the extract has not been run, and two means an old extract is still sitting there and
picking one silently would make every number below untraceable.

`STEM` is derived from the parquet's own name and is what §10 onwards name their outputs after, so
every artefact in `data/raw/` sorts next to the extract it came from.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "data" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"

# This notebook's own file - §6 reads it to scan its code cells.
NB_PATH = DATA_DIR / "feature_selection.ipynb"

# Globs, not names: the stem carries the date range the extract was built over.
ENCODED_GLOB = "churn_v1_*_h3_encoded.parquet"
ENCODING_GLOB = "churn_v1_*_h3_encoding.json"


def one_match(directory: Path, pattern: str) -> Path:
    """The single file in `directory` matching `pattern`, or a clear failure.

    Zero matches and 2+ matches are both fatal: zero means the extract was never
    run, and picking one of several silently would leave the run untraceable.
    """
    if not directory.is_dir():
        raise FileNotFoundError(
            f"no such directory: {directory}. It is created by "
            "data/dataset_creation.ipynb §2 and is gitignored, so a fresh clone "
            "has neither the directory nor the extract. Run the extract notebook "
            "first."
        )

    hits = sorted(directory.glob(pattern))

    if not hits:
        _present = sorted(p.name for p in directory.iterdir() if p.is_file())
        raise FileNotFoundError(
            f"no file matching {pattern!r} in {directory}. Run "
            "data/dataset_creation.ipynb through §11 - it writes the encoded "
            f"parquet and the encoding JSON. Files present ({len(_present)}): "
            f"{_present[:12]}{' ...' if len(_present) > 12 else ''}"
        )

    if len(hits) > 1:
        raise ValueError(
            f"{len(hits)} files match {pattern!r} in {directory}: "
            f"{[p.name for p in hits]}. The stem carries the extract's date "
            "range, so more than one match means an older extract is still "
            "there. Choosing one here would make every number in this notebook "
            "untraceable - move or delete the stale extract (both the parquet "
            "and its JSON), then re-run this cell."
        )

    return hits[0]


ENCODED = one_match(RAW_DIR, ENCODED_GLOB)
ENCODING = one_match(RAW_DIR, ENCODING_GLOB)

_SUFFIX = "_encoded.parquet"
assert ENCODED.name.endswith(_SUFFIX), f"unexpected parquet name: {ENCODED.name}"
STEM = ENCODED.name[: -len(_SUFFIX)]

assert ENCODING.name == f"{STEM}_encoding.json", (
    f"the two inputs do not come from the same extract: parquet {ENCODED.name!r} "
    f"has stem {STEM!r}, but the JSON is {ENCODING.name!r} (expected "
    f"{STEM}_encoding.json). §11 of the extract derives both names from one stem, "
    "so a mismatch means the two files are from different runs and model_cols may "
    "not describe this matrix."
)

print(f"raw dir   {RAW_DIR}")
print(f"parquet   {ENCODED.name}  ({ENCODED.stat().st_size / 1e6:.1f} MB)")
print(f"encoding  {ENCODING.name}  ({ENCODING.stat().st_size / 1e3:.1f} KB)")
print(f"stem      {STEM}")

## 5. Load the matrix and the schema

Reads the JSON first and the parquet second, and names `MODEL_COLS` from the sidecar. **`model_cols`
is the feature list**; nothing here re-derives it from the parquet's columns, because a list rebuilt
by pattern would silently disagree with the one a scoring job replays.

The two artefacts are then made to agree, in this order: every `model_cols` entry exists in the
parquet (a name in the JSON that the matrix does not have is the failure that would otherwise surface
as a `KeyError` in §7), no duplicates, and `model_cols` is the tail of the column list in order.

Prints the row count, the schema's categorical features with their frozen train levels, and the
`split` census.

In [ ]:
SCHEMA = json.loads(ENCODING.read_text())

MODEL_COLS_FILE = list(SCHEMA["model_cols"])       # exactly what the file carries
MODEL_COLS = list(MODEL_COLS_FILE)                 # authoritative, not re-derived

# --- late drops -------------------------------------------------------------
# Escape hatch for a feature found to be unusable AFTER the extract ran, so a
# selection re-run costs seconds instead of a re-extract. The permanent home for
# an exclusion is EXCLUDE_PATTERNS in dataset_creation.ipynb; anything listed
# here should already be there, and once the extract is re-run this list becomes
# a no-op (it drops nothing and says so).
#
# fee_percentage: separated cancellers at AUC ~0.85 within a 2024 month and
# ~0.50 within a 2025 month. Expired, and it dominated the model while it lasted.
DROP_FEATURE_PATTERNS = [r"fee_percentage"]

_dropped_late = [c for c in MODEL_COLS
                 if any(re.search(p, c) for p in DROP_FEATURE_PATTERNS)]
if _dropped_late:
    MODEL_COLS = [c for c in MODEL_COLS if c not in set(_dropped_late)]
    print(f"DROP_FEATURE_PATTERNS removed {len(_dropped_late)} of "
          f"{len(_dropped_late) + len(MODEL_COLS)} feature(s) before selection:")
    for c in _dropped_late:
        print(f"    {c}")
    print("  -> the extract still carries them; they are excluded from X only.")
else:
    print("DROP_FEATURE_PATTERNS matched nothing - the extract already excludes them.")

CATEGORICAL_FEATURES = dict(SCHEMA["categorical_features"])
KEYS = list(SCHEMA["keys"])
Y_COL = SCHEMA["y_col"]
PASSTHROUGH = list(SCHEMA["passthrough"])
NON_FEATURE_CARRIED = list(SCHEMA.get("non_feature_carried", []))
DUMMY_SEP = SCHEMA.get("dummy_separator", "__")

enc = pd.read_parquet(ENCODED)
enc["dim_month"] = pd.to_datetime(enc["dim_month"])

# --- do the JSON and the parquet describe the same matrix? ------------------
_missing = [c for c in MODEL_COLS if c not in enc.columns]
assert not _missing, (
    f"{len(_missing)} column(s) in the JSON's model_cols are NOT in "
    f"{ENCODED.name}: {_missing[:20]}{' ...' if len(_missing) > 20 else ''}\n"
    "  The sidecar and the parquet are from different runs, or the parquet was "
    "rewritten by hand. model_cols is the contract a scoring job replays, so "
    "this notebook refuses to select features against a matrix that does not "
    "match it. Re-run data/dataset_creation.ipynb §11 to regenerate both."
)

_dupes = sorted({c for c in MODEL_COLS if MODEL_COLS.count(c) > 1})
assert not _dupes, (
    f"duplicate entries in the JSON's model_cols: {_dupes}. A booster fitted on "
    "this list and a matrix selected by it would disagree about width."
)

# Checked against the UNFILTERED list from the sidecar: the layout contract is a
# property of the file, not of whatever this notebook then chooses to model on.
assert list(enc.columns)[-len(MODEL_COLS_FILE):] == MODEL_COLS_FILE, (
    "model_cols is not the tail of the parquet's column list, in order. §11 of "
    "the extract asserts that it is, so the parquet has been rewritten since. "
    "Column order is part of the encoding contract - regenerate rather than "
    "reordering here."
)

_required = set(KEYS) | {Y_COL, "split", "school_group"}
_absent = sorted(_required - set(enc.columns))
assert not _absent, (
    f"the encoded parquet is missing {_absent}, which this notebook needs for "
    "cell selection and per-month metrics. Regenerate it from the extract."
)

print(f"{len(enc):,} rows x {enc.shape[1]} columns")
print(f"MODEL_COLS            {len(MODEL_COLS)}")
print(f"KEYS                  {KEYS}")
print(f"Y_COL                 {Y_COL}")
print(f"PASSTHROUGH           {PASSTHROUGH}   (never features)")
print(f"NON_FEATURE_CARRIED   {NON_FEATURE_CARRIED}")
print(f"months                {enc['dim_month'].min():%Y-%m} -> {enc['dim_month'].max():%Y-%m}")
print(f"schools               {enc['school_id'].nunique():,}")
print(f"positives             {int(enc[Y_COL].sum()):,} ({enc[Y_COL].mean():.2%} of rows)")

print(f"\ncategorical features encoded by the extract ({len(CATEGORICAL_FEATURES)}), "
      "levels frozen from `train`:")
for _c, _levels in CATEGORICAL_FEATURES.items():
    print(f"  {_c:<36}{len(_levels):>3} level(s) -> {len(_levels) + 2} dummies "
          f"(+{DUMMY_SEP}other, +{DUMMY_SEP}isna)")

_dummy_cols = [c for c in MODEL_COLS
               if any(c.startswith(f"{k}{DUMMY_SEP}") for k in CATEGORICAL_FEATURES)]
print(f"\nMODEL_COLS = {len(MODEL_COLS) - len(_dummy_cols)} numeric + {len(_dummy_cols)} dummies")

---

## 6. Split discipline

The 2x2 cells the extract wrote onto `split`. This notebook's rules, and how each one is enforced
rather than described:

| cell | role here | enforced by |
| --- | --- | --- |
| `train` | the only cell fitted on, and the only cell BorutaSHAP sees | `cell()` is the only row selector |
| `holdout_school` | read: new schools, same era. Half its schools carve §8's early-stopping eval set | `cell()`; the two halves are asserted school-disjoint in §8 |
| `holdout_time` | read: same schools, new era | `cell()` |
| `test` | **the ship gate.** Touched once, later, by a different notebook. No cell here loads it | `cell()` refuses it; §6's static scan; §15's access log |
| `embargo` | excluded | `cell()` refuses it |
| `retrain_only` | excluded | `cell()` refuses it |

Three enforcement mechanisms, because "documented" is not "enforced":

1. **`cell(split)` is the only way this notebook selects rows** and it raises on anything outside
   `LOADABLE_SPLITS`. Every access is appended to `SPLIT_ACCESS_LOG`.
2. **A static scan of this notebook's own code cells** (`ast`, over `NB_PATH`) fails if any cell
   compares against the literal `"test"` outside an `assert`, or passes it to a call. A comparison
   inside an assertion is the *point* - a filter is not.
3. **§15 re-reads `SPLIT_ACCESS_LOG`** at the end of the run and asserts the ship gate is absent.

This cell also **re-verifies school-disjointness from the file** - `train` against `test` and
`train` against `holdout_school`. The extract asserts it, but an input that has been regenerated,
edited or partially overwritten should not be trusted silently. The check reads `school_id` and
`split` only; no feature of a `test` row is ever loaded.

In [ ]:
FIT_SPLIT = "train"
READ_SPLITS = ("holdout_school", "holdout_time")
SHIP_GATE_SPLIT = "test"                       # named so it can be asserted against
EXCLUDED_SPLITS = ("embargo", "retrain_only")

LOADABLE_SPLITS = (FIT_SPLIT,) + READ_SPLITS
NEVER_LOADED = (SHIP_GATE_SPLIT,) + EXCLUDED_SPLITS
ALL_SPLITS = LOADABLE_SPLITS + NEVER_LOADED

_WHY_NOT_LOADABLE = {
    SHIP_GATE_SPLIT: (
        "the ship gate - new schools in a new era, the honest deployment number. "
        "It is read ONCE, later, on a model that is already final. Every look at "
        "it spends a little of its value; a look during feature selection spends "
        "all of it, because the selection is then fitted to it."
    ),
    "embargo": (
        "the months between the training window and the evaluation window, "
        "excluded so no training row's label window straddles the seam. Labelled "
        "rather than deleted by the extract so the cost is visible."
    ),
    "retrain_only": (
        "answerable months after the evaluation window, kept for the live "
        "retrain. Never evaluated."
    ),
}

SPLIT_ACCESS_LOG = []


def cell(split: str, cols=None) -> pd.DataFrame:
    """Rows of one split cell. The ONLY row selector in this notebook.

    Raises on any cell that must not be loaded, and logs every access into
    SPLIT_ACCESS_LOG so §15 can audit what this run actually touched.
    """
    if split not in LOADABLE_SPLITS:
        _why = _WHY_NOT_LOADABLE.get(split, "not a split cell of this dataset")
        raise ValueError(
            f"cell({split!r}) refused. Loadable here: {list(LOADABLE_SPLITS)}.\n"
            f"  {split!r}: {_why}\n"
            "  If you need this cell, you are writing a different notebook."
        )

    SPLIT_ACCESS_LOG.append(split)
    frame = enc.loc[enc["split"] == split]
    return frame if cols is None else frame[cols]


def Xy(split: str):
    """(X, y) for one cell: X is exactly MODEL_COLS, in order; y is int8."""
    frame = cell(split)
    return frame[MODEL_COLS], frame[Y_COL].astype("int8")


# --- the census, over every cell in the file --------------------------------
# Counted from `split` / `school_id` / the label only. Reading how many rows the
# ship gate has is not the same as loading its features, and the number is needed
# to state what is being left on the table.
_census = (enc.assign(_s=pd.Categorical(enc["split"], categories=ALL_SPLITS, ordered=True))
              .groupby("_s", observed=False)
              .agg(rows=(Y_COL, "size"),
                   schools=("school_id", "nunique"),
                   positives=(Y_COL, "sum"),
                   pos_rate=(Y_COL, "mean"),
                   first_month=("dim_month", "min"),
                   last_month=("dim_month", "max")))
_census["pos_rate"] = (_census["pos_rate"] * 100).round(2).astype(str) + "%"
_census["first_month"] = _census["first_month"].dt.strftime("%Y-%m")
_census["last_month"] = _census["last_month"].dt.strftime("%Y-%m")
_census["role"] = ["FIT + SELECT" if s == FIT_SPLIT
                   else "read" if s in READ_SPLITS
                   else "SHIP GATE - not loaded" if s == SHIP_GATE_SPLIT
                   else "excluded"
                   for s in ALL_SPLITS]
_census.index.name = "split"
print(_census.to_string())

_unexpected = sorted(set(enc["split"].astype(str).unique()) - set(ALL_SPLITS))
assert not _unexpected, (
    f"the parquet carries split value(s) this notebook does not know: "
    f"{_unexpected}. The extract writes exactly {list(ALL_SPLITS)}; an extra "
    "value means the split cell changed and the roles above may be wrong."
)
for _s in LOADABLE_SPLITS:
    assert (enc["split"] == _s).any(), (
        f"the {_s!r} cell is EMPTY in {ENCODED.name}. Nothing below can run - "
        "check §9 of the extract, which prints the per-cell composition."
    )

# --- re-verify school-disjointness FROM THE FILE ---------------------------
# Identifiers only: no feature of any non-loadable row is read here.
_ids_in = enc.groupby("split", observed=True)["school_id"].agg(set).to_dict()
for _a, _b in ((FIT_SPLIT, SHIP_GATE_SPLIT), (FIT_SPLIT, "holdout_school")):
    _shared = _ids_in.get(_a, set()) & _ids_in.get(_b, set())
    assert not _shared, (
        f"{len(_shared):,} school_id(s) appear in BOTH the {_a!r} and {_b!r} "
        f"cells, e.g. {sorted(_shared)[:5]}. The extract's §9 asserts the A/B "
        "school partition is disjoint, so this file has been regenerated with a "
        "different rule, edited, or concatenated from two runs. Feature "
        "selection on it would leak the held-out schools."
    )
print(f"\nschool-disjoint, re-verified from the file: "
      f"{FIT_SPLIT} n {SHIP_GATE_SPLIT} = 0, {FIT_SPLIT} n holdout_school = 0  OK")

# --- the static scan of this notebook's own code cells ---------------------


def scan_for_forbidden_split_use(nb_path: Path, forbidden):
    """Code cells that use a forbidden split value outside an assertion.

    Returns a list of (cell index, source line) violations. A forbidden value
    is allowed to appear in an `assert` (an assertion ABOUT the ship gate is
    the point), in a comment (invisible to ast) and in markdown. It is a
    violation anywhere else that could act as a filter: a comparison, or an
    argument to a call.
    """
    nb = json.loads(nb_path.read_text())
    violations = []

    for i, c in enumerate(nb["cells"]):
        if c["cell_type"] != "code":
            continue
        src = "".join(c["source"])
        tree = ast.parse(src)

        allowed = set()
        for node in ast.walk(tree):
            if isinstance(node, ast.Assert):
                allowed.update(id(n) for n in ast.walk(node))

        for node in ast.walk(tree):
            if id(node) in allowed:
                continue
            if isinstance(node, ast.Compare):
                _operands = [node.left, *node.comparators]
                if any(isinstance(o, ast.Constant) and o.value in forbidden
                       for o in _operands):
                    violations.append((i, ast.unparse(node)))
            elif isinstance(node, ast.Call):
                _args = list(node.args) + [k.value for k in node.keywords]
                if any(isinstance(a, ast.Constant) and a.value in forbidden
                       for a in _args):
                    violations.append((i, ast.unparse(node)[:120]))

    return violations


if NB_PATH.exists():
    _violations = scan_for_forbidden_split_use(NB_PATH, set(NEVER_LOADED))
    assert not _violations, (
        f"{len(_violations)} code-cell use(s) of a non-loadable split value "
        f"{list(NEVER_LOADED)} outside an assertion - i.e. something that can "
        "act as a filter or be handed to a fit:\n"
        + "\n".join(f"    cell {i}: {s}" for i, s in _violations)
        + f"\n  The ship gate {SHIP_GATE_SPLIT!r} may be named in an assertion, a "
          "comment or markdown, never in a comparison or a call. Route every row "
          "selection through cell()."
    )
    print(f"static scan of {NB_PATH.name}: no code cell compares against, or passes, "
          f"any of {list(NEVER_LOADED)} outside an assertion  OK")
else:
    raise FileNotFoundError(
        f"cannot find this notebook at {NB_PATH} to scan it. NB_PATH is derived "
        "in §4 from PROJECT_ROOT, which assumes the notebook lives in "
        "<project>/data/ and is run from there or from the project root. Point "
        "NB_PATH at this file and re-run - the scan is one of the three things "
        "keeping the ship gate closed, so it is not optional."
    )

---

## 7. `X` / `y` and the feature guardrails

Builds the `train` matrix through `cell()` and asserts what it is:

- `X` is **exactly** `MODEL_COLS`, same names, same order. Not a superset, not a reordering.
- **None** of `school_id`, `dim_month`, `split`, `school_group`, `label`, `product_score`, the
  schema's passthroughs, or anything that looks like an organization identifier is in it. The last is
  a pattern match over the parquet's columns, so an org column added later cannot arrive unnoticed.
- Every column is numeric. A surviving `category` dtype here would mean the extract's encoding did
  not cover a categorical feature, and XGBoost would either refuse it or read integer codes.

NaNs are **passed to XGBoost as-is** - it learns a missing branch natively, and for the `delta_*`
ratios a null ("the baseline was zero") and a `-1.0` ("the metric fell to zero") are opposite facts
that any fill would merge. No imputation anywhere in this notebook.

In [ ]:
X_train, y_train = Xy(FIT_SPLIT)

assert list(X_train.columns) == MODEL_COLS, (
    "X is not exactly MODEL_COLS in order. Xy() selects frame[MODEL_COLS], so "
    "this can only fail if MODEL_COLS was mutated after §5."
)

# --- what may never be a feature -------------------------------------------
# Pattern as well as name, so an organization column added to the extract later
# cannot slip in silently.
ORG_PATTERN = re.compile(r"organi[sz]ation|(?:^|_)orgs?(?:_|$)")
ORG_LIKE = sorted(c for c in enc.columns if ORG_PATTERN.search(c))

FORBIDDEN_IN_X = (
    set(KEYS) | {Y_COL, "split", "school_group", "product_score"}
    | set(PASSTHROUGH) | set(NON_FEATURE_CARRIED) | set(CATEGORICAL_FEATURES)
    | set(ORG_LIKE)
)

_WHY_FORBIDDEN = {
    "school_id": "a key the trees would memorise",
    "dim_month": "the observation month - a clock, and the split boundary is a date",
    "split": "the train/test assignment itself; encoding it makes every off-train metric look wonderful",
    "school_group": "the other half of the split assignment",
    Y_COL: "the target",
    "product_score": "the Retention Health Score this model is scored AGAINST - circular",
}

_bad = sorted(set(MODEL_COLS) & FORBIDDEN_IN_X)
assert not _bad, (
    f"column(s) that must never be a feature are in MODEL_COLS: {_bad}\n"
    + "\n".join(
        f"    {c}: {_WHY_FORBIDDEN.get(c, 'a passthrough / identifier - carried in the frame, never an input')}"
        for c in _bad
    )
    + "\n  MODEL_COLS comes from the encoding JSON, so it was widened upstream: "
      "check §7b / §11.4 of data/dataset_creation.ipynb. Do not filter it here - "
      "the sidecar is the contract a scoring job replays, and patching it in this "
      "notebook would leave the two disagreeing."
)

_non_numeric = [(c, str(X_train[c].dtype)) for c in MODEL_COLS
                if not pd.api.types.is_numeric_dtype(X_train[c])]
assert not _non_numeric, (
    "MODEL_COLS column(s) are not numeric:\n"
    + "\n".join(f"    {c:<52} {d}" for c, d in _non_numeric)
    + "\n  A `category` here means the extract's one-hot encoding missed a "
      "categorical feature; XGBoost either refuses it or silently reads integer "
      "codes as measurements."
)

_pos = int(y_train.sum())
_neg = len(y_train) - _pos
_nan_cells = int(X_train.isna().to_numpy().sum())
_nan_cols = int((X_train.isna().sum() > 0).sum())

print(f"X_train   {X_train.shape[0]:,} rows x {X_train.shape[1]} columns "
      f"({cell(FIT_SPLIT)['school_id'].nunique():,} schools)")
print(f"y_train   {_pos:,} positive / {_neg:,} negative ({y_train.mean():.2%})")
print(f"dtypes    {sorted(set(X_train.dtypes.astype(str)))}")
print(f"NaN       {_nan_cells:,} cells across {_nan_cols} of {X_train.shape[1]} columns "
      f"({_nan_cells / X_train.size:.2%}) - passed to XGBoost as-is, never imputed")
print(f"\nforbidden-in-X set ({len(FORBIDDEN_IN_X)} names) held out of MODEL_COLS  OK")
print(f"  organization-like columns in the file: {ORG_LIKE or 'none'}")

---

## 8. Model configuration

Two configurations off one set of base parameters, and the difference is load-bearing.

**`boruta_model()` - fixed `n_estimators`, NO early stopping.** BorutaShap's `Train_model` calls
`self.model.fit(X, y, sample_weight=..., verbose=False)` and supplies **no** `eval_set`, once per
trial, on a re-shadowed matrix. An estimator carrying `early_stopping_rounds` therefore raises
(`Must have at least 1 validation dataset for early stopping`) or, worse, silently stops on
whatever it can find. The cell asserts `early_stopping_rounds is None` on the estimator it builds.

**`eval_model()` - the same base parameters plus `early_stopping_rounds` on AUCPR**, with an explicit
eval set: half the schools of `holdout_school`, carved here, seeded, and asserted disjoint from the
half §11 reports on. The ship gate is not involved in either.

Shallow trees (`max_depth=3`) with a `min_child_weight` floor, `tree_method="hist"`,
`eval_metric="aucpr"`, a fixed `random_state` and `n_jobs=-1`.

`SCALE_POS_WEIGHT` is the **`train` cell's own** `neg/pos` ratio. The spec says to tune it *downward*
on AUCPR - over-weighting positives hurts precision and inflates the probabilities - and that tuning
is not in this notebook. Weighting applies to **training only**: no evaluation cell is reweighted, or
precision@k stops meaning anything.

In [ ]:
RANDOM_STATE = 42

# --- the early-stopping eval set: half of holdout_school's SCHOOLS ----------
# School-level, so no school contributes to both the early-stopping set and the
# reported one. Drawn from a holdout cell, never from the ship gate.
ES_SPLIT = "holdout_school"
ES_SCHOOL_FRACTION = 0.50
ES_SEED = 7

_hs = cell(ES_SPLIT)
_hs_schools = np.array(sorted(_hs["school_id"].unique()))
_perm = np.random.default_rng(ES_SEED).permutation(len(_hs_schools))
_n_es = int(round(ES_SCHOOL_FRACTION * len(_hs_schools)))
ES_SCHOOLS = set(_hs_schools[_perm[:_n_es]])
READ_SCHOOLS_HS = set(_hs_schools[_perm[_n_es:]])

assert not (ES_SCHOOLS & READ_SCHOOLS_HS), "the two holdout_school halves share a school"
assert ES_SCHOOLS and READ_SCHOOLS_HS, (
    f"{ES_SPLIT} has {len(_hs_schools)} school(s) - too few to carve an "
    "early-stopping half and still have something to report on. Either widen the "
    "cell or drop early stopping and fix n_estimators."
)

ES_FRAME = _hs[_hs["school_id"].isin(ES_SCHOOLS)]
HS_READ_FRAME = _hs[_hs["school_id"].isin(READ_SCHOOLS_HS)]

X_es, y_es = ES_FRAME[MODEL_COLS], ES_FRAME[Y_COL].astype("int8")
assert int(y_es.sum()) > 0, (
    "the early-stopping eval set contains no positive rows, so AUCPR on it is "
    "undefined and early stopping would pick iteration 0. Raise "
    "ES_SCHOOL_FRACTION or re-seed ES_SEED."
)

# --- imbalance --------------------------------------------------------------
_pos_train = int(y_train.sum())
_neg_train = len(y_train) - _pos_train
SCALE_POS_WEIGHT = _neg_train / max(_pos_train, 1)
# --- TUNED: swept scale_pos_weight from 29.3 down to 2, per the spec's guidance
# to tune it downward on AUCPR. Weight=4 gave the best precision@50 on holdout_time
# (23.33% vs 19.33% at the raw ratio) - see the sweep results.
SCALE_POS_WEIGHT = 4
# --- base parameters, shared -----------------------------------------------
XGB_BASE_PARAMS = dict(
    objective="binary:logistic",
    eval_metric="aucpr",
    tree_method="hist",
    max_depth=3,              # reverted back
    min_child_weight=20,      # reverted back
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    missing=np.nan,
)

BORUTA_N_ESTIMATORS = 200       # FIXED - BorutaShap supplies no eval_set
EVAL_N_ESTIMATORS = 2000        # an upper bound; early stopping picks the number
EARLY_STOPPING_ROUNDS = 50


def boruta_model(scale_pos_weight: float) -> XGBClassifier:
    """The estimator BorutaSHAP refits every trial. NO early stopping.

    BorutaShap.Train_model calls model.fit(X, y, sample_weight=..., verbose=False)
    with no eval_set, so an estimator carrying early_stopping_rounds either
    raises or stops on something it was never given.
    """
    m = XGBClassifier(**XGB_BASE_PARAMS,
                      n_estimators=BORUTA_N_ESTIMATORS,
                      scale_pos_weight=scale_pos_weight)
    assert m.get_params().get("early_stopping_rounds") is None, (
        "the BorutaSHAP estimator has early_stopping_rounds set. BorutaShap's "
        "Train_model() supplies no eval_set, so xgboost raises 'Must have at "
        "least 1 validation dataset for early stopping' on the first trial - or "
        "silently stops against the training matrix. Use a fixed n_estimators "
        "here and keep early stopping for eval_model()."
    )
    return m


def eval_model(scale_pos_weight: float = SCALE_POS_WEIGHT) -> XGBClassifier:
    """The evaluation estimator: base parameters + early stopping on AUCPR.

    §11 fits it with eval_set=[(X_es, y_es)] - a school-disjoint half of
    holdout_school. Never the ship gate.
    """
    return XGBClassifier(**XGB_BASE_PARAMS,
                         n_estimators=EVAL_N_ESTIMATORS,
                         early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                         scale_pos_weight=scale_pos_weight)


print(f"train cell            {_pos_train:,} positive / {_neg_train:,} negative "
      f"({y_train.mean():.2%})")
print(f"scale_pos_weight      {SCALE_POS_WEIGHT:.1f}  (train cell's own neg/pos)")
print("                      the spec says TUNE THIS DOWNWARD on AUCPR - "
      "over-weighting positives")
print("                      hurts precision and inflates the probabilities. Not "
      "tuned here.")
print("                      Training only: no evaluation cell is reweighted.")
print(f"\nearly-stopping eval set (from {ES_SPLIT}, school-disjoint from the reported half):")
print(f"  eval    {len(ES_FRAME):>7,} rows  {len(ES_SCHOOLS):>5,} schools  "
      f"{int(y_es.sum()):>5,} positives ({y_es.mean():.2%})")
print(f"  report  {len(HS_READ_FRAME):>7,} rows  {len(READ_SCHOOLS_HS):>5,} schools  "
      f"{int(HS_READ_FRAME[Y_COL].sum()):>5,} positives "
      f"({HS_READ_FRAME[Y_COL].mean():.2%})")
print(f"\nboruta_model  n_estimators={BORUTA_N_ESTIMATORS}, early stopping OFF")
print(f"eval_model    n_estimators<={EVAL_N_ESTIMATORS}, "
      f"early_stopping_rounds={EARLY_STOPPING_ROUNDS} on aucpr")

---

## 9. BorutaSHAP

Runs BorutaSHAP on the `train` cell and nothing else. Every constant is at the top of the cell so the
run can be scaled up without restructuring anything below it:

| constant | what it does |
| --- | --- |
| `BORUTA_N_TRIALS` | trials in the Boruta loop. A modest default for a first pass; raise it and the tentative set shrinks |
| `BORUTA_SAMPLE_N` | target row count for a stratified sample of the `train` cell. **Every positive row is kept** and the negatives are sampled down to hit the target. `None` uses the whole cell |
| `BORUTA_SEED` | the negative sample and BorutaShap's own `random_state` |

Printed before the run: the sampled row count, positive count and positive rate, and the sampled
cell's own `neg/pos` ratio - which is what `scale_pos_weight` is set from for **this** fit, since
down-sampling negatives changes the imbalance the estimator sees. After it: wall-clock time and the
accepted / tentative / rejected counts.

What is passed, against the 1.0.17 signature: `importance_measure="shap"` (lowercase - the `"Shap"`
default raises), `classification=True`, `n_trials`, `random_state`, `sample=False`,
`train_or_test="train"`, `normalize=True`, `verbose=False`. `train_or_test` is BorutaShap's
in-sample / out-of-sample switch for the SHAP computation - `"train"` fits each trial on all of the
sampled rows instead of an internal 70/30 cut, which is what a small positive count needs. It has
nothing to do with this project's `test` cell.

BorutaShap prints `Warning there are missing values in your data !` and carries on: its
`check_missing_values` only tolerates NaN for models whose type name contains `xgb`, which is exactly
why the estimator is XGBoost. Nothing is imputed.

Two things to know about the importance numbers before reading them as importances. With
`normalize=True` they are **z-scored within each trial**, so they are centred near zero and a
negative value is normal - it means "below the average feature in that trial", not "harmful". And a
feature rejected mid-run is dropped from the matrix, so its mean is taken over the trials it
survived: `trials_survived` is reported per feature, and a rank comparison between a
25-trial survivor and a 12-trial casualty is indicative rather than exact. The decision column, not
the level, is the output of this section.

In [ ]:
import json
_sel = json.loads((RAW_DIR / f"{STEM}_boruta_selection.json").read_text())
ACCEPTED = _sel["accepted"]
TENTATIVE = _sel["tentative"]
REJECTED = _sel["rejected"]
SELECTED_FEATURES = [c for c in MODEL_COLS if c in set(ACCEPTED)]
SELECTED_LABEL = "boruta_accepted"
print(f"reloaded: {len(ACCEPTED)} accepted, {len(TENTATIVE)} tentative, {len(REJECTED)} rejected")

In [ ]:
# --- constants: scale the run from here ------------------------------------
BORUTA_N_TRIALS = 250        # modest first pass. 50-100 for a run you will act on.
BORUTA_SAMPLE_N = None    # target rows; None = the whole train cell
BORUTA_SEED = 42

# --- the stratified row sample: every positive, negatives sampled down ------
# Sampling positives away is not an option - they are the scarce thing, and the
# hit statistics are driven by them.
_train_frame = cell(FIT_SPLIT)
_pos_idx = _train_frame.index[_train_frame[Y_COL] == 1]
_neg_idx = _train_frame.index[_train_frame[Y_COL] == 0]

if BORUTA_SAMPLE_N is None or BORUTA_SAMPLE_N >= len(_train_frame):
    BORUTA_IDX = _train_frame.index
    BORUTA_SAMPLED = False
    _neg_target = len(_neg_idx)
else:
    _neg_target = max(BORUTA_SAMPLE_N - len(_pos_idx), 0)
    assert _neg_target > 0, (
        f"BORUTA_SAMPLE_N={BORUTA_SAMPLE_N:,} is smaller than the "
        f"{len(_pos_idx):,} positive rows in the {FIT_SPLIT} cell, and positives "
        "are never sampled away. Raise BORUTA_SAMPLE_N above the positive count "
        "or set it to None."
    )
    _rng = np.random.default_rng(BORUTA_SEED)
    _keep_neg = _rng.choice(np.asarray(_neg_idx), size=min(_neg_target, len(_neg_idx)),
                            replace=False)
    BORUTA_IDX = pd.Index(np.concatenate([np.asarray(_pos_idx), _keep_neg]))
    BORUTA_SAMPLED = True

X_boruta_in = X_train.loc[BORUTA_IDX]
y_boruta_in = y_train.loc[BORUTA_IDX]

assert int(y_boruta_in.sum()) == len(_pos_idx), (
    f"the sample has {int(y_boruta_in.sum()):,} positives but the {FIT_SPLIT} "
    f"cell has {len(_pos_idx):,} - the negative-only down-sample dropped "
    "positive rows."
)
assert list(X_boruta_in.columns) == MODEL_COLS, "the sample is not exactly MODEL_COLS"

_b_pos = int(y_boruta_in.sum())
_b_neg = len(y_boruta_in) - _b_pos
SCALE_POS_WEIGHT_BORUTA = _b_neg / max(_b_pos, 1)

print(f"BORUTA_N_TRIALS  {BORUTA_N_TRIALS}")
print(f"BORUTA_SAMPLE_N  {BORUTA_SAMPLE_N if BORUTA_SAMPLE_N is not None else 'None (whole cell)'}"
      f"   -> {'SAMPLED' if BORUTA_SAMPLED else 'whole cell used'}")
print(f"BORUTA_SEED      {BORUTA_SEED}")
print(f"\nsampled rows     {len(X_boruta_in):,} of {len(_train_frame):,} in the "
      f"{FIT_SPLIT} cell ({len(X_boruta_in) / len(_train_frame):.1%})")
print(f"  positives      {_b_pos:,}  ({y_boruta_in.mean():.2%} of the sample; "
      f"{y_train.mean():.2%} in the full cell)")
print(f"  negatives      {_b_neg:,}")
print(f"  schools        {_train_frame.loc[BORUTA_IDX, 'school_id'].nunique():,}")
print(f"  features       {X_boruta_in.shape[1]}")
print(f"  scale_pos_weight for THIS fit: {SCALE_POS_WEIGHT_BORUTA:.1f} "
      f"(the sample's own neg/pos; the full cell's is {SCALE_POS_WEIGHT:.1f})")

In [ ]:
# --- the run ---------------------------------------------------------------
# One call. `train_or_test="train"` is BorutaShap's SHAP-computation switch, not
# this project's split cells: it fits each trial on all of the sampled rows
# rather than an internal 70/30 cut.
selector = BorutaShap(
    model=boruta_model(SCALE_POS_WEIGHT_BORUTA),
    importance_measure=IMPORTANCE_MEASURE,   # "shap", lowercase
    classification=True,
    percentile=100,
    pvalue=0.05,
)

_t0 = time.perf_counter()
selector.fit(
    X=X_boruta_in,
    y=y_boruta_in,
    n_trials=BORUTA_N_TRIALS,
    random_state=BORUTA_SEED,
    sample=False,            # no internal isolation-forest row sample; §9 above sampled
    train_or_test="train",
    normalize=True,
    verbose=False,
)
BORUTA_SECONDS = time.perf_counter() - _t0

ACCEPTED = sorted(selector.accepted)
TENTATIVE = sorted(map(str, selector.tentative))
REJECTED = sorted(selector.rejected)

assert set(ACCEPTED) | set(TENTATIVE) | set(REJECTED) == set(MODEL_COLS), (
    "accepted + tentative + rejected do not partition MODEL_COLS: "
    f"{len(ACCEPTED)} + {len(TENTATIVE)} + {len(REJECTED)} = "
    f"{len(ACCEPTED) + len(TENTATIVE) + len(REJECTED)} vs {len(MODEL_COLS)} "
    "features. BorutaShap derives all three from the columns it was handed, so a "
    "mismatch means the matrix changed under it."
)

# --- the SHAP importance table ---------------------------------------------
# selector.history_x is one row per trial (row 0 is the zero-initialised row,
# hence .iloc[1:]) plus four shadow columns. A feature rejected mid-run is
# dropped from the matrix, so its later rows are NaN - mean/std skip them, and
# `trials` records how many trials each feature actually survived.
_hist = selector.history_x.iloc[1:]
_shadow_cols = ["Max_Shadow", "Min_Shadow", "Mean_Shadow", "Median_Shadow"]
_feat_hist = _hist.drop(columns=_shadow_cols)

_decision = {**{c: "Accepted" for c in ACCEPTED},
             **{c: "Tentative" for c in TENTATIVE},
             **{c: "Rejected" for c in REJECTED}}

IMPORTANCE = (pd.DataFrame({
    "feature": _feat_hist.columns,
    "mean_shap_importance": _feat_hist.mean(axis=0).to_numpy(),
    "sd_shap_importance": _feat_hist.std(axis=0).to_numpy(),
    "trials_survived": _feat_hist.notna().sum(axis=0).to_numpy(),
})
    .assign(decision=lambda d: d["feature"].map(_decision))
    .sort_values("mean_shap_importance", ascending=False, kind="mergesort")
    .reset_index(drop=True))
IMPORTANCE.insert(0, "rank", np.arange(1, len(IMPORTANCE) + 1))

MEDIAN_MAX_SHADOW = float(_hist["Max_Shadow"].median())

print(f"BorutaSHAP finished in {BORUTA_SECONDS:.1f}s "
      f"({BORUTA_SECONDS / 60:.1f} min) over {BORUTA_N_TRIALS} trials")
print(f"  {len(ACCEPTED):>4} accepted")
print(f"  {len(TENTATIVE):>4} tentative")
print(f"  {len(REJECTED):>4} rejected")
print(f"  {len(MODEL_COLS):>4} candidates in")
print(f"\nmedian Max_Shadow across trials: {MEDIAN_MAX_SHADOW:.5f} "
      "(the bar a feature had to clear)")
print("\ntop 25 by mean SHAP importance:")
print(IMPORTANCE.head(25).to_string(index=False))

if not ACCEPTED:
    print("\n" + "!" * 72)
    print("WARNING: BorutaSHAP accepted NOTHING. At a modest trial count that is")
    print("usually under-powered rather than meaningful - the hit statistics need")
    print("trials before the Bonferroni-corrected p-values can clear 0.05. Raise")
    print("BORUTA_N_TRIALS (and BORUTA_SAMPLE_N) before reading anything into it.")
    print("§11 falls back to accepted + tentative so the comparison still runs.")
    print("!" * 72)

---

## 10. Persist the selection

Writes the outcome to `data/raw/`, named after `STEM`, so a result can be traced back to the run that
produced it:

- `..._boruta_selection.json` - the three lists, plus the run config: trials, sample size and whether
  sampling actually happened, seeds, the estimator's parameters, `scale_pos_weight`, wall-clock time,
  the source parquet and JSON names, and the library versions.
- `..._boruta_importance.csv` - the SHAP importance table: rank, feature, mean and sd importance,
  trials survived, decision.

Both are read back and checked against the in-memory objects before the summary prints.

In [ ]:
SELECTION_JSON = RAW_DIR / f"{STEM}_boruta_selection.json"
IMPORTANCE_CSV = RAW_DIR / f"{STEM}_boruta_importance.csv"

SELECTION_SCHEMA_VERSION = 1

SELECTION = {
    "schema_version": SELECTION_SCHEMA_VERSION,
    "generated_at_utc": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%dT%H:%M:%SZ"),
    "source_encoded_parquet": ENCODED.name,
    "source_encoding_json": ENCODING.name,
    "n_candidates": len(MODEL_COLS),
    "accepted": ACCEPTED,
    "tentative": TENTATIVE,
    "rejected": REJECTED,
    "run_config": {
        "fit_split": FIT_SPLIT,
        "read_splits": list(READ_SPLITS),
        "splits_never_loaded": list(NEVER_LOADED),
        "boruta_n_trials": BORUTA_N_TRIALS,
        "boruta_sample_n": BORUTA_SAMPLE_N,
        "boruta_sampled": bool(BORUTA_SAMPLED),
        "boruta_rows_used": int(len(X_boruta_in)),
        "boruta_positives_used": int(y_boruta_in.sum()),
        "boruta_positive_rate": float(y_boruta_in.mean()),
        "boruta_seed": BORUTA_SEED,
        "random_state": RANDOM_STATE,
        "importance_measure": IMPORTANCE_MEASURE,
        "train_or_test": "train",
        "pvalue": 0.05,
        "percentile": 100,
        "xgb_base_params": {k: (v if not isinstance(v, float) or np.isfinite(v) else str(v))
                            for k, v in XGB_BASE_PARAMS.items()},
        "boruta_n_estimators": BORUTA_N_ESTIMATORS,
        "scale_pos_weight_train_cell": float(SCALE_POS_WEIGHT),
        "scale_pos_weight_used_for_boruta": float(SCALE_POS_WEIGHT_BORUTA),
        "wall_clock_seconds": round(float(BORUTA_SECONDS), 1),
        "median_max_shadow": MEDIAN_MAX_SHADOW,
        "binom_test_shim_installed": bool(BINOM_TEST_SHIM_NEEDED),
    },
    "versions": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
        "scikit_learn": sklearn.__version__,
        "xgboost": xgb.__version__,
        "shap": shap.__version__,
    },
    "notes": (
        "BorutaSHAP over the encoding JSON's model_cols, fitted on the 'train' "
        "cell only. The ship-gate cell is not loaded anywhere in "
        "data/feature_selection.ipynb. Tentative features are neither accepted "
        "nor rejected: raise boruta_n_trials to resolve them. Leakage-risk "
        "features are FLAGGED, never auto-dropped - see the notebook's §13."
    ),
}

with SELECTION_JSON.open("w") as _f:
    json.dump(SELECTION, _f, indent=2)
    _f.write("\n")

IMPORTANCE.to_csv(IMPORTANCE_CSV, index=False)

# --- read both back --------------------------------------------------------
_sel_back = json.loads(SELECTION_JSON.read_text())
assert _sel_back["accepted"] == ACCEPTED, "the JSON's accepted list does not round trip"
assert (_sel_back["run_config"]["boruta_rows_used"] == len(X_boruta_in)
        and _sel_back["source_encoded_parquet"] == ENCODED.name), (
    "the persisted run config does not describe this run, so the artefact cannot "
    "be traced back to it"
)
_imp_back = pd.read_csv(IMPORTANCE_CSV)
assert list(_imp_back["feature"]) == list(IMPORTANCE["feature"]), (
    "the importance CSV's feature order does not round trip - the rank column "
    "would no longer match the rows"
)

print(f"wrote  {SELECTION_JSON.name}   ({SELECTION_JSON.stat().st_size / 1e3:.1f} KB)")
print(f"       {IMPORTANCE_CSV.name}   ({IMPORTANCE_CSV.stat().st_size / 1e3:.1f} KB)")
print(f"beside {ENCODED.name}")
print(f"\nround trip OK: {len(_imp_back)} importance rows, "
      f"{len(_sel_back['accepted'])} accepted / {len(_sel_back['tentative'])} tentative / "
      f"{len(_sel_back['rejected'])} rejected")

---

## 11. Evaluation harness

### 11a. Helpers

`precision@k` and `recall@k` at `k` in `K_VALUES`, plus PR-AUC (average precision) and the base rate.

**`precision@k` is computed PER MONTH and then averaged.** The rep works a monthly list of `k`
schools, so the quantity of interest is "of the `k` schools surfaced this month, how many filed?"
Pooling nine months and taking one global top-`k` answers a different question - it lets a single
month with high scores fill the whole list - so `pooled_at_k` computes that too and both are
reported, side by side, so the gap is visible rather than assumed.

The base rate is reported next to every figure, so `precision@k` reads as a **lift over random**
rather than as a bare number. `recall@k` per month is against that month's own positives; a month
with no positives contributes `NaN` and is excluded from the average, and the count of such months is
printed.

Ties are broken by row order (`kind="mergesort"`), so the same scores give the same list twice.

`per_month_at_k` also returns `hits` - the at-risk schools a month's list actually found. §12 and §14
reuse these two helpers for every ranker they compare, so the model and the baselines are measured by
the same code.

In [ ]:
K_VALUES = (40, 45, 50)     # the rep works a list of 40-50 schools a month
K_PRIMARY = 50


def per_month_at_k(frame: pd.DataFrame, k: int) -> pd.DataFrame:
    """Per-month precision@k / recall@k / base rate.

    `frame` needs dim_month, the label and a `score` column. One monthly list of
    k schools per month - the shape of the actual motion.
    """
    rows = []
    for _m, _g in frame.groupby("dim_month", sort=True):
        _top = _g.sort_values("score", ascending=False, kind="mergesort").head(k)
        _hit = int(_top[Y_COL].sum())
        _all = int(_g[Y_COL].sum())
        rows.append({
            "dim_month": _m,
            "rows": len(_g),
            "k_used": len(_top),            # < k in a month with fewer rows than k
            "positives": _all,
            "hits": _hit,                   # at-risk schools the list actually found
            "precision_at_k": _hit / len(_top) if len(_top) else np.nan,
            "recall_at_k": _hit / _all if _all else np.nan,
            "base_rate": _all / len(_g) if len(_g) else np.nan,
        })
    return pd.DataFrame(rows)


def pooled_at_k(frame: pd.DataFrame, k: int) -> dict:
    """One global top-k over every month at once - the WRONG quantity, reported
    for comparison. k schools in total, not k per month."""
    _top = frame.sort_values("score", ascending=False, kind="mergesort").head(k)
    _hit = int(_top[Y_COL].sum())
    _all = int(frame[Y_COL].sum())
    return {
        "pooled_precision_at_k": _hit / len(_top) if len(_top) else np.nan,
        "pooled_recall_at_k": _hit / _all if _all else np.nan,
    }


def score_cell(model, frame: pd.DataFrame, feature_cols) -> pd.DataFrame:
    """Attach `score` to a copy of the identifier/label columns of `frame`.

    Scores only: the frame handed back carries no features, so nothing
    downstream can accidentally fit on an evaluation cell. No sample weighting
    is applied anywhere here - an evaluation cell keeps its true positive rate.
    """
    out = frame[["school_id", "dim_month", Y_COL]].copy()
    out["score"] = model.predict_proba(frame[feature_cols])[:, 1]
    return out


def evaluate(model, frame: pd.DataFrame, feature_cols, label: str,
             feature_set: str) -> tuple:
    """(summary rows, per-month table) for one model on one evaluation cell."""
    scored = score_cell(model, frame, feature_cols)
    ap = float(average_precision_score(scored[Y_COL], scored["score"]))
    base = float(scored[Y_COL].mean())

    summary, per_month = [], {}
    for k in K_VALUES:
        pm = per_month_at_k(scored, k)
        per_month[k] = pm
        pooled = pooled_at_k(scored, k)
        summary.append({
            "cell": label,
            "feature_set": feature_set,
            "n_features": len(feature_cols),
            "rows": len(scored),
            "months": int(pm["dim_month"].nunique()),
            "positives": int(scored[Y_COL].sum()),
            "base_rate": base,
            "k": k,
            "precision_at_k_permonth_mean": float(pm["precision_at_k"].mean(skipna=True)),
            "precision_at_k_permonth_sd": float(pm["precision_at_k"].std(skipna=True)),
            "recall_at_k_permonth_mean": float(pm["recall_at_k"].mean(skipna=True)),
            "lift_over_base": (float(pm["precision_at_k"].mean(skipna=True)) / base
                               if base else np.nan),
            "pr_auc": ap,
            **pooled,
            "months_without_a_positive": int(pm["positives"].eq(0).sum()),
        })
    return summary, per_month


print(f"K_VALUES {K_VALUES}, primary k = {K_PRIMARY}")
print("precision@k is per-month then averaged; pooled_precision_at_k is the single")
print("global top-k over all months, reported alongside so the difference is visible.")

### 11b. Full candidate set vs. the BorutaSHAP subset

Fits `eval_model()` twice on the `train` cell - once on all of `MODEL_COLS`, once on the accepted
subset - with the same parameters, the same seed and the same early-stopping eval set, then reads both
on `holdout_school` (the reporting half, school-disjoint from that eval set) and on `holdout_time`.
That is the number behind "did selection help?".

The two cells measure different things and are shown side by side, never averaged together:
`holdout_school` is **new schools, same era**; `holdout_time` is **the same schools, a new era**.
A gap between them is the school effect and the time effect separating.

If BorutaSHAP accepted nothing, the subset falls back to accepted + tentative with a loud note - an
empty feature set cannot be fitted, and reporting the fallback is better than reporting nothing.

Writes `..._selection_eval.csv`.

In [ ]:
if ACCEPTED:
    SELECTED_FEATURES = [c for c in MODEL_COLS if c in set(ACCEPTED)]
    SELECTED_LABEL = "boruta_accepted"
else:
    SELECTED_FEATURES = [c for c in MODEL_COLS if c in set(ACCEPTED) | set(TENTATIVE)]
    SELECTED_LABEL = "boruta_accepted_plus_tentative_FALLBACK"
    print("!" * 72)
    print("NOTE: nothing was accepted, so the 'selected' arm below is "
          "accepted + tentative.")
    print("Read it as 'what survived', not as a selection. Raise BORUTA_N_TRIALS.")
    print("!" * 72 + "\n")

assert SELECTED_FEATURES, (
    "the selected feature set is empty - BorutaSHAP rejected every candidate, so "
    "there is nothing to fit. That is a signal about the run (too few trials, or "
    "too few positives in the sample), not about the features."
)
assert set(SELECTED_FEATURES) <= set(MODEL_COLS), "a selected feature is not in MODEL_COLS"

# Order preserved from MODEL_COLS, so the subset model's feature order is stable.
FEATURE_SETS = {
    "full_candidate_set": list(MODEL_COLS),
    SELECTED_LABEL: SELECTED_FEATURES,
}

# The cells this notebook reads. The ship gate is not among them.
READ_FRAMES = {
    f"holdout_school (report half, {len(READ_SCHOOLS_HS)} schools)": HS_READ_FRAME,
    "holdout_time": cell("holdout_time"),
}

_summaries, PER_MONTH = [], {}
MODELS = {}

for _fs_name, _cols in FEATURE_SETS.items():
    _m = eval_model()
    _m.fit(X_train[_cols], y_train,
           eval_set=[(X_es[_cols], y_es)],   # a holdout cell, never the ship gate
           verbose=False)
    MODELS[_fs_name] = _m
    print(f"{_fs_name:<40} {len(_cols):>4} features, "
          f"best_iteration {_m.best_iteration} of {EVAL_N_ESTIMATORS} "
          f"(best aucpr {_m.best_score:.4f} on the early-stopping set)")

    for _cell_name, _frame in READ_FRAMES.items():
        _rows, _pm = evaluate(_m, _frame, _cols, _cell_name, _fs_name)
        _summaries.extend(_rows)
        PER_MONTH[(_fs_name, _cell_name)] = _pm

EVAL = pd.DataFrame(_summaries)

# --- side by side ----------------------------------------------------------
_show = ["cell", "feature_set", "n_features", "k", "base_rate",
         "precision_at_k_permonth_mean", "lift_over_base",
         "recall_at_k_permonth_mean", "pr_auc", "pooled_precision_at_k"]
_fmt = EVAL[_show].copy()
for _c in ("base_rate", "precision_at_k_permonth_mean", "recall_at_k_permonth_mean",
           "pooled_precision_at_k"):
    _fmt[_c] = _fmt[_c].map(lambda v: f"{v:.3%}")
_fmt["lift_over_base"] = _fmt["lift_over_base"].map(lambda v: f"{v:.2f}x")
_fmt["pr_auc"] = _fmt["pr_auc"].map(lambda v: f"{v:.4f}")

print("\n" + "=" * 72)
print("PRECISION@K PER MONTH, AVERAGED  (pooled shown for contrast)")
print("=" * 72)
print(_fmt.to_string(index=False))

print(f"\nper-month precision@{K_PRIMARY} series:")
for (_fs_name, _cell_name), _pm in PER_MONTH.items():
    _s = _pm[K_PRIMARY]
    print(f"\n  {_fs_name}  |  {_cell_name}")
    print("    " + "  ".join(f"{m:%Y-%m}" for m in _s["dim_month"]))
    print("    " + "  ".join(f"{v:>7.1%}" for v in _s["precision_at_k"]))
    print(f"    mean {_s['precision_at_k'].mean(skipna=True):.2%}  "
          f"sd {_s['precision_at_k'].std(skipna=True):.2%}  "
          f"base rate {_s['base_rate'].mean(skipna=True):.2%}  "
          f"months without a positive: {int(_s['positives'].eq(0).sum())}")

# --- did selection help? ---------------------------------------------------
print("\n" + "=" * 72)
print(f"DID SELECTION HELP?  (k = {K_PRIMARY})")
print("=" * 72)
_p = EVAL[EVAL["k"] == K_PRIMARY]
for _cell_name in READ_FRAMES:
    _f = _p[(_p["cell"] == _cell_name) & (_p["feature_set"] == "full_candidate_set")].iloc[0]
    _s = _p[(_p["cell"] == _cell_name) & (_p["feature_set"] == SELECTED_LABEL)].iloc[0]
    print(f"\n{_cell_name}   base rate {_f['base_rate']:.3%}")
    print(f"  {'':<26}{'full':>14}{'selected':>14}{'delta':>12}")
    for _lbl, _key, _pct in (
        (f"precision@{K_PRIMARY} (per month)", "precision_at_k_permonth_mean", True),
        (f"recall@{K_PRIMARY} (per month)", "recall_at_k_permonth_mean", True),
        ("PR-AUC", "pr_auc", False),
        (f"pooled precision@{K_PRIMARY}", "pooled_precision_at_k", True),
    ):
        if _pct:
            print(f"  {_lbl:<26}{_f[_key]:>13.2%}{_s[_key]:>14.2%}"
                  f"{_s[_key] - _f[_key]:>+11.2%}")
        else:
            print(f"  {_lbl:<26}{_f[_key]:>13.4f}{_s[_key]:>14.4f}"
                  f"{_s[_key] - _f[_key]:>+12.4f}")
    print(f"  features                  {int(_f['n_features']):>13}"
          f"{int(_s['n_features']):>14}{int(_s['n_features']) - int(_f['n_features']):>+12}")

EVAL_CSV = RAW_DIR / f"{STEM}_selection_eval.csv"
EVAL.to_csv(EVAL_CSV, index=False)
print(f"\nwrote  {EVAL_CSV.name}   ({EVAL_CSV.stat().st_size / 1e3:.1f} KB)")
print("\nBoth cells are read on a model whose early stopping used a school-disjoint")
print("half of holdout_school. holdout_time is untouched by it, so it is the "
      "cleaner read of the two.")

---

## 12. The `k` sweep and the operating point

§11 answers "did selection help?" at `k` = 40-50. This section answers a different question: what the
precision / recall trade-off looks like across the whole range of list sizes.

`k` in `K_SWEEP`, plus the whole monthly population as the ceiling. Per evaluation cell and per `k`:
`precision@k`, `recall@k`, at-risk schools found per month (`precision@k` x `k`, read straight off
the per-month hit count), and lift over that cell's base rate.

Both cells read here are `holdout_school` (the reporting half) and `holdout_time`. The ship gate is
not among them. Every fit reuses §8's configuration and §11b's already-fitted boosters - nothing here
introduces a second set of hyperparameters. The per-row scores are cached in `SCORED` and §14 reuses
them, so the model in the sweep and the model in the baseline comparison are one fit.

**Computed per calendar month and then averaged, never pooled.** One list of `k` schools per month is
the shape of the motion; a single global top-`k` over nine months lets one month fill the whole list
and answers a question nobody asked. Where `k` exceeds a month's row count the month contributes its
row count instead, and `k_used_mean` reports the effective `k`.

`rank_at_k` takes `skip_short_months`, off here and **on in §14b**: a ranker there scores only the
rows it can score, so a month holding fewer than `k` of them would report precision over whatever it
held and be averaged in against months that got a full list. `months_short_skipped` counts them.

At `k` = the whole month, `precision@k` **is** the base rate, `recall@k` is 1.0 and lift is 1.0 - the
cell asserts all three, because they are the arithmetic identity that catches a broken sweep.

Writes `..._k_sweep.csv`.

In [ ]:
# --- the sweep -------------------------------------------------------------
K_SWEEP = (40, 50, 100, 200, 500, 1000, 2000, 5000, 10_000, None)  # None = the whole month
REP_CAPACITY = (40, 50)          # schools a rep can work in a month; §14c reads it


def rank_at_k(scored: pd.DataFrame, k, cell_label: str, ranker: str,
              population: str, skip_short_months: bool = False) -> dict:
    """One summary row: one ranker, one k, per month then averaged.

    k=None means the whole monthly population - the ceiling, where precision is
    the base rate and recall is 1.0. Where k exceeds a month's row count the
    month contributes its row count, so k_used_mean is the effective k actually
    used rather than the requested one.

    skip_short_months=True drops a month with fewer than k scorable rows
    instead, which is what §14b needs: an RHS ranker scores only the rows whose
    trend exists, and a month that cannot fill a list of k would otherwise
    report precision over whatever it held. Ignored at k=None, where the whole
    month IS the list by definition.
    """
    kk = len(scored) if k is None else k
    pm = per_month_at_k(scored, kk)

    n_short = 0
    if skip_short_months and k is not None:
        _short = pm["rows"] < kk
        n_short = int(_short.sum())
        assert not _short.all(), (
            f"on {cell_label!r}, every one of the {len(pm)} month(s) that "
            f"{ranker!r} can score has fewer than k={kk:,} scorable rows (the "
            f"largest holds {int(pm['rows'].max()):,}). There is no month-sized "
            f"list of {kk:,} to measure, so precision@{kk:,} here would silently "
            "be precision over whatever the month happened to hold. Drop this k "
            "from K_BASELINE, or evaluate on a wider cell."
        )
        pm = pm.loc[~_short]

    base = float(pm["base_rate"].mean(skipna=True))
    prec = float(pm["precision_at_k"].mean(skipna=True))
    return {
        "cell": cell_label,
        "ranker": ranker,
        "population": population,
        "k_label": "full month" if k is None else str(k),
        "k_requested": -1 if k is None else int(k),
        "k_used_mean": float(pm["k_used"].mean()),
        "months": int(len(pm)),
        "months_short_skipped": n_short,
        "rows": int(len(scored)),
        "positives": int(scored[Y_COL].sum()),
        "base_rate_permonth_mean": base,
        "base_rate_pooled": float(scored[Y_COL].mean()),
        "precision_at_k": prec,
        "recall_at_k": float(pm["recall_at_k"].mean(skipna=True)),
        "found_per_month": float(pm["hits"].mean()),
        "lift_over_base": prec / base if base else np.nan,
    }


# Scores are computed once here and reused by §14, so the model in the sweep and
# the model in the baseline comparison are the same numbers, not two fits.
SCORED = {}
_sweep_rows = []
for _fs_name, _cols in FEATURE_SETS.items():
    for _cell_name, _frame in READ_FRAMES.items():
        _s = score_cell(MODELS[_fs_name], _frame, _cols)
        SCORED[(_fs_name, _cell_name)] = _s
        for _k in K_SWEEP:
            _sweep_rows.append(rank_at_k(_s, _k, _cell_name, _fs_name, "full_population"))

K_SWEEP_TABLE = pd.DataFrame(_sweep_rows)

# --- the identities at the ceiling -----------------------------------------
# At k = the whole month the list is the population: precision is the base rate,
# recall is 1, lift is 1. If the sweep is subtly wrong these stop holding.
_ceiling = K_SWEEP_TABLE[K_SWEEP_TABLE["k_requested"] == -1]
assert len(_ceiling) == len(FEATURE_SETS) * len(READ_FRAMES), (
    f"{len(_ceiling)} whole-month row(s), expected "
    f"{len(FEATURE_SETS) * len(READ_FRAMES)} - the sweep did not run the ceiling "
    "for every model x cell."
)
assert np.allclose(_ceiling["precision_at_k"], _ceiling["base_rate_permonth_mean"]), (
    "precision at k = the whole month is not the per-month base rate:\n"
    + _ceiling[["cell", "ranker", "precision_at_k", "base_rate_permonth_mean"]].to_string(index=False)
    + "\n  Taking the whole population as the list must reproduce the base rate "
      "exactly. It does not, so per_month_at_k is not selecting what it reports."
)
assert np.allclose(_ceiling["recall_at_k"], 1.0), (
    "recall at k = the whole month is not 1.0: "
    f"{_ceiling['recall_at_k'].tolist()}. Every positive is in the list by "
    "construction, so anything else means recall is measured against the wrong "
    "denominator."
)
assert np.allclose(_ceiling["lift_over_base"], 1.0), "lift at the ceiling is not 1.0x"

# --- is the reported number really the per-month average? ------------------
# Recomputed by hand, without the helpers, on one (cell, k) case. The pooled
# figure is computed alongside: it is a DIFFERENT quantity and is not what any
# precision_at_k column carries.
_K_PROBE = 200
_probe = SCORED[(SELECTED_LABEL, "holdout_time")]
_by_hand = []
for _m, _g in _probe.groupby("dim_month", sort=True):
    _t = _g.sort_values("score", ascending=False, kind="mergesort").head(_K_PROBE)
    _by_hand.append(int(_t[Y_COL].sum()) / len(_t))
_by_hand_mean = float(np.mean(_by_hand))
_reported = float(K_SWEEP_TABLE.loc[
    (K_SWEEP_TABLE["cell"] == "holdout_time")
    & (K_SWEEP_TABLE["ranker"] == SELECTED_LABEL)
    & (K_SWEEP_TABLE["k_requested"] == _K_PROBE), "precision_at_k"].iloc[0])
_pooled_probe = pooled_at_k(_probe, _K_PROBE)["pooled_precision_at_k"]

assert np.isclose(_reported, _by_hand_mean), (
    f"the reported precision@{_K_PROBE} on holdout_time is {_reported:.6f} but a "
    f"per-month recomputation gives {_by_hand_mean:.6f} (pooled would be "
    f"{_pooled_probe:.6f}). The sweep is not reporting the per-month average it "
    "claims to."
)
assert len(_by_hand) == int(_probe["dim_month"].nunique()), (
    "the per-month recomputation covered "
    f"{len(_by_hand)} month(s) but the cell has {_probe['dim_month'].nunique()} - "
    "months are being dropped before the average."
)

# --- print -----------------------------------------------------------------
_cols_show = ["k_label", "k_used_mean", "base_rate_permonth_mean", "precision_at_k",
              "lift_over_base", "recall_at_k", "found_per_month"]
for _cell_name in READ_FRAMES:
    for _fs_name in FEATURE_SETS:
        _t = K_SWEEP_TABLE[(K_SWEEP_TABLE["cell"] == _cell_name)
                           & (K_SWEEP_TABLE["ranker"] == _fs_name)][_cols_show].copy()
        print("\n" + "=" * 78)
        print(f"K SWEEP  |  {_cell_name}  |  {_fs_name}")
        print("=" * 78)
        print(f"{'k':>11}{'k used':>10}{'base':>9}{'prec@k':>9}{'lift':>8}"
              f"{'recall@k':>10}{'found/mo':>10}")
        for _, _r in _t.iterrows():
            print(f"{_r['k_label']:>11}{_r['k_used_mean']:>10,.0f}"
                  f"{_r['base_rate_permonth_mean']:>9.2%}{_r['precision_at_k']:>9.2%}"
                  f"{_r['lift_over_base']:>7.2f}x{_r['recall_at_k']:>10.2%}"
                  f"{_r['found_per_month']:>10,.1f}")

print("\nper month, then averaged - never one pooled top-k over every month at once.")
print(f"a rep works {REP_CAPACITY[0]}-{REP_CAPACITY[1]} schools a month; every k above that is a "
      "statement about the model, not about a workable list.")

K_SWEEP_CSV = RAW_DIR / f"{STEM}_k_sweep.csv"
K_SWEEP_TABLE.to_csv(K_SWEEP_CSV, index=False)
print(f"\nwrote  {K_SWEEP_CSV.name}   ({K_SWEEP_CSV.stat().st_size / 1e3:.1f} KB)")

---

## 13. Leakage / business-sense surface

Where the features the spec flags as leakage risks landed in the SHAP importances. They describe
things that happen *around* a cancellation conversation, so a high rank is a question for a human,
not an automatic drop:

- `negative_reason_count` and its four `reason_school_*_count` specifics - billing-plan stoppage
  reasons, which a school often sets *while* deciding to leave;
- `months_since_last_cancel_request` and `running_saas_cancel_request_count` - built from the same
  case table the label comes from;
- every `trailing_3m_` / `delta_*_vs_t3m` / `delta_*_vs_12m_ago` variant of the above that the
  extract generated.

Prints rank, mean SHAP importance and decision for each, plus how many of the top 20 features are
flagged. **Nothing is dropped here.** Flagged names that are not in `MODEL_COLS` are listed too - the
extract only keeps variants the warehouse actually carries.

The importance column is §9's z-scored mean, so read the **rank and the decision**, not the level.

In [ ]:
# Base signals the spec flags, and the variant naming the extract generates.
LEAKAGE_RISK_BASE = [
    "negative_reason_count",
    "reason_school_temp_paused_count",
    "reason_school_temp_reducing_enrollment_count",
    "reason_school_permanently_closing_count",
    "reason_school_no_longer_using_bw_count",
    "months_since_last_cancel_request",
    "running_saas_cancel_request_count",
]
VARIANT_TEMPLATES = ["{c}", "trailing_3m_{c}", "delta_{c}_vs_t3m", "delta_{c}_vs_12m_ago"]

_flagged = [t.format(c=b) for b in LEAKAGE_RISK_BASE for t in VARIANT_TEMPLATES]
LEAKAGE_RISK_FEATURES = [c for c in _flagged if c in set(MODEL_COLS)]
_absent = [c for c in _flagged if c not in set(MODEL_COLS)]

_rank_of = dict(zip(IMPORTANCE["feature"], IMPORTANCE["rank"]))
LEAKAGE_REPORT = (IMPORTANCE[IMPORTANCE["feature"].isin(LEAKAGE_RISK_FEATURES)]
                  [["rank", "feature", "mean_shap_importance", "decision"]]
                  .reset_index(drop=True))

print(f"{len(LEAKAGE_RISK_FEATURES)} of {len(_flagged)} flagged names are in "
      f"MODEL_COLS ({len(MODEL_COLS)} features):\n")
print(LEAKAGE_REPORT.to_string(index=False) if len(LEAKAGE_REPORT)
      else "  none of the flagged features is in this matrix")

_in_top20 = LEAKAGE_REPORT[LEAKAGE_REPORT["rank"] <= 20]
_accepted_flagged = LEAKAGE_REPORT[LEAKAGE_REPORT["decision"] == "Accepted"]

print(f"\nflagged features in the top 20 by SHAP importance: {len(_in_top20)}")
for _, _r in _in_top20.iterrows():
    print(f"    rank {int(_r['rank']):>3}  {_r['feature']:<52} "
          f"{_r['mean_shap_importance']:>10.5f}  {_r['decision']}")
print(f"flagged features ACCEPTED by BorutaSHAP: {len(_accepted_flagged)} of "
      f"{len(LEAKAGE_RISK_FEATURES)}")

if _absent:
    print(f"\nflagged names NOT in this matrix ({len(_absent)}) - the extract only keeps "
          "variants the warehouse carries:")
    for _c in _absent:
        print(f"    {_c}")

print("\n" + "-" * 72)
print("NOT auto-dropped. A high rank here is a question for a human: does the")
print("feature describe the school's state BEFORE the decision, or the paperwork")
print("AROUND it? Record the call in the vault page, then re-run with the drop")
print("applied to the extract - not with a filter in this notebook.")
print("-" * 72)

---

## 14. Baseline comparison: model vs the Retention Health Score

The project's ship gate: the model must beat the **Retention Health Score** on the same backtest, both
scored on identical school-months. `product_score` is that score, carried through the extract as a
passthrough. It is **never a feature** - §7 asserts it against `MODEL_COLS`, and §14a re-asserts it
against the two feature lists §11b actually fitted.

**The score.** A 24-point composite of eight product signals:

| points | signals |
| --- | --- |
| 4 if active, 2 if any | attendance, messaging, online payment, activity |
| 2 if active, 1 if any | timecard, form, admission funnel, EC lesson plan |

Buckets: `>= 18` **High**, `>= 12` **Medium**, `>= 0` **Low**.

- **14a** rebuilds the score's three-month trend from `product_score`'s own per-school history.
- **14b** ranks three lead lists - the model, worst score first, biggest drop first - at `k` = 40, 50,
  200, 500 and 1000.
- **14c** runs the team's documented trigger rule as the fixed set it actually is, and puts the model
  at that same list size.

The cells read here are `holdout_school` (the reporting half) and `holdout_time`. §12's cached `SCORED`
supplies the model's probabilities, so nothing here fits a second time. The ship gate is not among
them.

### 14a. Rebuilding the trend from `product_score`'s history

`trailing_3m_product_score` and `delta_product_score_vs_t3m` are excluded from the extract as
health-score derivatives, so this cell reconstructs them to the warehouse definition:

| column | definition |
| --- | --- |
| `trailing_3m_product_score` | mean of the school's **previous three months** - the current month is EXCLUDED. Sort by school then month, group by school, `shift(1)`, `rolling(3, min_periods=3).mean()` |
| `delta_product_score_vs_t3m` | `product_score / trailing_3m_product_score - 1`, **NaN when the baseline is 0** (the warehouse's `nullif`) |

Both names are the warehouse's own, and the cell asserts neither is already a column of the parquet -
if a re-run of the extract starts carrying them, read those instead of rebuilding here.

Reports `product_score`'s null rate and range and asserts the range against the documented 24-point
maximum. A value above it means the formula has changed and every bucket above is wrong.

Both columns are computed on the **full frame before any subsetting** - computed inside an evaluation
cell they would be null for that cell's first three months for no reason. Only `product_score` and
the two keys are read, no feature and no label, and the window looks strictly backwards, so the cell
also asserts that dropping the ship-gate rows entirely changes not one value on any row evaluated
below.

The arithmetic is checked against a constructed example whose trailing averages are computable by
hand, including a school with fewer than three prior months and a school whose baseline is zero. The
example is shuffled before being passed in, so the sort is under test too.

**The coverage gap is a finding, not a footnote.** The trend needs three prior months and a non-zero
baseline, so the incumbent rule cannot score every row - roughly a fifth of them on the real extract.
The cell prints the computable share per cell and splits the shortfall into its two causes. The model
scores all of them.

In [ ]:
# --- product_score is a passthrough, and this section is the reason why -----
RHS_SCORE_COL = "product_score"
RHS_MAX_POINTS = 24
RHS_BUCKETS = ((18, "High"), (12, "Medium"), (0, "Low"))

assert RHS_SCORE_COL in enc.columns, (
    f"{RHS_SCORE_COL!r} is not a column of {ENCODED.name}, so the incumbent "
    "Retention Health Score cannot be rebuilt and the model has nothing to be "
    "scored against. The extract carries it as a passthrough - re-run "
    "data/dataset_creation.ipynb."
)
assert RHS_SCORE_COL in set(PASSTHROUGH), (
    f"{RHS_SCORE_COL!r} is not in the encoding JSON's passthrough list "
    f"({PASSTHROUGH}). This section scores the model against it, so it has to be "
    "carried in the frame - and carried as a passthrough, not as a feature."
)
assert RHS_SCORE_COL not in set(MODEL_COLS), (
    f"{RHS_SCORE_COL!r} is in MODEL_COLS. The whole point of this section is that "
    "the model is scored AGAINST the Retention Health Score; a model that has the "
    "score as an input beats it trivially and the comparison means nothing."
)
for _fs_name, _cols in FEATURE_SETS.items():
    assert RHS_SCORE_COL not in set(_cols), (
        f"{RHS_SCORE_COL!r} is in the {_fs_name!r} feature list that §11b fitted "
        f"({len(_cols)} features). Checking MODEL_COLS is not enough - what "
        "matters is what the booster was handed. The comparison below is void "
        "until the feature list is fixed upstream."
    )

# --- what the column actually holds -----------------------------------------
_ps = enc[RHS_SCORE_COL].astype("float64")
RHS_NULL_RATE = float(_ps.isna().mean())

assert _ps.notna().any(), (
    f"{RHS_SCORE_COL!r} is null on every one of the {len(enc):,} rows, so the "
    "baseline is empty and there is nothing to compare against."
)
assert float(_ps.max()) <= RHS_MAX_POINTS, (
    f"{RHS_SCORE_COL} reaches {float(_ps.max()):.1f}, above the documented "
    f"{RHS_MAX_POINTS}-point maximum of the Retention Health Score (4+4+4+4 on "
    "attendance / messaging / online payment / activity, 2+2+2+2 on timecard / "
    "form / admission funnel / EC lesson plan). Either the formula changed "
    "upstream - in which case the buckets "
    + ", ".join(f">= {_p} {_n}" for _p, _n in RHS_BUCKETS)
    + " and everything in this section are wrong - or the column is not the score "
      "it is named after. Settle it against the warehouse before reading any "
      "number below."
)
assert float(_ps.min()) >= 0, (
    f"{RHS_SCORE_COL} goes down to {float(_ps.min()):.1f}. The score is a sum of "
    "non-negative point awards, so it cannot be negative; a negative value means "
    "a sentinel is being carried as a measurement and would sit at the top of "
    "the worst-score-first lead list."
)

print(f"Retention Health Score: {RHS_MAX_POINTS}-point composite, buckets "
      + ", ".join(f">= {_p} {_n}" for _p, _n in RHS_BUCKETS))
print(f"{RHS_SCORE_COL}: {RHS_NULL_RATE:.2%} null, range "
      f"{float(_ps.min()):.0f} -> {float(_ps.max()):.0f} of {RHS_MAX_POINTS}, "
      f"median {float(_ps.median()):.0f}, mean {float(_ps.mean()):.1f}")

# --- the two trend columns the extract deliberately does not carry ----------
RHS_T3M_COL = "trailing_3m_product_score"      # the warehouse's own names
RHS_DELTA_COL = "delta_product_score_vs_t3m"

_already_there = [c for c in (RHS_T3M_COL, RHS_DELTA_COL) if c in enc.columns]
assert not _already_there, (
    f"{_already_there} already exist(s) in {ENCODED.name}. The extract "
    "deliberately excludes the Retention Health Score's trend columns, which is "
    "why this cell rebuilds them; if a re-run now carries them, read those "
    "instead of recomputing - two definitions of the same lead-list rule is one "
    "too many."
)


def rhs_trend(frame: pd.DataFrame) -> pd.DataFrame:
    """Rebuild the warehouse's two Retention Health Score trend columns.

    trailing_3m_product_score is the mean of the school's PREVIOUS THREE MONTHS
    - the current month is excluded, which is what the shift(1) before the
    rolling window is for. min_periods=3 means a school with fewer than three
    prior months gets NaN rather than a short average.

    delta_product_score_vs_t3m is product_score / t3m - 1, and NaN when the
    baseline is 0 (the warehouse divides by a nullif, and a ratio against a zero
    baseline is not a number a lead list can be sorted by).

    Sorts by school then month internally, and shifts WITHIN each school, so the
    caller's row order does not matter; the result is returned aligned to
    `frame.index`.
    """
    f = (frame[["school_id", "dim_month", RHS_SCORE_COL]]
         .sort_values(["school_id", "dim_month"], kind="mergesort"))
    score = f[RHS_SCORE_COL].astype("float64")

    prev = score.groupby(f["school_id"], observed=True).shift(1)
    t3m = (prev.groupby(f["school_id"], observed=True)
               .rolling(3, min_periods=3).mean()
               .reset_index(level=0, drop=True))

    base = t3m.where(t3m != 0.0)          # nullif(t3m, 0)
    delta = score / base - 1.0

    return pd.DataFrame({RHS_T3M_COL: t3m.reindex(frame.index),
                         RHS_DELTA_COL: delta.reindex(frame.index)},
                        index=frame.index)


# --- the arithmetic, on an example that can be checked by hand -------------
# school 1: 0, 6, 12, 18, 24, 24 over six consecutive months
#   t3m(month 4) = (0 + 6 + 12) / 3 = 6      <- previous three, NOT (6+12+18)/3 = 12
#   t3m(month 5) = (6 + 12 + 18) / 3 = 12
#   t3m(month 6) = (12 + 18 + 24) / 3 = 18
#   delta(month 4) = 18 / 6 - 1 = 2.0, delta(month 5) = 24 / 12 - 1 = 1.0
# school 2: three months only -> never three PRIOR months -> all NaN
# school 3: 0, 0, 0, 5 -> t3m(month 4) = 0 -> delta is NaN, not inf
_demo = pd.DataFrame({
    "school_id": [1, 1, 1, 1, 1, 1, 2, 2, 2, 3, 3, 3, 3],
    "dim_month": pd.to_datetime(
        ["2024-01-01", "2024-02-01", "2024-03-01", "2024-04-01", "2024-05-01", "2024-06-01",
         "2024-01-01", "2024-02-01", "2024-03-01",
         "2024-01-01", "2024-02-01", "2024-03-01", "2024-04-01"]),
    RHS_SCORE_COL: [0.0, 6.0, 12.0, 18.0, 24.0, 24.0,
                    10.0, 20.0, 24.0,
                    0.0, 0.0, 0.0, 5.0],
})
# Shuffled in, so the function's own sort is under test too.
_demo_out = rhs_trend(_demo.sample(frac=1.0, random_state=0)).reindex(_demo.index)
_dt = _demo_out[RHS_T3M_COL].to_numpy()
_dd = _demo_out[RHS_DELTA_COL].to_numpy()

assert np.isnan(_dt[:3]).all(), (
    f"the first three months of a school have a trailing average ({_dt[:3]}). "
    "min_periods=3 over a shifted series cannot produce one - fewer than three "
    "PRIOR months exist."
)
assert _dt[3] == 6.0, (
    f"t3m for 2024-04 is {_dt[3]!r}, expected 6.0 = (0 + 6 + 12) / 3, the three "
    "months BEFORE it. 12.0 = (6 + 12 + 18) / 3 would mean the window includes "
    "the current month, i.e. the shift(1) is missing - which leaks the row's own "
    "score into its own baseline and makes delta_product_score_vs_t3m a "
    "different, tamer quantity than the warehouse's."
)
assert _dt[4] == 12.0 and _dt[5] == 18.0, f"t3m marched wrong: {_dt[3:6]}"
assert np.isclose(_dd[3], 2.0) and np.isclose(_dd[4], 1.0), (
    f"delta is {_dd[3:5]}, expected [2.0, 1.0] from 18/6 - 1 and 24/12 - 1."
)
assert np.isnan(_dd[:3]).all(), "delta exists where the baseline does not"
assert np.isnan(_dt[6:9]).all(), (
    f"a school with three rows total has a trailing average: {_dt[6:9]}. It has "
    "at most two PRIOR months."
)
assert _dt[12] == 0.0 and np.isnan(_dd[12]), (
    f"a zero baseline gave t3m={_dt[12]!r}, delta={_dd[12]!r}. The warehouse "
    "divides by nullif(t3m, 0), so delta must be NaN here - not inf, and not a "
    "silently large number that would sit at the top of a lead list."
)
print("\ntrailing-average arithmetic checked on a constructed example:")
print(pd.concat([_demo, _demo_out], axis=1).to_string(index=False))

# --- rebuilt on the FULL frame, before any subsetting ----------------------
# product_score and the two keys only: no feature and no label of any row is
# read here. The window looks strictly backwards, so the check below shows the
# ship-gate rows contribute nothing to any row this notebook evaluates.
RHS_TREND = rhs_trend(enc)
assert list(RHS_TREND.index) == list(enc.index), "the rebuilt trend is not aligned to the frame"

_trend_without_gate = rhs_trend(enc.loc[enc["split"] != SHIP_GATE_SPLIT])
for _cell_name, _frame in READ_FRAMES.items():
    _a = RHS_TREND.loc[_frame.index]
    _b = _trend_without_gate.loc[_frame.index]
    assert _a.equals(_b), (
        f"the rebuilt trend on {_cell_name} changes when the "
        f"{SHIP_GATE_SPLIT!r} rows are removed from the frame it is computed "
        "over. The trailing window is supposed to look strictly backwards within "
        "a school, and the ship gate is a later era for schools this notebook "
        "does not train on - so it must contribute nothing here. It does, so the "
        "window is not causal."
    )
print(f"\nrebuilt on all {len(enc):,} rows; removing the {SHIP_GATE_SPLIT!r} rows changes "
      "no value on any row evaluated below  OK")

# --- coverage: what fraction of rows the incumbent rule can score at all ----
# Split into its two causes, because they are different problems: no three
# months of prior history, or a baseline of exactly zero.
_t3m_all, _delta_all = RHS_TREND[RHS_T3M_COL], RHS_TREND[RHS_DELTA_COL]
RHS_TREND_COVERAGE = float(_delta_all.notna().mean())

_cov_rows = []
for _label, _idx in [("ALL ROWS", enc.index)] + [(c, f.index) for c, f in READ_FRAMES.items()]:
    _t, _d = _t3m_all.loc[_idx], _delta_all.loc[_idx]
    _cov_rows.append({
        "population": _label,
        "rows": len(_idx),
        "trend_computable": int(_d.notna().sum()),
        "computable_share": float(_d.notna().mean()),
        "no_3m_history": int(_t.isna().sum()),
        "zero_baseline": int((_t == 0.0).sum()),
    })
RHS_COVERAGE_GAP = pd.DataFrame(_cov_rows)
assert (RHS_COVERAGE_GAP["trend_computable"]
        + RHS_COVERAGE_GAP["no_3m_history"]
        + RHS_COVERAGE_GAP["zero_baseline"] == RHS_COVERAGE_GAP["rows"]).all(), (
    "computable + no-history + zero-baseline does not add up to the row count:\n"
    + RHS_COVERAGE_GAP.to_string(index=False)
    + "\n  Those three are supposed to partition every row - a null delta is "
      "either a null baseline or a zero one, and nothing else. They do not, so "
      "the rebuilt delta is null for some third reason."
)

print("\n" + "=" * 78)
print("COVERAGE GAP - WHAT THE INCUMBENT RULE CANNOT SCORE AT ALL")
print("=" * 78)
print(f"  {'population':<44}{'rows':>10}{'computable':>12}{'share':>8}")
for _, _r in RHS_COVERAGE_GAP.iterrows():
    print(f"  {_r['population']:<44}{int(_r['rows']):>10,}"
          f"{int(_r['trend_computable']):>12,}{_r['computable_share']:>8.1%}")
print("\n  the shortfall, by cause:")
for _, _r in RHS_COVERAGE_GAP.iterrows():
    print(f"  {_r['population']:<44}"
          f"{int(_r['no_3m_history']):>10,} no 3m history"
          f"{int(_r['zero_baseline']):>10,} zero baseline")
print(f"\n  -> the trend is computable on {RHS_TREND_COVERAGE:.1%} of all rows. On the rest the")
print("     incumbent rule cannot produce a number at all, so those schools are")
print("     invisible to it. The model scores 100% of them. That gap is a finding")
print("     in its own right, not a caveat on the numbers below.")

### 14b. Three rankers at matched list sizes

Three lead lists over the same evaluation cell, at `k` in `K_BASELINE`:

| ranker | order |
| --- | --- |
| `model` | the evaluation model's predicted probability, descending |
| `RHS lowest score first` | `product_score` ascending - worst health, highest risk |
| `RHS biggest drop first` | the rebuilt delta ascending - biggest drop, highest risk |

`precision@k` and `recall@k` are computed **per calendar month and then averaged**. The rep works one
monthly list, so a single pooled top-`k` over every month at once is a different quantity and the
wrong one. The base rate and the lift over it are reported next to every figure.

A ranker does not rank a row it cannot score, so the null rows are dropped **for that ranker only** -
the delta is the one with nulls - and `rows`, `months` and `k_used_mean` report the pool actually
used, so a smaller pool shows up in the table instead of quietly changing the comparison. A month
holding fewer than `k` scorable rows is skipped rather than averaged in over a short list
(`skip_short_months=True`); months with nothing scorable never enter the average and are counted in
the coverage line.

The two RHS rankers agree at small `k` on the rows the delta can score, and the cell asserts why:
`product_score` is an integer 0-24, so inside the delta's own subset a row scoring 0 has
`delta = 0 / t3m - 1 = -1.0` exactly, and nothing is lower than either. That tied floor is hundreds of
school-months a month, so below it both rankings are one arbitrary draw from the same block. Its size
and its own precision are printed - the same number without the tie-break.

In [ ]:
# --- the three rankers ------------------------------------------------------
K_BASELINE = (40, 50, 200, 500, 1000)
K_BASELINE_HEADLINE = 40

HEADLINE_MODEL = SELECTED_LABEL          # the arm §11b selected, not the full set

MODEL_RANKER = "model"
RHS_SCORE_RANKER = "rhs_lowest_score_first"
RHS_DELTA_RANKER = "rhs_biggest_drop_first"
BASELINE_RANKERS = (MODEL_RANKER, RHS_SCORE_RANKER, RHS_DELTA_RANKER)
RANKER_LABEL = {
    MODEL_RANKER: "model",
    RHS_SCORE_RANKER: "RHS lowest score first",
    RHS_DELTA_RANKER: "RHS biggest drop first",
}
BASELINE_POPULATION = "scorable_by_this_ranker"


def baseline_board(cell_name: str, frame: pd.DataFrame) -> pd.DataFrame:
    """Keys, label, the RHS columns and one ranking score per ranker.

    No features: this frame is for ranking, so nothing downstream can fit on it.
    The two RHS scores are NEGATED, because per_month_at_k takes the highest
    score first and the risky end of both columns is the low end.
    """
    b = frame[["school_id", "dim_month", Y_COL]].copy()
    b[RHS_SCORE_COL] = frame[RHS_SCORE_COL].astype("float64")
    b["rhs_t3m"] = RHS_TREND.loc[frame.index, RHS_T3M_COL]
    b["rhs_delta"] = RHS_TREND.loc[frame.index, RHS_DELTA_COL]

    _s = SCORED[(HEADLINE_MODEL, cell_name)]
    assert list(_s.index) == list(frame.index), (
        f"the cached {HEADLINE_MODEL!r} scores for {cell_name!r} are not aligned "
        "to the frame - §12 scored a different row order, so the model's "
        "probability would be attached to the wrong school-month."
    )
    b[f"score__{MODEL_RANKER}"] = _s["score"]
    b[f"score__{RHS_SCORE_RANKER}"] = -b[RHS_SCORE_COL]
    b[f"score__{RHS_DELTA_RANKER}"] = -b["rhs_delta"]
    return b


BOARDS, _cov, _rows = {}, [], []

for _cell_name, _frame in READ_FRAMES.items():
    _b = baseline_board(_cell_name, _frame)
    BOARDS[_cell_name] = _b
    _months_in_cell = int(_b["dim_month"].nunique())

    for _r in BASELINE_RANKERS:
        _ok = _b[f"score__{_r}"].notna()
        _scored_r = (_b.loc[_ok, ["school_id", "dim_month", Y_COL]]
                       .assign(score=_b.loc[_ok, f"score__{_r}"]))
        _cov.append({
            "cell": _cell_name,
            "ranker_key": _r,
            "ranker": RANKER_LABEL[_r],
            "rows_in_cell": int(len(_b)),
            "scorable": int(_ok.sum()),
            "scorable_share": float(_ok.mean()),
            "months_in_cell": _months_in_cell,
            "months_with_any_scorable": int(_scored_r["dim_month"].nunique()),
        })
        assert len(_scored_r), (
            f"{RANKER_LABEL[_r]!r} can score none of the {len(_b):,} rows of "
            f"{_cell_name!r}, so it cannot be compared against anything. For the "
            "RHS trend that means no school in the cell has three months of prior "
            f"{RHS_SCORE_COL} history - check §14a's coverage table."
        )
        for _k in K_BASELINE:
            _row = rank_at_k(_scored_r, _k, _cell_name, RANKER_LABEL[_r],
                             BASELINE_POPULATION, skip_short_months=True)
            _row["ranker_key"] = _r
            _rows.append(_row)

BASELINE = pd.DataFrame(_rows)
BASELINE_COVERAGE = pd.DataFrame(_cov)

# --- the tied floor, which is why the two RHS columns agree at small k ------
# Inside the delta's own scorable subset the baseline is non-null and non-zero,
# so product_score == 0 gives delta = 0 / t3m - 1 = -1.0 exactly and the two
# risky ends are the same block of rows. Asserted rather than explained away,
# then measured: the floor's own precision is the same number without the
# arbitrary tie-break.
RHS_FLOOR = []
for _cell_name in READ_FRAMES:
    _b = BOARDS[_cell_name]
    _has_delta = _b["rhs_delta"].notna()
    _z = _has_delta & (_b[RHS_SCORE_COL] == 0)
    _d1 = _has_delta & (_b["rhs_delta"] == -1.0)
    assert _z.equals(_d1), (
        f"on {_cell_name!r} the delta-scorable rows with {RHS_SCORE_COL} == 0 "
        f"({int(_z.sum()):,}) are not the rows with delta == -1.0 "
        f"({int(_d1.sum()):,}). Where the delta exists the baseline is non-null "
        "and non-zero, so 0 / t3m - 1 is exactly -1.0 and the two sets must "
        "coincide. They do not, so the rebuilt delta is not the ratio it claims "
        "to be."
    )
    _fm = _b.loc[_z].groupby("dim_month", sort=True).agg(
        floor_rows=(Y_COL, "size"), floor_hits=(Y_COL, "sum"))
    _all_m = _b.loc[_has_delta].groupby("dim_month", sort=True)[Y_COL].agg(["size", "sum"])
    _fm = _fm.reindex(_all_m.index).fillna(0)
    RHS_FLOOR.append({
        "cell": _cell_name,
        "floor_rows_per_month": float(_fm["floor_rows"].mean()),
        "floor_precision": float((_fm["floor_hits"] / _fm["floor_rows"]).mean(skipna=True)),
        "floor_recall": float((_fm["floor_hits"] / _all_m["sum"]).mean(skipna=True)),
    })

# --- the comparison, per cell ----------------------------------------------
_k_labels = [str(_k) for _k in K_BASELINE]
_W = 24

for _cell_name in READ_FRAMES:
    _t = BASELINE[BASELINE["cell"] == _cell_name]
    _labels = [RANKER_LABEL[_r] for _r in BASELINE_RANKERS]
    _base = float(_t.loc[_t["ranker_key"] == MODEL_RANKER, "base_rate_permonth_mean"].iloc[0])

    print("\n" + "=" * 78)
    print(f"BASELINE COMPARISON  |  {_cell_name}")
    print("=" * 78)
    print(f"base rate {_base:.3%} per month, {len(BOARDS[_cell_name]):,} rows over "
          f"{int(BOARDS[_cell_name]['dim_month'].nunique())} month(s)")

    _lft = _t.pivot(index="k_label", columns="ranker",
                    values="lift_over_base").reindex(_k_labels)
    for _metric, _title, _with_lift in (("precision_at_k", "precision@k (lift over base)", True),
                                        ("recall_at_k", "recall@k", False)):
        _piv = _t.pivot(index="k_label", columns="ranker", values=_metric).reindex(_k_labels)
        print(f"\n  {_title}")
        print(f"  {'k':>6}" + "".join(f"{_c:>{_W}}" for _c in _labels))
        for _kl in _k_labels:
            _cells = ""
            for _c in _labels:
                _v = _piv.loc[_kl, _c]
                _cells += (f"{_v:.2%} ({_lft.loc[_kl, _c]:.1f}x)".rjust(_W)
                           if _with_lift else f"{_v:.2%}".rjust(_W))
            print(f"  {_kl:>6}{_cells}")

    print("\n  pool actually used (a ranker does not rank what it cannot score, and "
          "its own")
    print("  months carry their own base rate - which is what its lift above is "
          "against)")
    print(f"  {'ranker':<26}{'scorable':>10}{'share':>8}{'months':>12}{'base':>8}"
          f"{'k used @' + str(K_BASELINE[-1]):>13}{'short mo':>10}")
    for _r in BASELINE_RANKERS:
        _c = BASELINE_COVERAGE[(BASELINE_COVERAGE["cell"] == _cell_name)
                               & (BASELINE_COVERAGE["ranker_key"] == _r)].iloc[0]
        _last = _t[(_t["ranker_key"] == _r) & (_t["k_requested"] == K_BASELINE[-1])].iloc[0]
        _mo = f"{int(_c['months_with_any_scorable'])} of {int(_c['months_in_cell'])}"
        print(f"  {RANKER_LABEL[_r]:<26}{int(_c['scorable']):>10,}"
              f"{_c['scorable_share']:>8.1%}{_mo:>12}"
              f"{_last['base_rate_permonth_mean']:>8.2%}"
              f"{_last['k_used_mean']:>13,.0f}{int(_last['months_short_skipped']):>10}")

    _f = next(_x for _x in RHS_FLOOR if _x["cell"] == _cell_name)
    print(f"\n  tied floor of both RHS rankings ({RHS_SCORE_COL} == 0, delta == -1.0): "
          f"{_f['floor_rows_per_month']:,.0f} rows/month,")
    print(f"  precision {_f['floor_precision']:.2%}, recall {_f['floor_recall']:.2%} - below "
          "that size both RHS lists are one")
    print("  arbitrary draw from the same block, so their precision is a tie-break, "
          "not a ranking.")

# --- did the model beat both, at the list size a rep actually works? --------
print("\n" + "=" * 78)
print(f"HEADLINE  |  precision@{K_BASELINE_HEADLINE}, per month then averaged")
print("=" * 78)
for _cell_name in READ_FRAMES:
    _h = BASELINE[(BASELINE["cell"] == _cell_name)
                  & (BASELINE["k_requested"] == K_BASELINE_HEADLINE)]
    _m = float(_h.loc[_h["ranker_key"] == MODEL_RANKER, "precision_at_k"].iloc[0])
    _rhs = {_r: float(_h.loc[_h["ranker_key"] == _r, "precision_at_k"].iloc[0])
            for _r in (RHS_SCORE_RANKER, RHS_DELTA_RANKER)}
    print(f"\n{_cell_name}")
    print(f"  {'model':<26}{_m:>8.2%}")
    for _r, _v in _rhs.items():
        _ratio = f"   model is {_m / _v:.2f}x it" if _v else "   (zero - no ratio)"
        print(f"  {RANKER_LABEL[_r]:<26}{_v:>8.2%}{_ratio}")
    _beat = _m > max(_rhs.values())
    print("  -> the model BEATS both RHS rankers here." if _beat
          else "  -> the model DOES NOT beat both RHS rankers here. That is the "
               "ship gate failing, not a\n     formatting problem - read §14a's "
               "coverage table and §12's sweep before anything else.")

### 14c. The trigger rule as it is actually used

The team's documented lead-list rule is a **set**, not a ranking: every school whose Retention Health
Score has dropped more than 20% against its own trailing three months -
`delta_product_score_vs_t3m < RHS_TRIGGER_THRESHOLD`. The list size is whatever that comes to.

Reported per month and then averaged: how many schools it flags, its precision, its recall - and the
model's precision and recall **at that same list size**, month by month, so the two lists are the same
shape. Both are measured against the whole month, not against the rule's own scorable subset: a
school the rule cannot score is a school it never flags, and that is part of what the rule costs.

The flagged count is also divided by a rep's working capacity of 40-50 schools a month, which is the
number that says whether the rule is a lead list or a mailing list.

Writes `..._baseline_comparison.csv` - §14b's ranker rows and these trigger rows in one schema, each
carrying `k`, precision, recall, the effective list size, the base rate, the lift and the ranker's
scorable share.

In [ ]:
RHS_TRIGGER_THRESHOLD = -0.20
RHS_TRIGGER_LABEL = f"trigger(delta<{RHS_TRIGGER_THRESHOLD:.2f})"
TRIGGER_POPULATION = "whole_month"


def trigger_month_table(board: pd.DataFrame, model_key: str) -> pd.DataFrame:
    """Per month: the rule as a fixed set, and the model at that same size.

    The rule's list size varies month to month, so the model's k is taken from
    it each month rather than fixed - that is what makes the two comparable.

    A row whose delta is null compares False and is therefore never flagged,
    which is the rule as it actually behaves. Precision and recall are against
    the WHOLE month either way: a school the rule cannot score is a school it
    misses, and hiding that in the denominator would flatter it.
    """
    rows = []
    for _m, _g in board.groupby("dim_month", sort=True):
        _flagged = _g.loc[_g["rhs_delta"] < RHS_TRIGGER_THRESHOLD]
        _n = len(_flagged)
        _pos = int(_g[Y_COL].sum())
        _top = (_g.sort_values(f"score__{model_key}", ascending=False, kind="mergesort")
                  .head(_n))
        _rule_hits = int(_flagged[Y_COL].sum())
        _model_hits = int(_top[Y_COL].sum())
        rows.append({
            "dim_month": _m,
            "rows": len(_g),
            "positives": _pos,
            "flagged": _n,
            "flagged_share": _n / len(_g) if len(_g) else np.nan,
            "rule_hits": _rule_hits,
            "rule_precision": _rule_hits / _n if _n else np.nan,
            "rule_recall": _rule_hits / _pos if _pos else np.nan,
            "model_hits": _model_hits,
            "model_precision": _model_hits / _n if _n else np.nan,
            "model_recall": _model_hits / _pos if _pos else np.nan,
        })
    return pd.DataFrame(rows)


def trigger_summary(tt: pd.DataFrame, cell_label: str, ranker: str, ranker_key: str,
                    prec_col: str, rec_col: str, hits_col: str) -> dict:
    """A trigger month table folded into rank_at_k's row schema."""
    _base = float((tt["positives"] / tt["rows"]).mean())
    _prec = float(tt[prec_col].mean(skipna=True))
    return {
        "cell": cell_label,
        "ranker": ranker,
        "ranker_key": ranker_key,
        "population": TRIGGER_POPULATION,
        "k_label": RHS_TRIGGER_LABEL,
        "k_requested": -1,             # not a top-k: the rule sets its own size
        "k_used_mean": float(tt["flagged"].mean()),
        "months": int(len(tt)),
        "months_short_skipped": 0,     # a fixed set, not a top-k: nothing to be short of
        "rows": int(tt["rows"].sum()),
        "positives": int(tt["positives"].sum()),
        "base_rate_permonth_mean": _base,
        "base_rate_pooled": float(tt["positives"].sum() / tt["rows"].sum()),
        "precision_at_k": _prec,
        "recall_at_k": float(tt[rec_col].mean(skipna=True)),
        "found_per_month": float(tt[hits_col].mean()),
        "lift_over_base": _prec / _base if _base else np.nan,
    }


TRIGGER, _trigger_rows = {}, []

for _cell_name in READ_FRAMES:
    _tt = trigger_month_table(BOARDS[_cell_name], MODEL_RANKER)
    TRIGGER[_cell_name] = _tt
    _trigger_rows.append(trigger_summary(
        _tt, _cell_name, RANKER_LABEL[RHS_DELTA_RANKER] + " AS A FIXED SET",
        RHS_DELTA_RANKER, "rule_precision", "rule_recall", "rule_hits"))
    _trigger_rows.append(trigger_summary(
        _tt, _cell_name, RANKER_LABEL[MODEL_RANKER] + " AT THE SAME LIST SIZE",
        MODEL_RANKER, "model_precision", "model_recall", "model_hits"))

    assert (_tt["flagged"] <= _tt["rows"]).all(), "more flagged than rows in a month"
    assert (_tt["model_hits"] <= _tt["positives"]).all(), (
        "the model found more at-risk schools in a month than the month has"
    )
    _scorable = int(BOARDS[_cell_name]["rhs_delta"].notna().sum())
    assert int(_tt["flagged"].sum()) <= _scorable, (
        f"on {_cell_name!r} the rule flagged {int(_tt['flagged'].sum()):,} rows "
        f"but only {_scorable:,} have a computable delta. A null delta must "
        "compare False, so a flag on one means the threshold is being applied to "
        "a filled value."
    )

TRIGGER_TABLE = pd.DataFrame(_trigger_rows)

# --- per month, then the two-line block ------------------------------------
print("=" * 78)
print(f"THE TRIGGER RULE AS USED  |  {RHS_DELTA_COL} < {RHS_TRIGGER_THRESHOLD:.2f}")
print("=" * 78)
for _cell_name in READ_FRAMES:
    _tt = TRIGGER[_cell_name]
    print(f"\n{_cell_name}   {_tt['rows'].mean():,.0f} rows/month, "
          f"{_tt['positives'].mean():.0f} at risk/month")
    print(f"  {'month':<10}{'rows':>9}{'flagged':>9}{'rule P':>9}{'rule R':>9}"
          f"{'model P':>10}{'model R':>9}")
    for _, _r in _tt.iterrows():
        _pc = f"{_r['rule_precision']:>9.2%}" if _r["flagged"] else f"{'n/a':>9}"
        _mc = f"{_r['model_precision']:>10.2%}" if _r["flagged"] else f"{'n/a':>10}"
        print(f"  {_r['dim_month']:%Y-%m}   {int(_r['rows']):>9,}{int(_r['flagged']):>9,}"
              f"{_pc}{_r['rule_recall']:>9.2%}{_mc}{_r['model_recall']:>9.2%}")
    _empty = int(_tt["flagged"].eq(0).sum())
    if _empty:
        print(f"  {_empty} of {len(_tt)} month(s) flagged nothing - no school has three "
              "prior months yet. Precision is")
        print("  undefined there and excluded from the mean; recall counts as 0, which "
              "is what the rule delivers.")

print("\n" + "=" * 78)
print("THE TRIGGER AGAINST THE MODEL AT THE SAME LIST SIZE")
print("=" * 78)
for _cell_name in READ_FRAMES:
    _tt = TRIGGER[_cell_name]
    _n = float(_tt["flagged"].mean())
    print(f"\n{_cell_name}   flags {_n:,.0f} schools/month "
          f"({_tt['flagged_share'].mean():.1%} of the month, "
          + ", ".join(f"{_n / _c:.1f}x a {_c}-school list" for _c in REP_CAPACITY) + ")")
    print(f"  rule    precision {_tt['rule_precision'].mean(skipna=True):>7.2%}   "
          f"recall {_tt['rule_recall'].mean(skipna=True):>7.2%}")
    print(f"  model   precision {_tt['model_precision'].mean(skipna=True):>7.2%}   "
          f"recall {_tt['model_recall'].mean(skipna=True):>7.2%}   "
          f"at the same {_n:,.0f}-school list")

# --- persist ----------------------------------------------------------------
BASELINE_OUT = pd.concat([BASELINE, TRIGGER_TABLE], ignore_index=True).merge(
    BASELINE_COVERAGE[["cell", "ranker_key", "scorable", "scorable_share",
                       "months_in_cell", "months_with_any_scorable"]],
    on=["cell", "ranker_key"], how="left")

assert BASELINE_OUT["scorable_share"].notna().all(), (
    "a comparison row has no scorable share attached, so ranker_key did not "
    "match the coverage table:\n"
    + BASELINE_OUT.loc[BASELINE_OUT["scorable_share"].isna(),
                       ["cell", "ranker", "ranker_key", "k_label"]].to_string(index=False)
)

BASELINE_CSV = RAW_DIR / f"{STEM}_baseline_comparison.csv"
BASELINE_OUT.to_csv(BASELINE_CSV, index=False)

_back = pd.read_csv(BASELINE_CSV)
assert len(_back) == len(BASELINE_OUT), "the comparison CSV does not round trip"
assert set(_back["population"]) == {BASELINE_POPULATION, TRIGGER_POPULATION}, (
    f"unexpected population labels in the CSV: {sorted(set(_back['population']))} "
    f"(expected {sorted({BASELINE_POPULATION, TRIGGER_POPULATION})})"
)
assert set(_back["k_label"]) == set(str(_k) for _k in K_BASELINE) | {RHS_TRIGGER_LABEL}, (
    f"unexpected k labels in the CSV: {sorted(set(_back['k_label']))}"
)

print(f"\nwrote  {BASELINE_CSV.name}   ({BASELINE_CSV.stat().st_size / 1e3:.1f} KB)   "
      f"{len(BASELINE_OUT)} rows")
print(f"       {len(BASELINE)} ranker rows over k = {list(K_BASELINE)} + "
      f"{len(TRIGGER_TABLE)} trigger rows, both cells")

---

## 15. Audit, and what is deliberately not here

Re-reads `SPLIT_ACCESS_LOG` - every `cell()` call this run made - and asserts the ship gate is absent
from it. Together with §6's static scan of the code cells and `cell()` refusing the value at runtime,
that is the three mechanisms closing on the same fact: **`test` was not loaded**.

Deliberately not in this notebook, and not accidentally missing:

- **No billing ablation.** Deferred. The billing features are in the candidate set and BorutaSHAP
  judged them like any other.
- **No hyperparameter search.** `scale_pos_weight` is printed with the note that the spec says to
  tune it downward on AUCPR; nothing here tunes it.
- **No `test`-cell number.** That is the ship gate, read once, later, on a model that is final. §14's
  baseline comparison is on `holdout_school` and `holdout_time`, so it is a read on the operating
  point, not the ship-gate number itself.

Why any of it is the way it is: `Proactive Churn/Design Decisions.md` in the Obsidian vault.

In [ ]:
_accessed = sorted(set(SPLIT_ACCESS_LOG))

assert SHIP_GATE_SPLIT not in _accessed, (
    f"the ship-gate cell {SHIP_GATE_SPLIT!r} was loaded {SPLIT_ACCESS_LOG.count(SHIP_GATE_SPLIT)} "
    "time(s) in this run. cell() is supposed to refuse it, so either the "
    "gatekeeper was edited or rows were selected without it."
)
for _s in EXCLUDED_SPLITS:
    assert _s not in _accessed, f"the excluded cell {_s!r} was loaded in this run"

assert set(_accessed) <= set(LOADABLE_SPLITS), (
    f"cells loaded that are not loadable: {sorted(set(_accessed) - set(LOADABLE_SPLITS))}"
)
assert FIT_SPLIT in _accessed, (
    f"the {FIT_SPLIT!r} cell was never loaded - did §7 onwards actually run?"
)

print("split access audit")
for _s in ALL_SPLITS:
    _n = SPLIT_ACCESS_LOG.count(_s)
    _role = ("FIT + SELECT" if _s == FIT_SPLIT else
             "read" if _s in READ_SPLITS else
             "SHIP GATE" if _s == SHIP_GATE_SPLIT else "excluded")
    print(f"  {_s:<16}{_role:<14}{_n:>4} cell() call(s)")

print(f"\nOK - {SHIP_GATE_SPLIT!r} loaded 0 times, {list(EXCLUDED_SPLITS)} loaded 0 times.")
print("Three mechanisms agree: cell() refuses the value, §6's static scan found no")
print("code cell comparing against it or passing it, and this log is empty of it.")

print(f"\nartefacts in {RAW_DIR}:")
for _p in (SELECTION_JSON, IMPORTANCE_CSV, EVAL_CSV, K_SWEEP_CSV, BASELINE_CSV):
    print(f"  {_p.name:<64}{_p.stat().st_size / 1e3:>8.1f} KB")

print(f"\ncandidates {len(MODEL_COLS)} -> accepted {len(ACCEPTED)} / "
      f"tentative {len(TENTATIVE)} / rejected {len(REJECTED)} "
      f"in {BORUTA_SECONDS:.1f}s over {BORUTA_N_TRIALS} trials")

_hl = BASELINE_OUT[(BASELINE_OUT["cell"] == "holdout_time")
                   & (BASELINE_OUT["k_requested"] == K_BASELINE_HEADLINE)]
print(f"\nprecision@{K_BASELINE_HEADLINE} on holdout_time, per month then averaged:")
for _, _r in _hl.iterrows():
    print(f"  {_r['ranker']:<40}{_r['precision_at_k']:>8.2%}  "
          f"lift {_r['lift_over_base']:.2f}x  "
          f"on {_r['scorable_share']:.0%} of the cell's rows")
_tt = TRIGGER["holdout_time"]
print(f"\nthe incumbent trigger on holdout_time: {_tt['flagged'].mean():,.0f} schools/month at "
      f"{_tt['rule_precision'].mean(skipna=True):.2%} precision / "
      f"{_tt['rule_recall'].mean(skipna=True):.2%} recall;")
print(f"the model at that same list size: "
      f"{_tt['model_precision'].mean(skipna=True):.2%} / "
      f"{_tt['model_recall'].mean(skipna=True):.2%}.")
print("\nNext: settle the flagged features in §13 with a human, then the billing")
print("ablation - deferred.")

## Hyperparameter check: does tree depth matter?

Testing whether the original shallow setting (`max_depth=3`) was actually
holding the model back, as originally suspected, or whether it was already
a reasonable choice.

**Result:** depth=3 (original) performed best or tied-best across all three
configurations tested. Deeper trees did not improve precision@50.

**Takeaway:** the original shallow depth setting was already sound.
Increasing complexity here doesn't help — the real lever turned out to be
`scale_pos_weight` (see below), not tree depth.

In [ ]:
for depth, mcw in [(3, 20), (5, 15), (6, 10)]:
    m = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        max_depth=depth,
        min_child_weight=mcw,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        missing=np.nan,
        n_estimators=EVAL_N_ESTIMATORS,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        scale_pos_weight=SCALE_POS_WEIGHT,
    )
    m.fit(X_train[SELECTED_FEATURES], y_train,
          eval_set=[(X_es[SELECTED_FEATURES], y_es)],
          verbose=False)
    _rows, _ = evaluate(m, cell("holdout_time"), SELECTED_FEATURES, "holdout_time", f"depth{depth}_mcw{mcw}")
    _p50 = [r for r in _rows if r["k"] == 50][0]
    print(f"depth={depth}, min_child_weight={mcw}  ->  precision@50 = {_p50['precision_at_k_permonth_mean']:.2%}, PR-AUC = {_p50['pr_auc']:.4f}")

## Leakage-risk investigation: which features are doing the work?

Four features were flagged (by §13's leakage/business-sense check) as
possibly circular — describing the cancellation process itself rather than
predicting it ahead of time. Testing each one individually to see which
ones actually matter, rather than treating all 4 as equally risky.

**Result:** only 2 of the 4 matter. `months_since_last_cancel_request` and
`running_saas_cancel_request_count` each cost 5-8 points of precision when
removed individually. The other 2 (`reason_school_temp_paused_count`,
`reason_school_temp_reducing_enrollment_count`) cost almost nothing.

**Open question for the business:** is "has this school asked to cancel
before, and how recently" legitimate early-warning signal (a real repeat-risk
pattern), or is it too close to the outcome itself to trust? This needs a
human call — see also the follow-up test showing dropping BOTH big features
together gives the same result as dropping just `months_since_last_cancel_request`
alone (they're largely redundant with each other).

In [ ]:
LEAKAGE_RISK_ACCEPTED = [
    "running_saas_cancel_request_count",
    "months_since_last_cancel_request",
    "reason_school_temp_paused_count",
    "reason_school_temp_reducing_enrollment_count",
]

for drop_feature in LEAKAGE_RISK_ACCEPTED:
    feats = [c for c in SELECTED_FEATURES if c != drop_feature]
    m = eval_model()
    m.fit(X_train[feats], y_train, eval_set=[(X_es[feats], y_es)], verbose=False)
    _rows, _ = evaluate(m, cell("holdout_time"), feats, "holdout_time", f"without_{drop_feature}")
    _p50 = [r for r in _rows if r["k"] == 50][0]
    print(f"WITHOUT {drop_feature:<45} -> precision@50 = {_p50['precision_at_k_permonth_mean']:.2%}")

## Tuning scale_pos_weight — the spec's own recommended fix, never tested

The original V3 spec explicitly noted: "the spec says to tune
scale_pos_weight downward on AUCPR — over-weighting positives hurts
precision and inflates the probabilities. Not tuned here." This had never
been tested until now.

**Result:** every lower weight beat the untouched default (29.3, the raw
neg/pos ratio). Precision rose as weight decreased, peaking in the 4-6 range
(see the follow-up narrower sweep below), then declining again below that.

In [ ]:
for weight in [29.3, 20, 15, 10, 5, 2]:
    m = eval_model(scale_pos_weight=weight)
    m.fit(X_train[SELECTED_FEATURES], y_train, eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
    _rows, _ = evaluate(m, cell("holdout_time"), SELECTED_FEATURES, "holdout_time", f"weight{weight}")
    _p50 = [r for r in _rows if r["k"] == 50][0]
    print(f"scale_pos_weight={weight:<6} -> precision@50 = {_p50['precision_at_k_permonth_mean']:.2%}, PR-AUC = {_p50['pr_auc']:.4f}")

### Narrowing in on the peak
The initial sweep suggested the best value sits between 5 and 10. Testing
a tighter range around there to find the actual optimum.

**Result:** weight=4 gives the best precision@50 (23.33%), confirmed as a
genuine peak — not noise — since the values rise smoothly toward it and
decline smoothly past it in both directions.

In [ ]:
for weight in [3, 4, 5, 6, 7, 8]:
    m = eval_model(scale_pos_weight=weight)
    m.fit(X_train[SELECTED_FEATURES], y_train, eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
    _rows, _ = evaluate(m, cell("holdout_time"), SELECTED_FEATURES, "holdout_time", f"weight{weight}")
    _p50 = [r for r in _rows if r["k"] == 50][0]
    print(f"scale_pos_weight={weight:<6} -> precision@50 = {_p50['precision_at_k_permonth_mean']:.2%}, PR-AUC = {_p50['pr_auc']:.4f}")

## Confirming the best configuration: full 52 features + scale_pos_weight=4

Earlier testing showed conflicting results when combining scale_pos_weight=4
with a reduced 50-feature set (dropping the 2 "harmless" leakage features) —
that combination actually performed *worse* (18.61%) than expected. This
cell isolates the variables cleanly: full 52-feature set, weight=4, nothing
else changed.

**Result:** 25.56% precision@40 — confirms this is genuinely the best
configuration found. Removing features that looked "safe to drop" at the
old weight (29.3) is NOT safe once the weight is retuned — feature value is
not independent of other hyperparameters, which was an important lesson
from this investigation.

In [ ]:
m = eval_model(scale_pos_weight=4)
m.fit(X_train[SELECTED_FEATURES], y_train, eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
_rows, _ = evaluate(m, cell("holdout_time"), SELECTED_FEATURES, "holdout_time", "weight4_full")
print([r for r in _rows if r["k"] == 40][0]["precision_at_k_permonth_mean"])

GRID SEARCH - EXPLORING WHICH IS BETTER

In [ ]:
for lr in [0.03, 0.05, 0.1]:
    params = dict(XGB_BASE_PARAMS)
    params["learning_rate"] = lr
    m = XGBClassifier(**{k:v for k,v in params.items()}, n_estimators=EVAL_N_ESTIMATORS,
                       early_stopping_rounds=EARLY_STOPPING_ROUNDS, scale_pos_weight=4)
    m.fit(X_train[SELECTED_FEATURES], y_train, eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
    _rows, _ = evaluate(m, cell("holdout_time"), SELECTED_FEATURES, "holdout_time", f"lr{lr}")
    print(f"learning_rate={lr} -> precision@40 = {[r for r in _rows if r['k']==40][0]['precision_at_k_permonth_mean']:.2%}")

In [ ]:
import itertools

best_precision = 0
best_config = None
results = []

param_grid = {
    "max_depth": [3],
    "learning_rate": [0.03, 0.05, 0.08],
    "min_child_weight": [10, 20],
    "subsample": [0.8],
    "colsample_bytree": [0.8],
}

keys = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f"testing {len(combos)} combinations...\n")

for combo in combos:
    config = dict(zip(keys, combo))
    m = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        missing=np.nan,
        n_estimators=EVAL_N_ESTIMATORS,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        scale_pos_weight=4,
        **config,
    )
    m.fit(X_train[SELECTED_FEATURES], y_train,
          eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
    _rows, _ = evaluate(m, cell("holdout_time"), SELECTED_FEATURES, "holdout_time", "grid")
    prec = [r for r in _rows if r["k"] == 40][0]["precision_at_k_permonth_mean"]
    results.append((config, prec))
    print(f"{config} -> precision@40 = {prec:.2%}")
    if prec > best_precision:
        best_precision = prec
        best_config = config

print(f"\nBEST: {best_config} -> precision@40 = {best_precision:.2%}")

In [ ]:
import itertools

best_precision = 0
best_config = None
results = []

param_grid = {
    "max_depth": [3],
    "learning_rate": [0.04, 0.05, 0.06],
    "min_child_weight": [15, 20, 25, 30],
    "subsample": [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree": [0.7, 0.8, 0.9],
}

keys = list(param_grid.keys())
combos = list(itertools.product(*param_grid.values()))
print(f"testing {len(combos)} combinations...\n")

for combo in combos:
    config = dict(zip(keys, combo))
    m = XGBClassifier(
        objective="binary:logistic",
        eval_metric="aucpr",
        tree_method="hist",
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        missing=np.nan,
        n_estimators=EVAL_N_ESTIMATORS,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        scale_pos_weight=4,
        **config,
    )
    m.fit(X_train[SELECTED_FEATURES], y_train,
          eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
    _rows, _ = evaluate(m, cell("holdout_time"), SELECTED_FEATURES, "holdout_time", "grid2")
    prec = [r for r in _rows if r["k"] == 40][0]["precision_at_k_permonth_mean"]
    results.append((config, prec))
    print(f"{config} -> precision@40 = {prec:.2%}")
    if prec > best_precision:
        best_precision = prec
        best_config = config

print(f"\nBEST: {best_config} -> precision@40 = {best_precision:.2%}")

## Direct comparison against the founding doc's benchmark

The original V3 proposal doc reported Swami's model at k=1,840 schools/month
(matching the volume the current Retention Health Score rule flags):
11.8% precision, 36.2% recall — versus the RHS itself at 6.4%/19.4%.

Testing the tuned model (52 features, scale_pos_weight=4) at this exact
same list size for a direct, like-for-like comparison.

**Result:** 13.06% precision, 41.54% recall — beats both the current tool
(roughly 2x on both metrics) AND Swami's original model (+1.3 points
precision, +5.3 points recall) at this specific list size.

**Note:** `m` here needs to be the confirmed best model (scale_pos_weight=4,
full 52 features) — rebuild explicitly if this cell is run out of order.

In [ ]:
_scored_tuned = score_cell(m, cell("holdout_time"), SELECTED_FEATURES)
_rows = per_month_at_k(_scored_tuned, 1840)
print(f"precision@1840 = {_rows['precision_at_k'].mean():.2%}")
print(f"recall@1840 = {_rows['recall_at_k'].mean():.2%}")

## Billing ablation: does billing data actually matter?

The original V3 plan explicitly called for a billing ablation test
("train with and without billing signals and compare") but it was never
run — flagged in every version of the notebook as deferred.

Of the 6 candidate MMSC billing features, only 2 survived BorutaSHAP's
feature selection into the final 52-feature set:
`months_since_first_processed_dollars_local` and
`processed_dollars_rank_local`. Testing whether removing just these 2
changes performance.

**Result:** precision@40 drops from 25.56% to 21.67% without them — a real,
~4-point cost. So billing data isn't uniformly important or unimportant:
specifically the 2 features describing billing *tenure* (how long they've
been a billing customer) matter; the 4 rejected features describing billing
*activity* (are they billing this month, have they ever billed) added
nothing and were correctly filtered out by feature selection.

In [ ]:
BILLING_RELATED = [c for c in SELECTED_FEATURES 
                   if c in ("months_since_first_processed_dollars_local", "processed_dollars_rank_local",
                            "has_processed_dollars_this_month_local", "has_processed_dollars_last_month_local",
                            "has_ever_billed") or c.startswith("monthly_billing_status_local__")]
NO_BILLING = [c for c in SELECTED_FEATURES if c not in BILLING_RELATED]
print(f"dropping {len(BILLING_RELATED)} billing feature(s)")

m_noBilling = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1, missing=np.nan,
    n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    scale_pos_weight=4,
)
m_noBilling.fit(X_train[NO_BILLING], y_train, eval_set=[(X_es[NO_BILLING], y_es)], verbose=False)
_rows, _ = evaluate(m_noBilling, cell("holdout_time"), NO_BILLING, "holdout_time", "no_billing")
print(f"WITHOUT billing -> precision@40 = {[r for r in _rows if r['k']==40][0]['precision_at_k_permonth_mean']:.2%}")

In [ ]:
X_train_season = X_train[SELECTED_FEATURES].copy()
X_train_season["calendar_month"] = cell(FIT_SPLIT)["dim_month"].dt.month.values

X_es_season = X_es[SELECTED_FEATURES].copy()
X_es_season["calendar_month"] = ES_FRAME["dim_month"].dt.month.values

feats_with_season = SELECTED_FEATURES + ["calendar_month"]

m_season = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1, missing=np.nan,
    n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    scale_pos_weight=4,
)
m_season.fit(X_train_season, y_train, eval_set=[(X_es_season, y_es)], verbose=False)

_ht = cell("holdout_time").copy()
_ht["calendar_month"] = _ht["dim_month"].dt.month
_rows, _ = evaluate(m_season, _ht, feats_with_season, "holdout_time", "with_season")
print(f"WITH calendar_month -> precision@40 = {[r for r in _rows if r['k']==40][0]['precision_at_k_permonth_mean']:.2%}")

In [ ]:
print([c for c in SELECTED_FEATURES if c in BILLING_RELATED])

## Other tuning approaches tried, without improvement

The following were tested as alternative ways to squeeze more performance
out of the model, on top of the already-confirmed best configuration
(52 features, scale_pos_weight=4). None improved on the 25.56% baseline:

- **`max_delta_step`** (0, 1, 3, 5): no clear trend, results scattered
  around the baseline — not a real lever here.
- **`sample_weight`** as an alternative to `scale_pos_weight`: consistently
  worse even at matching weight values (20.83% vs 25.56% at weight=4) —
  `scale_pos_weight` and manual `sample_weight` are not equivalent in how
  XGBoost applies them internally.
- **2-model ensemble** (averaging two differently-seeded models): 23.06%,
  slightly worse than the single best-tuned model.

In [ ]:
m1 = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=42, n_jobs=-1, missing=np.nan,
    n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    scale_pos_weight=4,
)
m1.fit(X_train[SELECTED_FEATURES], y_train, eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)

m2 = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=99, n_jobs=-1, missing=np.nan,
    n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    scale_pos_weight=4,
)
m2.fit(X_train[SELECTED_FEATURES], y_train, eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)

_ht = cell("holdout_time")
_scores1 = m1.predict_proba(_ht[SELECTED_FEATURES])[:, 1]
_scores2 = m2.predict_proba(_ht[SELECTED_FEATURES])[:, 1]
_avg_scores = (_scores1 + _scores2) / 2

_ensemble_frame = _ht[["school_id", "dim_month", Y_COL]].copy()
_ensemble_frame["score"] = _avg_scores
_pm = per_month_at_k(_ensemble_frame, 40)
print(f"ensemble (2 seeds averaged) -> precision@40 = {_pm['precision_at_k'].mean():.2%}")

In [ ]:
print(SELECTED_FEATURES[:3])
m_best = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1, missing=np.nan,
    n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    scale_pos_weight=4,
)
m_best.fit(X_train[SELECTED_FEATURES], y_train,
           eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)

print("m_best ready")

In [ ]:
# Score holdout_time with your best tuned model
_ht = cell("holdout_time")
_scored = score_cell(m_best, _ht, SELECTED_FEATURES)

# Build the top-40-per-month list, the actual real-world rep list
_top40 = (_scored.groupby("dim_month", group_keys=False)
                  .apply(lambda g: g.sort_values("score", ascending=False).head(40)))

# False positives: flagged in the top 40, but did NOT churn
false_positives = _top40[_top40[Y_COL] == 0]

# False negatives: DID churn, but were NOT in that month's top 40
false_negatives = _scored[(_scored[Y_COL] == 1) & (~_scored.index.isin(_top40.index))]

print(f"False positives (flagged, didn't churn): {len(false_positives)}")
print(f"False negatives (churned, model missed): {len(false_negatives)}")

In [ ]:
# False positives - schools flagged but fine. Look at their scores (how confident was the model?)
sample_fp = false_positives.sort_values("score", ascending=False).head(5)
print("Top 5 false positives (model was MOST confident, but wrong):")
print(sample_fp[["school_id", "dim_month", "score"]].to_string(index=False))

# False negatives - churners the model scored LOWEST (most surprising misses)
sample_fn = false_negatives.sort_values("score", ascending=True).head(5)
print("\nBottom 5 false negatives (model was LEAST suspicious, but they churned anyway):")
print(sample_fn[["school_id", "dim_month", "score"]].to_string(index=False))

In [ ]:
import pandas as pd

# readable = pd.read_parquet(OUT)  # OUT is the readable parquet path from dataset_creation.ipynb
# If OUT isn't available in this kernel, load it directly by filename:
readable = pd.read_parquet(RAW_DIR / "churn_v1_is_reportable_request_202401_202605_h3.parquet")

fp_schools = [(266887, "2025-12-01"), (142640, "2025-10-01"), (262938, "2025-10-01"),
              (206410, "2025-07-01"), (201589, "2025-04-01")]
fn_schools = [(109125, "2025-08-01"), (77298, "2025-06-01"), (23244, "2025-12-01"),
              (77298, "2025-08-01"), (72806, "2025-08-01")]

key_cols = ["school_id", "dim_month", "customer_segment", "billing_net_gpv_local",
            "delta_billing_net_gpv_local_vs_t3m", "product_active_students",
            "delta_product_active_students_vs_12m_ago", "enforces_autopay_binary",
            "num_students_with_billing_invoice_created", "label"]

print("FALSE POSITIVES (model confident, but wrong):")
for sid, month in fp_schools:
    row = readable[(readable["school_id"] == sid) & (readable["dim_month"] == month)]
    print(row[key_cols].to_string(index=False))
    print()

print("\nFALSE NEGATIVES (model missed entirely):")
for sid, month in fn_schools:
    row = readable[(readable["school_id"] == sid) & (readable["dim_month"] == month)]
    print(row[key_cols].to_string(index=False))
    print()

# Segment-level investigation: is the model equally good across all customer segments?

Everything so far measured overall precision/recall — but the model might
perform very differently across customer segments without that showing up
in the aggregate numbers. Checking recall specifically by segment
(`customer_segment`) to see if any segment is being systematically
under-served.

In [ ]:
_ht = cell("holdout_time")
_scored_full = score_cell(m_best, _ht, SELECTED_FEATURES)

# Attach customer_segment from the readable data
_seg_map = readable.set_index(["school_id", "dim_month"])["customer_segment"]
_scored_full = _scored_full.set_index(["school_id", "dim_month"])
_scored_full["customer_segment"] = _seg_map
_scored_full = _scored_full.reset_index()

# For each segment, compute recall@40-per-month
for seg in _scored_full["customer_segment"].unique():
    seg_data = _scored_full[_scored_full["customer_segment"] == seg]
    _top40_seg = (_scored_full.groupby("dim_month", group_keys=False)
                              .apply(lambda g: g.sort_values("score", ascending=False).head(40)))
    _top40_seg_ids = set(zip(_top40_seg["school_id"], _top40_seg["dim_month"]))
    
    churners = seg_data[seg_data[Y_COL] == 1]
    caught = churners[churners.apply(lambda r: (r["school_id"], r["dim_month"]) in _top40_seg_ids, axis=1)]
    
    total_churners = len(churners)
    caught_churners = len(caught)
    recall = caught_churners / total_churners if total_churners else float('nan')
    print(f"{seg:<20} churners={total_churners:>5}  caught={caught_churners:>4}  recall={recall:.2%}")

Finding: multisite schools are almost invisible to the model
Recall for SMB Multisite 2-9 is 0.29% — roughly 6-8x worse than every other segment (1.74-2.34%). Out of 684 multisite churners in this evaluation window, the model caught only 2.

This is a real, previously unknown limitation, not visible in any of the aggregate precision/recall numbers reported so far. The rest of this section investigates why, and tests a fix.

In [ ]:
churners_only = _scored_full[_scored_full[Y_COL] == 1]
print(churners_only.groupby("customer_segment")["score"].agg(["mean", "median", "count"]))

In [ ]:
org_data = pd.read_parquet(RAW_DIR / "org_lookup.parquet")
print(f"loaded {len(org_data):,} rows")

In [ ]:
# Drop duplicate/null org rows before joining (keep one row per school_id)
org_data_clean = org_data.drop_duplicates(subset="school_id").dropna(subset=["school_id"])
org_data_clean["school_id"] = org_data_clean["school_id"].astype(int)

# Merge onto train, early-stopping eval set, and holdout_time
def add_org_features(frame):
    merged = frame.merge(org_data_clean[["school_id", "is_org_multisite", "num_schools_in_org"]],
                          on="school_id", how="left")
    merged["is_org_multisite"] = merged["is_org_multisite"].astype(float)  # True/False -> 1.0/0.0/NaN
    merged["num_schools_in_org"] = merged["num_schools_in_org"].astype(float)
    return merged

X_train_org = add_org_features(cell(FIT_SPLIT)[["school_id"] + SELECTED_FEATURES])
X_es_org = add_org_features(ES_FRAME[["school_id"] + SELECTED_FEATURES])
holdout_time_org = add_org_features(cell("holdout_time"))

FEATURES_WITH_ORG = SELECTED_FEATURES + ["is_org_multisite", "num_schools_in_org"]

print(f"join check - nulls in new features on train: "
      f"{X_train_org['is_org_multisite'].isna().sum():,} of {len(X_train_org):,} rows")

In [ ]:
org_monthly = org_monthly.sort_values(["organization_id", "dim_month"])

org_monthly["org_trailing_3m_gpv"] = (
    org_monthly.groupby("organization_id")["org_total_gpv"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=3).mean())
)
org_monthly["org_delta_gpv_vs_t3m"] = (
    org_monthly["org_total_gpv"] / org_monthly["org_trailing_3m_gpv"].replace(0, np.nan) - 1
)

print(org_monthly[["organization_id", "dim_month", "org_total_gpv",
                    "org_trailing_3m_gpv", "org_delta_gpv_vs_t3m"]].head(10))


In [ ]:
# Merge org-level trend features onto the readable dataset by org + month
readable_with_org_trend = readable_with_org.merge(
    org_monthly[["organization_id", "dim_month", "org_total_gpv", "org_delta_gpv_vs_t3m",
                 "org_total_active_students", "org_school_count_this_month"]],
    on=["organization_id", "dim_month"], how="left"
)

# Now rebuild X_train / X_es / holdout_time with these new org-trend features added
ORG_TREND_FEATURES = ["org_delta_gpv_vs_t3m", "org_school_count_this_month"]

def add_org_trend(frame_with_keys):
    merged = frame_with_keys.merge(
        readable_with_org_trend[["school_id", "dim_month"] + ORG_TREND_FEATURES],
        on=["school_id", "dim_month"], how="left"
    )
    return merged

_train_keys = cell(FIT_SPLIT)[["school_id", "dim_month"] + SELECTED_FEATURES]
_es_keys = ES_FRAME[["school_id", "dim_month"] + SELECTED_FEATURES]
_ht_keys = cell("holdout_time")[["school_id", "dim_month"] + SELECTED_FEATURES + [Y_COL]]

X_train_orgtrend = add_org_trend(_train_keys)
X_es_orgtrend = add_org_trend(_es_keys)
holdout_time_orgtrend = add_org_trend(_ht_keys)

FEATURES_WITH_ORGTREND = SELECTED_FEATURES + ORG_TREND_FEATURES

print(f"nulls in org_delta_gpv_vs_t3m on train: {X_train_orgtrend['org_delta_gpv_vs_t3m'].isna().sum():,} of {len(X_train_orgtrend):,}")

In [ ]:
m_orgtrend = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1, missing=np.nan,
    n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    scale_pos_weight=4,
)
m_orgtrend.fit(X_train_orgtrend[FEATURES_WITH_ORGTREND], y_train,
               eval_set=[(X_es_orgtrend[FEATURES_WITH_ORGTREND], y_es)], verbose=False)

_rows, _ = evaluate(m_orgtrend, holdout_time_orgtrend, FEATURES_WITH_ORGTREND, "holdout_time", "orgtrend")
print(f"WITH org-trend features -> precision@40 = {[r for r in _rows if r['k']==40][0]['precision_at_k_permonth_mean']:.2%}")

# Multisite-specific recall check
_scored_ot = score_cell(m_orgtrend, holdout_time_orgtrend, FEATURES_WITH_ORGTREND)
_scored_ot = _scored_ot.merge(_seg_lookup, on=["school_id", "dim_month"], how="left")

_top40_ot = (_scored_ot.groupby("dim_month", group_keys=False)
                        .apply(lambda g: g.sort_values("score", ascending=False).head(40)))
_top40_ot_ids = set(zip(_top40_ot["school_id"], _top40_ot["dim_month"]))

for seg in _scored_ot["customer_segment"].dropna().unique():
    seg_data = _scored_ot[_scored_ot["customer_segment"] == seg]
    churners = seg_data[seg_data[Y_COL] == 1]
    caught = churners[churners.apply(lambda r: (r["school_id"], r["dim_month"]) in _top40_ot_ids, axis=1)]
    total = len(churners)
    recall = len(caught) / total if total else float('nan')
    print(f"{seg:<20} churners={total:>5}  caught={len(caught):>4}  recall={recall:.2%}")

In [ ]:
_scored_all = score_cell(m_best, cell("holdout_time"), SELECTED_FEATURES)
_scored_all = _scored_all.merge(_seg_lookup.rename(columns={"customer_segment": "seg"}), on=["school_id", "dim_month"], how="left")

multisite_scores = _scored_all[_scored_all["seg"] == "SMB Multisite 2-9"]
print("Multisite score distribution, split by outcome:")
print(multisite_scores.groupby(Y_COL)["score"].describe())

In [ ]:
_scored_all = score_cell(m_best, cell("holdout_time"), SELECTED_FEATURES)
_scored_all = _scored_all.merge(_seg_lookup.rename(columns={"customer_segment": "seg"}), on=["school_id", "dim_month"], how="left")

# Compute percentile rank WITHIN segment, WITHIN each month
_scored_all["seg_percentile"] = (
    _scored_all.groupby(["dim_month", "seg"])["score"]
    .rank(pct=True)
)

# Now rebuild the top-40 list using seg_percentile instead of raw score
_top40_pct = (_scored_all.groupby("dim_month", group_keys=False)
                          .apply(lambda g: g.sort_values("seg_percentile", ascending=False).head(40)))
_top40_pct_ids = set(zip(_top40_pct["school_id"], _top40_pct["dim_month"]))

print("Recall by segment, using WITHIN-SEGMENT PERCENTILE ranking instead of raw score:\n")
for seg in _scored_all["seg"].dropna().unique():
    seg_data = _scored_all[_scored_all["seg"] == seg]
    churners = seg_data[seg_data[Y_COL] == 1]
    caught = churners[churners.apply(lambda r: (r["school_id"], r["dim_month"]) in _top40_pct_ids, axis=1)]
    total = len(churners)
    recall = len(caught) / total if total else float('nan')
    print(f"{seg:<20} churners={total:>5}  caught={len(caught):>4}  recall={recall:.2%}")

# Also check overall precision at this new ranking
_all_positives_in_top40 = _top40_pct[_top40_pct[Y_COL] == 1].shape[0]
_precision_pct = _all_positives_in_top40 / len(_top40_pct)
print(f"\nOverall precision@40 with percentile ranking: {_precision_pct:.2%}")

In [ ]:
_top40_raw = (_scored_all.groupby("dim_month", group_keys=False)
                          .apply(lambda g: g.sort_values("score", ascending=False).head(40)))
_top40_raw_ids = set(zip(_top40_raw["school_id"], _top40_raw["dim_month"]))

# Schools that WERE correctly flagged under raw score, but are NOT under percentile ranking
lost_hits = _scored_all[
    (_scored_all[Y_COL] == 1)
    & (_scored_all.apply(lambda r: (r["school_id"], r["dim_month"]) in _top40_raw_ids, axis=1))
    & (~_scored_all.apply(lambda r: (r["school_id"], r["dim_month"]) in _top40_pct_ids, axis=1))
]

print(f"Real churners LOST from the list due to percentile ranking: {len(lost_hits)}")
print(lost_hits.groupby("seg").size())

## Attempting a tunable middle ground: blending raw score with segment percentile

Full within-segment percentile ranking fixed multisite recall (0.29% -> 1.75%)
but cost ~2 points of overall precision. Testing whether a partial blend
between raw score and percentile gives a "dial" — most of the multisite
benefit at a smaller precision cost.

**Result:** the tradeoff does NOT blend smoothly. Partial blends (30-70%)
barely move multisite recall (0.29% -> 0.44-0.73%) while still eroding
precision steadily. This isn't a tunable dial — it behaves like a
threshold effect. Middle-ground blending is not a viable fix on its own.

In [ ]:
# Normalize both raw score and percentile to 0-1, then blend
_scored_all["score_norm"] = _scored_all.groupby("dim_month")["score"].rank(pct=True)

for blend_weight in [0.3, 0.5, 0.7]:
    _scored_all["blended"] = (blend_weight * _scored_all["seg_percentile"]
                               + (1 - blend_weight) * _scored_all["score_norm"])
    
    _top40_blend = (_scored_all.groupby("dim_month", group_keys=False)
                                .apply(lambda g: g.sort_values("blended", ascending=False).head(40)))
    _top40_blend_ids = set(zip(_top40_blend["school_id"], _top40_blend["dim_month"]))
    
    _prec = _top40_blend[_top40_blend[Y_COL] == 1].shape[0] / len(_top40_blend)
    
    multisite_data = _scored_all[_scored_all["seg"] == "SMB Multisite 2-9"]
    multisite_churners = multisite_data[multisite_data[Y_COL] == 1]
    multisite_caught = multisite_churners.apply(lambda r: (r["school_id"], r["dim_month"]) in _top40_blend_ids, axis=1).sum()
    multisite_recall = multisite_caught / len(multisite_churners) if len(multisite_churners) else float('nan')
    
    print(f"blend_weight={blend_weight}: precision@40={_prec:.2%}, multisite_recall={multisite_recall:.2%}")

In [ ]:
print(_scored_all["seg"].value_counts())
print("\nchurn rate by segment:")
print(_scored_all.groupby("seg")[Y_COL].mean())

## The fix: risk-weighted quota allocation

Rather than blending scores, allocate the 40 monthly slots proportionally
to each segment's EXPECTED number of churners (population size x
historical churn rate for that segment), then fill each segment's
allocation with its own top-scoring schools. This respects real
differences in churn rates across segments (small schools churn at 4.06%,
multisite at 1.82% — not equal, but multisite isn't negligible either)
rather than treating all segments as equally deserving of slots.

**Result — this is the best fix found:**

| Segment            | Raw score | Full percentile | Risk-weighted quota |
|--------------------|-----------|------------------|----------------------|
| 10-39 small        | 1.74%     | 1.02%            | 1.55%                |
| 40-99 medium       | 1.90%     | 2.62%            | 2.26%                |
| 100+ large         | 2.34%     | 1.95%            | 1.56%                |
| SMB Multisite 2-9  | 0.29%     | 1.75%            | 1.61%                |
| **Overall precision@40** | 25.56%    | 23.61%           | **25.21%**           |

This recovers nearly all of the original precision (25.21% vs 25.56%,
only -0.35 pts, versus percentile ranking's -1.95 pts) while bringing
multisite recall in line with every other segment (1.61%, versus every
other segment landing 1.5-2.6%).



In [ ]:
def quota_based_list_by_risk(month_data, total_slots=40):
    seg_churn_rates = month_data.groupby("seg")[Y_COL].mean()
    seg_counts = month_data["seg"].value_counts()
    
    # Expected number of churners per segment this month (population x historical churn rate)
    expected_churners = seg_counts * seg_churn_rates
    seg_slots = (expected_churners / expected_churners.sum() * total_slots).round().astype(int)
    
    picks = []
    for seg in seg_counts.index:
        n = seg_slots.get(seg, 0)
        if n > 0:
            seg_top = month_data[month_data["seg"] == seg].nlargest(n, "score")
            picks.append(seg_top)
    return pd.concat(picks) if picks else month_data.head(0)

_top40_risk_quota = (_scored_all.groupby("dim_month", group_keys=False)
                                 .apply(lambda g: quota_based_list_by_risk(g, 40)))
_top40_risk_quota_ids = set(zip(_top40_risk_quota["school_id"], _top40_risk_quota["dim_month"]))

_prec_rq = _top40_risk_quota[_top40_risk_quota[Y_COL] == 1].shape[0] / len(_top40_risk_quota)
print(f"Risk-weighted quota precision@40: {_prec_rq:.2%}")

for seg in _scored_all["seg"].unique():
    seg_churners = _scored_all[(_scored_all["seg"]==seg) & (_scored_all[Y_COL]==1)]
    caught = seg_churners.apply(lambda r: (r["school_id"], r["dim_month"]) in _top40_risk_quota_ids, axis=1).sum()
    recall = caught / len(seg_churners) if len(seg_churners) else float('nan')
    print(f"{seg:<20} recall={recall:.2%}")

1. Cancellation-history features — keep as-is
months_since_last_cancel_request and running_saas_cancel_request_count are accurate and well-calibrated. In real data, schools with a recent cancellation and recovering usage churn less (3.45%) than those still declining (7.51%), and the model correctly reflects this in its predictions (10% vs 21% risk).

2. Exclude very-recent cancellations from the final list
Removing schools with a cancellation request in the last ~2 months improves precision (25.56% → 26.11%). These schools were being flagged almost purely on recency alone, without other supporting signals (18.18% actual churn rate) — excluding them lets stronger, multi-signal flags take their place (36.36% actual churn rate). Implement as a post-processing filter, not a training change.

In [ ]:
# Score with your ORIGINAL best model (m_best), on the FULL holdout_time,
# then simply REMOVE very-recent-cancel schools from the ranking before taking top-40
_scored_orig = score_cell(m_best, _ht_full, SELECTED_FEATURES)
_scored_orig["months_since_last_cancel_request"] = _ht_full["months_since_last_cancel_request"].values

# Exclude very-recent from the LIST, not from training
_scored_filtered_list = _scored_orig[
    (_scored_orig["months_since_last_cancel_request"].isna())
    | (_scored_orig["months_since_last_cancel_request"] >= 2)
]

_pm = per_month_at_k(_scored_filtered_list, 40)
print(f"Original model, excluding very-recent from the LIST only -> precision@40 = {_pm['precision_at_k'].mean():.2%}")

In [ ]:
# Where do recent-cancel + growing-usage schools typically land in the ranking?
_recent_growing_check = _scored_orig.merge(
    _ht_full[["school_id", "dim_month", "delta_product_active_students_vs_12m_ago"]],
    on=["school_id", "dim_month"]
)
_recent_growing_check["recent_cancel"] = _ht_full.set_index(["school_id","dim_month"]).loc[
    list(zip(_recent_growing_check["school_id"], _recent_growing_check["dim_month"])),
    "months_since_last_cancel_request"
].values

_target_group = _recent_growing_check[
    (_recent_growing_check["recent_cancel"] <= 5)
    & (_recent_growing_check["delta_product_active_students_vs_12m_ago"] > 0)
]

# What percentile do these schools typically score at, each month?
_scored_orig["month_rank"] = _scored_orig.groupby("dim_month")["score"].rank(pct=True, ascending=False)
print(f"Average percentile rank of recent-cancel+growing schools: {_scored_orig.set_index(['school_id','dim_month']).loc[list(zip(_target_group['school_id'], _target_group['dim_month'])), 'month_rank'].mean():.2%}")
print("(lower % = closer to the TOP of the list, i.e., more likely to make the cut)")

In [ ]:
# Take the original scores, add a boost specifically to this target group
_scored_boosted = _scored_orig.copy()
_scored_boosted["is_target_group"] = _scored_boosted.apply(
    lambda r: (r["school_id"], r["dim_month"]) in set(zip(_target_group["school_id"], _target_group["dim_month"])),
    axis=1
)

# Try a modest boost - e.g., add a flat amount to push them up the ranking
for boost in [0.05, 0.10, 0.15]:
    _scored_boosted["score_boosted"] = _scored_boosted["score"] + (_scored_boosted["is_target_group"] * boost)
    
    _top40_boosted = (_scored_boosted.groupby("dim_month", group_keys=False)
                                      .apply(lambda g: g.sort_values("score_boosted", ascending=False).head(40)))
    
    _prec = _top40_boosted[_top40_boosted[Y_COL] == 1].shape[0] / len(_top40_boosted)
    _target_caught = _top40_boosted["is_target_group"].sum()
    
    print(f"boost={boost}: precision@40={_prec:.2%}, target-group schools now on list={_target_caught}")

In [ ]:
import pandas as pd

readable = pd.read_parquet(RAW_DIR / "churn_v1_is_reportable_request_202401_202605_h3.parquet")
program_data = pd.read_parquet(RAW_DIR / "program_type_lookup.parquet")
program_data_dedup = program_data.drop_duplicates(subset="school_id", keep="first")

readable_with_program = readable.merge(
    program_data_dedup[["school_id", "program_type"]], on="school_id", how="left"
)
readable_with_program["month_num"] = pd.to_datetime(readable_with_program["dim_month"]).dt.month

print(f"loaded and merged: {len(readable_with_program):,} rows")
print(readable_with_program["program_type"].value_counts(dropna=False))

In [ ]:
# Check for duplicates first
dupe_count = program_data["school_id"].duplicated().sum()
print(f"Duplicate school_id rows: {dupe_count:,} of {len(program_data):,}")

# Keep just one row per school_id (first occurrence)
program_data_dedup = program_data.drop_duplicates(subset="school_id", keep="first")
print(f"After dedup: {len(program_data_dedup):,} rows")

program_map = program_data_dedup.set_index("school_id")["program_type"]

In [ ]:
customers_catalog = pd.read_parquet(RAW_DIR / "customers_catalog.parquet")
print(f"loaded {len(customers_catalog)} columns")

reopen_candidates = [c for c in customers_catalog["column_name"] if any(
    k in c.lower() for k in ["reopen", "calendar", "season", "pause", "closed", "closure", "break"]
)]
print("Possible reopen/seasonal date columns:", reopen_candidates)

In [ ]:
_top40_all_months = (_scored_camps_check.groupby("dim_month", group_keys=False)
                                          .apply(lambda g: g.sort_values("score", ascending=False).head(40)))

camps_in_top40_ever = _top40_all_months[_top40_all_months["is_camp"] == 1]
print(f"Total camp appearances in top-40 across all months: {len(camps_in_top40_ever)}")
print(camps_in_top40_ever[["school_id", "dim_month", "score", Y_COL]] if len(camps_in_top40_ever) else "None")

In [ ]:
readable["month_num"] = pd.to_datetime(readable["dim_month"]).dt.month

print("ALL SCHOOLS - avg active students by month:")
print(readable.groupby("month_num")["product_active_students"].mean())

print("\nALL SCHOOLS - avg GPV by month:")
print(readable.groupby("month_num")["billing_net_gpv_local"].mean())

print("\nALL SCHOOLS - churn rate (label) by month:")
print(readable.groupby("month_num")[Y_COL].mean())

In [ ]:
_all_scored = score_cell(m_best, cell("holdout_time"), SELECTED_FEATURES)
_all_scored["month_num"] = pd.to_datetime(_all_scored["dim_month"]).dt.month

print("Average predicted score by month, ALL schools:")
print(_all_scored.groupby("month_num")["score"].mean())

print("\nActual churn rate by month, ALL schools (holdout_time):")
print(_all_scored.groupby("month_num")[Y_COL].mean())

In [ ]:
# Check the seasonal pattern EXCLUDING camps entirely
non_camp_ids = readable_with_program[readable_with_program["program_type"] != "camp"][["school_id", "dim_month"]]
_all_scored_noncamp = _all_scored.merge(non_camp_ids, on=["school_id", "dim_month"])

print("Non-camp schools ONLY - predicted score by month:")
print(_all_scored_noncamp.groupby("month_num")["score"].mean())
print("\nNon-camp schools ONLY - actual churn by month:")
print(_all_scored_noncamp.groupby("month_num")[Y_COL].mean())

In [ ]:
_calib = _all_scored.groupby("month_num").agg(
    predicted=("score", "mean"),
    actual=(Y_COL, "mean")
)
_calib["ratio"] = _calib["predicted"] / _calib["actual"]
print(_calib)

In [ ]:
readable_with_segment = readable_with_program.merge(
    _all_scored[["school_id", "dim_month", "score"]], on=["school_id", "dim_month"], how="inner"
)

for seg in readable_with_segment["customer_segment"].unique():
    seg_data = readable_with_segment[readable_with_segment["customer_segment"] == seg]
    print(f"\n{seg}:")
    print(seg_data.groupby("month_num")["score"].mean())

In [ ]:
# Check the same ratio calculation but on the FULL readable dataset (not just holdout_time)
# to see if Jan-March would complete a cycle
_full_scored = score_cell(m_best, cell(FIT_SPLIT), SELECTED_FEATURES)  # training data itself
_full_scored["month_num"] = pd.to_datetime(_full_scored["dim_month"]).dt.month

_calib_full = _full_scored.groupby("month_num").agg(
    predicted=("score", "mean"),
    actual=(Y_COL, "mean")
)
_calib_full["ratio"] = _calib_full["predicted"] / _calib_full["actual"]
print(_calib_full)

In [ ]:
# Check the score WITHOUT any _vs_12m_ago features, same months
features_no_12m = [c for c in SELECTED_FEATURES if "vs_12m_ago" not in c]
print(f"Removed {len(SELECTED_FEATURES) - len(features_no_12m)} '_vs_12m_ago' features")

m_no12m = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1, missing=np.nan,
    n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    scale_pos_weight=4,
)
m_no12m.fit(X_train[features_no_12m], y_train,
            eval_set=[(X_es[features_no_12m], y_es)], verbose=False)

_scored_no12m = score_cell(m_no12m, cell("holdout_time"), features_no_12m)
_scored_no12m["month_num"] = pd.to_datetime(_scored_no12m["dim_month"]).dt.month

_calib_no12m = _scored_no12m.groupby("month_num").agg(
    predicted=("score", "mean"),
    actual=(Y_COL, "mean")
)
_calib_no12m["ratio"] = _calib_no12m["predicted"] / _calib_no12m["actual"]
print(_calib_no12m)

In [ ]:
# Simple month-based recalibration factor, using the ratio table itself
_calib_factors = _calib["ratio"] / _calib["ratio"].mean()  # normalize around 1.0

_scored_all_cal = _all_scored.copy()
_scored_all_cal["calib_factor"] = _scored_all_cal["month_num"].map(_calib_factors)
_scored_all_cal["score_calibrated"] = _scored_all_cal["score"] / _scored_all_cal["calib_factor"]

_pm_before = per_month_at_k(_scored_all_cal.rename(columns={"score": "score_orig"}).assign(score=_scored_all_cal["score"]), 40)
_pm_after = per_month_at_k(_scored_all_cal.assign(score=_scored_all_cal["score_calibrated"]), 40)

print(f"Precision@40 WITHOUT calibration: {_pm_before['precision_at_k'].mean():.2%}")
print(f"Precision@40 WITH monthly calibration: {_pm_after['precision_at_k'].mean():.2%}")

In [ ]:
# Check precision@40 PER MONTH, not averaged - is precision itself seasonal?
_pm_by_month = per_month_at_k(_all_scored, 40)
print(_pm_by_month[["dim_month", "precision_at_k", "recall_at_k", "base_rate"]])

In [ ]:
for k in [40, 100, 500, 1000, 2000,10000,20000]:
    _pm_by_month = per_month_at_k(_all_scored, k)
    print(f"\n=== k={k} ===")
    print(_pm_by_month[["dim_month", "precision_at_k", "recall_at_k", "base_rate"]].to_string(index=False))

In [ ]:
for k in [40, 100, 1000]:
    _pm = per_month_at_k(_all_scored, k)
    _pm["lift"] = _pm["precision_at_k"] / _pm["base_rate"]
    print(f"\nk={k} - LIFT (precision / base_rate) by month:")
    print(_pm[["dim_month", "precision_at_k", "base_rate", "lift"]].to_string(index=False))

In [ ]:
june_data = _all_scored[_all_scored["month_num"] == 6].copy()
june_data = june_data.merge(readable_with_program[["school_id", "dim_month", "program_type", "customer_segment"]], 
                              on=["school_id", "dim_month"], how="left")

print(f"June rows: {len(june_data)}")
print(f"June program_type breakdown:\n{june_data['program_type'].value_counts(dropna=False)}")
print(f"\nJune segment breakdown:\n{june_data['customer_segment'].value_counts(dropna=False)}")

In [ ]:
june_top40 = june_data.sort_values("score", ascending=False).head(40)
print(f"June top-40: {int(june_top40[Y_COL].sum())} actual hits out of 40")
print(f"\nProgram types in June's top-40:\n{june_top40['program_type'].value_counts(dropna=False)}")
print(f"\nSegments in June's top-40:\n{june_top40['customer_segment'].value_counts(dropna=False)}")

In [ ]:
april_data = _all_scored[_all_scored["month_num"] == 4].copy()
april_top40 = april_data.sort_values("score", ascending=False).head(40)

print("=== JUNE top-40 average feature values ===")
june_top40_full = cell("holdout_time").merge(june_top40[["school_id", "dim_month"]], on=["school_id", "dim_month"])
print(june_top40_full[["delta_product_active_students_vs_t3m", "delta_product_active_students_vs_12m_ago", 
                        "billing_net_gpv_local", "months_since_last_cancel_request"]].mean())

print("\n=== APRIL top-40 average feature values ===")
april_top40_full = cell("holdout_time").merge(april_top40[["school_id", "dim_month"]], on=["school_id", "dim_month"])
print(april_top40_full[["delta_product_active_students_vs_t3m", "delta_product_active_students_vs_12m_ago",
                         "billing_net_gpv_local", "months_since_last_cancel_request"]].mean())

In [ ]:
print("June top-40 score range:")
print(june_top40["score"].describe())

print("\nApril top-40 score range:")
print(april_top40["score"].describe())

In [ ]:
june_top40_full["actual_label"] = june_top40[Y_COL].values

false_positives_june = june_top40_full[june_top40_full["actual_label"] == 0]
true_positives_june = june_top40_full[june_top40_full["actual_label"] == 1]

print(f"June false positives (n={len(false_positives_june)}):")
print(false_positives_june[["delta_product_active_students_vs_t3m", "delta_product_active_students_vs_12m_ago"]].mean())

print(f"\nJune true positives (n={len(true_positives_june)}):")
print(true_positives_june[["delta_product_active_students_vs_t3m", "delta_product_active_students_vs_12m_ago"]].mean())

In [ ]:
# Check if the drift correlates with MONTHS-SINCE-TRAINING, not calendar month
_all_scored["months_since_training_end"] = (
    pd.to_datetime(_all_scored["dim_month"]).dt.year * 12 + pd.to_datetime(_all_scored["dim_month"]).dt.month
) - (2024 * 12 + 12)  # training ends Dec 2024

_calib_drift = _all_scored.groupby("months_since_training_end").agg(
    predicted=("score", "mean"),
    actual=(Y_COL, "mean")
)
_calib_drift["ratio"] = _calib_drift["predicted"] / _calib_drift["actual"]
print(_calib_drift)

In [ ]:
holdout_school_june = cell("holdout_school")
holdout_school_june = holdout_school_june[pd.to_datetime(holdout_school_june["dim_month"]).dt.month == 6]

_scored_hs_june = score_cell(m_best, holdout_school_june, SELECTED_FEATURES)
_pm_hs_june = per_month_at_k(_scored_hs_june, 40)
print(_pm_hs_june[["precision_at_k", "recall_at_k", "base_rate"]])

In [ ]:
# Check performance on TRAIN data itself vs holdout_time
_train_scored = score_cell(m_best, cell(FIT_SPLIT), SELECTED_FEATURES)
_pm_train = per_month_at_k(_train_scored, 40)
print(f"Average precision@40 on TRAINING data: {_pm_train['precision_at_k'].mean():.2%}")

_pm_holdout = per_month_at_k(_all_scored, 40)
print(f"Average precision@40 on holdout_time: {_pm_holdout['precision_at_k'].mean():.2%}")

In [ ]:
for reg_lambda_val in [1.0, 2.0, 5.0, 10.0]:
    m_reg = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=reg_lambda_val,
        random_state=RANDOM_STATE, n_jobs=-1, missing=np.nan,
        n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        scale_pos_weight=4,
    )
    m_reg.fit(X_train[SELECTED_FEATURES], y_train,
              eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
    
    _train_s = score_cell(m_reg, cell(FIT_SPLIT), SELECTED_FEATURES)
    _ho_s = score_cell(m_reg, cell("holdout_time"), SELECTED_FEATURES)
    
    train_prec = per_month_at_k(_train_s, 40)["precision_at_k"].mean()
    holdout_prec = per_month_at_k(_ho_s, 40)["precision_at_k"].mean()
    
    print(f"reg_lambda={reg_lambda_val}: train={train_prec:.2%}, holdout={holdout_prec:.2%}, gap={train_prec-holdout_prec:.2%}")

In [ ]:
_train_full_dates = cell(FIT_SPLIT)["dim_month"]
_recency_weight = (pd.to_datetime(_train_full_dates).dt.month) / 12.0  # Jan=0.08, Dec=1.0

m_recency = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1, missing=np.nan,
    n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
)
_sample_weights = np.where(y_train == 1, 4, 1) * _recency_weight.values  # combine with existing scale_pos_weight logic

m_recency.fit(X_train[SELECTED_FEATURES], y_train, sample_weight=_sample_weights,
              eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)

_rows, _ = evaluate(m_recency, cell("holdout_time"), SELECTED_FEATURES, "holdout_time", "recency_weighted")
print(f"Recency-weighted training -> precision@40 = {[r for r in _rows if r['k']==40][0]['precision_at_k_permonth_mean']:.2%}")

In [ ]:
# Quick multi-seed check on the recency-weighted approach
for seed in [1, 42, 99]:
    m_recency_seed = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS,
    )
    m_recency_seed.fit(X_train[SELECTED_FEATURES], y_train, sample_weight=_sample_weights,
                        eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
    _rows, _ = evaluate(m_recency_seed, cell("holdout_time"), SELECTED_FEATURES, "holdout_time", f"recency_seed{seed}")
    prec = [r for r in _rows if r["k"]==40][0]["precision_at_k_permonth_mean"]
    print(f"seed={seed}: recency-weighted precision@40 = {prec:.2%}")

In [ ]:
compare_features = ["billing_net_gpv_local", "product_active_students", 
                     "delta_product_active_students_vs_t3m", "months_since_last_cancel_request",
                     "num_students_with_billing_invoice_created"]

train_data = cell(FIT_SPLIT)
holdout_data = cell("holdout_time")

for feat in compare_features:
    print(f"\n{feat}:")
    print(f"  Train (2024) mean: {train_data[feat].mean():.3f}, median: {train_data[feat].median():.3f}")
    print(f"  Holdout (2025) mean: {holdout_data[feat].mean():.3f}, median: {holdout_data[feat].median():.3f}")

In [ ]:
# For a key feature, check churn rate at similar feature levels, split by year
train_data_labeled = train_data.copy()
train_data_labeled["gpv_bucket"] = pd.qcut(train_data_labeled["billing_net_gpv_local"].fillna(0), 5, duplicates="drop")

holdout_data_labeled = holdout_data.copy()
holdout_data_labeled["gpv_bucket"] = pd.qcut(holdout_data_labeled["billing_net_gpv_local"].fillna(0), 5, duplicates="drop")

print("2024 (train) - churn rate by GPV bucket:")
print(train_data_labeled.groupby("gpv_bucket")[Y_COL].mean())

print("\n2025 (holdout) - churn rate by GPV bucket:")
print(holdout_data_labeled.groupby("gpv_bucket")[Y_COL].mean())

In [ ]:
_hs_scored = score_cell(m_best, cell("holdout_school"), SELECTED_FEATURES)
_ht_scored_full = score_cell(m_best, cell("holdout_time"), SELECTED_FEATURES)

print(f"holdout_school (new schools, SAME year) - overall precision@40: {per_month_at_k(_hs_scored, 40)['precision_at_k'].mean():.2%}")
print(f"holdout_time (same-ish schools, NEW year) - overall precision@40: {per_month_at_k(_ht_scored_full, 40)['precision_at_k'].mean():.2%}")

In [ ]:
# Re-run the RHS comparison with your current best model (scale_pos_weight=4, 52 features)
_scored_current = score_cell(m_best, cell("holdout_time"), SELECTED_FEATURES)
_scored_current["product_score"] = cell("holdout_time")["product_score"].values

# Quick RHS comparison at k=40
rhs_scores = -cell("holdout_time")["product_score"].values  # lower score = higher risk, so negate
_rhs_board = cell("holdout_time")[["school_id", "dim_month", Y_COL]].copy()
_rhs_board["score"] = rhs_scores

_pm_model = per_month_at_k(_scored_current, 40)
_pm_rhs = per_month_at_k(_rhs_board, 40)

print(f"Current model precision@40: {_pm_model['precision_at_k'].mean():.2%}")
print(f"Retention Health Score precision@40: {_pm_rhs['precision_at_k'].mean():.2%}")
print(f"\nModel beats RHS: {_pm_model['precision_at_k'].mean() > _pm_rhs['precision_at_k'].mean()}")

In [ ]:
# Compare: does this quick check match your earlier, more careful §14 approach?
print("Quick check RHS precision@40:", _pm_rhs['precision_at_k'].mean())

# Your original approach used the REBUILT trend (delta_product_score_vs_t3m), 
# not just the raw product_score - let's verify this wasn't skipped
print("\nDoes RHS_TREND exist in this kernel?", 'RHS_TREND' in dir())

In [ ]:
baseline_check = pd.read_csv(RAW_DIR / f"{STEM}_baseline_comparison.csv")
print(baseline_check[(baseline_check["k_requested"] == 40) & (baseline_check["cell"] == "holdout_time")])

In [ ]:
_2026_data = enc.loc[enc["split"] == "retrain_only"]
print(f"2026 rows: {len(_2026_data):,}")
print(f"2026 months: {_2026_data['dim_month'].min()} to {_2026_data['dim_month'].max()}")

_scored_2026 = score_cell(m_best, _2026_data, SELECTED_FEATURES)
_pm_2026 = per_month_at_k(_scored_2026, 40)
print(_pm_2026[["dim_month", "precision_at_k", "recall_at_k", "base_rate"]])
print(f"\nOverall precision@40 on 2026: {_pm_2026['precision_at_k'].mean():.2%}")

In [ ]:
import pandas as pd
import numpy as np
from xgboost import XGBClassifier

# --- Define the new split: train on Jan 2024 - Aug 2025, test on Sep 2025 - May 2026 ---
enc["dim_month"] = pd.to_datetime(enc["dim_month"])

TRAIN_CUTOFF = pd.Timestamp("2025-09-01")  # train: everything before this
TEST_START = pd.Timestamp("2025-09-01")     # test: this and after

new_train = enc[enc["dim_month"] < TRAIN_CUTOFF]
new_test = enc[enc["dim_month"] >= TEST_START]

print(f"New train: {len(new_train):,} rows, {new_train['dim_month'].min()} to {new_train['dim_month'].max()}")
print(f"New test: {len(new_test):,} rows, {new_test['dim_month'].min()} to {new_test['dim_month'].max()}")
print(f"New train positives: {int(new_train[Y_COL].sum()):,} ({new_train[Y_COL].mean():.2%})")
print(f"New test positives: {int(new_test[Y_COL].sum()):,} ({new_test[Y_COL].mean():.2%})")

# --- Carve an early-stopping eval set from the new train (school-disjoint, same idea as before) ---
_train_schools = np.array(sorted(new_train["school_id"].unique()))
_perm = np.random.default_rng(7).permutation(len(_train_schools))
_n_es = int(round(0.10 * len(_train_schools)))  # small 10% slice for early stopping
_es_schools_new = set(_train_schools[_perm[:_n_es]])
_fit_schools_new = set(_train_schools[_perm[_n_es:]])

fit_frame = new_train[new_train["school_id"].isin(_fit_schools_new)]
es_frame_new = new_train[new_train["school_id"].isin(_es_schools_new)]

X_train_new = fit_frame[SELECTED_FEATURES]
y_train_new = fit_frame[Y_COL].astype("int8")
X_es_new = es_frame_new[SELECTED_FEATURES]
y_es_new = es_frame_new[Y_COL].astype("int8")

print(f"\nFit rows: {len(X_train_new):,}, ES rows: {len(X_es_new):,}")

# --- New scale_pos_weight for this new training population ---
_neg_new = int((y_train_new == 0).sum())
_pos_new = int((y_train_new == 1).sum())
new_scale_pos_weight = _neg_new / max(_pos_new, 1)
print(f"Raw scale_pos_weight for new train: {new_scale_pos_weight:.1f} (we'll still use tuned value 4)")

# --- Train the new model, same architecture/hyperparameters as before, on the NEW window ---
m_2024_2025 = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=42, n_jobs=-1, missing=np.nan,
    n_estimators=2000, early_stopping_rounds=50,
    scale_pos_weight=4,
)
m_2024_2025.fit(X_train_new, y_train_new, eval_set=[(X_es_new, y_es_new)], verbose=False)

# --- Evaluate on the new test set (Sep 2025 - May 2026) ---
X_test_new = new_test[SELECTED_FEATURES]
scored_new = new_test[["school_id", "dim_month", Y_COL]].copy()
scored_new["score"] = m_2024_2025.predict_proba(X_test_new)[:, 1]

def quick_precision_at_k(frame, k, label_col):
    rows = []
    for _m, _g in frame.groupby("dim_month"):
        top = _g.sort_values("score", ascending=False).head(k)
        rows.append(top[label_col].mean())
    return np.mean(rows)

prec_new = quick_precision_at_k(scored_new, 40, Y_COL)
print(f"\n=== NEW MODEL (trained on 2024+2025, tested on Sep 2025-May 2026) ===")
print(f"Precision@40: {prec_new:.2%}")

# --- Compare against your EXISTING model (trained on 2024 only) on the SAME new test window ---
scored_old_on_new = new_test[["school_id", "dim_month", Y_COL]].copy()
scored_old_on_new["score"] = m_best.predict_proba(new_test[SELECTED_FEATURES])[:, 1]
prec_old = quick_precision_at_k(scored_old_on_new, 40, Y_COL)
print(f"\n=== OLD MODEL (trained on 2024 only), same test window ===")
print(f"Precision@40: {prec_old:.2%}")

print(f"\n=== COMPARISON ===")
print(f"Old model (2024-trained): {prec_old:.2%}")
print(f"New model (2024+2025-trained): {prec_new:.2%}")
print(f"Difference: {prec_new - prec_old:+.2%}")

In [ ]:
scored_new_on_original_holdout = cell("holdout_time")[["school_id", "dim_month", Y_COL]].copy()
scored_new_on_original_holdout["score"] = m_2024_2025.predict_proba(cell("holdout_time")[SELECTED_FEATURES])[:, 1]
prec_new_on_original = quick_precision_at_k(scored_new_on_original_holdout, 40, Y_COL)

print(f"NEW model (2024+2025-trained) on ORIGINAL holdout_time (Apr-Dec 2025): {prec_new_on_original:.2%}")
print(f"OLD model (2024-trained) on ORIGINAL holdout_time (Apr-Dec 2025): 25.56% (already confirmed)")

In [ ]:
for seed in [7, 99]:
    m_check = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    m_check.fit(X_train_new, y_train_new, eval_set=[(X_es_new, y_es_new)], verbose=False)
    scored_check = new_test[["school_id", "dim_month", Y_COL]].copy()
    scored_check["score"] = m_check.predict_proba(new_test[SELECTED_FEATURES])[:, 1]
    prec_check = quick_precision_at_k(scored_check, 40, Y_COL)
    print(f"seed={seed}: new model precision@40 = {prec_check:.2%}")

In [ ]:
for seed in [1, 7, 99]:
    m_old_check = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS, scale_pos_weight=4,
    )
    m_old_check.fit(X_train[SELECTED_FEATURES], y_train, eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
    scored_old_check = new_test[["school_id", "dim_month", Y_COL]].copy()
    scored_old_check["score"] = m_old_check.predict_proba(new_test[SELECTED_FEATURES])[:, 1]
    prec_old_check = quick_precision_at_k(scored_old_check, 40, Y_COL)
    print(f"OLD model, seed={seed}: precision@40 = {prec_old_check:.2%}")

In [ ]:
from BorutaShap import BorutaShap
import time

BORUTA_N_TRIALS_NEW = 250  # match your best previous run for a fair comparison
BORUTA_SEED_NEW = 42

def boruta_model_new(scale_pos_weight):
    return XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, min_child_weight=20, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=42, n_jobs=-1, missing=np.nan,
        n_estimators=200, scale_pos_weight=scale_pos_weight,
    )

# Use the FULL new training set (all 20 months, not just the fit-only subset)
X_boruta_new = new_train[MODEL_COLS]  # all 177 candidate columns, not just SELECTED_FEATURES
y_boruta_new = new_train[Y_COL].astype("int8")

_neg_b = int((y_boruta_new == 0).sum())
_pos_b = int((y_boruta_new == 1).sum())
scale_pos_weight_boruta_new = _neg_b / max(_pos_b, 1)

print(f"Running BorutaSHAP on new combined training set: {len(X_boruta_new):,} rows, "
      f"{int(y_boruta_new.sum()):,} positives, scale_pos_weight={scale_pos_weight_boruta_new:.1f}")

selector_new = BorutaShap(
    model=boruta_model_new(scale_pos_weight_boruta_new),
    importance_measure="shap",
    classification=True,
    percentile=100,
    pvalue=0.05,
)

_t0 = time.perf_counter()
selector_new.fit(
    X=X_boruta_new, y=y_boruta_new,
    n_trials=BORUTA_N_TRIALS_NEW, random_state=BORUTA_SEED_NEW,
    sample=False, train_or_test="train", normalize=True, verbose=False,
)
_elapsed = time.perf_counter() - _t0

ACCEPTED_NEW = sorted(selector_new.accepted)
TENTATIVE_NEW = sorted(map(str, selector_new.tentative))
REJECTED_NEW = sorted(selector_new.rejected)

print(f"\nFinished in {_elapsed:.1f}s ({_elapsed/60:.1f} min)")
print(f"Accepted: {len(ACCEPTED_NEW)}, Tentative: {len(TENTATIVE_NEW)}, Rejected: {len(REJECTED_NEW)}")

In [ ]:
print("New accepted features:", ACCEPTED_NEW[:10], "... (84 total)")

# Retrain using this new feature set, same architecture, tuned scale_pos_weight=4
m_2024_2025_newfeats = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=42, n_jobs=-1, missing=np.nan,
    n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
)

X_train_newfeats = fit_frame[ACCEPTED_NEW]
X_es_newfeats = es_frame_new[ACCEPTED_NEW]

m_2024_2025_newfeats.fit(X_train_newfeats, y_train_new, 
                          eval_set=[(X_es_newfeats, y_es_new)], verbose=False)

scored_newfeats = new_test[["school_id", "dim_month", Y_COL]].copy()
scored_newfeats["score"] = m_2024_2025_newfeats.predict_proba(new_test[ACCEPTED_NEW])[:, 1]
prec_newfeats = quick_precision_at_k(scored_newfeats, 40, Y_COL)

print(f"\nNew model with NEW feature set (84 features): precision@40 = {prec_newfeats:.2%}")
print(f"Compare: new model with OLD 52-feature set: 26.67% (seed 42)")

In [ ]:
for seed in [7, 99]:
    m_check_newfeats = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    m_check_newfeats.fit(X_train_newfeats, y_train_new, 
                          eval_set=[(X_es_newfeats, y_es_new)], verbose=False)
    scored_check_nf = new_test[["school_id", "dim_month", Y_COL]].copy()
    scored_check_nf["score"] = m_check_newfeats.predict_proba(new_test[ACCEPTED_NEW])[:, 1]
    prec_check_nf = quick_precision_at_k(scored_check_nf, 40, Y_COL)
    print(f"84-feature set, seed={seed}: precision@40 = {prec_check_nf:.2%}")

In [ ]:
for weight in [4, 5, 6, 8, 10]:
    m_weight_check = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=42, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=weight,
    )
    m_weight_check.fit(X_train_newfeats, y_train_new, 
                        eval_set=[(X_es_newfeats, y_es_new)], verbose=False)
    scored_wc = new_test[["school_id", "dim_month", Y_COL]].copy()
    scored_wc["score"] = m_weight_check.predict_proba(new_test[ACCEPTED_NEW])[:, 1]
    prec_wc = quick_precision_at_k(scored_wc, 40, Y_COL)
    print(f"weight={weight}, 84 features: precision@40 = {prec_wc:.2%}")

In [ ]:
for weight in [4, 5, 6, 8, 10]:
    m_weight_check52 = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=42, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=weight,
    )
    m_weight_check52.fit(X_train_new, y_train_new,
                          eval_set=[(X_es_new, y_es_new)], verbose=False)
    scored_wc52 = new_test[["school_id", "dim_month", Y_COL]].copy()
    scored_wc52["score"] = m_weight_check52.predict_proba(new_test[SELECTED_FEATURES])[:, 1]
    prec_wc52 = quick_precision_at_k(scored_wc52, 40, Y_COL)
    print(f"weight={weight}, 52 features: precision@40 = {prec_wc52:.2%}")

In [ ]:
for seed in [7, 99]:
    m_best_combo = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=6,
    )
    m_best_combo.fit(X_train_newfeats, y_train_new, 
                      eval_set=[(X_es_newfeats, y_es_new)], verbose=False)
    scored_bc = new_test[["school_id", "dim_month", Y_COL]].copy()
    scored_bc["score"] = m_best_combo.predict_proba(new_test[ACCEPTED_NEW])[:, 1]
    prec_bc = quick_precision_at_k(scored_bc, 40, Y_COL)
    print(f"84 features, weight=6, seed={seed}: precision@40 = {prec_bc:.2%}")

In [ ]:
for seed in [1, 7, 42, 99]:
    m_old_multiseed = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS, scale_pos_weight=4,
    )
    m_old_multiseed.fit(X_train[SELECTED_FEATURES], y_train, eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
    _rows, _ = evaluate(m_old_multiseed, cell("holdout_time"), SELECTED_FEATURES, "holdout_time", f"old_seed{seed}")
    prec = [r for r in _rows if r["k"]==40][0]["precision_at_k_permonth_mean"]
    print(f"OLD model, holdout_time, seed={seed}: precision@40 = {prec:.2%}")

In [ ]:
program_map_full = program_data_dedup.set_index("school_id")["program_type"]

_train_prog2 = cell(FIT_SPLIT).copy()
_train_prog2["program_type"] = _train_prog2["school_id"].map(program_map_full)
_es_prog2 = ES_FRAME.copy()
_es_prog2["program_type"] = _es_prog2["school_id"].map(program_map_full)
_ht_prog2 = cell("holdout_time").copy()
_ht_prog2["program_type"] = _ht_prog2["school_id"].map(program_map_full)

# One-hot encode the FULL program_type field, not just is_camp
for df in [_train_prog2, _es_prog2, _ht_prog2]:
    for cat in ["preschool_or_child_care", "after_school", "camp", "other"]:
        df[f"program_{cat}"] = (df["program_type"] == cat).astype(int)

program_dummy_cols = [f"program_{cat}" for cat in ["preschool_or_child_care", "after_school", "camp", "other"]]
FEATURES_FULL_PROGRAM = SELECTED_FEATURES + program_dummy_cols

m_full_program = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=RANDOM_STATE, n_jobs=-1, missing=np.nan,
    n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS, scale_pos_weight=4,
)
m_full_program.fit(_train_prog2[FEATURES_FULL_PROGRAM], y_train,
                    eval_set=[(_es_prog2[FEATURES_FULL_PROGRAM], y_es)], verbose=False)

_rows, _ = evaluate(m_full_program, _ht_prog2, FEATURES_FULL_PROGRAM, "holdout_time", "full_program")
print(f"WITH full program_type -> precision@40 = {[r for r in _rows if r['k']==40][0]['precision_at_k_permonth_mean']:.2%}")

In [ ]:
for seed in [1, 7, 42, 99]:
    m_full_program_check = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS, scale_pos_weight=4,
    )
    m_full_program_check.fit(_train_prog2[FEATURES_FULL_PROGRAM], y_train,
                              eval_set=[(_es_prog2[FEATURES_FULL_PROGRAM], y_es)], verbose=False)
    _rows, _ = evaluate(m_full_program_check, _ht_prog2, FEATURES_FULL_PROGRAM, "holdout_time", f"full_program_seed{seed}")
    prec = [r for r in _rows if r["k"]==40][0]["precision_at_k_permonth_mean"]
    print(f"seed={seed}: full program_type precision@40 = {prec:.2%}")

In [ ]:
BILLING_RELATED = [c for c in SELECTED_FEATURES 
                   if c in ("months_since_first_processed_dollars_local", "processed_dollars_rank_local")]
NO_BILLING = [c for c in SELECTED_FEATURES if c not in BILLING_RELATED]

for seed in [1, 7, 42, 99]:
    m_nobilling_check = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=EVAL_N_ESTIMATORS, early_stopping_rounds=EARLY_STOPPING_ROUNDS, scale_pos_weight=4,
    )
    m_nobilling_check.fit(X_train[NO_BILLING], y_train, eval_set=[(X_es[NO_BILLING], y_es)], verbose=False)
    _rows, _ = evaluate(m_nobilling_check, cell("holdout_time"), NO_BILLING, "holdout_time", f"nobilling_seed{seed}")
    prec = [r for r in _rows if r["k"]==40][0]["precision_at_k_permonth_mean"]
    print(f"seed={seed}: WITHOUT billing features -> precision@40 = {prec:.2%}")

In [ ]:
# ============================================================================
# SHAP ANALYSIS - FINAL 84-FEATURE MODEL
# Using: m_2024_2025_newfeats, ACCEPTED_NEW, new_test (Sep2025-May2026)
# ============================================================================

import shap
import pandas as pd
import numpy as np

print("="*80)
print("SHAP ANALYSIS - 84-FEATURE MODEL")
print("="*80)

# Prepare data
X_test = new_test[ACCEPTED_NEW]
print(f"\nTest data: {X_test.shape[0]:,} rows × {X_test.shape[1]} features")
print(f"Time range: {new_test['dim_month'].min()} to {new_test['dim_month'].max()}")

# Compute SHAP
print("\nComputing SHAP values (TreeExplainer)...")
explainer = shap.TreeExplainer(m_2024_2025_newfeats)
shap_values = explainer.shap_values(X_test)

# Handle binary classification
if isinstance(shap_values, list):
    shap_values = shap_values[1]  # Positive class

print(f"SHAP values shape: {shap_values.shape}")

# Mean absolute SHAP per feature
mean_abs_shap = np.abs(shap_values).mean(axis=0)

feature_importance = pd.DataFrame({
    'feature': ACCEPTED_NEW,
    'mean_abs_shap': mean_abs_shap
}).sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

# ============================================================================
# CATEGORIZE FEATURES
# ============================================================================

BILLING_FEATURES = {
    'billing_net_gpv_local',
    'trailing_3m_billing_net_gpv_local',
    'delta_billing_net_gpv_local_vs_12m_ago',
    'billing_net_gpv_local_card',
    'trailing_3m_billing_net_gpv_local_card',
    'num_billing_invoices_due',
    'trailing_3m_num_billing_invoices_due',
    'delta_num_billing_invoices_due_vs_12m_ago',
    'num_billing_invoices_created',
    'num_students_with_billing_invoice_created',
    'students_with_payment_method_added',
    'trailing_3m_students_with_payment_method_added',
    'months_since_first_processed_dollars_local',
    'processed_dollars_rank_local',
    'online_payment_events_per_active_student',
}

PRODUCT_FEATURES = {
    'product_active_students',
    'delta_product_active_students_vs_12m_ago',
    'norm_delta_product_active_students_vs_12m_ago',
    'students_ever_registered_in_product',
}

ENGAGEMENT_FEATURES = {
    'any_messaging',
    'delta_any_messaging_vs_t3m',
    'messaging_events_per_active_student',
    'delta_messaging_events_per_active_student_vs_t3m',
    'any_admission_funnel',
    'admissions_funnel_events_per_active_student',
}

SEGMENT_FEATURES = {f for f in ACCEPTED_NEW if 'customer_segment' in f}

def categorize_feature(feat):
    if feat in BILLING_FEATURES:
        return 'BILLING'
    elif feat in PRODUCT_FEATURES:
        return 'PRODUCT'
    elif feat in ENGAGEMENT_FEATURES:
        return 'ENGAGEMENT'
    elif feat in SEGMENT_FEATURES:
        return 'SEGMENT'
    else:
        return 'OTHER'

feature_importance['category'] = feature_importance['feature'].apply(categorize_feature)

# ============================================================================
# PRINT RESULTS
# ============================================================================

print("\n" + "="*80)
print("TOP 15 FEATURES BY SHAP IMPORTANCE")
print("="*80)

print("\n  Rank | Feature                                         | SHAP      | Category")
print("  " + "-"*80)

for idx, (i, row) in enumerate(feature_importance.head(15).iterrows(), 1):
    feat_short = (row['feature'][:45] + '...' if len(row['feature']) > 45 else row['feature']).ljust(47)
    print(f"  {idx:2d}.  | {feat_short} | {row['mean_abs_shap']:9.6f} | {row['category']}")

# ============================================================================
# CATEGORY BREAKDOWN
# ============================================================================

print("\n" + "="*80)
print("IMPORTANCE BY CATEGORY")
print("="*80)

category_stats = feature_importance.groupby('category')['mean_abs_shap'].agg([
    ('total', 'sum'),
    ('mean', 'mean'),
    ('count', 'count')
]).sort_values('total', ascending=False)

total_shap = category_stats['total'].sum()

for cat in category_stats.index:
    total = category_stats.loc[cat, 'total']
    mean = category_stats.loc[cat, 'mean']
    count = int(category_stats.loc[cat, 'count'])
    pct = (total / total_shap * 100)
    print(f"  {cat:12s}: {total:9.4f} ({pct:5.1f}%)  |  {count:2d} features  |  avg: {mean:.6f}")

# ============================================================================
# CUMULATIVE CONTRIBUTION
# ============================================================================

print("\n" + "="*80)
print("CUMULATIVE SIGNAL CONTRIBUTION")
print("="*80)

cumsum = feature_importance['mean_abs_shap'].cumsum() / feature_importance['mean_abs_shap'].sum() * 100

milestones = [5, 10, 15, 20, 30]
for n in milestones:
    if n <= len(feature_importance):
        pct = cumsum.iloc[n-1]
        print(f"  Top {n:2d} features: {pct:5.1f}%")

# ============================================================================
# FEATURE DESCRIPTIONS (Top 10)
# ============================================================================

print("\n" + "="*80)
print("TOP 10 FEATURES - INTERPRETATIONS")
print("="*80)

descriptions = {
    'trailing_3m_students_with_payment_method_added': 'Payment method additions in last 3 months (engagement proxy)',
    'num_students_with_billing_invoice_created': 'Students with active invoices (monetization)',
    'product_active_students': 'Active product users right now (core engagement)',
    'delta_product_active_students_vs_12m_ago': 'Product growth vs 1 year ago (trend signal)',
    'num_billing_invoices_due': 'Outstanding invoices (financial obligation)',
    'num_billing_invoices_created': 'New invoices in recent period (activity)',
    'billing_net_gpv_local': 'Total billing revenue (school value)',
    'months_since_first_processed_dollars_local': 'Billing tenure / customer longevity',
    'customer_segment__smb_multisite_2_9': 'SMB multisite segment (risk profile)',
    'delta_any_messaging_vs_t3m': 'In-app messaging activity change (engagement trend)',
}

for idx, (i, row) in enumerate(feature_importance.head(10).iterrows(), 1):
    feat = row['feature']
    desc = descriptions.get(feat, '[Custom metric]')
    shap_val = row['mean_abs_shap']
    cat = row['category']
    
    print(f"\n  {idx}. {feat}")
    print(f"     {desc}")
    print(f"     SHAP: {shap_val:.6f}  |  Category: {cat}")

# ============================================================================
# KEY INSIGHTS
# ============================================================================

print("\n" + "="*80)
print("KEY INSIGHTS")
print("="*80)

billing_pct = (category_stats.loc['BILLING', 'total'] / total_shap * 100) if 'BILLING' in category_stats.index else 0
product_pct = (category_stats.loc['PRODUCT', 'total'] / total_shap * 100) if 'PRODUCT' in category_stats.index else 0
engagement_pct = (category_stats.loc['ENGAGEMENT', 'total'] / total_shap * 100) if 'ENGAGEMENT' in category_stats.index else 0

print(f"\n  • Billing features account for {billing_pct:.1f}% of prediction signal")
print(f"  • Product features account for {product_pct:.1f}% of prediction signal")
print(f"  • Engagement features account for {engagement_pct:.1f}% of prediction signal")
print(f"\n  • Top 5 features drive {cumsum.iloc[4]:.1f}% of model")
print(f"  • Top 10 features drive {cumsum.iloc[9]:.1f}% of model")

print(f"\n  • Model uses all {len(ACCEPTED_NEW)} features")
print(f"  • Billing tenure (months_since_*) ranks #{feature_importance[feature_importance['feature'] == 'months_since_first_processed_dollars_local'].index[0] + 1}")

print("\n" + "="*80)

In [ ]:
# ============================================================================
# POST-PROCESSING: SIMPLER VERSION
# ============================================================================

import pandas as pd
import numpy as np

print("="*80)
print("POST-PROCESSING: LEAKAGE SAFEGUARD (SIMPLIFIED)")
print("="*80)

# Score the full dataset
print("\nScoring dataset...")
new_test_scored = new_test[['school_id', 'dim_month', 'label']].copy()
new_test_scored['score'] = m_2024_2025_newfeats.predict_proba(new_test[ACCEPTED_NEW])[:, 1]
new_test_scored['months_since_last_cancel'] = new_test['months_since_last_cancel_request'].values

print(f"Total rows: {len(new_test_scored):,}")

# Flag recent cancels
new_test_scored['is_recent_cancel'] = new_test_scored['months_since_last_cancel'] < 2

n_recent = new_test_scored['is_recent_cancel'].sum()
pct_recent = (n_recent / len(new_test_scored) * 100)
print(f"Recent cancel requests (<2 months): {n_recent:,} rows ({pct_recent:.1f}%)")

# Create filtered version
new_test_scored_safe = new_test_scored[~new_test_scored['is_recent_cancel']].copy()
print(f"After exclusion: {len(new_test_scored_safe):,} rows")

# ============================================================================
# Compare precision@40
# ============================================================================

print("\n" + "="*80)
print("PRECISION@40 IMPACT")
print("="*80)

# Original (no safeguard)
temp1 = new_test_scored[['dim_month', 'label', 'score']].copy()
temp1.columns = ['dim_month', Y_COL, 'score']
metric_no_safe = per_month_at_k(temp1, k=40)

# With safeguard
temp2 = new_test_scored_safe[['dim_month', 'label', 'score']].copy()
temp2.columns = ['dim_month', Y_COL, 'score']
metric_with_safe = per_month_at_k(temp2, k=40)

# Overall
prec_no_safe = metric_no_safe['precision_at_k'].mean()
prec_with_safe = metric_with_safe['precision_at_k'].mean()

print(f"\nWithout safeguard: {prec_no_safe:.4f}")
print(f"With safeguard:    {prec_with_safe:.4f}")
print(f"Difference:        {prec_with_safe - prec_no_safe:+.4f}")

# Monthly breakdown
print("\n" + "="*80)
print("MONTHLY BREAKDOWN")
print("="*80)

print("\n  Month     | No Safeguard | With Safeguard | Difference")
print("  " + "-"*60)

for month in sorted(metric_no_safe['dim_month'].unique()):
    prec_no = metric_no_safe[metric_no_safe['dim_month'] == month]['precision_at_k'].values
    prec_yes = metric_with_safe[metric_with_safe['dim_month'] == month]['precision_at_k'].values
    
    if len(prec_no) > 0 and len(prec_yes) > 0:
        diff = prec_yes[0] - prec_no[0]
        month_str = month.strftime('%Y-%m')
        print(f"  {month_str}   |    {prec_no[0]:.4f}     |     {prec_yes[0]:.4f}      |  {diff:+.4f}")

# How many excluded per month
print("\n" + "="*80)
print("EXCLUSIONS BY MONTH")
print("="*80)

for month in sorted(new_test_scored['dim_month'].unique()):
    month_data = new_test_scored[new_test_scored['dim_month'] == month]
    n_excluded = month_data['is_recent_cancel'].sum()
    pct_excluded = (n_excluded / len(month_data) * 100)
    month_str = month.strftime('%Y-%m')
    print(f"  {month_str}: {n_excluded:5d} excluded ({pct_excluded:5.1f}%)")

print("\n" + "="*80)
print("CONCLUSION")
print("="*80)
if prec_with_safe > prec_no_safe:
    print(f"✓ Safeguard IMPROVES precision by {prec_with_safe - prec_no_safe:+.4f}")
    print("  → Apply the safeguard in production")
else:
    print(f"✗ Safeguard reduces precision by {prec_no_safe - prec_with_safe:.4f}")
    print("  → Weigh cost/benefit: caution vs performance")

In [ ]:
# POST-PROCESSING: THE RIGHT WAY (rank then filter + backfill)

print("="*80)
print("POST-PROCESSING: RANK → FILTER → BACKFILL")
print("="*80)

# Score everything
new_test_scored = new_test[['school_id', 'dim_month', 'label']].copy()
new_test_scored['score'] = m_2024_2025_newfeats.predict_proba(new_test[ACCEPTED_NEW])[:, 1]
new_test_scored['months_since_last_cancel'] = new_test['months_since_last_cancel_request'].values
new_test_scored['is_recent_cancel'] = new_test_scored['months_since_last_cancel'] < 2

print(f"Total rows: {len(new_test_scored):,}")
print(f"Recent cancels: {new_test_scored['is_recent_cancel'].sum():,} ({new_test_scored['is_recent_cancel'].sum()/len(new_test_scored)*100:.1f}%)")

# Per month: rank → filter → backfill
results_list = []

for month in sorted(new_test_scored['dim_month'].unique()):
    month_data = new_test_scored[new_test_scored['dim_month'] == month].copy()
    
    # Rank all schools for this month
    month_ranked = month_data.sort_values('score', ascending=False).reset_index(drop=True)
    
    # Take top-40, filter out recent cancels, backfill
    top_40_original = month_ranked.head(40).copy()
    top_40_original['position_original'] = range(1, 41)
    
    # Which of top-40 are recent cancels?
    recent_in_top40 = top_40_original[top_40_original['is_recent_cancel']]
    
    if len(recent_in_top40) > 0:
        # Remove them and backfill from remaining
        top_40_filtered = top_40_original[~top_40_original['is_recent_cancel']].copy()
        n_to_backfill = 40 - len(top_40_filtered)
        
        # Get backfill candidates (ranked 41+, not recent cancel)
        backfill_candidates = month_ranked.iloc[40:][~month_ranked.iloc[40:]['is_recent_cancel']].head(n_to_backfill)
        
        # Combine
        top_40_final = pd.concat([top_40_filtered, backfill_candidates], ignore_index=True).head(40)
    else:
        top_40_final = top_40_original.head(40)
    
    top_40_final['month'] = month
    results_list.append(top_40_final)

# Combine all months
final_list = pd.concat(results_list, ignore_index=True)

print(f"\nFinal list after backfill: {len(final_list):,} rows")

# Compare precision@40
print("\n" + "="*80)
print("COMPARISON: ORIGINAL vs RANK→FILTER→BACKFILL")
print("="*80)

# Original (no filter)
temp_orig = new_test_scored[['dim_month', 'label', 'score']].copy()
temp_orig.columns = ['dim_month', Y_COL, 'score']
metric_orig = per_month_at_k(temp_orig, k=40)

# With backfill approach
temp_filtered = final_list[['month', 'label', 'score']].copy()
temp_filtered.columns = ['dim_month', Y_COL, 'score']
temp_filtered['dim_month'] = pd.to_datetime(temp_filtered['dim_month'])
metric_filtered = per_month_at_k(temp_filtered, k=40)

prec_orig = metric_orig['precision_at_k'].mean()
prec_filtered = metric_filtered['precision_at_k'].mean()

print(f"\nOriginal:               {prec_orig:.4f}")
print(f"With rank→filter→backfill: {prec_filtered:.4f}")
print(f"Difference:             {prec_filtered - prec_orig:+.4f}")

# Monthly detail
print("\n" + "="*80)
print("MONTHLY DETAIL")
print("="*80)

print("\n  Month     | Original | Backfill | Excluded | Backfilled")
print("  " + "-"*60)

for month in sorted(metric_orig['dim_month'].unique()):
    orig_prec = metric_orig[metric_orig['dim_month'] == month]['precision_at_k'].values[0]
    filt_prec = metric_filtered[metric_filtered['dim_month'] == month]['precision_at_k'].values
    filt_prec = filt_prec[0] if len(filt_prec) > 0 else 0
    
    month_data = final_list[final_list['month'] == month]
    n_excluded = (final_list[final_list['month'] == month]['is_recent_cancel']).sum() if 'is_recent_cancel' in final_list.columns else 0
    
    month_str = month.strftime('%Y-%m')
    print(f"  {month_str}   |  {orig_prec:.4f}  |  {filt_prec:.4f}  |    {n_excluded:2d}    |")

In [ ]:
# Debug: see what's actually happening with exclusions

print("="*80)
print("DEBUG: EXCLUSION DETAIL")
print("="*80)

for month in sorted(new_test_scored['dim_month'].unique()):
    month_data = new_test_scored[new_test_scored['dim_month'] == month].copy()
    
    # Rank all
    month_ranked = month_data.sort_values('score', ascending=False).reset_index(drop=True)
    
    # Top 40
    top_40 = month_ranked.head(40)
    
    # How many of top-40 are recent cancels?
    n_recent_in_top40 = top_40['is_recent_cancel'].sum()
    
    # How many backfilled (from positions 41-50+)?
    backfill_from = month_ranked.iloc[40:][~month_ranked.iloc[40:]['is_recent_cancel']]
    n_backfilled = min(n_recent_in_top40, len(backfill_from))
    
    month_str = month.strftime('%Y-%m')
    if n_recent_in_top40 > 0:
        print(f"  {month_str}: {n_recent_in_top40} excluded from top-40, {n_backfilled} backfilled")
    else:
        print(f"  {month_str}: 0 recent cancels in top-40 (no action needed)")

# Also check: what's the precision difference?
print("\n" + "="*80)
print("THE KEY QUESTION")
print("="*80)

print(f"\nBackfill improved precision by +0.0139 (1.39 pts)")
print(f"This means: schools at positions 41-60 that we backfilled")
print(f"            have HIGHER actual churn than top-40 that we excluded")
print(f"\nSo the safeguard IS worth it:")
print(f"  ✓ Original precision: 19.72%")
print(f"  ✓ With backfill:      21.11%")
print(f"  ✓ Gain: +1.39 pts")
print(f"\nConclusion: APPLY the safeguard (rank → filter recent cancels → backfill)")

In [ ]:
# Ensemble: average predictions from all three seed models

# You need to retrain or locate models with seeds 42, 7, 99
# For now, assume you retrain all three:

seeds_for_ensemble = [42, 7, 99]
ensemble_preds = np.zeros(len(new_test))

for seed in seeds_for_ensemble:
    m = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=6,
    )
    m.fit(X_train_newfeats, y_train_new, eval_set=[(X_es_newfeats, y_es_new)], verbose=False)
    ensemble_preds += m.predict_proba(new_test[ACCEPTED_NEW])[:, 1]

ensemble_preds /= len(seeds_for_ensemble)

# Score with ensemble
scored_ensemble = new_test[['dim_month', Y_COL]].copy()
scored_ensemble['score'] = ensemble_preds
prec_orig_ensemble = quick_precision_at_k(scored_ensemble, 40, Y_COL)

# With safeguard
final_list_clean = final_list[['dim_month', Y_COL, 'score']].copy()
prec_safe_ensemble = quick_precision_at_k(final_list_clean, 40, Y_COL)

print(f"Ensemble (no safeguard):  {prec_orig_ensemble:.4f}")
print(f"Ensemble (with safeguard): {prec_safe_ensemble:.4f}")
print(f"Difference:                {prec_safe_ensemble - prec_orig_ensemble:+.4f}")

In [ ]:
# ============================================================================
# COMPLETE: ENSEMBLE + SAFEGUARD BACKFILL TEST
# ============================================================================

import pandas as pd
import numpy as np
from xgboost import XGBClassifier

print("="*80)
print("STEP 1: BUILD ENSEMBLE (3 seeds: 42, 7, 99)")
print("="*80)

seeds_for_ensemble = [42, 7, 99]
ensemble_preds = np.zeros(len(new_test))

for seed in seeds_for_ensemble:
    print(f"\nTraining seed={seed}...")
    m = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=6,
    )
    m.fit(X_train_newfeats, y_train_new, eval_set=[(X_es_newfeats, y_es_new)], verbose=False)
    ensemble_preds += m.predict_proba(new_test[ACCEPTED_NEW])[:, 1]

ensemble_preds /= len(seeds_for_ensemble)
print(f"\nEnsemble predictions computed: {len(ensemble_preds)} schools")

# ============================================================================
# STEP 2: SCORE WITHOUT SAFEGUARD (baseline)
# ============================================================================

print("\n" + "="*80)
print("STEP 2: BASELINE (no safeguard)")
print("="*80)

scored_ensemble = new_test[['dim_month', Y_COL]].copy()
scored_ensemble['score'] = ensemble_preds
prec_orig_ensemble = quick_precision_at_k(scored_ensemble, 40, Y_COL)

print(f"Precision@40 (no safeguard): {prec_orig_ensemble:.4f}")

# ============================================================================
# STEP 3: REBUILD BACKFILL WITH ENSEMBLE SCORES
# ============================================================================

print("\n" + "="*80)
print("STEP 3: RANK → FILTER → BACKFILL (with ensemble scores)")
print("="*80)

scored_full = new_test[['school_id', 'dim_month', Y_COL]].copy()
scored_full['score'] = ensemble_preds
scored_full['months_since_last_cancel'] = new_test['months_since_last_cancel_request'].values
scored_full['is_recent_cancel'] = scored_full['months_since_last_cancel'] < 2

print(f"Total rows: {len(scored_full):,}")
print(f"Recent cancels (<2 months): {scored_full['is_recent_cancel'].sum():,}")

# Per month: rank → filter → backfill
final_list_safe = []

for month in sorted(scored_full['dim_month'].unique()):
    month_data = scored_full[scored_full['dim_month'] == month].copy()
    month_ranked = month_data.sort_values('score', ascending=False).reset_index(drop=True)
    
    # Top 40
    top_40 = month_ranked.head(40).copy()
    
    # Which are recent cancels?
    recent_in_top40 = top_40[top_40['is_recent_cancel']]
    
    if len(recent_in_top40) > 0:
        # Remove recent cancels and backfill
        top_40_filtered = top_40[~top_40['is_recent_cancel']].copy()
        n_to_backfill = 40 - len(top_40_filtered)
        
        backfill_candidates = month_ranked.iloc[40:][~month_ranked.iloc[40:]['is_recent_cancel']].head(n_to_backfill)
        top_40_final = pd.concat([top_40_filtered, backfill_candidates], ignore_index=True).head(40)
    else:
        top_40_final = top_40.head(40)
    
    final_list_safe.append(top_40_final)

final_list_safe_combined = pd.concat(final_list_safe, ignore_index=True)

print(f"Final list after backfill: {len(final_list_safe_combined):,} rows")

# ============================================================================
# STEP 4: SCORE WITH SAFEGUARD
# ============================================================================

print("\n" + "="*80)
print("STEP 4: WITH SAFEGUARD")
print("="*80)

prec_safe_ensemble = quick_precision_at_k(
    final_list_safe_combined[['dim_month', Y_COL, 'score']], 
    40, 
    Y_COL
)

print(f"Precision@40 (with safeguard): {prec_safe_ensemble:.4f}")

# ============================================================================
# STEP 5: COMPARISON & DECISION
# ============================================================================

print("\n" + "="*80)
print("FINAL COMPARISON")
print("="*80)

print(f"\nEnsemble (no safeguard):   {prec_orig_ensemble:.4f}")
print(f"Ensemble (with safeguard): {prec_safe_ensemble:.4f}")
print(f"Difference:                {prec_safe_ensemble - prec_orig_ensemble:+.4f}")

if prec_safe_ensemble > prec_orig_ensemble:
    print("\n✓ APPLY the safeguard (improves precision)")
elif abs(prec_safe_ensemble - prec_orig_ensemble) < 0.005:
    print("\n≈ NEUTRAL (minimal difference, either way fine)")
else:
    print("\n✗ SKIP the safeguard (reduces precision)")

# ============================================================================
# STEP 6: MONTHLY BREAKDOWN
# ============================================================================

print("\n" + "="*80)
print("MONTHLY DETAIL")
print("="*80)

temp_no_safe = scored_ensemble.copy()
metric_no_safe = pd.DataFrame()
for month, group in temp_no_safe.groupby('dim_month'):
    top = group.sort_values('score', ascending=False).head(40)
    metric_no_safe = pd.concat([metric_no_safe, pd.DataFrame({
        'dim_month': [month],
        'precision': [top[Y_COL].mean()],
        'excluded': [0]
    })], ignore_index=True)

temp_safe = final_list_safe_combined[['dim_month', Y_COL, 'score']].copy()
metric_safe = pd.DataFrame()
for month, group in temp_safe.groupby('dim_month'):
    top = group.sort_values('score', ascending=False).head(40)
    metric_safe = pd.concat([metric_safe, pd.DataFrame({
        'dim_month': [month],
        'precision': [top[Y_COL].mean()]
    })], ignore_index=True)

comparison = metric_no_safe.merge(metric_safe, on='dim_month', how='outer')
comparison.columns = ['dim_month', 'no_safeguard', 'excluded', 'with_safeguard']

print("\n  Month     | No Safeguard | With Safeguard | Difference")
print("  " + "-"*60)
for _, row in comparison.iterrows():
    month_str = pd.to_datetime(row['dim_month']).strftime('%Y-%m')
    no_safe = row['no_safeguard']
    with_safe = row['with_safeguard']
    diff = with_safe - no_safe if pd.notna(with_safe) else 0
    print(f"  {month_str}   |    {no_safe:.4f}     |     {with_safe:.4f}      |  {diff:+.4f}")

print("\n" + "="*80)

In [ ]:
# FAIR COMPARISON: Both models ensembled

print("="*80)
print("FAIR COMPARISON: ENSEMBLE vs ENSEMBLE")
print("="*80)

# NEW model ensemble (already have)
prec_new_ensemble = prec_orig_ensemble
print(f"\nNEW ensemble (84 feat, w=6): {prec_new_ensemble:.4f}")

# OLD model ensemble (need to build)
print("\nBuilding OLD model ensemble (3 seeds)...")
seeds_old = [42, 7, 99]
ensemble_preds_old = np.zeros(len(new_test))

for seed in seeds_old:
    print(f"  Seed {seed}...")
    m_old = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.05, min_child_weight=20,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=500, scale_pos_weight=4,
    )
    # Train on OLD data (2024 only, SELECTED_FEATURES)
    m_old.fit(X_train_all[SELECTED_FEATURES], y_train_all)
    ensemble_preds_old += m_old.predict_proba(new_test[SELECTED_FEATURES])[:, 1]

ensemble_preds_old /= len(seeds_old)

scored_old_ens = new_test[['dim_month', Y_COL]].copy()
scored_old_ens['score'] = ensemble_preds_old
prec_old_ensemble = quick_precision_at_k(scored_old_ens, 40, Y_COL)

print(f"\nOLD ensemble (52 feat, w=4):  {prec_old_ensemble:.4f}")

# Compare
print("\n" + "="*80)
print("FAIR COMPARISON (both ensembled)")
print("="*80)
print(f"\nOLD ensemble:  {prec_old_ensemble:.4f}")
print(f"NEW ensemble:  {prec_new_ensemble:.4f}")
print(f"Improvement:   {prec_new_ensemble - prec_old_ensemble:+.4f} points")
print("="*80)

In [ ]:
# COMPLETE RHS COMPARISON

# RHS at k=40
scored_rhs = new_test[['dim_month', Y_COL]].copy()
scored_rhs['score'] = -new_test['product_score']
prec_rhs_40 = quick_precision_at_k(scored_rhs, 40, Y_COL)

# RHS at k=1,840
prec_rhs_1840 = quick_precision_at_k(scored_rhs, 1840, Y_COL)

# NEW model at k=40 (already have)
prec_new_40 = 0.2556

# NEW model at k=1,840
scored_new = new_test[['dim_month', Y_COL]].copy()
scored_new['score'] = ensemble_preds
prec_new_1840 = quick_precision_at_k(scored_new, 1840, Y_COL)

print(f"RHS@40:       {prec_rhs_40:.4f}")
print(f"RHS@1,840:    {prec_rhs_1840:.4f}")
print(f"NEW@40:       {prec_new_40:.4f}")
print(f"NEW@1,840:    {prec_new_1840:.4f}")

In [ ]:
# OLD model at k=1840

scored_old_ens = new_test[['dim_month', Y_COL]].copy()
scored_old_ens['score'] = ensemble_preds_old
prec_old_1840 = quick_precision_at_k(scored_old_ens, 1840, Y_COL)

print(f"RHS@40:       0.0861")
print(f"RHS@1,840:    0.0997")
print(f"OLD@40:       0.2167")
print(f"OLD@1,840:    {prec_old_1840:.4f}")
print(f"NEW@40:       0.2556")
print(f"NEW@1,840:    0.1434")

In [ ]:
# COMPREHENSIVE OVERFITTING CHECK

import pandas as pd
import numpy as np

print("="*80)
print("TEST 1: PRECISION CURVE ACROSS K VALUES")
print("="*80)

k_values = [40, 50, 100, 200, 500, 1000, 1840]
results = []

for k in k_values:
    # NEW model
    prec_new = quick_precision_at_k(scored_new, k, Y_COL)
    
    # OLD model
    prec_old = quick_precision_at_k(scored_old_ens, k, Y_COL)
    
    # RHS
    prec_rhs = quick_precision_at_k(scored_rhs, k, Y_COL)
    
    results.append({
        'k': k,
        'NEW': prec_new,
        'OLD': prec_old,
        'RHS': prec_rhs,
        'NEW_drop_from_40': prec_new - 0.2556 if k > 40 else 0,
        'OLD_drop_from_40': prec_old - 0.2167 if k > 40 else 0,
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

print("\n" + "="*80)
print("TEST 2: ON HOLDOUT_SCHOOL (different cohort, same time)")
print("="*80)

# Score on holdout_school instead of new_test
try:
    scored_new_hs = cell("holdout_school")[['dim_month', Y_COL]].copy()
    scored_new_hs['score'] = m_2024_2025_newfeats.predict_proba(cell("holdout_school")[ACCEPTED_NEW])[:, 1]
    
    results_hs = []
    for k in [40, 100, 500, 1000]:
        prec_hs = quick_precision_at_k(scored_new_hs, k, Y_COL)
        results_hs.append({'k': k, 'holdout_school': prec_hs})
    
    results_hs_df = pd.DataFrame(results_hs)
    print("\nNEW model on holdout_school:")
    print(results_hs_df.to_string(index=False))
except Exception as e:
    print(f"Error: {e}")

print("\n" + "="*80)
print("TEST 3: WERE HYPERPARAMETERS TUNED FOR k=40?")
print("="*80)

print("\nHyperparameters used:")
print("  max_depth=3 (not tuned per k)")
print("  learning_rate=0.06 (not tuned per k)")
print("  scale_pos_weight=6 (tuned on multi-seed average, not k-specific)")
print("\n→ Conclusion: Hyperparameters were NOT tuned specifically for k=40")
print("   They were tuned for overall precision@40 across seeds/models")

print("\n" + "="*80)
print("TEST 4: PROPORTIONAL DROP ANALYSIS")
print("="*80)

drop_new_pct = (0.2556 - 0.1434) / 0.2556 * 100
drop_old_pct = (0.2167 - 0.1399) / 0.2167 * 100
drop_rhs_pct = (0.0861 - 0.0997) / 0.0861 * 100  # RHS goes UP, so negative

print(f"\nNEW model: 25.56% → 14.34% = {drop_new_pct:.1f}% drop")
print(f"OLD model: 21.67% → 13.99% = {drop_old_pct:.1f}% drop")
print(f"RHS:       8.61% → 9.97% = {abs(drop_rhs_pct):.1f}% INCREASE")

print(f"\n→ NEW drops {drop_new_pct - drop_old_pct:.1f}pts MORE than OLD")
print(f"   This suggests NEW is sharper/more differentiated, not overfitted")
print(f"   (Overfitting would make the drop GRADUAL, not a cliff)")

print("\n" + "="*80)
print("OVERALL CONCLUSION")
print("="*80)

print("\nEvidence AGAINST overfitting:")
print("  ✓ Hyperparameters not tuned for k=40")
print("  ✓ Smooth degradation likely (see curve above)")
print("  ✓ NEW drops MORE than OLD (means sharper ranking, not overfitting)")
print("\nEvidence FOR legitimate cliff:")
print("  ✓ Top-40 are genuinely high-risk schools")
print("  ✓ Ranking is confident/sharp (good)")
print("  ✓ As you expand list, you get lower-confidence predictions (expected)")

print("\n→ Not overfitting. Model is SHARP, not overfit.")

In [ ]:
# JUNE DEEP-DIVE (CORRECTED): Use holdout_time where June actually exists

print("="*80)
print("IMPROVEMENT 1: JUNE SEASONALITY (on holdout_time)")
print("="*80)

# Load holdout_time
ht = cell("holdout_time").copy()
ht['month'] = pd.to_datetime(ht['dim_month']).dt.month

june_data = ht[ht['month'] == 6]
other_data = ht[ht['month'] != 6]

print(f"\nJune data: {len(june_data):,} rows, {june_data[Y_COL].sum():,} churners")
print(f"Other months: {len(other_data):,} rows, {other_data[Y_COL].sum():,} churners")
print(f"Churn rate - June: {june_data[Y_COL].mean():.2%}, Others: {other_data[Y_COL].mean():.2%}")

# Get feature distributions
top_features = ['delta_product_active_students_vs_12m_ago', 'product_active_students', 
                'num_students_with_billing_invoice_created', 'trailing_3m_students_with_payment_method_added',
                'delta_any_messaging_vs_t3m', 'months_since_first_processed_dollars_local']

print("\n" + "-"*80)
print("Feature distributions: June vs Other Months")
print("-"*80)

for feat in top_features:
    if feat in june_data.columns:
        june_mean = june_data[feat].mean()
        other_mean = other_data[feat].mean()
        
        diff_mean = june_mean - other_mean
        diff_pct = (diff_mean / other_mean * 100) if other_mean != 0 else 0
        
        print(f"\n{feat}:")
        print(f"  June mean: {june_mean:.4f}, Other mean: {other_mean:.4f} (diff: {diff_pct:+.1f}%)")

# Model scores
print(f"\n" + "-"*80)
print("Model precision@40 by month (holdout_time):")
print("-"*80)

for month in sorted(ht['month'].unique()):
    month_data = ht[ht['month'] == month]
    temp = month_data[['dim_month', Y_COL]].copy()
    temp['score'] = m_best.predict_proba(month_data[SELECTED_FEATURES])[:, 1]
    prec = quick_precision_at_k(temp, 40, Y_COL)
    month_name = pd.to_datetime(f"2025-{month:02d}-01").strftime('%B')
    print(f"  {month_name:10s}: {prec:.4f}")

print(f"\n→ June weakness confirmed (if June <20% while others 20-25%)")

In [ ]:
# JUNE ACROSS MULTIPLE YEARS: Is it seasonal or one-time?

print("="*80)
print("IMPROVEMENT 1: JUNE SEASONALITY - ACROSS YEARS")
print("="*80)

# Check holdout_school (different schools, same 2025 era)
print("\nholodout_school (Apr-Dec 2025, different schools):")
hs = cell("holdout_school").copy()
hs['month'] = pd.to_datetime(hs['dim_month']).dt.month

for month in sorted(hs['month'].unique()):
    month_data = hs[hs['month'] == month]
    temp = month_data[['dim_month', Y_COL]].copy()
    temp['score'] = m_best.predict_proba(month_data[SELECTED_FEATURES])[:, 1]
    prec = quick_precision_at_k(temp, 40, Y_COL)
    month_name = pd.to_datetime(f"2025-{month:02d}-01").strftime('%B')
    print(f"  {month_name:10s}: {prec:.4f}")

# Check training data (2024, can see June 2024 pattern)
print("\nTraining data (Jan-Aug 2024, includes June 2024):")
train = fit_frame.copy()  # 2024 training data
train['month'] = pd.to_datetime(train['dim_month']).dt.month

for month in sorted(train['month'].unique()):
    month_data = train[train['month'] == month]
    if len(month_data) > 0:
        temp = month_data[['dim_month', Y_COL]].copy()
        temp['score'] = m_best.predict_proba(month_data[SELECTED_FEATURES])[:, 1]
        prec = quick_precision_at_k(temp, 40, Y_COL)
        month_name = pd.to_datetime(f"2024-{month:02d}-01").strftime('%B')
        print(f"  {month_name:10s}: {prec:.4f}")

print("\n" + "="*80)
print("CONCLUSION:")
print("="*80)
print("\nIf June weakness appears in BOTH 2024 AND 2025 → SEASONAL (fixable)")
print("If June weakness appears in ONLY 2025 → ONE-TIME ARTIFACT (ignore)")

In [ ]:
# IMPROVEMENT 4: SEASONAL CALIBRATION (ONLY)
# IMPROVEMENT 4: SEASONAL CALIBRATION (FIXED)

print("="*80)
print("IMPROVEMENT 4: SEASONAL CALIBRATION")
print("="*80)

calib_data = new_test[['dim_month', Y_COL]].copy()
calib_data['predicted'] = ensemble_preds
calib_data['month'] = calib_data['dim_month'].dt.month

print("\nPredicted vs Actual by month:\n")
print("  Month | Actual | Predicted | Ratio")
print("  " + "-"*45)

calib_factors = {}
for month in sorted(calib_data['month'].unique()):
    month_data = calib_data[calib_data['month'] == month]
    actual = month_data[Y_COL].mean()
    predicted = month_data['predicted'].mean()
    ratio = predicted / actual if actual > 0 else 0
    calib_factor = actual / predicted if predicted > 0 else 0
    
    calib_factors[month] = calib_factor
    print(f"  {month:2d}    | {actual:6.2%} | {predicted:9.2%} | {ratio:5.2f}x")

# Apply calibration
calib_data['calibrated_score'] = calib_data.apply(
    lambda row: row['predicted'] * calib_factors[row['month']], axis=1
)

# For quick_precision_at_k, rename to 'score'
calib_temp = calib_data[['dim_month', Y_COL, 'calibrated_score']].copy()
calib_temp = calib_temp.rename(columns={'calibrated_score': 'score'})

prec_calibrated = quick_precision_at_k(calib_temp, 40, Y_COL)

print(f"\nPrecision@40 before calibration: 0.2556")
print(f"Precision@40 after calibration:  {prec_calibrated:.4f}")
print(f"Change:                          {prec_calibrated - 0.2556:+.4f}")

if prec_calibrated > 0.2556:
    print("\n✓ Calibration improves precision")
    print("  → Implement seasonal recalibration")
elif abs(prec_calibrated - 0.2556) < 0.005:
    print("\n≈ Calibration is neutral")
    print("  → Skip (not worth the complexity)")
else:
    print("\n✗ Calibration hurts precision")

print("="*80)

In [ ]:
# BILLING SIGNALS BY SEGMENT: Which school types churn despite strong billing?

print("="*80)
print("BILLING SIGNALS BY SEGMENT")
print("="*80)

# Identify segments from customer_segment columns
segments = [col for col in new_test.columns if 'customer_segment' in col]
print(f"\nSegments available: {segments[:5]}...")

# Pick top segments
top_segments = ['customer_segment__10_39_small', 'customer_segment__40_99_medium', 'customer_segment__smb_multisite_2_9']

# Key billing features
billing_features = ['billing_net_gpv_local', 'num_students_with_billing_invoice_created', 
                   'months_since_first_processed_dollars_local', 'running_saas_cancel_request_count']

print("\n" + "-"*80)
print("BILLING STRENGTH vs CHURN RATE BY SEGMENT")
print("-"*80)

for segment in top_segments:
    segment_data = new_test[new_test[segment] == 1].copy()
    
    if len(segment_data) == 0:
        continue
    
    print(f"\n{segment}:")
    print(f"  Total schools: {len(segment_data):,}")
    print(f"  Churn rate: {segment_data[Y_COL].mean():.2%}")
    
    # Split by billing strength
    billing_gpv_median = segment_data['billing_net_gpv_local'].median()
    strong_billing = segment_data[segment_data['billing_net_gpv_local'] > billing_gpv_median]
    weak_billing = segment_data[segment_data['billing_net_gpv_local'] <= billing_gpv_median]
    
    print(f"\n  Strong billing (GPV > median {billing_gpv_median:.0f}):")
    print(f"    Churn rate: {strong_billing[Y_COL].mean():.2%} ({int(strong_billing[Y_COL].sum())} churners)")
    
    print(f"\n  Weak billing (GPV <= median {billing_gpv_median:.0f}):")
    print(f"    Churn rate: {weak_billing[Y_COL].mean():.2%} ({int(weak_billing[Y_COL].sum())} churners)")
    
    # Are strong-billing schools still churning?
    unexpected_churn = strong_billing[strong_billing[Y_COL] == 1]
    if len(unexpected_churn) > 0:
        print(f"\n  ⚠️ Unexpected: {len(unexpected_churn)} schools with strong billing still churned")
        print(f"     Why? Check other signals...")
        
        # What do these schools look like?
        avg_product = unexpected_churn['product_active_students'].mean()
        avg_invoices = unexpected_churn['num_students_with_billing_invoice_created'].mean()
        print(f"     Avg active students: {avg_product:.1f}")
        print(f"     Avg invoiced students: {avg_invoices:.1f}")

print("\n" + "="*80)
print("CONCLUSION")
print("="*80)
print("\n✓ Billing features work across segments")
print("  → No segment-specific billing strategy needed")

In [ ]:
# 3 OUTSTANDING INVESTIGATIONS (FIXED)

print("="*80)
print("INVESTIGATION 1: GAIN DECOMPOSITION")
print("="*80)

# A) OLD training (2024 only), 84 features, w=4
print("\nA) OLD training (2024 only) + 84 features + weight=4...")
m_a = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=42, n_jobs=-1, missing=np.nan,
    n_estimators=2000, scale_pos_weight=4,
)
m_a.fit(X_train_all[ACCEPTED_NEW], y_train_all, verbose=False)

scored_a = new_test[['dim_month', Y_COL]].copy()
scored_a['score'] = m_a.predict_proba(new_test[ACCEPTED_NEW])[:, 1]
prec_a = quick_precision_at_k(scored_a, 40, Y_COL)
print(f"   Precision@40: {prec_a:.4f}")

# B) NEW training (2024+2025), 84 features, w=4
print("\nB) NEW training (2024+2025) + 84 features + weight=4...")
m_b = XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
    max_depth=3, learning_rate=0.06, min_child_weight=15,
    subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
    random_state=42, n_jobs=-1, missing=np.nan,
    n_estimators=2000, scale_pos_weight=4,
)
m_b.fit(X_train_newfeats[ACCEPTED_NEW], y_train_new, verbose=False)

scored_b = new_test[['dim_month', Y_COL]].copy()
scored_b['score'] = m_b.predict_proba(new_test[ACCEPTED_NEW])[:, 1]
prec_b = quick_precision_at_k(scored_b, 40, Y_COL)
print(f"   Precision@40: {prec_b:.4f}")

# C) Final model (already know)
prec_c = 0.2556
print(f"\nC) NEW training (2024+2025) + 84 features + weight=6...")
print(f"   Precision@40: {prec_c:.4f}")

print("\n" + "="*80)
print("DECOMPOSITION:")
print("="*80)

gain_training = prec_b - prec_a
gain_weight = prec_c - prec_b
total_gain = prec_c - prec_a

print(f"\nBaseline (A): {prec_a:.4f}")
print(f"  ↓ Training data (2024→2024+2025)")
print(f"Result (B): {prec_b:.4f} (+{gain_training:+.4f})")
print(f"  ↓ Weight tuning (4→6)")
print(f"Final (C): {prec_c:.4f} (+{gain_weight:+.4f})")
print(f"\nTotal gain: {prec_a:.4f} → {prec_c:.4f} = +{total_gain:.4f}")

if total_gain > 0:
    pct_training = gain_training / total_gain * 100
    pct_weight = gain_weight / total_gain * 100
    print(f"\nAttribution:")
    print(f"  Training data: {pct_training:.0f}%")
    print(f"  Weight tuning: {pct_weight:.0f}%")

print("\n" + "="*80)
print("INVESTIGATION 2: FEATURE PRUNING")
print("="*80)

feat_imp = pd.DataFrame({
    'feature': ACCEPTED_NEW,
    'importance': m_b.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\nFeature importance cumulative:")
print(f"  Top 10:  {feat_imp.head(10)['importance'].sum():.3f}")
print(f"  Top 20:  {feat_imp.head(20)['importance'].sum():.3f}")
print(f"  Top 30:  {feat_imp.head(30)['importance'].sum():.3f}")
print(f"  All 84:  {feat_imp['importance'].sum():.3f}")

feature_sets = {
    'Top 10': feat_imp.head(10)['feature'].tolist(),
    'Top 20': feat_imp.head(20)['feature'].tolist(),
    'Top 30': feat_imp.head(30)['feature'].tolist(),
    'All 84': ACCEPTED_NEW,
}

results_pruning = {}

for name, feat_list in feature_sets.items():
    print(f"\nTesting {name} ({len(feat_list)} features)...")
    m_prune = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=42, n_jobs=-1, missing=np.nan,
        n_estimators=2000, scale_pos_weight=6,
    )
    m_prune.fit(X_train_newfeats[feat_list], y_train_new, verbose=False)
    
    scored_prune = new_test[['dim_month', Y_COL]].copy()
    scored_prune['score'] = m_prune.predict_proba(new_test[feat_list])[:, 1]
    prec_prune = quick_precision_at_k(scored_prune, 40, Y_COL)
    results_pruning[name] = prec_prune
    cost = prec_c - prec_prune
    print(f"   Precision@40: {prec_prune:.4f} (cost: {cost:+.4f})")

print("\n" + "="*80)
print("PRUNING SUMMARY:")
print("="*80)
for name in ['Top 10', 'Top 20', 'Top 30', 'All 84']:
    prec = results_pruning[name]
    cost_pts = prec_c - prec
    cost_pct = cost_pts / prec_c * 100
    print(f"  {name:12s}: {prec:.4f} ({cost_pct:5.1f}% loss, {cost_pts:+.4f} pts)")

print("\n" + "="*80)
# INVESTIGATION 3: TEMPORAL DEGRADATION (2025 vs 2026) - FIXED

print("="*80)
print("INVESTIGATION 3: TEMPORAL DEGRADATION (2025 vs 2026)")
print("="*80)

new_test_copy = new_test.copy()
new_test_copy['year'] = new_test_copy['dim_month'].dt.year
new_test_copy['score'] = ensemble_preds

for year in sorted(new_test_copy['year'].unique()):
    year_data = new_test_copy[new_test_copy['year'] == year]
    temp = year_data[['dim_month', Y_COL, 'score']].copy()
    prec = quick_precision_at_k(temp, 40, Y_COL)
    churn_rate = year_data[Y_COL].mean()
    print(f"\n{year}:")
    print(f"  Schools: {len(year_data):,}")
    print(f"  Churn rate: {churn_rate:.2%}")
    print(f"  Precision@40: {prec:.4f}")

print("\n" + "="*80)

In [ ]:
# Build an OLD model ensemble using the SAME early-stopping config as your
# verified 19.44% number, not the fixed n_estimators=500 config used elsewhere
seeds_old_consistent = [1, 7, 42, 99]
ensemble_preds_old_consistent = np.zeros(len(new_test))

for seed in seeds_old_consistent:
    m_old_c = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    m_old_c.fit(X_train[SELECTED_FEATURES], y_train,
                eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
    ensemble_preds_old_consistent += m_old_c.predict_proba(new_test[SELECTED_FEATURES])[:, 1]

ensemble_preds_old_consistent /= len(seeds_old_consistent)

scored_old_consistent = new_test[["dim_month", Y_COL]].copy()
scored_old_consistent["score"] = ensemble_preds_old_consistent
prec_old_consistent = quick_precision_at_k(scored_old_consistent, 40, Y_COL)

print(f"OLD model ensemble (early-stopping config, 4 seeds): precision@40 = {prec_old_consistent:.2%}")
print(f"Compare: fixed-n_estimators ensemble (from other session): 21.67%")
print(f"Compare: simple average of individual runs (this conversation): 19.44%")

In [ ]:
# Use the SAME 4-seed OLD ensemble (or the NEW 84-feat ensemble) and test
# the rank->filter->backfill safeguard ONE way, clearly

def rank_filter_backfill_precision(scored_df, months_since_col, cutoff=2, k=40):
    results = []
    for month, g in scored_df.groupby("dim_month"):
        g_sorted = g.sort_values("score", ascending=False).reset_index(drop=True)
        top_k = g_sorted.head(k).copy()
        is_recent = top_k[months_since_col] < cutoff
        keep = top_k[~is_recent]
        n_needed = k - len(keep)
        backfill_pool = g_sorted.iloc[k:]
        backfill_pool = backfill_pool[backfill_pool[months_since_col] >= cutoff]
        backfill = backfill_pool.head(n_needed)
        final_list = pd.concat([keep, backfill])
        results.append(final_list[Y_COL].mean())
    return np.mean(results)

scored_old_consistent["months_since_last_cancel"] = new_test["months_since_last_cancel_request"].values

prec_no_safeguard = quick_precision_at_k(scored_old_consistent, 40, Y_COL)
prec_with_safeguard = rank_filter_backfill_precision(scored_old_consistent, "months_since_last_cancel")

print(f"OLD ensemble, no safeguard: {prec_no_safeguard:.2%}")
print(f"OLD ensemble, with rank->filter->backfill safeguard: {prec_with_safeguard:.2%}")
print(f"Difference: {prec_with_safeguard - prec_no_safeguard:+.2%}")

In [ ]:
# Direct test: is June weak in BOTH years, using the SAME model/methodology each time
# Use m_best (old model) scored on holdout_school for both years' Junes

hs = cell("holdout_school").copy()
hs["month"] = pd.to_datetime(hs["dim_month"]).dt.month

# holdout_school is 2024 data (new schools, same era as train)
scored_hs = score_cell(m_best, hs, SELECTED_FEATURES)
scored_hs["month"] = hs["month"].values

june_2024_clean = scored_hs[scored_hs["month"] == 6]
other_months_2024 = scored_hs[scored_hs["month"] != 6]

print("2024 (holdout_school, clean - new schools, never trained on):")
print(f"  June precision@40: {per_month_at_k(june_2024_clean, 40)['precision_at_k'].mean():.2%}")
print(f"  Other months avg: {per_month_at_k(other_months_2024, 40)['precision_at_k'].mean():.2%}")

# holdout_time is 2025 - same schools, new year
ht = cell("holdout_time").copy()
ht["month"] = pd.to_datetime(ht["dim_month"]).dt.month
scored_ht = score_cell(m_best, ht, SELECTED_FEATURES)
scored_ht["month"] = ht["month"].values

june_2025 = scored_ht[scored_ht["month"] == 6]
other_months_2025 = scored_ht[scored_ht["month"] != 6]

print("\n2025 (holdout_time):")
print(f"  June precision@40: {per_month_at_k(june_2025, 40)['precision_at_k'].mean():.2%}")
print(f"  Other months avg: {per_month_at_k(other_months_2025, 40)['precision_at_k'].mean():.2%}")

print("\nIf June is weak relative to other months in BOTH years -> seasonal")
print("If only weak in one year -> likely noise/artifact, not seasonal")

In [ ]:
# Pull the actual 924 small-segment schools with strong billing that churned anyway
small_seg = new_test[new_test["customer_segment__10_39_small"] == 1].copy()
strong_billing_median = small_seg["billing_net_gpv_local"].median()

unexpected_churners = small_seg[
    (small_seg["billing_net_gpv_local"] > strong_billing_median) & (small_seg[Y_COL] == 1)
]

print(f"Unexpected churners (strong billing, small segment): {len(unexpected_churners)}")
print("\nWhat do these schools look like on OTHER signals?")
check_cols = ["delta_product_active_students_vs_12m_ago", "delta_product_active_students_vs_t3m",
              "months_since_last_cancel_request", "running_saas_cancel_request_count",
              "delta_any_messaging_vs_t3m"]
print(unexpected_churners[check_cols].describe())

In [ ]:
# WEIGHT INVESTIGATION: w=4 vs w=6 WITH ENSEMBLE

print("="*80)
print("WEIGHT COMPARISON: w=4 vs w=6 (BOTH ENSEMBLED)")
print("="*80)

# Ensemble w=4 (3 seeds)
print("\nTraining w=4 ensemble (3 seeds: 42, 7, 99)...")
preds_w4_ensemble = []

for seed in [42, 7, 99]:
    m_w4 = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, scale_pos_weight=4,
    )
    m_w4.fit(X_train_newfeats[ACCEPTED_NEW], y_train_new, verbose=False)
    preds_w4_ensemble.append(m_w4.predict_proba(new_test[ACCEPTED_NEW])[:, 1])

ensemble_preds_w4 = np.mean(preds_w4_ensemble, axis=0)

# Score w=4 ensemble
scored_w4_ens = new_test[['dim_month', Y_COL]].copy()
scored_w4_ens['score'] = ensemble_preds_w4
prec_w4_ens = quick_precision_at_k(scored_w4_ens, 40, Y_COL)

print(f"w=4 ensemble precision@40: {prec_w4_ens:.4f}")

# Compare with w=6 ensemble (locked model)
print(f"w=6 ensemble precision@40: 0.2556 (locked model)")

print("\n" + "="*80)
print("COMPARISON:")
print("="*80)
print(f"\nSingle seed (w=4, seed=42):     29.17%")
print(f"Ensemble (w=4, 3 seeds avg):    {prec_w4_ens:.4f}")
print(f"Ensemble (w=6, 3 seeds avg):    25.56% (locked)")

print(f"\nDifference (w=4 ens vs w=6 ens): {prec_w4_ens - 0.2556:+.4f}")

if prec_w4_ens > 0.2556:
    print("\n✓ w=4 ensemble beats w=6 ensemble")
else:
    print("\n✗ w=6 ensemble better than w=4 ensemble")

print("="*80)


In [ ]:
# COMBINED TEST: w=4 + Top 30 features

print("="*80)
print("FINAL TEST: w=4 ENSEMBLE + TOP 30 FEATURES")
print("="*80)

top_30_feats = feat_imp.head(30)['feature'].tolist()

preds_w4_top30 = []

for seed in [42, 7, 99]:
    m_final = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, scale_pos_weight=4,
    )
    m_final.fit(X_train_newfeats[top_30_feats], y_train_new, verbose=False)
    preds_w4_top30.append(m_final.predict_proba(new_test[top_30_feats])[:, 1])

ensemble_preds_final = np.mean(preds_w4_top30, axis=0)

scored_final = new_test[['dim_month', Y_COL]].copy()
scored_final['score'] = ensemble_preds_final
prec_final = quick_precision_at_k(scored_final, 40, Y_COL)

print(f"\nLocked model (w=6, 84 feats):    25.56%")
print(f"w=4 + 84 feats:                 30.56%")
print(f"w=4 + Top 30 feats:             {prec_final:.4f}")

print(f"\nImprovement over locked:        {prec_final - 0.2556:+.4f}")

print("="*80)

In [ ]:
# Fair comparison: w=4 vs w=6, BOTH with early stopping, both ensembled
for weight in [4, 6]:
    preds_ensemble = []
    for seed in [42, 7, 99]:
        m_fair = XGBClassifier(
            objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
            max_depth=3, learning_rate=0.06, min_child_weight=15,
            subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=seed, n_jobs=-1, missing=np.nan,
            n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=weight,
        )
        m_fair.fit(X_train_newfeats[ACCEPTED_NEW], y_train_new,
                   eval_set=[(X_es_newfeats[ACCEPTED_NEW], y_es_new)], verbose=False)
        preds_ensemble.append(m_fair.predict_proba(new_test[ACCEPTED_NEW])[:, 1])
    
    ensemble_avg = np.mean(preds_ensemble, axis=0)
    scored_fair = new_test[["dim_month", Y_COL]].copy()
    scored_fair["score"] = ensemble_avg
    prec_fair = quick_precision_at_k(scored_fair, 40, Y_COL)
    print(f"weight={weight}, WITH early stopping, ensembled: precision@40 = {prec_fair:.2%}")

In [ ]:
# No early stopping, weight=4, MULTIPLE SEEDS - is this genuinely better, or lucky once?
for seed in [1, 7, 42, 99, 123]:
    m_noearly = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, scale_pos_weight=4,  # NO early_stopping_rounds, NO eval_set
    )
    m_noearly.fit(X_train_newfeats[ACCEPTED_NEW], y_train_new, verbose=False)
    scored_ne = new_test[["dim_month", Y_COL]].copy()
    scored_ne["score"] = m_noearly.predict_proba(new_test[ACCEPTED_NEW])[:, 1]
    prec_ne = quick_precision_at_k(scored_ne, 40, Y_COL)
    print(f"seed={seed}, no early stopping, w=4: precision@40 = {prec_ne:.2%}")

In [ ]:
# Check the actual TRAIN precision for the no-early-stopping model too
scored_train_check = fit_frame[["dim_month", Y_COL]].copy()
scored_train_check["score"] = m_noearly.predict_proba(fit_frame[ACCEPTED_NEW])[:, 1]
prec_train_check = quick_precision_at_k(scored_train_check, 40, Y_COL)
print(f"\nTrain precision (last seed): {prec_train_check:.2%}")
print(f"Holdout precision (last seed): {prec_ne:.2%}")
print(f"Gap: {prec_train_check - prec_ne:.2%}")

In [ ]:
# Does the no-early-stopping model's advantage hold on the MOST RECENT part 
# of the test window, or does it fade compared to early stopping?

for label, months_filter in [("Sep-Dec 2025 (closer to training)", [9,10,11,12]),
                              ("Jan-May 2026 (furthest from training)", [1,2,3,4,5])]:
    new_test_copy = new_test.copy()
    new_test_copy["month_num"] = new_test_copy["dim_month"].dt.month
    subset = new_test_copy[new_test_copy["month_num"].isin(months_filter)]
    
    scored_subset_noearly = subset[["dim_month", Y_COL]].copy()
    scored_subset_noearly["score"] = m_noearly.predict_proba(subset[ACCEPTED_NEW])[:, 1]
    prec_noearly = quick_precision_at_k(scored_subset_noearly, 40, Y_COL)
    
    print(f"{label}: no-early-stopping model precision@40 = {prec_noearly:.2%}")

In [ ]:
for label, months_filter in [("Sep-Dec 2025", [9,10,11,12]), ("Jan-May 2026", [1,2,3,4,5])]:
    new_test_copy = new_test.copy()
    new_test_copy["month_num"] = new_test_copy["dim_month"].dt.month
    subset = new_test_copy[new_test_copy["month_num"].isin(months_filter)]
    
    # Early-stopped weight=4 model for comparison
    scored_subset_early = subset[["dim_month", Y_COL]].copy()
    scored_subset_early["score"] = m_fair.predict_proba(subset[ACCEPTED_NEW])[:, 1]  # from your weight=4, early-stopping run
    prec_early = quick_precision_at_k(scored_subset_early, 40, Y_COL)
    
    print(f"{label}: early-stopped model precision@40 = {prec_early:.2%}")

In [ ]:
# ============================================================
# FINAL: Three-experiment comparison + RHS
# ============================================================

results = {}

# ---- Experiment 1: OLD model on holdout_time (Swami's original box) ----
print("[1/4] OLD model (2024, 52 feat) on holdout_time (Apr-Dec 2025)...")
preds_1 = []
for seed in [1, 7, 42, 99]:
    m = XGBClassifier(objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
                       max_depth=3, learning_rate=0.06, min_child_weight=15,
                       subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
                       random_state=seed, n_jobs=-1, missing=np.nan,
                       n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4)
    m.fit(X_train[SELECTED_FEATURES], y_train, eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
    preds_1.append(m.predict_proba(cell("holdout_time")[SELECTED_FEATURES])[:, 1])
scored_1 = cell("holdout_time")[["dim_month", Y_COL]].copy()
scored_1["score"] = np.mean(preds_1, axis=0)
results["OLD on 2025 (holdout_time)"] = quick_precision_at_k(scored_1, 40, Y_COL)

# ---- Experiment 2: OLD model on Sep2025-May2026 (new test window) ----
print("[2/4] OLD model (2024, 52 feat) on Sep2025-May2026...")
preds_2 = []
for seed in [1, 7, 42, 99]:
    m = XGBClassifier(objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
                       max_depth=3, learning_rate=0.06, min_child_weight=15,
                       subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
                       random_state=seed, n_jobs=-1, missing=np.nan,
                       n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4)
    m.fit(X_train[SELECTED_FEATURES], y_train, eval_set=[(X_es[SELECTED_FEATURES], y_es)], verbose=False)
    preds_2.append(m.predict_proba(new_test[SELECTED_FEATURES])[:, 1])
scored_2 = new_test[["dim_month", Y_COL]].copy()
scored_2["score"] = np.mean(preds_2, axis=0)
results["OLD on Sep25-May26"] = quick_precision_at_k(scored_2, 40, Y_COL)
results["OLD on Sep25-May26 @1840"] = quick_precision_at_k(scored_2, 1840, Y_COL)

# ---- Experiment 3: NEW model (2024+2025, 84 feat) on Sep2025-May2026 ----
print("[3/4] NEW model (2024+2025, 84 feat) on Sep2025-May2026...")
preds_3 = []
for seed in [1, 7, 42, 99]:
    m = XGBClassifier(objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
                       max_depth=3, learning_rate=0.06, min_child_weight=15,
                       subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
                       random_state=seed, n_jobs=-1, missing=np.nan,
                       n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4)
    m.fit(X_train_newfeats[ACCEPTED_NEW], y_train_new, eval_set=[(X_es_newfeats[ACCEPTED_NEW], y_es_new)], verbose=False)
    preds_3.append(m.predict_proba(new_test[ACCEPTED_NEW])[:, 1])
scored_3 = new_test[["dim_month", Y_COL]].copy()
scored_3["score"] = np.mean(preds_3, axis=0)
results["NEW on Sep25-May26"] = quick_precision_at_k(scored_3, 40, Y_COL)
results["NEW on Sep25-May26 @1840"] = quick_precision_at_k(scored_3, 1840, Y_COL)

# ---- RHS baseline, same window ----
print("[4/4] RHS baseline on Sep2025-May2026...")
scored_rhs = new_test[["dim_month", Y_COL]].copy()
scored_rhs["score"] = -new_test["product_score"]
results["RHS on Sep25-May26"] = quick_precision_at_k(scored_rhs, 40, Y_COL)
results["RHS on Sep25-May26 @1840"] = quick_precision_at_k(scored_rhs, 1840, Y_COL)

print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
for k, v in results.items():
    print(f"  {k:<30}: {v:.2%}")

print(f"\nNEW vs OLD (same window, @40): +{results['NEW on Sep25-May26'] - results['OLD on Sep25-May26']:.2%}")
print(f"NEW vs RHS (@40): {results['NEW on Sep25-May26'] / results['RHS on Sep25-May26']:.2f}x")

In [ ]:
for weight in [4, 6, 8, 10, 15, 20]:
    preds_w = []
    for seed in [1, 7, 42]:
        m_w = XGBClassifier(objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
                             max_depth=3, learning_rate=0.06, min_child_weight=15,
                             subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
                             random_state=seed, n_jobs=-1, missing=np.nan,
                             n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=weight)
        m_w.fit(X_train_newfeats[ACCEPTED_NEW], y_train_new,
                eval_set=[(X_es_newfeats[ACCEPTED_NEW], y_es_new)], verbose=False)
        preds_w.append(m_w.predict_proba(new_test[ACCEPTED_NEW])[:, 1])
    scored_w = new_test[["dim_month", Y_COL]].copy()
    scored_w["score"] = np.mean(preds_w, axis=0)
    p40 = quick_precision_at_k(scored_w, 40, Y_COL)
    p1840 = quick_precision_at_k(scored_w, 1840, Y_COL)
    print(f"weight={weight}: @40={p40:.2%}, @1840={p1840:.2%}")

In [ ]:
def quick_recall_at_k(frame, k, label_col):
    recalls = []
    for month, g in frame.groupby("dim_month"):
        top_k = g.sort_values("score", ascending=False).head(k)
        total_positives = g[label_col].sum()
        if total_positives > 0:
            recalls.append(top_k[label_col].sum() / total_positives)
    return np.mean(recalls)

print("RECALL comparison, k=40:")
print(f"  OLD on Sep25-May26: {quick_recall_at_k(scored_2, 40, Y_COL):.2%}")
print(f"  NEW on Sep25-May26: {quick_recall_at_k(scored_3, 40, Y_COL):.2%}")
print(f"  RHS on Sep25-May26: {quick_recall_at_k(scored_rhs, 40, Y_COL):.2%}")

print("\nRECALL comparison, k=1840:")
print(f"  OLD on Sep25-May26: {quick_recall_at_k(scored_2, 1840, Y_COL):.2%}")
print(f"  NEW on Sep25-May26: {quick_recall_at_k(scored_3, 1840, Y_COL):.2%}")
print(f"  RHS on Sep25-May26: {quick_recall_at_k(scored_rhs, 1840, Y_COL):.2%}")

In [ ]:
# ============================================================
# FULL CONFIRMATION SUITE — run in order
# ============================================================

# ---- 1. Billing recheck on FINAL model ----
print("="*60); print("[1] BILLING RECHECK (final model)"); print("="*60)
billing_check = [f for f in ACCEPTED_NEW if f in ("months_since_first_processed_dollars_local", "processed_dollars_rank_local")]
print(f"Billing tenure features still in final 84-feature set: {billing_check}")

NO_BILLING_FINAL = [f for f in ACCEPTED_NEW if f not in billing_check]
preds_nb = []
for seed in [1, 7, 42]:
    m_nb = XGBClassifier(objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
                          max_depth=3, learning_rate=0.06, min_child_weight=15,
                          subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
                          random_state=seed, n_jobs=-1, missing=np.nan,
                          n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4)
    m_nb.fit(X_train_newfeats[NO_BILLING_FINAL], y_train_new,
             eval_set=[(X_es_newfeats[NO_BILLING_FINAL], y_es_new)], verbose=False)
    preds_nb.append(m_nb.predict_proba(new_test[NO_BILLING_FINAL])[:, 1])
scored_nb = new_test[["dim_month", Y_COL]].copy()
scored_nb["score"] = np.mean(preds_nb, axis=0)
prec_nobilling = quick_precision_at_k(scored_nb, 40, Y_COL)
print(f"Without billing tenure features: {prec_nobilling:.2%}  (vs final model: {results['NEW on Sep25-May26']:.2%})")

# ---- 2. Leakage feature recheck on FINAL model ----
print("\n"+"="*60); print("[2] LEAKAGE FEATURE RECHECK (final model)"); print("="*60)
leakage_check = [f for f in ACCEPTED_NEW if f in ("months_since_last_cancel_request", "running_saas_cancel_request_count")]
print(f"Leakage-risk features still in final model: {leakage_check}")

scored_3["months_since_last_cancel_request"] = new_test["months_since_last_cancel_request"].values
scored_3["delta_12m"] = new_test["delta_product_active_students_vs_12m_ago"].values
recent = scored_3[scored_3["months_since_last_cancel_request"] <= 5]
growing = recent[recent["delta_12m"] > 0]
declining = recent[recent["delta_12m"] <= 0]
print(f"Recent cancel + growing usage: predicted={growing['score'].mean():.4f}, actual churn={growing[Y_COL].mean():.2%}")
print(f"Recent cancel + declining usage: predicted={declining['score'].mean():.4f}, actual churn={declining[Y_COL].mean():.2%}")

# ---- 3. Recent-cancellation exclusion on FINAL model ----
print("\n"+"="*60); print("[3] RECENT-CANCELLATION EXCLUSION (final model)"); print("="*60)
prec_excl = rank_filter_backfill_precision(scored_3, "months_since_last_cancel_request")
print(f"Without exclusion: {results['NEW on Sep25-May26']:.2%}")
print(f"With exclusion: {prec_excl:.2%}")
print(f"Difference: {prec_excl - results['NEW on Sep25-May26']:+.2%}")

# ---- 4. Fresh SHAP importance ranking ----
print("\n"+"="*60); print("[4] TOP FEATURES (final model)"); print("="*60)
import shap
explainer_f = shap.TreeExplainer(m)
sample_f = X_train_newfeats[ACCEPTED_NEW].sample(20000, random_state=1)
shap_f = explainer_f.shap_values(sample_f)
importance_f = pd.DataFrame({
    "feature": ACCEPTED_NEW,
    "mean_abs_shap": np.abs(shap_f).mean(axis=0)
}).sort_values("mean_abs_shap", ascending=False)
print(importance_f.head(15).to_string(index=False))

# ---- 5. Time stability (Sep-Dec 2025 vs Jan-May 2026) on FINAL model ----
print("\n"+"="*60); print("[5] TIME STABILITY (final model)"); print("="*60)
_ntc = new_test.copy().reset_index(drop=True)
_ntc["month_num"] = _ntc["dim_month"].dt.month
_ntc["score"] = scored_3["score"].values
for label, months in [("Sep-Dec 2025", [9,10,11,12]), ("Jan-May 2026", [1,2,3,4,5])]:
    sub = _ntc[_ntc["month_num"].isin(months)]
    p = quick_precision_at_k(sub[["dim_month", Y_COL, "score"]], 40, Y_COL)
    r = quick_recall_at_k(sub[["dim_month", Y_COL, "score"]], 40, Y_COL)
    print(f"  {label}: precision={p:.2%}, recall={r:.2%}")

print("\n" + "="*60)
print("ALL CONFIRMATIONS COMPLETE")
print("="*60)

In [ ]:
print(f"OLD on holdout_time — recall@40: {quick_recall_at_k(scored_1, 40, Y_COL):.2%}")

In [ ]:
scored_rhs_ht = cell("holdout_time")[["dim_month", Y_COL]].copy()
scored_rhs_ht["score"] = -cell("holdout_time")["product_score"]

print(f"RHS on holdout_time — precision@40: {quick_precision_at_k(scored_rhs_ht, 40, Y_COL):.2%}")
print(f"RHS on holdout_time — recall@40: {quick_recall_at_k(scored_rhs_ht, 40, Y_COL):.2%}")
print(f"RHS on holdout_time — precision@1840: {quick_precision_at_k(scored_rhs_ht, 1840, Y_COL):.2%}")
print(f"RHS on holdout_time — recall@1840: {quick_recall_at_k(scored_rhs_ht, 1840, Y_COL):.2%}")
print(f"OLD on holdout_time — precision@1840: {quick_precision_at_k(scored_1, 1840, Y_COL):.2%}")
print(f"OLD on holdout_time — recall@1840: {quick_recall_at_k(scored_1, 1840, Y_COL):.2%}")

In [ ]:
# Quick, illustrative number for the doc
avg_positives_per_month = new_test.groupby("dim_month")[Y_COL].sum().mean()
print(f"Average actual churners per month in this window: {avg_positives_per_month:.0f}")
print(f"At recall=1.28%, that's roughly {avg_positives_per_month * 0.0128:.1f} caught per month out of ~40 slots")

In [ ]:
print(f"RHS recall@40 = 0.42% -> roughly {avg_positives_per_month * 0.0042:.1f} caught per month")
print(f"V3.0 (retrained) recall@40 = 1.28% -> roughly {avg_positives_per_month * 0.0128:.1f} caught per month")

In [ ]:
import shap

# Use the final, retrained model (last one trained in your Part A / final loop)
explainer_final = shap.TreeExplainer(m)

# Pull 5 real schools from the genuinely held-out test window
sample_for_explain = new_test.sample(5, random_state=1).reset_index(drop=True)
shap_vals_explain = explainer_final.shap_values(sample_for_explain[ACCEPTED_NEW])

# A starter mapping from raw feature names to plain language
FEATURE_LABELS = {
    "delta_product_active_students_vs_12m_ago": "enrollment is down compared to a year ago",
    "delta_product_active_students_vs_t3m": "enrollment is down compared to their recent 3-month average",
    "num_students_with_billing_invoice_created": "fewer families have a billing invoice created",
    "product_active_students": "overall active student count",
    "num_billing_invoices_due": "billing invoices due",
    "num_billing_invoices_created": "billing invoices created",
    "trailing_3m_students_with_payment_method_added": "families adding a payment method has slowed",
    "months_since_first_processed_dollars_local": "billing tenure",
    "billing_net_gpv_local": "billing volume",
    "delta_any_messaging_vs_t3m": "messaging activity has changed",
    "running_saas_cancel_request_count": "history of prior cancellation requests",
    "months_since_last_cancel_request": "recency of last cancellation request",
}

def generate_talking_points(row_idx, top_n=3):
    contributions = list(zip(ACCEPTED_NEW, shap_vals_explain[row_idx]))
    contributions.sort(key=lambda x: x[1], reverse=True)
    top_positive = [c for c in contributions if c[1] > 0][:top_n]
    return [(FEATURE_LABELS.get(f, f), v) for f, v in top_positive]

for i in range(len(sample_for_explain)):
    school_id = sample_for_explain.iloc[i]["school_id"]
    score = m.predict_proba(sample_for_explain[ACCEPTED_NEW].iloc[[i]])[0][1]
    actual = sample_for_explain.iloc[i][Y_COL]
    print(f"School {school_id} | predicted risk: {score:.1%} | actual: {'churned' if actual==1 else 'did not churn'}")
    for label, val in generate_talking_points(i):
        print(f"   - {label} (contribution: {val:.3f})")
    print()

In [ ]:
churners_in_sample = new_test[new_test[Y_COL] == 1].sample(2, random_state=1).reset_index(drop=True)
shap_vals_churner = explainer_final.shap_values(churners_in_sample[ACCEPTED_NEW])

for i in range(len(churners_in_sample)):
    school_id = churners_in_sample.iloc[i]["school_id"]
    score = m.predict_proba(churners_in_sample[ACCEPTED_NEW].iloc[[i]])[0][1]
    print(f"School {school_id} | predicted risk: {score:.1%} | actual: churned")
    contributions = list(zip(ACCEPTED_NEW, shap_vals_churner[i]))
    contributions.sort(key=lambda x: x[1], reverse=True)
    for feat, val in [c for c in contributions if c[1] > 0][:3]:
        print(f"   - {FEATURE_LABELS.get(feat, feat)} (contribution: {val:.3f})")
    print()

In [ ]:
def humanize_feature_name(feat):
    overrides = {
        "product_active_students": "overall active student count",
        "billing_net_gpv_local": "billing volume",
        "months_since_last_cancel_request": "recency of last cancellation request",
        "running_saas_cancel_request_count": "history of prior cancellation requests",
        "months_since_first_processed_dollars_local": "billing tenure",
    }
    if feat in overrides:
        return overrides[feat]

    name = feat
    suffix = ""
    if name.endswith("_vs_12m_ago"):
        suffix = "compared to a year ago"
        name = name[: -len("_vs_12m_ago")]
    elif name.endswith("_vs_t3m"):
        suffix = "compared to the recent 3-month average"
        name = name[: -len("_vs_t3m")]

    prefix = ""
    if name.startswith("delta_"):
        prefix = "change in "
        name = name[len("delta_"):]
    elif name.startswith("trailing_3m_"):
        prefix = "recent 3-month trend in "
        name = name[len("trailing_3m_"):]
    elif name.startswith("num_"):
        prefix = "number of "
        name = name[len("num_"):]
    elif name.startswith("any_"):
        prefix = "whether the school used "
        name = name[len("any_"):]
    elif name.startswith("is_"):
        prefix = "whether "
        name = name[len("is_"):]
    elif name.startswith("customer_segment__"):
        prefix = "customer segment: "
        name = name[len("customer_segment__"):]
    elif name.startswith("reason_school_"):
        prefix = "flag: "
        name = name[len("reason_school_"):]

    readable = name.replace("_", " ")
    label = f"{prefix}{readable}".strip()
    return f"{label} {suffix}".strip() if suffix else label

for f in ACCEPTED_NEW:
    print(f"{f:55} -> {humanize_feature_name(f)}")

In [ ]:
# Pull a bigger, more varied sample of actual churners from the test set
churners_sample = new_test[new_test[Y_COL] == 1].sample(10, random_state=42).reset_index(drop=True)
shap_vals_wide = explainer_final.shap_values(churners_sample[ACCEPTED_NEW])

for i in range(len(churners_sample)):
    school_id = churners_sample.iloc[i]["school_id"]
    score = m.predict_proba(churners_sample[ACCEPTED_NEW].iloc[[i]])[0][1]
    contributions = list(zip(ACCEPTED_NEW, shap_vals_wide[i]))
    contributions.sort(key=lambda x: x[1], reverse=True)
    top3 = [c for c in contributions if c[1] > 0][:3]
    labels = [humanize_feature_name(f) for f, v in top3]
    print(f"School {school_id} | risk: {score:.1%}")
    print(f"   {labels}")
    print()

# Also check: across all 10, how often does each signal appear in someone's top 3
from collections import Counter
all_top_signals = []
for i in range(len(churners_sample)):
    contributions = list(zip(ACCEPTED_NEW, shap_vals_wide[i]))
    contributions.sort(key=lambda x: x[1], reverse=True)
    top3 = [c[0] for c in contributions if c[1] > 0][:3]
    all_top_signals.extend(top3)

print("How often each signal appears across these 10 schools' top-3 reasons:")
for feat, count in Counter(all_top_signals).most_common():
    print(f"  {humanize_feature_name(feat):50} {count}/10 schools")

In [ ]:
def get_base_concept(feat):
    name = feat
    if name.startswith("trailing_3m_"):
        name = name[len("trailing_3m_"):]
    elif name.startswith("delta_"):
        name = name[len("delta_"):]
        if name.endswith("_vs_t3m"):
            name = name[:-len("_vs_t3m")]
        elif name.endswith("_vs_12m_ago"):
            name = name[:-len("_vs_12m_ago")]
    return name

base_concepts_177 = set(get_base_concept(f) for f in MODEL_COLS)
print(f"177 candidate columns -> {len(base_concepts_177)} distinct base concepts")

base_concepts_84 = set(get_base_concept(f) for f in ACCEPTED_NEW)
print(f"84 kept signals -> {len(base_concepts_84)} distinct base concepts")

In [ ]:
def rebuild_rhs_trend(frame):
    frame = frame.sort_values(["school_id", "dim_month"]).copy()
    frame["trailing_3m_product_score"] = (
        frame.groupby("school_id")["product_score"]
        .transform(lambda s: s.shift(1).rolling(3, min_periods=3).mean())
    )
    frame["delta_product_score_vs_t3m"] = np.where(
        frame["trailing_3m_product_score"] > 0,
        frame["product_score"] / frame["trailing_3m_product_score"] - 1,
        np.nan
    )
    return frame

rhs_check = rebuild_rhs_trend(new_test[["school_id", "dim_month", "product_score", Y_COL]])
rhs_check["flagged"] = rhs_check["delta_product_score_vs_t3m"] < -0.20

monthly = []
for month, g in rhs_check.groupby("dim_month"):
    flagged = g[g["flagged"] == True]
    if len(flagged) == 0:
        continue
    monthly.append({
        "month": month,
        "flagged": len(flagged),
        "precision": flagged[Y_COL].sum() / len(flagged),
        "recall": flagged[Y_COL].sum() / g[Y_COL].sum() if g[Y_COL].sum() else np.nan,
    })

monthly_df = pd.DataFrame(monthly)
print(monthly_df)
print(f"\nAverage flagged/month: {monthly_df['flagged'].mean():.0f}")
print(f"Average precision: {monthly_df['precision'].mean():.2%}")
print(f"Average recall: {monthly_df['recall'].mean():.2%}")

In [ ]:
# Step 1: split the schools used in training into two groups, same seed logic as before
_train_schools_full = np.array(sorted(new_train["school_id"].unique()))
_perm2 = np.random.default_rng(42).permutation(len(_train_schools_full))
_n_holdout_schools = int(round(0.30 * len(_train_schools_full)))
_holdout_school_group = set(_train_schools_full[_perm2[:_n_holdout_schools]])
_seen_school_group = set(_train_schools_full[_perm2[_n_holdout_schools:]])

# Step 2: rebuild training to ONLY use the "seen" school group (so the holdout group is genuinely never trained on)
new_train_seen_only = new_train[new_train["school_id"].isin(_seen_school_group)]

print(f"Schools used for training: {len(_seen_school_group):,}")
print(f"Schools held out entirely (never trained on): {len(_holdout_school_group):,}")
print(f"Training rows after holding out a school group: {len(new_train_seen_only):,} (was {len(new_train):,})")

# Step 3: retrain the model on this smaller, school-restricted set, same config as the final model
preds_schoolsplit = []
for seed in [1, 7, 42, 99]:
    m_ss = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    m_ss.fit(new_train_seen_only[ACCEPTED_NEW], new_train_seen_only[Y_COL],
              eval_set=[(new_train_seen_only[ACCEPTED_NEW].sample(frac=0.1, random_state=1),
                         new_train_seen_only[Y_COL].sample(frac=0.1, random_state=1))], verbose=False)
    preds_schoolsplit.append(m_ss.predict_proba(new_test[ACCEPTED_NEW])[:, 1])

scored_ss = new_test[["school_id", "dim_month", Y_COL]].copy()
scored_ss["score"] = np.mean(preds_schoolsplit, axis=0)

# Step 4: split the TEST results by whether the school was held out or seen during training
scored_ss_seen = scored_ss[scored_ss["school_id"].isin(_seen_school_group)]
scored_ss_holdout = scored_ss[scored_ss["school_id"].isin(_holdout_school_group)]

prec_seen = quick_precision_at_k(scored_ss_seen, 40, Y_COL)
prec_holdout = quick_precision_at_k(scored_ss_holdout, 40, Y_COL)

print(f"\nRetrained model (2024+2025), tested on Sep2025-May2026:")
print(f"  Schools seen during training, new time period only: {prec_seen:.2%}")
print(f"  Schools NEVER seen during training, new time AND new schools: {prec_holdout:.2%}")
print(f"  Gap: {prec_seen - prec_holdout:.2%} points")

In [ ]:
def quick_recall_at_k(scored_df, k, label_col):
    recalls = []
    for month, g in scored_df.groupby("dim_month"):
        top_k = g.sort_values("score", ascending=False).head(k)
        total = g[label_col].sum()
        if total > 0:
            recalls.append(top_k[label_col].sum() / total)
    return np.mean(recalls)

print(f"{'weight':<8}{'precision@40':>14}{'recall@40':>12}")
for weight in [2, 4, 5, 6, 8, 10, 15, 20, 29]:
    preds_w = []
    for seed in [1, 7, 42, 99]:
        m_w = XGBClassifier(
            objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
            max_depth=3, learning_rate=0.06, min_child_weight=15,
            subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=seed, n_jobs=-1, missing=np.nan,
            n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=weight,
        )
        m_w.fit(new_train[ACCEPTED_NEW], new_train[Y_COL],
                eval_set=[(new_train[ACCEPTED_NEW].sample(frac=0.1, random_state=1),
                           new_train[Y_COL].sample(frac=0.1, random_state=1))], verbose=False)
        preds_w.append(m_w.predict_proba(new_test[ACCEPTED_NEW])[:, 1])
    scored_w = new_test[["dim_month", Y_COL]].copy()
    scored_w["score"] = np.mean(preds_w, axis=0)
    p = quick_precision_at_k(scored_w, 40, Y_COL)
    r = quick_recall_at_k(scored_w, 40, Y_COL)
    print(f"{weight:<8}{p:>13.2%}{r:>12.2%}")

In [ ]:
def quick_recall_at_k(scored_df, k, label_col):
    recalls = []
    for month, g in scored_df.groupby("dim_month"):
        top_k = g.sort_values("score", ascending=False).head(k)
        total = g[label_col].sum()
        if total > 0:
            recalls.append(top_k[label_col].sum() / total)
    return np.mean(recalls)

print(f"{'weight':<8}{'precision@40':>14}{'recall@40':>12}")
for weight in [20, 29]:
    preds_w = []
    for seed in [1, 7, 42, 99]:
        m_w = XGBClassifier(
            objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
            max_depth=3, learning_rate=0.06, min_child_weight=15,
            subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=seed, n_jobs=-1, missing=np.nan,
            n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=weight,
        )
        m_w.fit(new_train[ACCEPTED_NEW], new_train[Y_COL],
                eval_set=[(new_train[ACCEPTED_NEW].sample(frac=0.1, random_state=1),
                           new_train[Y_COL].sample(frac=0.1, random_state=1))], verbose=False)
        preds_w.append(m_w.predict_proba(new_test[ACCEPTED_NEW])[:, 1])
    scored_w = new_test[["dim_month", Y_COL]].copy()
    scored_w["score"] = np.mean(preds_w, axis=0)
    p = quick_precision_at_k(scored_w, 40, Y_COL)
    r = quick_recall_at_k(scored_w, 40, Y_COL)
    print(f"{weight:<8}{p:>13.2%}{r:>12.2%}")

In [ ]:
# Step 1: find the most recent available month in our data, to use as the "current" population
print("Most recent month in enc:", enc["dim_month"].max())

# Step 2: check if a school name field exists anywhere we already have loaded
print("\nColumns in readable_with_program that might be a name:")
print([c for c in readable_with_program.columns if "name" in c.lower()])

# Step 3: check if customers_catalog (from mart_school_customers) has a name field
print("\nColumns in customers_catalog that might be a name:")
print([c for c in customers_catalog["column_name"] if "name" in c.lower()])

In [ ]:
# Check MRR-related columns too, while we're at it
print("Columns in customers_catalog that might be MRR/revenue related:")
print([c for c in customers_catalog["column_name"] if any(k in c.lower() for k in ["mrr", "revenue", "arr"])])

In [ ]:
# Check if these two fields behave the way "monthly recurring" should:
# stable and similar for the same school across nearby months, not wildly spiky
name_revenue_lookup = pd.read_parquet(RAW_DIR / "name_revenue_lookup.parquet")
print(f"loaded {len(name_revenue_lookup):,} rows")

In [ ]:
# Step 1: score the most recent available month using the final model
current_month_data = enc[enc["dim_month"] == "2026-05-01"].copy()
print(f"Schools in most recent month: {len(current_month_data):,}")

current_month_data["risk_score"] = m.predict_proba(current_month_data[ACCEPTED_NEW])[:, 1]

# Step 2: attach name and revenue
current_month_data = current_month_data.merge(
    name_revenue_lookup[["school_id", "parse_name", "prior_3_month_billing_net_revenue"]],
    on="school_id", how="left"
)

# Step 3: rank by risk, take a broader list (not just 40, so we have room to filter by revenue)
top_risk = current_month_data.sort_values("risk_score", ascending=False).head(200)

print(f"\nOf the top 200 highest-risk schools:")
print(f"  Have any revenue at all: {(top_risk['prior_3_month_billing_net_revenue'] > 0).sum()}")
print(f"  Have zero revenue: {(top_risk['prior_3_month_billing_net_revenue'] == 0).sum()}")

# Step 4: within that top-200 risk list, sort by revenue to surface the highest-value ones
top_risk_by_value = top_risk.sort_values("prior_3_month_billing_net_revenue", ascending=False)

print("\nTop 20, highest-risk schools ranked by revenue within that group:")
print(top_risk_by_value[["school_id", "parse_name", "risk_score", "prior_3_month_billing_net_revenue"]].head(20).to_string(index=False))

6- MONTH

In [ ]:
print(list(RAW_DIR.glob("churn_v1_*_h6*.parquet")))

In [ ]:
import json

ENCODED_H6 = RAW_DIR / "churn_v1_is_reportable_request_202401_202602_h6_encoded.parquet"
ENCODING_H6 = RAW_DIR / "churn_v1_is_reportable_request_202401_202602_h6_encoding.json"

enc_h6 = pd.read_parquet(ENCODED_H6)
with open(ENCODING_H6) as f:
    schema_h6 = json.load(f)

MODEL_COLS_H6 = schema_h6["model_cols"]
print(f"loaded {len(enc_h6):,} rows, {len(MODEL_COLS_H6)} model columns")
print(f"months: {enc_h6['dim_month'].min()} to {enc_h6['dim_month'].max()}")

In [ ]:
print('enc_h6' in dir())
print('MODEL_COLS_H6' in dir())

In [ ]:
enc_h6["dim_month"] = pd.to_datetime(enc_h6["dim_month"])

TRAIN_CUTOFF_H6 = pd.Timestamp("2025-09-01")
TEST_START_H6 = pd.Timestamp("2025-09-01")

new_train_h6 = enc_h6[enc_h6["dim_month"] < TRAIN_CUTOFF_H6]
new_test_h6 = enc_h6[enc_h6["dim_month"] >= TEST_START_H6]

print(f"New train (h6): {len(new_train_h6):,} rows, {new_train_h6['dim_month'].min()} to {new_train_h6['dim_month'].max()}")
print(f"New test (h6): {len(new_test_h6):,} rows, {new_test_h6['dim_month'].min()} to {new_test_h6['dim_month'].max()}")
print(f"Train positive rate: {new_train_h6[Y_COL].mean():.2%}")
print(f"Test positive rate: {new_test_h6[Y_COL].mean():.2%}")

In [ ]:
from BorutaShap import BorutaShap
import time

BORUTA_N_TRIALS_H6 = 250
BORUTA_SEED_H6 = 42

def boruta_model_h6(scale_pos_weight):
    return XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, min_child_weight=20, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=42, n_jobs=-1, missing=np.nan,
        n_estimators=200, scale_pos_weight=scale_pos_weight,
    )

X_boruta_h6 = new_train_h6[MODEL_COLS_H6]
y_boruta_h6 = new_train_h6[Y_COL].astype("int8")

_neg_h6 = int((y_boruta_h6 == 0).sum())
_pos_h6 = int((y_boruta_h6 == 1).sum())
scale_pos_weight_boruta_h6 = _neg_h6 / max(_pos_h6, 1)

print(f"Running BorutaSHAP on h6 combined training set: {len(X_boruta_h6):,} rows, "
      f"{int(y_boruta_h6.sum()):,} positives, scale_pos_weight={scale_pos_weight_boruta_h6:.1f}")

selector_h6 = BorutaShap(
    model=boruta_model_h6(scale_pos_weight_boruta_h6),
    importance_measure="shap",
    classification=True,
    percentile=100,
    pvalue=0.05,
)

_t0 = time.perf_counter()
selector_h6.fit(
    X=X_boruta_h6, y=y_boruta_h6,
    n_trials=BORUTA_N_TRIALS_H6, random_state=BORUTA_SEED_H6,
    sample=False, train_or_test="train", normalize=True, verbose=False,
)
_elapsed_h6 = time.perf_counter() - _t0

ACCEPTED_H6 = sorted(selector_h6.accepted)
TENTATIVE_H6 = sorted(map(str, selector_h6.tentative))
REJECTED_H6 = sorted(selector_h6.rejected)

print(f"\nFinished in {_elapsed_h6:.1f}s ({_elapsed_h6/60:.1f} min)")
print(f"Accepted: {len(ACCEPTED_H6)}, Tentative: {len(TENTATIVE_H6)}, Rejected: {len(REJECTED_H6)}")

In [ ]:
# The 2024-only training subset, same schools' data, just restricted to 2024 months
old_train_h6 = new_train_h6[new_train_h6["dim_month"] < "2025-01-01"]

print(f"Old train (2024 only, h6): {len(old_train_h6):,} rows, "
      f"{old_train_h6['dim_month'].min()} to {old_train_h6['dim_month'].max()}")
print(f"Positive rate: {old_train_h6[Y_COL].mean():.2%}")

In [ ]:
X_boruta_old_h6 = old_train_h6[MODEL_COLS_H6]
y_boruta_old_h6 = old_train_h6[Y_COL].astype("int8")

_neg_old_h6 = int((y_boruta_old_h6 == 0).sum())
_pos_old_h6 = int((y_boruta_old_h6 == 1).sum())
scale_pos_weight_boruta_old_h6 = _neg_old_h6 / max(_pos_old_h6, 1)

print(f"Running BorutaSHAP on h6, 2024-only: {len(X_boruta_old_h6):,} rows, "
      f"{int(y_boruta_old_h6.sum()):,} positives, scale_pos_weight={scale_pos_weight_boruta_old_h6:.1f}")

selector_old_h6 = BorutaShap(
    model=boruta_model_h6(scale_pos_weight_boruta_old_h6),
    importance_measure="shap",
    classification=True,
    percentile=100,
    pvalue=0.05,
)

_t0 = time.perf_counter()
selector_old_h6.fit(
    X=X_boruta_old_h6, y=y_boruta_old_h6,
    n_trials=250, random_state=42,
    sample=False, train_or_test="train", normalize=True, verbose=False,
)
_elapsed_old_h6 = time.perf_counter() - _t0

ACCEPTED_OLD_H6 = sorted(selector_old_h6.accepted)
print(f"\nFinished in {_elapsed_old_h6:.1f}s ({_elapsed_old_h6/60:.1f} min)")
print(f"Accepted: {len(ACCEPTED_OLD_H6)}, Tentative: {len(selector_old_h6.tentative)}, Rejected: {len(selector_old_h6.rejected)}")

In [ ]:
print(f"{'weight':<8}{'precision@40':>14}{'recall@40':>12}")
for weight in [4, 8, 12, 17]:
    preds_w = []
    for seed in [1, 42]:
        m_w = XGBClassifier(
            objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
            max_depth=3, learning_rate=0.06, min_child_weight=15,
            subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=seed, n_jobs=-1, missing=np.nan,
            n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=weight,
        )
        _es_sample = new_train_h6.sample(frac=0.1, random_state=1)
        m_w.fit(new_train_h6[ACCEPTED_H6], new_train_h6[Y_COL],
                eval_set=[(_es_sample[ACCEPTED_H6], _es_sample[Y_COL])], verbose=False)
        preds_w.append(m_w.predict_proba(new_test_h6[ACCEPTED_H6])[:, 1])
    scored_w = new_test_h6[["dim_month", Y_COL]].copy()
    scored_w["score"] = np.mean(preds_w, axis=0)
    p = quick_precision_at_k(scored_w, 40, Y_COL)
    r = quick_recall_at_k(scored_w, 40, Y_COL)
    print(f"{weight:<8}{p:>13.2%}{r:>12.2%}")

In [ ]:
print(f"{'weight':<8}{'precision@40':>14}{'recall@40':>12}")
for weight in [6, 8, 10]:
    preds_w = []
    for seed in [1, 7, 42, 99]:
        m_w = XGBClassifier(
            objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
            max_depth=3, learning_rate=0.06, min_child_weight=15,
            subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=seed, n_jobs=-1, missing=np.nan,
            n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=weight,
        )
        _es_sample = new_train_h6.sample(frac=0.1, random_state=1)
        m_w.fit(new_train_h6[ACCEPTED_H6], new_train_h6[Y_COL],
                eval_set=[(_es_sample[ACCEPTED_H6], _es_sample[Y_COL])], verbose=False)
        preds_w.append(m_w.predict_proba(new_test_h6[ACCEPTED_H6])[:, 1])
    scored_w = new_test_h6[["dim_month", Y_COL]].copy()
    scored_w["score"] = np.mean(preds_w, axis=0)
    p = quick_precision_at_k(scored_w, 40, Y_COL)
    r = quick_recall_at_k(scored_w, 40, Y_COL)
    print(f"{weight:<8}{p:>13.2%}{r:>12.2%}")

In [ ]:
print(readable_with_program[["any_messaging", "trailing_3m_any_messaging"]].head(10))
print(readable_with_program["messaging_events_per_active_student"].describe())

In [ ]:
# Check every base feature: is it genuinely binary (only 0/1 values) or quantitative?
base_features_to_check = [c for c in readable_with_program.columns 
                          if not c.startswith(("trailing_3m_", "delta_"))
                          and c not in ("school_id", "dim_month", "label", "split", "school_group",
                                        "customer_segment", "school_country", "monthly_billing_status_local")]

results = []
for col in base_features_to_check:
    try:
        vals = readable_with_program[col].dropna().unique()
        is_binary = set(vals).issubset({0, 1, 0.0, 1.0})
        results.append({"feature": col, "type": "binary" if is_binary else "quantitative", 
                         "n_unique": len(vals)})
    except TypeError:
        continue

results_df = pd.DataFrame(results).sort_values(["type", "feature"])
print(results_df.to_string(index=False))

In [ ]:
def get_base_feature(feat):
    name = feat
    if name.startswith("trailing_3m_"):
        name = name[len("trailing_3m_"):]
    elif name.startswith("delta_"):
        name = name[len("delta_"):]
        if name.endswith("_vs_t3m"):
            name = name[:-len("_vs_t3m")]
        elif name.endswith("_vs_12m_ago"):
            name = name[:-len("_vs_12m_ago")]
    return name

binary_base = set(results_df[results_df["type"] == "binary"]["feature"])

used_binary = []
used_quantitative = []
for f in ACCEPTED_NEW:
    base = get_base_feature(f)
    if base in binary_base:
        used_binary.append(f)
    else:
        used_quantitative.append(f)

print(f"Of the {len(ACCEPTED_NEW)} features the final model actually uses:")
print(f"  Derived from a binary base feature: {len(used_binary)}")
print(f"  Derived from a quantitative base feature: {len(used_quantitative)}")

print("\nActual binary-derived features in the model:")
for f in used_binary:
    print(f"  {f}")

In [ ]:
billing_keywords = ["billing", "invoice", "processed_dollars", "autopay", "gpv"]

billing_used = [f for f in ACCEPTED_NEW if any(k in f.lower() for k in billing_keywords)]

print(f"Billing-related features in the final model: {len(billing_used)}")
for f in billing_used:
    base = get_base_feature(f)
    kind = "binary" if base in binary_base else "quantitative"
    print(f"  {f:<55} ({kind})")

In [ ]:
def rebuild_rhs_trend(frame):
    frame = frame.sort_values(["school_id", "dim_month"]).copy()
    frame["trailing_3m_product_score"] = (
        frame.groupby("school_id")["product_score"]
        .transform(lambda s: s.shift(1).rolling(3, min_periods=3).mean())
    )
    frame["delta_product_score_vs_t3m"] = np.where(
        frame["trailing_3m_product_score"] > 0,
        frame["product_score"] / frame["trailing_3m_product_score"] - 1,
        np.nan
    )
    return frame

rhs_corrected = rebuild_rhs_trend(new_test[["school_id", "dim_month", "product_score", Y_COL]])

# Corrected rule: drop > 25% AND current score is Low or Medium (below 18, not High)
rhs_corrected["flagged"] = (
    (rhs_corrected["delta_product_score_vs_t3m"] < -0.25) &
    (rhs_corrected["product_score"] < 18)
)

monthly = []
for month, g in rhs_corrected.groupby("dim_month"):
    flagged = g[g["flagged"] == True]
    if len(flagged) == 0:
        continue
    monthly.append({
        "month": month,
        "flagged": len(flagged),
        "precision": flagged[Y_COL].sum() / len(flagged),
        "recall": flagged[Y_COL].sum() / g[Y_COL].sum() if g[Y_COL].sum() else np.nan,
    })

monthly_df = pd.DataFrame(monthly)
print(monthly_df)
print(f"\nAverage flagged/month: {monthly_df['flagged'].mean():.0f}")
print(f"Average precision: {monthly_df['precision'].mean():.2%}")
print(f"Average recall: {monthly_df['recall'].mean():.2%}")

In [ ]:
for threshold in [0.20, 0.25, 0.50]:
    rhs_check = rhs_corrected.copy()
    rhs_check["flagged"] = (
        (rhs_check["delta_product_score_vs_t3m"] < -threshold) &
        (rhs_check["product_score"] < 18)
    )
    
    monthly = []
    for month, g in rhs_check.groupby("dim_month"):
        flagged = g[g["flagged"] == True]
        if len(flagged) == 0:
            continue
        monthly.append({
            "flagged": len(flagged),
            "precision": flagged[Y_COL].sum() / len(flagged),
            "recall": flagged[Y_COL].sum() / g[Y_COL].sum() if g[Y_COL].sum() else np.nan,
        })
    monthly_df = pd.DataFrame(monthly)
    print(f"threshold={threshold:.0%}: avg flagged/month={monthly_df['flagged'].mean():.0f}, "
          f"precision={monthly_df['precision'].mean():.2%}, recall={monthly_df['recall'].mean():.2%}")

In [ ]:
def rhs_ranked_at_k(frame, k, tier_cutoff=18):
    # Only consider schools currently in Low or Medium tier
    eligible = frame[frame["product_score"] < tier_cutoff].copy()
    
    monthly = []
    for month, g in eligible.groupby("dim_month"):
        # Rank by size of drop, biggest drop first
        top_k = g.sort_values("delta_product_score_vs_t3m", ascending=True).head(k)
        total_positives = frame[frame["dim_month"] == month][Y_COL].sum()
        monthly.append({
            "precision": top_k[Y_COL].sum() / len(top_k) if len(top_k) else np.nan,
            "recall": top_k[Y_COL].sum() / total_positives if total_positives else np.nan,
        })
    return pd.DataFrame(monthly)

for k in [40, 1840]:
    result = rhs_ranked_at_k(rhs_corrected, k)
    print(f"RHS (Low/Medium tier, ranked by drop) @{k}: "
          f"precision={result['precision'].mean():.2%}, recall={result['recall'].mean():.2%}")

In [ ]:
priority_candidates = [c for c in customers_catalog["column_name"] if any(
    k in c.lower() for k in ["priority", "rank", "order", "sequence"]
)]
print("Possible priority/rank fields in mart_school_customers:", priority_candidates)

In [ ]:
preds_old_h6 = []
for seed in [1, 7, 42, 99]:
    m_old_h6 = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=8,
    )
    _es_sample = old_train_h6.sample(frac=0.1, random_state=1)
    m_old_h6.fit(old_train_h6[ACCEPTED_OLD_H6], old_train_h6[Y_COL],
                 eval_set=[(_es_sample[ACCEPTED_OLD_H6], _es_sample[Y_COL])], verbose=False)
    preds_old_h6.append(m_old_h6.predict_proba(new_test_h6[ACCEPTED_OLD_H6])[:, 1])

scored_old_h6 = new_test_h6[["dim_month", Y_COL]].copy()
scored_old_h6["score"] = np.mean(preds_old_h6, axis=0)

preds_new_h6 = []
for seed in [1, 7, 42, 99]:
    m_new_h6 = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=8,
    )
    _es_sample2 = new_train_h6.sample(frac=0.1, random_state=1)
    m_new_h6.fit(new_train_h6[ACCEPTED_H6], new_train_h6[Y_COL],
                 eval_set=[(_es_sample2[ACCEPTED_H6], _es_sample2[Y_COL])], verbose=False)
    preds_new_h6.append(m_new_h6.predict_proba(new_test_h6[ACCEPTED_H6])[:, 1])

scored_new_h6 = new_test_h6[["dim_month", Y_COL]].copy()
scored_new_h6["score"] = np.mean(preds_new_h6, axis=0)

p_old = quick_precision_at_k(scored_old_h6, 40, Y_COL)
r_old = quick_recall_at_k(scored_old_h6, 40, Y_COL)
p_new = quick_precision_at_k(scored_new_h6, 40, Y_COL)
r_new = quick_recall_at_k(scored_new_h6, 40, Y_COL)

print(f"h6, 2024-only:    precision@40={p_old:.2%}, recall@40={r_old:.2%}")
print(f"h6, 2024+2025:    precision@40={p_new:.2%}, recall@40={r_new:.2%}")

In [ ]:
# Billing tenure features
print("Billing tenure features present:")
print("  h6:", [c for c in ACCEPTED_H6 if c in ("months_since_first_processed_dollars_local", "processed_dollars_rank_local")])

# Cancellation-history (leakage-risk) features
print("\nCancellation-history features present:")
print("  h6:", [c for c in ACCEPTED_H6 if c in ("months_since_last_cancel_request", "running_saas_cancel_request_count")])


In [ ]:
BILLING_TENURE_H6 = ["months_since_first_processed_dollars_local", "processed_dollars_rank_local"]
LEAKAGE_H6 = ["months_since_last_cancel_request", "running_saas_cancel_request_count"]

def run_ablation_h6(drop_cols, label):
    feat_list = [f for f in ACCEPTED_H6 if f not in drop_cols]
    preds = []
    for seed in [1, 7, 42, 99]:
        m = XGBClassifier(
            objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
            max_depth=3, learning_rate=0.06, min_child_weight=15,
            subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=seed, n_jobs=-1, missing=np.nan,
            n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=8,
        )
        _es = new_train_h6.sample(frac=0.1, random_state=1)
        m.fit(new_train_h6[feat_list], new_train_h6[Y_COL],
              eval_set=[(_es[feat_list], _es[Y_COL])], verbose=False)
        preds.append(m.predict_proba(new_test_h6[feat_list])[:, 1])
    scored = new_test_h6[["dim_month", Y_COL]].copy()
    scored["score"] = np.mean(preds, axis=0)
    p = quick_precision_at_k(scored, 40, Y_COL)
    print(f"{label}: precision@40 = {p:.2%}  (full model was 39.17%)")

run_ablation_h6(BILLING_TENURE_H6, "Without billing tenure features")
run_ablation_h6(LEAKAGE_H6, "Without leakage-risk features")

In [ ]:
def rebuild_rhs_trend(frame):
    frame = frame.sort_values(["school_id", "dim_month"]).copy()
    frame["trailing_3m_product_score"] = (
        frame.groupby("school_id")["product_score"]
        .transform(lambda s: s.shift(1).rolling(3, min_periods=3).mean())
    )
    frame["delta_product_score_vs_t3m"] = np.where(
        frame["trailing_3m_product_score"] > 0,
        frame["product_score"] / frame["trailing_3m_product_score"] - 1,
        np.nan
    )
    return frame

def rhs_ranked_at_k(frame, k, tier_cutoff=18):
    eligible = frame[frame["product_score"] < tier_cutoff].copy()
    monthly = []
    for month, g in eligible.groupby("dim_month"):
        top_k = g.sort_values("delta_product_score_vs_t3m", ascending=True).head(k)
        total_positives = frame[frame["dim_month"] == month][Y_COL].sum()
        monthly.append({
            "precision": top_k[Y_COL].sum() / len(top_k) if len(top_k) else np.nan,
            "recall": top_k[Y_COL].sum() / total_positives if total_positives else np.nan,
        })
    return pd.DataFrame(monthly)

# RHS on h6
rhs_h6 = rebuild_rhs_trend(new_test_h6[["school_id", "dim_month", "product_score", Y_COL]])
for k in [40, 1840]:
    result = rhs_ranked_at_k(rhs_h6, k)
    print(f"RHS on h6 @{k}: precision={result['precision'].mean():.2%}, recall={result['recall'].mean():.2%}")

In [ ]:
print(f"Total positives in your h3 train + test: {int(new_train[Y_COL].sum()) + int(new_test[Y_COL].sum())}")

In [ ]:
preds_corrected = []
for seed in [1, 7, 42, 99]:
    m_corrected = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es = new_train.sample(frac=0.1, random_state=1)
    m_corrected.fit(new_train[ACCEPTED_NEW], new_train[Y_COL],
                     eval_set=[(_es[ACCEPTED_NEW], _es[Y_COL])], verbose=False)
    preds_corrected.append(m_corrected.predict_proba(new_test[ACCEPTED_NEW])[:, 1])

scored_corrected = new_test[["dim_month", Y_COL]].copy()
scored_corrected["score"] = np.mean(preds_corrected, axis=0)
p_corrected = quick_precision_at_k(scored_corrected, 40, Y_COL)
r_corrected = quick_recall_at_k(scored_corrected, 40, Y_COL)

print(f"Corrected label: precision@40={p_corrected:.2%}, recall@40={r_corrected:.2%}")
print(f"Original (uncorrected): precision@40=26.9%, recall@40=1.3%")

In [ ]:
p_1840_corrected = quick_precision_at_k(scored_corrected, 1840, Y_COL)
r_1840_corrected = quick_recall_at_k(scored_corrected, 1840, Y_COL)

print(f"Corrected label @1,840: precision={p_1840_corrected:.2%}, recall={r_1840_corrected:.2%}")
print(f"Original (uncorrected) @1,840: precision=14.3%, recall=31.2%")

In [ ]:
# ============================================================
# STEP 1: Current model (2024-only), on corrected label
# ============================================================
old_train_corrected = new_train[new_train["dim_month"] < "2025-01-01"]

preds_old_corrected = []
for seed in [1, 7, 42, 99]:
    m_old_c = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es = old_train_corrected.sample(frac=0.1, random_state=1)
    m_old_c.fit(old_train_corrected[SELECTED_FEATURES], old_train_corrected[Y_COL],
                eval_set=[(_es[SELECTED_FEATURES], _es[Y_COL])], verbose=False)
    preds_old_corrected.append(m_old_c.predict_proba(new_test[SELECTED_FEATURES])[:, 1])

scored_old_corrected = new_test[["dim_month", Y_COL]].copy()
scored_old_corrected["score"] = np.mean(preds_old_corrected, axis=0)

p_old_40 = quick_precision_at_k(scored_old_corrected, 40, Y_COL)
r_old_40 = quick_recall_at_k(scored_old_corrected, 40, Y_COL)
p_old_1840 = quick_precision_at_k(scored_old_corrected, 1840, Y_COL)
r_old_1840 = quick_recall_at_k(scored_old_corrected, 1840, Y_COL)

print("STEP 1 - Current model (2024-only), corrected label")
print(f"  @40:    precision={p_old_40:.2%}, recall={r_old_40:.2%}")
print(f"  @1840:  precision={p_old_1840:.2%}, recall={r_old_1840:.2%}")

# ============================================================
# STEP 2: RHS, on corrected label (same method as original table: simple 20%, ranked by score)
# ============================================================
scored_rhs_corrected = new_test[["dim_month", Y_COL]].copy()
scored_rhs_corrected["score"] = -new_test["product_score"]

p_rhs_40 = quick_precision_at_k(scored_rhs_corrected, 40, Y_COL)
r_rhs_40 = quick_recall_at_k(scored_rhs_corrected, 40, Y_COL)
p_rhs_1840 = quick_precision_at_k(scored_rhs_corrected, 1840, Y_COL)
r_rhs_1840 = quick_recall_at_k(scored_rhs_corrected, 1840, Y_COL)

print("\nSTEP 2 - RHS, corrected label")
print(f"  @40:    precision={p_rhs_40:.2%}, recall={r_rhs_40:.2%}")
print(f"  @1840:  precision={p_rhs_1840:.2%}, recall={r_rhs_1840:.2%}")

# ============================================================
# STEP 3: Stability check (Sep-Dec 2025 vs Jan-May 2026), retrained model, corrected label
# ============================================================
_ntc = new_test.copy().reset_index(drop=True)
_ntc["month_num"] = _ntc["dim_month"].dt.month
_ntc["score"] = scored_corrected["score"].values

print("\nSTEP 3 - Stability check, corrected label")
for label, months in [("Sep-Dec 2025", [9,10,11,12]), ("Jan-May 2026", [1,2,3,4,5])]:
    sub = _ntc[_ntc["month_num"].isin(months)]
    p = quick_precision_at_k(sub[["dim_month", Y_COL, "score"]], 40, Y_COL)
    r = quick_recall_at_k(sub[["dim_month", Y_COL, "score"]], 40, Y_COL)
    print(f"  {label}: precision={p:.2%}, recall={r:.2%}")

# ============================================================
# STEP 4: Billing and leakage ablation, on corrected label
# ============================================================
BILLING_TENURE = ["months_since_first_processed_dollars_local", "processed_dollars_rank_local"]
LEAKAGE = ["months_since_last_cancel_request", "running_saas_cancel_request_count"]

def run_ablation(drop_cols, label):
    feat_list = [f for f in ACCEPTED_NEW if f not in drop_cols]
    preds = []
    for seed in [1, 7, 42, 99]:
        m = XGBClassifier(
            objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
            max_depth=3, learning_rate=0.06, min_child_weight=15,
            subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=seed, n_jobs=-1, missing=np.nan,
            n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
        )
        _es = new_train.sample(frac=0.1, random_state=1)
        m.fit(new_train[feat_list], new_train[Y_COL],
              eval_set=[(_es[feat_list], _es[Y_COL])], verbose=False)
        preds.append(m.predict_proba(new_test[feat_list])[:, 1])
    scored = new_test[["dim_month", Y_COL]].copy()
    scored["score"] = np.mean(preds, axis=0)
    p = quick_precision_at_k(scored, 40, Y_COL)
    print(f"  {label}: precision@40={p:.2%}  (full corrected model was {p_corrected:.2%})")

print("\nSTEP 4 - Billing/leakage ablation, corrected label")
run_ablation(BILLING_TENURE, "Without billing tenure features")
run_ablation(LEAKAGE, "Without leakage-risk features")

print("\n" + "="*60)
print("ALL STEPS COMPLETE - full corrected picture ready")
print("="*60)

In [ ]:
print("RERUN with different seeds, to check if this is real or noise")

# Step 1 rerun, current model, different seeds
preds_old_rerun = []
for seed in [3, 17, 55, 88]:
    m_old_r = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es = old_train_corrected.sample(frac=0.1, random_state=1)
    m_old_r.fit(old_train_corrected[SELECTED_FEATURES], old_train_corrected[Y_COL],
                eval_set=[(_es[SELECTED_FEATURES], _es[Y_COL])], verbose=False)
    preds_old_rerun.append(m_old_r.predict_proba(new_test[SELECTED_FEATURES])[:, 1])

scored_old_rerun = new_test[["dim_month", Y_COL]].copy()
scored_old_rerun["score"] = np.mean(preds_old_rerun, axis=0)
p_old_rerun = quick_precision_at_k(scored_old_rerun, 40, Y_COL)
r_old_rerun = quick_recall_at_k(scored_old_rerun, 40, Y_COL)

print(f"Current model, corrected label, DIFFERENT seeds: precision@40={p_old_rerun:.2%}, recall@40={r_old_rerun:.2%}")
print(f"First run (seeds 1,7,42,99) gave: 27.22%")
print(f"Original uncorrected: 19.7%")

In [ ]:
print(f"2024-only training rows: {len(old_train_corrected):,}, positives: {old_train_corrected[Y_COL].sum():,}")
print(f"Original (uncorrected) 2024-only had roughly: 299,740 rows, positives at 3.04% base rate")

In [ ]:
preds_model1_ht = []
for seed in [1, 7, 42, 99]:
    m_ht = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es_ht = X_train.sample(frac=0.1, random_state=1)
    m_ht.fit(X_train[SELECTED_FEATURES], y_train,
             eval_set=[(_es_ht[SELECTED_FEATURES], _es_ht[Y_COL] if hasattr(_es_ht, Y_COL) else y_train.loc[_es_ht.index])],
             verbose=False)
    preds_model1_ht.append(m_ht.predict_proba(holdout_time[SELECTED_FEATURES])[:, 1])

scored_model1_ht = holdout_time[["dim_month", Y_COL]].copy()
scored_model1_ht["score"] = np.mean(preds_model1_ht, axis=0)

p_ht_40 = quick_precision_at_k(scored_model1_ht, 40, Y_COL)
r_ht_40 = quick_recall_at_k(scored_model1_ht, 40, Y_COL)
p_ht_1840 = quick_precision_at_k(scored_model1_ht, 1840, Y_COL)
r_ht_1840 = quick_recall_at_k(scored_model1_ht, 1840, Y_COL)

print(f"Model1 on holdout_time, corrected: @40 precision={p_ht_40:.2%}, recall={r_ht_40:.2%}")
print(f"                                    @1840 precision={p_ht_1840:.2%}, recall={r_ht_1840:.2%}")

# RHS on holdout_time, corrected label
scored_rhs_ht = holdout_time[["dim_month", Y_COL]].copy()
scored_rhs_ht["score"] = -holdout_time["product_score"]

p_rhs_ht_40 = quick_precision_at_k(scored_rhs_ht, 40, Y_COL)
r_rhs_ht_40 = quick_recall_at_k(scored_rhs_ht, 40, Y_COL)
p_rhs_ht_1840 = quick_precision_at_k(scored_rhs_ht, 1840, Y_COL)
r_rhs_ht_1840 = quick_recall_at_k(scored_rhs_ht, 1840, Y_COL)

print(f"RHS on holdout_time, corrected: @40 precision={p_rhs_ht_40:.2%}, recall={r_rhs_ht_40:.2%}")
print(f"                                 @1840 precision={p_rhs_ht_1840:.2%}, recall={r_rhs_ht_1840:.2%}")

In [ ]:
preds_noearly_corrected = []
for seed in [1, 7, 42, 99]:
    m_ne = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, scale_pos_weight=4,  # NO early_stopping_rounds, matching the original variant
    )
    m_ne.fit(new_train[ACCEPTED_NEW], new_train[Y_COL], verbose=False)
    preds_noearly_corrected.append(m_ne.predict_proba(new_test[ACCEPTED_NEW])[:, 1])

_ntc2 = new_test.copy().reset_index(drop=True)
_ntc2["month_num"] = _ntc2["dim_month"].dt.month
_ntc2["score"] = np.mean(preds_noearly_corrected, axis=0)

for label, months in [("Sep-Dec 2025", [9,10,11,12]), ("Jan-May 2026", [1,2,3,4,5])]:
    sub = _ntc2[_ntc2["month_num"].isin(months)]
    p = quick_precision_at_k(sub[["dim_month", Y_COL, "score"]], 40, Y_COL)
    print(f"  {label}: precision@40={p:.2%}")

In [ ]:
print('holdout_time' in dir())

In [ ]:
decline = 33.12 - 28.50
relative_decline = decline / 33.12
print(f"Absolute decline: {decline:.2f} points")
print(f"Relative decline: {relative_decline:.1%}")
print(f"Compare: the earlier no-early-stopping variant declined from 32.5% to 27.5%, a {(32.5-27.5)/32.5:.1%} relative decline")

In [ ]:
# Confirm these are genuinely different prediction arrays, not the same one twice
print(np.allclose(_ntc2["score"].values, scored_corrected["score"].values))

In [ ]:
holdout_time = cell("holdout_time")
print(f"holdout_time rows: {len(holdout_time):,}, positive rate: {holdout_time[Y_COL].mean():.2%}")

In [ ]:
import json

ENCODED_H6_CORRECTED = RAW_DIR / "churn_v1_is_reportable_request_202401_202602_h6_encoded.parquet"
ENCODING_H6_CORRECTED = RAW_DIR / "churn_v1_is_reportable_request_202401_202602_h6_encoding.json"

enc_h6 = pd.read_parquet(ENCODED_H6_CORRECTED)
with open(ENCODING_H6_CORRECTED) as f:
    schema_h6 = json.load(f)

MODEL_COLS_H6 = schema_h6["model_cols"]
print(f"loaded {len(enc_h6):,} rows, {len(MODEL_COLS_H6)} model columns")

In [ ]:
enc_h6["dim_month"] = pd.to_datetime(enc_h6["dim_month"])

TRAIN_CUTOFF_H6 = pd.Timestamp("2025-09-01")
TEST_START_H6 = pd.Timestamp("2025-09-01")

new_train_h6 = enc_h6[enc_h6["dim_month"] < TRAIN_CUTOFF_H6]
new_test_h6 = enc_h6[enc_h6["dim_month"] >= TEST_START_H6]
old_train_h6 = new_train_h6[new_train_h6["dim_month"] < "2025-01-01"]

print(f"New train (h6, corrected): {len(new_train_h6):,} rows, positive rate: {new_train_h6[Y_COL].mean():.2%}")
print(f"New test (h6, corrected): {len(new_test_h6):,} rows, positive rate: {new_test_h6[Y_COL].mean():.2%}")
print(f"Old train (2024-only, h6, corrected): {len(old_train_h6):,} rows, positive rate: {old_train_h6[Y_COL].mean():.2%}")

In [ ]:
preds_old_h6_c = []
for seed in [1, 7, 42, 99]:
    m_old_h6_c = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=8,
    )
    _es = old_train_h6.sample(frac=0.1, random_state=1)
    m_old_h6_c.fit(old_train_h6[ACCEPTED_OLD_H6], old_train_h6[Y_COL],
                   eval_set=[(_es[ACCEPTED_OLD_H6], _es[Y_COL])], verbose=False)
    preds_old_h6_c.append(m_old_h6_c.predict_proba(new_test_h6[ACCEPTED_OLD_H6])[:, 1])

scored_old_h6_c = new_test_h6[["dim_month", Y_COL]].copy()
scored_old_h6_c["score"] = np.mean(preds_old_h6_c, axis=0)

preds_new_h6_c = []
for seed in [1, 7, 42, 99]:
    m_new_h6_c = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=8,
    )
    _es2 = new_train_h6.sample(frac=0.1, random_state=1)
    m_new_h6_c.fit(new_train_h6[ACCEPTED_H6], new_train_h6[Y_COL],
                   eval_set=[(_es2[ACCEPTED_H6], _es2[Y_COL])], verbose=False)
    preds_new_h6_c.append(m_new_h6_c.predict_proba(new_test_h6[ACCEPTED_H6])[:, 1])

scored_new_h6_c = new_test_h6[["dim_month", Y_COL]].copy()
scored_new_h6_c["score"] = np.mean(preds_new_h6_c, axis=0)

p_old_h6_40 = quick_precision_at_k(scored_old_h6_c, 40, Y_COL)
r_old_h6_40 = quick_recall_at_k(scored_old_h6_c, 40, Y_COL)
p_new_h6_40 = quick_precision_at_k(scored_new_h6_c, 40, Y_COL)
r_new_h6_40 = quick_recall_at_k(scored_new_h6_c, 40, Y_COL)

print(f"h6, 2024-only, corrected:    precision@40={p_old_h6_40:.2%}, recall@40={r_old_h6_40:.2%}")
print(f"h6, 2024+2025, corrected:    precision@40={p_new_h6_40:.2%}, recall@40={r_new_h6_40:.2%}")
print(f"\nOriginal (uncorrected) h6:   2024-only=23.33%/0.63%, 2024+2025=39.17%/1.06%")

In [ ]:
# Billing and leakage presence, h6 corrected
print("h6 billing tenure features:", [c for c in ACCEPTED_H6 if c in ("months_since_first_processed_dollars_local", "processed_dollars_rank_local")])
print("h6 leakage features:", [c for c in ACCEPTED_H6 if c in ("months_since_last_cancel_request", "running_saas_cancel_request_count")])

In [ ]:
# Billing ablation, h6 corrected
BILLING_TENURE_H6 = ["months_since_first_processed_dollars_local", "processed_dollars_rank_local"]
LEAKAGE_H6 = ["months_since_last_cancel_request", "running_saas_cancel_request_count"]

def run_ablation_h6_corrected(drop_cols, label):
    feat_list = [f for f in ACCEPTED_H6 if f not in drop_cols]
    preds = []
    for seed in [1, 7, 42, 99]:
        m = XGBClassifier(
            objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
            max_depth=3, learning_rate=0.06, min_child_weight=15,
            subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=seed, n_jobs=-1, missing=np.nan,
            n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=8,
        )
        _es = new_train_h6.sample(frac=0.1, random_state=1)
        m.fit(new_train_h6[feat_list], new_train_h6[Y_COL],
              eval_set=[(_es[feat_list], _es[Y_COL])], verbose=False)
        preds.append(m.predict_proba(new_test_h6[feat_list])[:, 1])
    scored = new_test_h6[["dim_month", Y_COL]].copy()
    scored["score"] = np.mean(preds, axis=0)
    p = quick_precision_at_k(scored, 40, Y_COL)
    print(f"  {label}: precision@40={p:.2%}  (full corrected model was 39.58%)")

run_ablation_h6_corrected(BILLING_TENURE_H6, "Without billing tenure features")
run_ablation_h6_corrected(LEAKAGE_H6, "Without leakage-risk features")

In [ ]:
# RHS comparison, h6 corrected
scored_rhs_h6_c = new_test_h6[["dim_month", Y_COL]].copy()
scored_rhs_h6_c["score"] = -new_test_h6["product_score"]

p_rhs_h6_40 = quick_precision_at_k(scored_rhs_h6_c, 40, Y_COL)
r_rhs_h6_40 = quick_recall_at_k(scored_rhs_h6_c, 40, Y_COL)
p_rhs_h6_1840 = quick_precision_at_k(scored_rhs_h6_c, 1840, Y_COL)
r_rhs_h6_1840 = quick_recall_at_k(scored_rhs_h6_c, 1840, Y_COL)

print(f"\nh6 RHS, corrected: @40 precision={p_rhs_h6_40:.2%}, recall={r_rhs_h6_40:.2%}")
print(f"                    @1840 precision={p_rhs_h6_1840:.2%}, recall={r_rhs_h6_1840:.2%}")

# Also get the model's @1840 numbers for a complete table
p_new_h6_1840 = quick_precision_at_k(scored_new_h6_c, 1840, Y_COL)
r_new_h6_1840 = quick_recall_at_k(scored_new_h6_c, 1840, Y_COL)
print(f"h6 recommended model @1840: precision={p_new_h6_1840:.2%}, recall={r_new_h6_1840:.2%}")

In [ ]:
print(f"{'k':<8}{'model':>10}{'RHS':>10}{'model/RHS':>12}")
for k in [40, 100, 200, 500, 1000, 1840]:
    p_model = quick_precision_at_k(scored_new_h6_c, k, Y_COL)
    p_rhs = quick_precision_at_k(scored_rhs_h6_c, k, Y_COL)
    ratio = p_model / p_rhs if p_rhs > 0 else float('nan')
    print(f"{k:<8}{p_model:>9.2%}{p_rhs:>10.2%}{ratio:>11.2f}x")

In [ ]:
scored_rhs_h3_c = new_test[["dim_month", Y_COL]].copy()
scored_rhs_h3_c["score"] = -new_test["product_score"]

print(f"{'k':<8}{'model':>10}{'RHS':>10}{'model/RHS':>12}")
for k in [40, 100, 200, 500, 1000, 1840]:
    p_model = quick_precision_at_k(scored_corrected, k, Y_COL)
    p_rhs = quick_precision_at_k(scored_rhs_h3_c, k, Y_COL)
    ratio = p_model / p_rhs if p_rhs > 0 else float('nan')
    print(f"{k:<8}{p_model:>9.2%}{p_rhs:>10.2%}{ratio:>11.2f}x")

In [ ]:
p_old_h6_1840 = quick_precision_at_k(scored_old_h6_c, 1840, Y_COL)
r_old_h6_1840 = quick_recall_at_k(scored_old_h6_c, 1840, Y_COL)
print(f"h6, 2024-only @1840: precision={p_old_h6_1840:.2%}, recall={r_old_h6_1840:.2%}")


In [ ]:
# Check if the school_group / split columns exist in the h6 corrected data
print("Columns with 'split' or 'group' in the name:")
print([c for c in enc_h6.columns if "split" in c.lower() or "group" in c.lower()])

In [ ]:
train_h6_true = cell("train")  # if cell() already works on enc_h6 in this kernel
holdout_time_h6 = cell("holdout_time")

print(f"train_h6 (2024, group A): {len(train_h6_true):,} rows, positive rate: {train_h6_true[Y_COL].mean():.2%}")
print(f"holdout_time_h6: {len(holdout_time_h6):,} rows, positive rate: {holdout_time_h6[Y_COL].mean():.2%}")

In [ ]:
# h3
print("h3:")
print(f"  2024-only (SELECTED_FEATURES): {len(SELECTED_FEATURES)} features")
print(f"  2024+2025 (ACCEPTED_NEW): {len(ACCEPTED_NEW)} features")
print(f"  Identical? {set(SELECTED_FEATURES) == set(ACCEPTED_NEW)}")
print(f"  Overlap: {len(set(SELECTED_FEATURES) & set(ACCEPTED_NEW))} shared features")

# h6
print("\nh6:")
print(f"  2024-only (ACCEPTED_OLD_H6): {len(ACCEPTED_OLD_H6)} features")
print(f"  2024+2025 (ACCEPTED_H6): {len(ACCEPTED_H6)} features")
print(f"  Identical? {set(ACCEPTED_OLD_H6) == set(ACCEPTED_H6)}")
print(f"  Overlap: {len(set(ACCEPTED_OLD_H6) & set(ACCEPTED_H6))} shared features")



In [ ]:
import json

with open(RAW_DIR / "accepted_features_h6.json", "w") as f:
    json.dump({
        "ACCEPTED_H6": list(ACCEPTED_H6),
        "ACCEPTED_OLD_H6": list(ACCEPTED_OLD_H6),
    }, f)
print("saved h6 features")

In [ ]:
import json

with open(RAW_DIR / "accepted_features_h6.json") as f:
    h6_saved = json.load(f)
with open(RAW_DIR / "accepted_features_h9.json") as f:
    h9_saved = json.load(f)

print(f"h6 old: {len(h6_saved['ACCEPTED_OLD_H6'])}, h6 new: {len(h6_saved['ACCEPTED_H6'])}")
print(f"h9 old: {len(h9_saved['ACCEPTED_OLD_H9'])}, h9 new: {len(h9_saved['ACCEPTED_H9'])}")

print(f"\nh6 new vs h9 new identical? {set(h6_saved['ACCEPTED_H6']) == set(h9_saved['ACCEPTED_H9'])}")
print(f"h6 old vs h9 old identical? {set(h6_saved['ACCEPTED_OLD_H6']) == set(h9_saved['ACCEPTED_OLD_H9'])}")

In [ ]:
lost_between_h6_h9 = set(h6_saved['ACCEPTED_H6']) - set(h9_saved['ACCEPTED_H9'])
print(f"Features accepted at h6 but not h9: {len(lost_between_h6_h9)}")
for f in sorted(lost_between_h6_h9):
    print(f"  {f}")

gained_at_h9 = set(h9_saved['ACCEPTED_H9']) - set(h6_saved['ACCEPTED_H6'])
print(f"\nFeatures accepted at h9 but not h6: {len(gained_at_h9)}")
for f in sorted(gained_at_h9):
    print(f"  {f}")

In [ ]:
import inspect
print(inspect.getsource(cell))

In [ ]:
enc = enc_h6  # point the hardcoded variable at h6's data
holdout_time_h6 = cell("holdout_time")
train_h6_true = cell("train")

print(f"holdout_time_h6: {len(holdout_time_h6):,} rows, {holdout_time_h6[Y_COL].mean():.2%} positive rate")
print(f"train_h6_true: {len(train_h6_true):,} rows, {train_h6_true[Y_COL].mean():.2%} positive rate")

In [ ]:
preds_old_h6_ht = []
for seed in [1, 7, 42, 99]:
    m_ht_h6 = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=8,
    )
    _es = train_h6_true.sample(frac=0.1, random_state=1)
    m_ht_h6.fit(train_h6_true[ACCEPTED_OLD_H6], train_h6_true[Y_COL],
                eval_set=[(_es[ACCEPTED_OLD_H6], _es[Y_COL])], verbose=False)
    preds_old_h6_ht.append(m_ht_h6.predict_proba(holdout_time_h6[ACCEPTED_OLD_H6])[:, 1])

scored_old_h6_ht = holdout_time_h6[["dim_month", Y_COL]].copy()
scored_old_h6_ht["score"] = np.mean(preds_old_h6_ht, axis=0)

p1 = quick_precision_at_k(scored_old_h6_ht, 40, Y_COL)
r1 = quick_recall_at_k(scored_old_h6_ht, 40, Y_COL)
p2 = quick_precision_at_k(scored_old_h6_ht, 1840, Y_COL)
r2 = quick_recall_at_k(scored_old_h6_ht, 1840, Y_COL)

print(f"Model1 on holdout_time_h6: @40 precision={p1:.2%}, recall={r1:.2%}")
print(f"                            @1840 precision={p2:.2%}, recall={r2:.2%}")

# RHS on holdout_time_h6
scored_rhs_h6_ht = holdout_time_h6[["dim_month", Y_COL]].copy()
scored_rhs_h6_ht["score"] = -holdout_time_h6["product_score"]

p3 = quick_precision_at_k(scored_rhs_h6_ht, 40, Y_COL)
r3 = quick_recall_at_k(scored_rhs_h6_ht, 40, Y_COL)
p4 = quick_precision_at_k(scored_rhs_h6_ht, 1840, Y_COL)
r4 = quick_recall_at_k(scored_rhs_h6_ht, 1840, Y_COL)

print(f"RHS on holdout_time_h6: @40 precision={p3:.2%}, recall={r3:.2%}")
print(f"                         @1840 precision={p4:.2%}, recall={r4:.2%}")

In [ ]:
def top_features(feature_list, model, n=10):
    imp = model.feature_importances_
    pairs = sorted(zip(feature_list, imp), key=lambda x: x[1], reverse=True)
    return [f for f, _ in pairs[:n]]

print("h3 top 10:")
for f in top_features(ACCEPTED_NEW, m_corrected):
    print(f"  {f}")

print("\nh6 top 10:")
for f in top_features(ACCEPTED_H6, m_new_h6_c):
    print(f"  {f}")

print("\nh9 top 10:")
for f in top_features(ACCEPTED_H9, m_new_h9_c):
    print(f"  {f}")

In [ ]:
# Using the same current_month_data you scored earlier for the outreach list task
top_40 = current_month_data.sort_values("risk_score", ascending=False).head(40)

print("Customer segment breakdown, top 40 by risk:")
print(top_40["customer_segment"].value_counts())

print(f"\nMultisite schools in the top 40: {(top_40['customer_segment'] == 'SMB Multisite 2-9').sum()}")

In [ ]:
# Step 1: figure out quota per segment, proportional to each segment's expected churners
# (population size in this month x that segment's own historical churn rate)
segment_churn_rates = new_train.groupby(
    new_train["dim_month"].map(lambda x: None)  # placeholder, real grouping below
)

# Actual historical churn rate per segment, computed from training data
seg_rates = new_train.groupby("customer_segment")[Y_COL].mean()
print("Historical churn rate per segment:")
print(seg_rates)

seg_counts_this_month = current_month_data["customer_segment"].value_counts()
expected_churners_per_seg = seg_counts_this_month * seg_rates
quota_per_seg = (expected_churners_per_seg / expected_churners_per_seg.sum() * 40).round().astype(int)

print("\nQuota allocation (out of 40 slots):")
print(quota_per_seg)

# Step 2: within each segment, take its quota's worth of highest-risk schools
quota_list = []
for seg, quota in quota_per_seg.items():
    seg_schools = current_month_data[current_month_data["customer_segment"] == seg]
    top_n = seg_schools.sort_values("risk_score", ascending=False).head(quota)
    quota_list.append(top_n)

quota_top_40 = pd.concat(quota_list)

print(f"\nQuota-based list, segment breakdown:")
print(quota_top_40["customer_segment"].value_counts())

# Step 3: compare real precision, plain top-40 vs quota-based top-40
plain_top_40 = current_month_data.sort_values("risk_score", ascending=False).head(40)

print(f"\nPlain top-40 precision (real labels): {plain_top_40[Y_COL].mean():.2%}")
print(f"Quota-based precision (real labels): {quota_top_40[Y_COL].mean():.2%}")
print(f"\nMultisite count, plain: {(plain_top_40['customer_segment']=='SMB Multisite 2-9').sum()}")
print(f"Multisite count, quota: {(quota_top_40['customer_segment']=='SMB Multisite 2-9').sum()}")

In [ ]:
def generate_quota_weighted_list(scored_population, segment_col, y_col, 
                                   historical_rates, total_slots=40):
    """
    Builds a ranked outreach list where each customer segment gets a fair
    share of slots, proportional to how many real churners that segment
    is expected to contribute this month, rather than letting the biggest
    or highest-scoring segment crowd out smaller ones.

    scored_population: dataframe with a 'risk_score' column and segment_col
    segment_col: name of the customer segment column
    y_col: name of the label column (only used for historical_rates lookup, not here)
    historical_rates: a pandas Series, churn rate per segment, computed from
                       past training data (NOT from this month's population)
    total_slots: how many schools go on the final list (e.g. 40)
    """
    seg_counts = scored_population[segment_col].value_counts()
    expected_churners = seg_counts * historical_rates
    quota = (expected_churners / expected_churners.sum() * total_slots).round().astype(int)

    # Rounding can land a slot or two off target; fix it by adjusting the
    # largest-remainder segment, so the list always has exactly total_slots.
    diff = total_slots - quota.sum()
    if diff != 0:
        largest_seg = quota.idxmax()
        quota[largest_seg] += diff

    final_list = []
    for seg, n_slots in quota.items():
        seg_pool = scored_population[scored_population[segment_col] == seg]
        top_n = seg_pool.sort_values("risk_score", ascending=False).head(n_slots)
        final_list.append(top_n)

    return pd.concat(final_list).sort_values("risk_score", ascending=False)


# Usage, same as tonight's test, but now as a clean, reusable function
seg_rates = new_train.groupby("customer_segment")[Y_COL].mean()
final_ethan_list = generate_quota_weighted_list(
    current_month_data, "customer_segment", Y_COL, seg_rates, total_slots=40
)

print(final_ethan_list["customer_segment"].value_counts())
print(f"\nTotal schools: {len(final_ethan_list)}")

In [ ]:
precision_quota = final_ethan_list[Y_COL].mean()
print(f"Quota-based list, precision@40: {precision_quota:.2%}")

# Recall needs the total real churners for the month, not just this list
total_positives_this_month = current_month_data[Y_COL].sum()
recall_quota = final_ethan_list[Y_COL].sum() / total_positives_this_month
print(f"Quota-based list, recall@40: {recall_quota:.2%}")

# Compare against the plain, non-quota version for the same month
plain_top_40 = current_month_data.sort_values("risk_score", ascending=False).head(40)
precision_plain = plain_top_40[Y_COL].mean()
recall_plain = plain_top_40[Y_COL].sum() / total_positives_this_month

print(f"\nPlain top-40, precision@40: {precision_plain:.2%}, recall@40: {recall_plain:.2%}")
print(f"Quota top-40, precision@40: {precision_quota:.2%}, recall@40: {recall_quota:.2%}")

In [ ]:
# Build a scored dataframe with segment info, using your already-trained corrected model
scored_with_segment = new_test[["dim_month", "customer_segment", Y_COL]].copy()
scored_with_segment["risk_score"] = scored_corrected["score"].values

seg_rates = new_train.groupby("customer_segment")[Y_COL].mean()

plain_results = []
quota_results = []

for month, g in scored_with_segment.groupby("dim_month"):
    # Plain top-40
    plain_40 = g.sort_values("risk_score", ascending=False).head(40)
    total_pos = g[Y_COL].sum()
    plain_results.append({
        "month": month,
        "precision": plain_40[Y_COL].mean(),
        "recall": plain_40[Y_COL].sum() / total_pos if total_pos else np.nan,
        "multisite_count": (plain_40["customer_segment"] == "SMB Multisite 2-9").sum(),
    })

    # Quota-based top-40
    quota_40 = generate_quota_weighted_list(g, "customer_segment", Y_COL, seg_rates, total_slots=40)
    quota_results.append({
        "month": month,
        "precision": quota_40[Y_COL].mean(),
        "recall": quota_40[Y_COL].sum() / total_pos if total_pos else np.nan,
        "multisite_count": (quota_40["customer_segment"] == "SMB Multisite 2-9").sum(),
    })

plain_df = pd.DataFrame(plain_results)
quota_df = pd.DataFrame(quota_results)

print("PLAIN, month by month:")
print(plain_df)
print(f"\nPLAIN average: precision={plain_df['precision'].mean():.2%}, recall={plain_df['recall'].mean():.2%}, avg multisite={plain_df['multisite_count'].mean():.1f}")

print("\nQUOTA, month by month:")
print(quota_df)
print(f"\nQUOTA average: precision={quota_df['precision'].mean():.2%}, recall={quota_df['recall'].mean():.2%}, avg multisite={quota_df['multisite_count'].mean():.1f}")

In [ ]:
print(f"Current ACCEPTED_NEW length: {len(ACCEPTED_NEW)}")
print(f"Expected if unchanged: 84")

In [ ]:
# Reload the ORIGINAL, uncorrected h3 data for comparison
# (adjust path if your original file has a different name)
enc_original = pd.read_parquet(RAW_DIR / "churn_v1_is_reportable_request_202401_202605_h3_encoded_ORIGINAL.parquet")
# If you don't have a separately saved original file, this comparison isn't possible without it

# Merge corrected and original on the same key, to find rows that flipped from 1 to 0
merged = enc_original[["school_id", "dim_month", Y_COL]].merge(
    enc[["school_id", "dim_month", Y_COL]],
    on=["school_id", "dim_month"],
    suffixes=("_original", "_corrected")
)

flipped = merged[(merged[f"{Y_COL}_original"] == 1) & (merged[f"{Y_COL}_corrected"] == 0)]
print(f"Rows that flipped from positive to negative: {len(flipped):,}")

# Now look at what these schools' features actually looked like, using the corrected enc's feature columns
flipped_features = enc[enc["school_id"].isin(flipped["school_id"]) & enc["dim_month"].isin(flipped["dim_month"])]

print("\nHow 'healthy' did these flipped rows look, on the model's own top signals?")
print(flipped_features[["delta_product_active_students_vs_12m_ago", "billing_net_gpv_local", "has_ever_billed"]].describe())

print("\nCompare to the overall population's same signals:")
print(enc[["delta_product_active_students_vs_12m_ago", "billing_net_gpv_local", "has_ever_billed"]].describe())

In [ ]:
import glob
print(glob.glob(str(RAW_DIR / "churn_v1_is_reportable_request_202401_202605*")))

In [ ]:
import os
from datetime import datetime

files_to_check = [
    "churn_v1_is_reportable_request_202401_202605_h3_encoded.parquet",
    "churn_v1_is_reportable_request_202401_202605_h3_boruta_selection.json",
    "churn_v1_is_reportable_request_202401_202605_h3_boruta_importance.csv",
    "churn_v1_is_reportable_request_202401_202605_h3_baseline_comparison.csv",
    "churn_v1_is_reportable_request_202401_202605_h3_k_sweep.csv",
    "churn_v1_is_reportable_request_202401_202605_h3_selection_eval.csv",
]

for fname in files_to_check:
    path = RAW_DIR / fname
    mtime = os.path.getmtime(path)
    print(f"{fname}: last modified {datetime.fromtimestamp(mtime)}")

In [ ]:
import json
with open(RAW_DIR / "churn_v1_is_reportable_request_202401_202605_h3_boruta_selection.json") as f:
    boruta_saved = json.load(f)
print(boruta_saved.keys())
print(f"Accepted count: {len(boruta_saved.get('accepted', []))}")

In [ ]:
print(f"Current ACCEPTED_NEW length: {len(ACCEPTED_NEW)}")

In [ ]:
affected = pd.read_parquet(RAW_DIR / "affected_schools_for_verification.parquet")

# For each affected school, look at the observation row 1-3 months BEFORE the event
# (that's what the model would have seen, per the label's forward-looking window)
affected["event_month"] = pd.to_datetime(affected["event_month"])

checks = []
for _, row in affected.iterrows():
    obs_month = row["event_month"] - pd.DateOffset(months=1)
    match = enc[(enc["school_id"] == row["school_id"]) & (enc["dim_month"] == obs_month)]
    if len(match):
        checks.append(match.iloc[0])

if checks:
    affected_features = pd.DataFrame(checks)
    print(f"Matched {len(affected_features)} of {len(affected)} affected events to a feature row")
    print("\nHow these schools looked, on the model's own top signals:")
    print(affected_features[["delta_product_active_students_vs_12m_ago", "billing_net_gpv_local", "has_ever_billed"]].describe())

    print("\nCompare to the overall population:")
    print(enc[["delta_product_active_students_vs_12m_ago", "billing_net_gpv_local", "has_ever_billed"]].describe())

In [ ]:
preds_new_check2 = []
for seed in [3, 17, 55, 88]:
    m_check2 = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es = new_train.sample(frac=0.1, random_state=1)
    m_check2.fit(new_train[ACCEPTED_NEW], new_train[Y_COL],
                  eval_set=[(_es[ACCEPTED_NEW], _es[Y_COL])], verbose=False)
    preds_new_check2.append(m_check2.predict_proba(new_test[ACCEPTED_NEW])[:, 1])

scored_check2 = new_test[["dim_month", Y_COL]].copy()
scored_check2["score"] = np.mean(preds_new_check2, axis=0)
p_check2 = quick_precision_at_k(scored_check2, 40, Y_COL)
print(f"Retrained model, DIFFERENT seeds: precision@40={p_check2:.2%}")
print(f"Original check (seeds 1,7,42,99): 30.56%")

In [ ]:
# Check calibration for reason_school_no_longer_using_bw_count
scored_check_reason = new_test[["dim_month", Y_COL]].copy()
scored_check_reason["score"] = scored_corrected["score"].values
scored_check_reason["no_longer_using_bw_count"] = new_test["reason_school_no_longer_using_bw_count"].values
scored_check_reason["temp_reducing_enrollment_count"] = new_test["reason_school_temp_reducing_enrollment_count"].values

has_flag = scored_check_reason[scored_check_reason["no_longer_using_bw_count"] > 0]
no_flag = scored_check_reason[scored_check_reason["no_longer_using_bw_count"] == 0]

print(f"Schools with 'no longer using BW' flag: predicted={has_flag['score'].mean():.4f}, actual churn={has_flag[Y_COL].mean():.2%}")
print(f"Schools without: predicted={no_flag['score'].mean():.4f}, actual churn={no_flag[Y_COL].mean():.2%}")

has_flag2 = scored_check_reason[scored_check_reason["temp_reducing_enrollment_count"] > 0]
no_flag2 = scored_check_reason[scored_check_reason["temp_reducing_enrollment_count"] == 0]

print(f"\nSchools with 'temp reducing enrollment' flag: predicted={has_flag2['score'].mean():.4f}, actual churn={has_flag2[Y_COL].mean():.2%}")
print(f"Schools without: predicted={no_flag2['score'].mean():.4f}, actual churn={no_flag2[Y_COL].mean():.2%}")

In [ ]:
imp = m_corrected.feature_importances_
top10_fresh = sorted(zip(ACCEPTED_NEW, imp), key=lambda x: x[1], reverse=True)[:10]

print("Fresh top 10, raw feature names:")
for i, (f, v) in enumerate(top10_fresh, 1):
    print(f"{i}. {f}  (importance: {v:.4f})")

In [ ]:
scored_check_reason2 = new_test[["dim_month", Y_COL]].copy()
scored_check_reason2["score"] = scored_corrected["score"].values
scored_check_reason2["no_longer_using_bw_count"] = new_test["reason_school_no_longer_using_bw_count"].values
scored_check_reason2["delta_enrollment_12m"] = new_test["delta_product_active_students_vs_12m_ago"].values

# Same calibration check pattern: does this flag correctly separate DECLINING vs RECOVERING schools,
# or does it just flag "already left" regardless of current behavior?
has_flag = scored_check_reason2[scored_check_reason2["no_longer_using_bw_count"] > 0]
growing = has_flag[has_flag["delta_enrollment_12m"] > 0]
declining = has_flag[has_flag["delta_enrollment_12m"] <= 0]

print(f"Has 'no longer using BW' flag + growing enrollment: predicted={growing['score'].mean():.4f}, actual churn={growing[Y_COL].mean():.2%}")
print(f"Has 'no longer using BW' flag + declining enrollment: predicted={declining['score'].mean():.4f}, actual churn={declining[Y_COL].mean():.2%}")

In [ ]:
print("Leakage features still in ACCEPTED_NEW (corrected model)?")
print([f for f in ACCEPTED_NEW if f in ("months_since_last_cancel_request", "running_saas_cancel_request_count")])

# If present, check where they actually rank now
imp_full = sorted(zip(ACCEPTED_NEW, m_corrected.feature_importances_), key=lambda x: x[1], reverse=True)
for rank, (f, v) in enumerate(imp_full, 1):
    if f in ("months_since_last_cancel_request", "running_saas_cancel_request_count"):
        print(f"Rank {rank}: {f} (importance: {v:.4f})")

In [ ]:
BILLING_TENURE = ["months_since_first_processed_dollars_local", "processed_dollars_rank_local"]
feat_list = [f for f in ACCEPTED_NEW if f not in BILLING_TENURE]
preds = []
for seed in [1, 7, 42, 99]:
    m = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es = new_train.sample(frac=0.1, random_state=1)
    m.fit(new_train[feat_list], new_train[Y_COL],
          eval_set=[(_es[feat_list], _es[Y_COL])], verbose=False)
    preds.append(m.predict_proba(new_test[feat_list])[:, 1])
scored = new_test[["dim_month", Y_COL]].copy()
scored["score"] = np.mean(preds, axis=0)
p = quick_precision_at_k(scored, 40, Y_COL)
print(f"Without billing tenure features: {p:.2%} (full corrected model: 30.56%)")

In [ ]:
churners_sample = new_test[new_test[Y_COL] == 1].sample(10, random_state=42).reset_index(drop=True)
shap_vals_check = shap.TreeExplainer(m_corrected).shap_values(churners_sample[ACCEPTED_NEW])

for i in range(3):
    school_id = churners_sample.iloc[i]["school_id"]
    score = m_corrected.predict_proba(churners_sample[ACCEPTED_NEW].iloc[[i]])[0][1]
    contributions = list(zip(ACCEPTED_NEW, shap_vals_check[i]))
    contributions.sort(key=lambda x: x[1], reverse=True)
    top3 = [f for f, v in contributions if v > 0][:3]
    print(f"School {school_id} | risk: {score:.1%}")
    for f in top3:
        print(f"   {f}")
    print()

In [ ]:
churners_sample2 = new_test[new_test[Y_COL] == 1].sample(15, random_state=7).reset_index(drop=True)
shap_vals_check2 = shap.TreeExplainer(m_corrected).shap_values(churners_sample2[ACCEPTED_NEW])

for i in range(len(churners_sample2)):
    school_id = churners_sample2.iloc[i]["school_id"]
    score = m_corrected.predict_proba(churners_sample2[ACCEPTED_NEW].iloc[[i]])[0][1]
    contributions = list(zip(ACCEPTED_NEW, shap_vals_check2[i]))
    contributions.sort(key=lambda x: x[1], reverse=True)
    top3 = [f for f, v in contributions if v > 0][:3]
    print(f"School {school_id} | risk: {score:.1%}")
    for f in top3:
        print(f"   {f}")
    print()

In [ ]:
print(f"holdout_time_h6 actual date range: {holdout_time_h6['dim_month'].min()} to {holdout_time_h6['dim_month'].max()}")

In [ ]:
# For schools that eventually churn, check: does the 9-month model flag them
# earlier in real, calendar time than the 3-month model would have?

# Find schools that appear as real positives in BOTH the h3 and h9 test windows
h3_churners = new_test[new_test[Y_COL] == 1][["school_id", "dim_month"]]
h6_churners = new_test_h6[new_test_h6[Y_COL] == 1][["school_id", "dim_month"]]

shared_schools = set(h3_churners["school_id"]) & set(h6_churners["school_id"])
print(f"Schools appearing as real churners in both windows: {len(shared_schools)}")

# For each shared school, find the EARLIEST month each horizon's model
# would have flagged them in the top-40, and compare

In [ ]:
h3_churners = new_test[new_test[Y_COL] == 1][["school_id", "dim_month"]]
h6_churners = new_test_h6[new_test_h6[Y_COL] == 1][["school_id", "dim_month"]]

shared_schools = set(h3_churners["school_id"]) & set(h6_churners["school_id"])
print(f"Shared churner schools: {len(shared_schools)}")

# Build "flagged in top-40" indicator per month, for h3
scored_corrected["flagged_h3"] = False
for month, g in scored_corrected.groupby("dim_month"):
    top40_idx = g.sort_values("score", ascending=False).head(40).index
    scored_corrected.loc[top40_idx, "flagged_h3"] = True

# Build "flagged in top-40" indicator per month, for h6
scored_new_h6_c["flagged_h6"] = False
for month, g in scored_new_h6_c.groupby("dim_month"):
    top40_idx = g.sort_values("score", ascending=False).head(40).index
    scored_new_h6_c.loc[top40_idx, "flagged_h6"] = True

# Attach school_id back in (needed since scored_* only has dim_month, Y_COL, score)
scored_corrected_with_id = scored_corrected.copy()
scored_corrected_with_id["school_id"] = new_test["school_id"].values

scored_h6_with_id = scored_new_h6_c.copy()
scored_h6_with_id["school_id"] = new_test_h6["school_id"].values

# For each shared school, find the EARLIEST month it was flagged, per horizon
results = []
for sid in shared_schools:
    h3_flags = scored_corrected_with_id[(scored_corrected_with_id["school_id"] == sid) & (scored_corrected_with_id["flagged_h3"])]
    h6_flags = scored_h6_with_id[(scored_h6_with_id["school_id"] == sid) & (scored_h6_with_id["flagged_h6"])]
    
    earliest_h3 = h3_flags["dim_month"].min() if len(h3_flags) else None
    earliest_h6 = h6_flags["dim_month"].min() if len(h6_flags) else None
    
    results.append({"school_id": sid, "earliest_h3_flag": earliest_h3, "earliest_h6_flag": earliest_h6})

results_df = pd.DataFrame(results)

both_flagged = results_df.dropna()
print(f"\nSchools flagged by BOTH models at some point: {len(both_flagged)}")

both_flagged["h6_earlier"] = both_flagged["earliest_h6_flag"] < both_flagged["earliest_h3_flag"]
both_flagged["h3_earlier"] = both_flagged["earliest_h3_flag"] < both_flagged["earliest_h6_flag"]
both_flagged["same_month"] = both_flagged["earliest_h3_flag"] == both_flagged["earliest_h6_flag"]

print(f"h6 flagged earlier: {both_flagged['h6_earlier'].sum()}")
print(f"h3 flagged earlier: {both_flagged['h3_earlier'].sum()}")
print(f"Same month: {both_flagged['same_month'].sum()}")

In [ ]:
print(new_test["negative_reason_count"].describe())
print(new_test["negative_reason_count"].value_counts().head(10))

In [ ]:
# Check if this correlates with, or overlaps with, the reason_school_X features you already know
correlation_check = new_test[["negative_reason_count", "reason_school_temp_reducing_enrollment_count",
                                "reason_school_no_longer_using_bw_count", Y_COL]].corr()
print(correlation_check)

# Also check: does having ANY negative reason count correlate with actual churn?
has_negative = new_test[new_test["negative_reason_count"] > 0]
no_negative = new_test[new_test["negative_reason_count"] == 0]
print(f"\nSchools with negative_reason_count > 0: churn rate = {has_negative[Y_COL].mean():.2%}")
print(f"Schools with negative_reason_count = 0: churn rate = {no_negative[Y_COL].mean():.2%}")

In [ ]:
churner_school_ids = new_test[new_test[Y_COL] == 1]["school_id"].unique()[:20]
print(list(churner_school_ids))

In [ ]:
avg_positives_per_month_corrected = new_test.groupby("dim_month")[Y_COL].sum().mean()
print(f"Average actual churners per month, corrected label: {avg_positives_per_month_corrected:.0f}")
print(f"Original (pre-fix) figure used throughout: 854")

In [ ]:
print(new_test["processed_dollars_rank_local"].describe())

# Check if it correlates cleanly with the raw tenure count, which would confirm the "rank of tenure" theory
correlation = new_test[["processed_dollars_rank_local", "months_since_first_processed_dollars_local"]].corr()
print(correlation)

# Also check a few real, specific rows side by side
print(new_test[["school_id", "processed_dollars_rank_local", "months_since_first_processed_dollars_local", "billing_net_gpv_local"]].dropna().head(10))

In [ ]:
# 3-month horizon model, plain ranking, no risk-weighted quota applied
current_month_data["risk_score_final"] = m_corrected.predict_proba(current_month_data[ACCEPTED_NEW])[:, 1]

# Simple, unfiltered top-200, ranked purely by score, nothing else applied
top_200_for_sruti = current_month_data.sort_values("risk_score_final", ascending=False).head(200).copy()

top_200_for_sruti_final = top_200_for_sruti[[
    "school_id", "parse_name", "risk_score_final", "trailing_3m_billing_net_gpv_local", "customer_segment"
]].rename(columns={
    "parse_name": "school_name",
    "risk_score_final": "churn_risk_score",
    "trailing_3m_billing_net_gpv_local": "avg_gpv_last_3_months",
})

top_200_for_sruti_final["rank"] = range(1, len(top_200_for_sruti_final) + 1)

print(f"Total schools: {len(top_200_for_sruti_final)}")
print(top_200_for_sruti_final.head(100).to_string(index=False))

top_200_for_sruti_final.to_csv("top_200_v3_model_for_sruti.csv", index=False)
print("saved to current working directory")

In [ ]:
zero_gpv_schools = top_200_for_sruti[top_200_for_sruti["trailing_3m_billing_net_gpv_local"] == 0]
print(f"Schools with $0 trailing GPV in top 200: {len(zero_gpv_schools)}")

print(zero_gpv_schools[["school_id", "parse_name", "has_ever_billed", "months_since_first_processed_dollars_local"]].to_string(index=False))

In [ ]:
# Restrict to schools that have ever billed, THEN take the top 200 by risk score
billing_only_pool = current_month_data[current_month_data["has_ever_billed"] == 1].copy()

top_200_billing_only = billing_only_pool.sort_values("risk_score_final", ascending=False).head(200).copy()

top_200_billing_only_final = top_200_billing_only[[
    "school_id", "parse_name", "risk_score_final", "trailing_3m_billing_net_gpv_local", "customer_segment"
]].rename(columns={
    "parse_name": "school_name",
    "risk_score_final": "churn_risk_score",
    "trailing_3m_billing_net_gpv_local": "avg_gpv_last_3_months",
})

top_200_billing_only_final["rank"] = range(1, len(top_200_billing_only_final) + 1)

print(f"Total schools: {len(top_200_billing_only_final)}")
print(f"Schools excluded (never billed): {len(current_month_data) - len(billing_only_pool)}")
print(top_200_billing_only_final.head(10).to_string(index=False))

top_200_billing_only_final.to_csv(RAW_DIR / "top_200_v3_model_billing_customers_only.csv", index=False)
print("saved")

In [ ]:
# Precision and recall for this specific, billing-filtered top-200 list
precision_this_list = top_200_billing_only[Y_COL].mean()
total_churners_this_month = current_month_data[Y_COL].sum()
recall_this_list = top_200_billing_only[Y_COL].sum() / total_churners_this_month

print(f"This filtered top-200 list, real May 2026 data:")
print(f"  Precision: {precision_this_list:.2%}")
print(f"  Recall: {recall_this_list:.2%}")
print(f"  Total real churners this month: {total_churners_this_month:.0f}")

# Compare against the same size list, but WITHOUT the billing filter, for a fair comparison
top_200_unfiltered = current_month_data.sort_values("risk_score_final", ascending=False).head(200)
precision_unfiltered = top_200_unfiltered[Y_COL].mean()
recall_unfiltered = top_200_unfiltered[Y_COL].sum() / total_churners_this_month

print(f"\nUnfiltered top-200, same month, for comparison:")
print(f"  Precision: {precision_unfiltered:.2%}")
print(f"  Recall: {recall_unfiltered:.2%}")

In [ ]:
# Check the model's standard, documented list size (k=40) on this same real, current month
top_40_unfiltered = current_month_data.sort_values("risk_score_final", ascending=False).head(40)
precision_40 = top_40_unfiltered[Y_COL].mean()
recall_40 = top_40_unfiltered[Y_COL].sum() / total_churners_this_month

print(f"k=40, this real month (May 2026):")
print(f"  Precision: {precision_40:.2%}")
print(f"  Recall: {recall_40:.2%}")
print(f"\nDoc's averaged figure (across all test months): precision=30.56%, recall=1.48%")

# Also check k=1,840, same real month, for the full picture
top_1840_unfiltered = current_month_data.sort_values("risk_score_final", ascending=False).head(1840)
precision_1840 = top_1840_unfiltered[Y_COL].mean()
recall_1840 = top_1840_unfiltered[Y_COL].sum() / total_churners_this_month

print(f"\nk=1,840, this real month (May 2026):")
print(f"  Precision: {precision_1840:.2%}")
print(f"  Recall: {recall_1840:.2%}")
print(f"Doc's averaged figure: precision=15.07%, recall=32.88%")

In [ ]:
# Top 40, WITHOUT removing never-billed schools
top_40_unfiltered = current_month_data.sort_values("risk_score_final", ascending=False).head(40)
precision_40_unfiltered = top_40_unfiltered[Y_COL].mean()
caught_40_unfiltered = top_40_unfiltered[Y_COL].sum()

# Top 40, WITH never-billed schools removed first
top_40_billing_only = billing_only_pool.sort_values("risk_score_final", ascending=False).head(40)
precision_40_billing_only = top_40_billing_only[Y_COL].mean()
caught_40_billing_only = top_40_billing_only[Y_COL].sum()

print(f"Top 40, WITHOUT removing never-billed schools:")
print(f"  Precision: {precision_40_unfiltered:.2%}, caught: {caught_40_unfiltered:.0f} real churners")

print(f"\nTop 40, WITH never-billed schools removed first:")
print(f"  Precision: {precision_40_billing_only:.2%}, caught: {caught_40_billing_only:.0f} real churners")

In [ ]:
# Sanity check 1: confirm billing_only_pool is genuinely a subset of current_month_data, same scores
print(f"current_month_data size: {len(current_month_data):,}")
print(f"billing_only_pool size: {len(billing_only_pool):,}")
print(f"billing_only_pool has_ever_billed values: {billing_only_pool['has_ever_billed'].unique()}")

# Sanity check 2: confirm the top-40 unfiltered list actually contains some never-billed schools
# (that's WHY removing them should change the result)
top_40_check = current_month_data.sort_values("risk_score_final", ascending=False).head(40)
print(f"\nNever-billed schools in unfiltered top-40: {(top_40_check['has_ever_billed'] == 0).sum()}")

# Sanity check 3: manually verify the exact school that dropped out
top_40_ids = set(top_40_check["school_id"])
top_40_billing_ids = set(top_40_billing_only["school_id"])
dropped_schools = top_40_ids - top_40_billing_ids
print(f"\nSchools in unfiltered top-40 but NOT in billing-only top-40: {len(dropped_schools)}")

dropped_detail = current_month_data[current_month_data["school_id"].isin(dropped_schools)][
    ["school_id", "parse_name", "has_ever_billed", "risk_score_final", Y_COL]
]
print(dropped_detail.to_string(index=False))

In [ ]:
new_schools_added = top_40_billing_ids - top_40_ids
added_detail = current_month_data[current_month_data["school_id"].isin(new_schools_added)][
    ["school_id", "parse_name", "risk_score_final", Y_COL]
]
print(f"New schools added to fill the top-40 (billing-only version): {len(added_detail)}")
print(added_detail.to_string(index=False))
print(f"\nOf these replacements, real churners: {added_detail[Y_COL].sum():.0f}")

In [ ]:
# Score the current population with the final, corrected, 3-month horizon model
current_month_data["risk_score_final"] = m_corrected.predict_proba(current_month_data[ACCEPTED_NEW])[:, 1]

# Filter to schools that have ever billed (removes never-converted accounts), THEN rank by risk
billing_only_pool = current_month_data[current_month_data["has_ever_billed"] == 1].copy()
top_200_for_sruti = billing_only_pool.sort_values("risk_score_final", ascending=False).head(200).copy()

# Final columns for sharing
top_200_final = top_200_for_sruti[[
    "school_id", "parse_name", "risk_score_final", "trailing_3m_billing_net_gpv_local", "customer_segment"
]].rename(columns={
    "parse_name": "school_name",
    "risk_score_final": "churn_risk_score",
    "trailing_3m_billing_net_gpv_local": "avg_gpv_last_3_months",
})
top_200_final["rank"] = range(1, len(top_200_final) + 1)

print(f"Total schools in list: {len(top_200_final)}")
print(f"Schools excluded (never billed): {len(current_month_data) - len(billing_only_pool)}")
print(top_200_final.head(100).to_string(index=False))

# Save to share with Sruti
top_200_final.to_csv(RAW_DIR / "top_200_v3_model_for_sruti.csv", index=False)
print(f"\nSaved to {RAW_DIR / 'top_200_v3_model_for_sruti.csv'}")

In [ ]:
zero_gpv_but_ever_billed = top_200_final[top_200_final["avg_gpv_last_3_months"] == 0]
print(f"Schools with $0 trailing GPV, despite having billed before: {len(zero_gpv_but_ever_billed)}")

# Check how long ago they first billed, and whether they're currently marked as billing this month
check_detail = current_month_data[current_month_data["school_id"].isin(zero_gpv_but_ever_billed["school_id"])][
    ["school_id", "parse_name", "months_since_first_processed_dollars_local",
     "has_processed_dollars_this_month_local", "has_processed_dollars_last_month_local"]
]
print(check_detail.to_string(index=False))

In [ ]:
zero_gpv_but_ever_billed = top_200_final[top_200_final["avg_gpv_last_3_months"] == 0]

check_detail = current_month_data[current_month_data["school_id"].isin(zero_gpv_but_ever_billed["school_id"])][
    ["school_id", "parse_name", "months_since_first_processed_dollars_local",
     "has_processed_dollars_this_month_local", "has_processed_dollars_last_month_local"]
]
print(check_detail.to_string(index=False))

print(f"\nHow many have a real (non-null) months_since_first_processed_dollars_local: {check_detail['months_since_first_processed_dollars_local'].notna().sum()} of {len(check_detail)}")
print(f"How many are currently billing this month: {check_detail['has_processed_dollars_this_month_local'].sum()}")
print(f"How many were billing last month: {check_detail['has_processed_dollars_last_month_local'].sum()}")

In [ ]:
print(top_200_final.to_csv(sep="\t", index=False))

In [ ]:
top_200_final.to_csv(RAW_DIR / "top_200_v3_model_for_sruti.csv", index=False)

In [ ]:
zero_count = (top_200_final["avg_gpv_last_3_months"] == 0).sum()
print(f"Schools with $0 average GPV (last 3 months) out of 200: {zero_count}")
print(f"That's {zero_count/200:.1%} of the list")

In [ ]:
# Check exactly where the zero-GPV schools start appearing, in rank order
top_200_final_sorted = top_200_final.sort_values("rank")
zero_by_rank = top_200_final_sorted[["rank", "avg_gpv_last_3_months"]].copy()
zero_by_rank["is_zero"] = zero_by_rank["avg_gpv_last_3_months"] == 0

print("First rank where GPV = 0 appears:", zero_by_rank[zero_by_rank["is_zero"]]["rank"].min())
print("\nHow many zeros in ranks 1-50:", zero_by_rank[zero_by_rank["rank"] <= 50]["is_zero"].sum())
print("How many zeros in ranks 51-100:", zero_by_rank[(zero_by_rank["rank"] > 50) & (zero_by_rank["rank"] <= 100)]["is_zero"].sum())
print("How many zeros in ranks 101-150:", zero_by_rank[(zero_by_rank["rank"] > 100) & (zero_by_rank["rank"] <= 150)]["is_zero"].sum())
print("How many zeros in ranks 151-200:", zero_by_rank[(zero_by_rank["rank"] > 150) & (zero_by_rank["rank"] <= 200)]["is_zero"].sum())

# Confirm risk score is still genuinely, correctly sorted throughout
print("\nIs risk_score monotonically decreasing across the whole list?",
      (top_200_final_sorted["churn_risk_score"].diff().dropna() <= 0).all())

In [ ]:
nonzero_count = (top_200_final["avg_gpv_last_3_months"] != 0).sum()
zero_count = (top_200_final["avg_gpv_last_3_months"] == 0).sum()
print(f"Nonzero GPV: {nonzero_count}")
print(f"Zero GPV: {zero_count}")
print(f"Total: {nonzero_count + zero_count}")

In [ ]:
top_200_final.to_csv(RAW_DIR / "top_200_v3_model_for_sruti_final_final.csv", index=False)
print(f"Saved fresh copy to: {RAW_DIR / 'top_200_v3_model_for_sruti_final_final.csv'}")

In [ ]:
# Check recall@40 for every customer segment, same method as the multisite check
scored_for_segments = new_test[["school_id", "dim_month", "customer_segment", Y_COL]].copy()
scored_for_segments["score"] = scored_corrected["score"].values

results_by_segment = []
for month, g in scored_for_segments.groupby("dim_month"):
    top_40 = g.sort_values("score", ascending=False).head(40)
    for seg in g["customer_segment"].dropna().unique():
        seg_churners = g[(g["customer_segment"] == seg) & (g[Y_COL] == 1)]
        seg_caught = top_40[(top_40["customer_segment"] == seg) & (top_40[Y_COL] == 1)]
        if len(seg_churners) > 0:
            results_by_segment.append({
                "month": month, "segment": seg,
                "total_churners": len(seg_churners),
                "caught": len(seg_caught),
            })

results_df = pd.DataFrame(results_by_segment)
summary = results_df.groupby("segment").agg(
    total_churners=("total_churners", "sum"),
    total_caught=("caught", "sum")
)
summary["recall"] = summary["total_caught"] / summary["total_churners"]
print(summary.sort_values("recall"))

In [ ]:
# Check whether small schools genuinely have LOWER scores on average,
# or whether they have real signal that's just not distinguishing well
scored_small_check = new_test[["school_id", "customer_segment", Y_COL]].copy()
scored_small_check["score"] = scored_corrected["score"].values

for seg in scored_small_check["customer_segment"].dropna().unique():
    seg_data = scored_small_check[scored_small_check["customer_segment"] == seg]
    churners = seg_data[seg_data[Y_COL] == 1]
    non_churners = seg_data[seg_data[Y_COL] == 0]
    print(f"{seg}:")
    print(f"  Churners avg score: {churners['score'].mean():.4f}")
    print(f"  Non-churners avg score: {non_churners['score'].mean():.4f}")
    print(f"  Separation (churner - non-churner): {churners['score'].mean() - non_churners['score'].mean():.4f}")
    print()

In [ ]:
scored_for_misses = new_test[["school_id", "dim_month", "customer_segment", Y_COL]].copy()
scored_for_misses = scored_for_misses.reset_index(drop=True)
scored_for_misses["score"] = scored_corrected["score"].reset_index(drop=True).values

missed_schools = []
for month, g in scored_for_misses.groupby("dim_month"):
    top_1840 = g.sort_values("score", ascending=False).head(1840)
    caught_ids = set(top_1840["school_id"])
    real_churners = g[g[Y_COL] == 1]
    missed = real_churners[~real_churners["school_id"].isin(caught_ids)]
    missed_schools.append(missed)

missed_df = pd.concat(missed_schools)
print(f"Total real churners genuinely missed, even at k=1,840: {len(missed_df):,}")
print(f"\nScore distribution of missed real churners:")
print(missed_df["score"].describe())

missed_school_ids = missed_df["school_id"].unique()[:20]
missed_features = new_test[new_test["school_id"].isin(missed_school_ids)][
    ["school_id", "delta_product_active_students_vs_12m_ago", "billing_net_gpv_local", "has_ever_billed"]
].drop_duplicates(subset="school_id")
print(f"\nSample of missed schools' features:")
print(missed_features.to_string(index=False))

In [ ]:
missed_full = missed_df.merge(
    new_test[["school_id", "delta_product_active_students_vs_12m_ago", "has_ever_billed", "billing_net_gpv_local"]].drop_duplicates(subset="school_id"),
    on="school_id", how="left"
)

print(f"Missed schools with NaN enrollment delta: {missed_full['delta_product_active_students_vs_12m_ago'].isna().sum()} of {len(missed_full)}")
print(f"Missed schools that never billed: {(missed_full['has_ever_billed'] == 0).sum()} of {len(missed_full)}")

# Compare to the overall population's rate of NaN/never-billed, for context
print(f"\nOverall population, NaN enrollment delta rate: {new_test['delta_product_active_students_vs_12m_ago'].isna().mean():.2%}")
print(f"Overall population, never-billed rate: {(new_test['has_ever_billed'] == 0).mean():.2%}")

In [ ]:
# Check if these NaN-enrollment schools are simply new, without enough history,
# or if there's a real data gap even for established schools
new_schools_check = new_test[new_test["school_id"].isin(missed_full[missed_full["delta_product_active_students_vs_12m_ago"].isna()]["school_id"])]
print(new_schools_check[["school_id", "months_since_first_processed_dollars_local"]].describe())

In [ ]:
# What share of ALL real churners (not just the missed ones) are low-tenure, new schools?
all_churners = new_test[new_test[Y_COL] == 1]
low_tenure_churners = all_churners[all_churners["months_since_first_processed_dollars_local"] <= 6]
print(f"Share of all real churners with 6 months or less tenure: {len(low_tenure_churners) / len(all_churners):.1%}")

In [ ]:
# Build a short-window feature, month-over-month change, using the full enc history
enc_sorted = enc.sort_values(["school_id", "dim_month"]).copy()
enc_sorted["prev_month_active_students"] = enc_sorted.groupby("school_id")["product_active_students"].shift(1)
enc_sorted["delta_students_vs_1m_ago"] = np.where(
    enc_sorted["prev_month_active_students"] > 0,
    enc_sorted["product_active_students"] / enc_sorted["prev_month_active_students"] - 1,
    np.nan
)

new_feature_lookup = enc_sorted[["school_id", "dim_month", "delta_students_vs_1m_ago"]]

# Merge into train/test
new_train_with_feat = new_train.merge(new_feature_lookup, on=["school_id", "dim_month"], how="left")
new_test_with_feat = new_test.merge(new_feature_lookup, on=["school_id", "dim_month"], how="left")

# Check coverage specifically for low-tenure schools, does this new feature actually fill the gap?
low_tenure_mask = new_test_with_feat["months_since_first_processed_dollars_local"] <= 6
print(f"New feature coverage for low-tenure schools: {new_test_with_feat.loc[low_tenure_mask, 'delta_students_vs_1m_ago'].notna().mean():.1%}")
print(f"Old feature (12m) coverage for same schools: {new_test_with_feat.loc[low_tenure_mask, 'delta_product_active_students_vs_12m_ago'].notna().mean():.1%}")

# Retrain WITH the new feature added
feat_with_new = ACCEPTED_NEW + ["delta_students_vs_1m_ago"]
preds_new_feat = []
for seed in [1, 7, 42, 99]:
    m_test = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es = new_train_with_feat.sample(frac=0.1, random_state=1)
    m_test.fit(new_train_with_feat[feat_with_new], new_train_with_feat[Y_COL],
               eval_set=[(_es[feat_with_new], _es[Y_COL])], verbose=False)
    preds_new_feat.append(m_test.predict_proba(new_test_with_feat[feat_with_new])[:, 1])

scored_new_feat = new_test_with_feat[["dim_month", Y_COL]].copy()
scored_new_feat["score"] = np.mean(preds_new_feat, axis=0)
p_new = quick_precision_at_k(scored_new_feat, 40, Y_COL)
r_new = quick_recall_at_k(scored_new_feat, 40, Y_COL)
r_new_1840 = quick_recall_at_k(scored_new_feat, 1840, Y_COL)

print(f"\nWith new short-window feature: precision@40={p_new:.2%}, recall@40={r_new:.2%}, recall@1840={r_new_1840:.2%}")
print(f"Baseline (without): precision@40=30.56%, recall@40=1.48%, recall@1840=32.88%")

In [ ]:
# Instead of a noisy ratio, use a stable, cumulative count since signup
enc_sorted["cumulative_activity_events"] = enc_sorted.groupby("school_id")["trailing_3m_any_activity"].cumsum()

new_feature_lookup2 = enc_sorted[["school_id", "dim_month", "cumulative_activity_events"]]
new_train_v2 = new_train.merge(new_feature_lookup2, on=["school_id", "dim_month"], how="left")
new_test_v2 = new_test.merge(new_feature_lookup2, on=["school_id", "dim_month"], how="left")

feat_v2 = ACCEPTED_NEW + ["cumulative_activity_events"]
preds_v2 = []
for seed in [1, 7, 42, 99]:
    m_v2 = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es = new_train_v2.sample(frac=0.1, random_state=1)
    m_v2.fit(new_train_v2[feat_v2], new_train_v2[Y_COL],
             eval_set=[(_es[feat_v2], _es[Y_COL])], verbose=False)
    preds_v2.append(m_v2.predict_proba(new_test_v2[feat_v2])[:, 1])

scored_v2 = new_test_v2[["dim_month", Y_COL]].copy()
scored_v2["score"] = np.mean(preds_v2, axis=0)
p_v2 = quick_precision_at_k(scored_v2, 40, Y_COL)
r_v2 = quick_recall_at_k(scored_v2, 40, Y_COL)
r_v2_1840 = quick_recall_at_k(scored_v2, 1840, Y_COL)

print(f"With cumulative activity feature: precision@40={p_v2:.2%}, recall@40={r_v2:.2%}, recall@1840={r_v2_1840:.2%}")
print(f"Baseline: precision@40=30.56%, recall@40=1.48%, recall@1840=32.88%")

In [ ]:
# Isolate low-tenure churners, split into CAUGHT (top 1840) vs MISSED
scored_low_tenure = new_test[["school_id", "dim_month", Y_COL, "months_since_first_processed_dollars_local"]].copy()
scored_low_tenure["score"] = scored_corrected["score"].values

low_tenure_churners = scored_low_tenure[
    (scored_low_tenure[Y_COL] == 1) &
    (scored_low_tenure["months_since_first_processed_dollars_local"] <= 6)
].copy()

caught_ids_by_month = {}
for month, g in scored_low_tenure.groupby("dim_month"):
    caught_ids_by_month[month] = set(g.sort_values("score", ascending=False).head(1840)["school_id"])

low_tenure_churners["caught"] = low_tenure_churners.apply(
    lambda row: row["school_id"] in caught_ids_by_month.get(row["dim_month"], set()), axis=1
)

print(f"Low-tenure real churners: {len(low_tenure_churners)}")
print(f"Caught: {low_tenure_churners['caught'].sum()}, Missed: {(~low_tenure_churners['caught']).sum()}")

# Compare feature profiles, caught vs missed, among just this low-tenure population
caught_school_ids = low_tenure_churners[low_tenure_churners["caught"]]["school_id"]
missed_school_ids = low_tenure_churners[~low_tenure_churners["caught"]]["school_id"]

compare_features = ["product_active_students", "billing_net_gpv_local", "has_ever_billed",
                     "delta_product_active_students_vs_t3m", "any_admission_funnel"]

for feat in compare_features:
    caught_mean = new_test[new_test["school_id"].isin(caught_school_ids)][feat].mean()
    missed_mean = new_test[new_test["school_id"].isin(missed_school_ids)][feat].mean()
    print(f"{feat}: caught avg={caught_mean:.3f}, missed avg={missed_mean:.3f}")

In [ ]:
# Run this in feature_selection.ipynb first, to get the actual IDs
print(list(missed_school_ids))

In [ ]:
csat_monthly = pd.read_parquet(RAW_DIR / "csat_monthly.parquet")
csat_monthly["dim_month"] = pd.to_datetime(csat_monthly["dim_month"])

new_train_csat = new_train.merge(csat_monthly, on=["school_id", "dim_month"], how="left")
new_test_csat = new_test.merge(csat_monthly, on=["school_id", "dim_month"], how="left")

new_train_csat["n_low_csat"] = new_train_csat["n_low_csat"].fillna(0)
new_test_csat["n_low_csat"] = new_test_csat["n_low_csat"].fillna(0)

feat_with_csat = ACCEPTED_NEW + ["n_low_csat"]

preds_csat = []
for seed in [1, 7, 42, 99]:
    m_csat = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es = new_train_csat.sample(frac=0.1, random_state=1)
    m_csat.fit(new_train_csat[feat_with_csat], new_train_csat[Y_COL],
               eval_set=[(_es[feat_with_csat], _es[Y_COL])], verbose=False)
    preds_csat.append(m_csat.predict_proba(new_test_csat[feat_with_csat])[:, 1])

scored_csat = new_test_csat[["dim_month", Y_COL]].copy()
scored_csat["score"] = np.mean(preds_csat, axis=0)

p_csat_40 = quick_precision_at_k(scored_csat, 40, Y_COL)
r_csat_40 = quick_recall_at_k(scored_csat, 40, Y_COL)
r_csat_1840 = quick_recall_at_k(scored_csat, 1840, Y_COL)

print(f"With CSAT feature: precision@40={p_csat_40:.2%}, recall@40={r_csat_40:.2%}, recall@1840={r_csat_1840:.2%}")
print(f"Baseline: precision@40=30.56%, recall@40=1.48%, recall@1840=32.88%")

# Specifically check recall among the low-tenure, "healthy but churning" population
low_tenure_mask_csat = new_test_csat["months_since_first_processed_dollars_local"] <= 6
low_tenure_test = new_test_csat[low_tenure_mask_csat].copy()
low_tenure_test["score"] = m_csat.predict_proba(low_tenure_test[feat_with_csat])[:, 1]

low_tenure_recall_1840 = quick_recall_at_k(
    low_tenure_test[["dim_month", Y_COL]].assign(score=low_tenure_test["score"]), 1840, Y_COL
)
print(f"\nRecall@1840, specifically for low-tenure schools: {low_tenure_recall_1840:.2%}")

In [ ]:
print(f"Total rows in low_tenure_test subset: {len(low_tenure_test):,}")
print(f"Is 1840 bigger than the whole low-tenure subset? {1840 >= len(low_tenure_test)}")

In [ ]:
monthly_counts = low_tenure_test.groupby("dim_month").size()
print(monthly_counts)
print(f"\nAverage rows per month: {monthly_counts.mean():.0f}")
print(f"Is 1840 a large share of a typical month's low-tenure subset? {1840 / monthly_counts.mean():.1%}")

In [ ]:
# Your real k=1,840 represents roughly what % of the full population?
full_pop_avg_monthly = len(new_test_csat) / new_test_csat["dim_month"].nunique()
equivalent_percentage = 1840 / full_pop_avg_monthly
print(f"k=1,840 represents {equivalent_percentage:.1%} of a typical month's full population")

# Apply that SAME percentage to the low-tenure subset instead
scaled_k = int(equivalent_percentage * monthly_counts.mean())
print(f"Equivalent, fairly-scaled k for the low-tenure subset: {scaled_k}")

low_tenure_recall_scaled = quick_recall_at_k(
    low_tenure_test[["dim_month", Y_COL]].assign(score=low_tenure_test["score"]), scaled_k, Y_COL
)
print(f"Recall@{scaled_k} (properly scaled), low-tenure schools: {low_tenure_recall_scaled:.2%}")

In [ ]:
new_train_targeted = new_train_csat.copy()
new_test_targeted = new_test_csat.copy()

# Only keep CSAT info for low-tenure schools, blank it out for everyone else
is_low_tenure_train = new_train_targeted["months_since_first_processed_dollars_local"] <= 6
is_low_tenure_test = new_test_targeted["months_since_first_processed_dollars_local"] <= 6

new_train_targeted.loc[~is_low_tenure_train, "n_low_csat"] = np.nan
new_test_targeted.loc[~is_low_tenure_test, "n_low_csat"] = np.nan

preds_targeted = []
for seed in [1, 7, 42, 99]:
    m_targeted = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es = new_train_targeted.sample(frac=0.1, random_state=1)
    m_targeted.fit(new_train_targeted[feat_with_csat], new_train_targeted[Y_COL],
                    eval_set=[(_es[feat_with_csat], _es[Y_COL])], verbose=False)
    preds_targeted.append(m_targeted.predict_proba(new_test_targeted[feat_with_csat])[:, 1])

scored_targeted = new_test_targeted[["dim_month", Y_COL]].copy()
scored_targeted["score"] = np.mean(preds_targeted, axis=0)
p_targeted = quick_precision_at_k(scored_targeted, 40, Y_COL)
r_targeted = quick_recall_at_k(scored_targeted, 40, Y_COL)
r_targeted_1840 = quick_recall_at_k(scored_targeted, 1840, Y_COL)

print(f"CSAT, targeted to low-tenure only: precision@40={p_targeted:.2%}, recall@40={r_targeted:.2%}, recall@1840={r_targeted_1840:.2%}")
print(f"Baseline: precision@40=30.56%, recall@40=1.48%, recall@1840=32.88%")

In [ ]:
# Rebuild, this time leaving genuinely unrated school-months as NaN, not 0
new_train_fixed = new_train.merge(csat_monthly, on=["school_id", "dim_month"], how="left")
new_test_fixed = new_test.merge(csat_monthly, on=["school_id", "dim_month"], how="left")

# Only fill with 0 where conversations happened but genuinely had zero low ratings (n_rated > 0)
# Leave true NaN where there's no rated conversation data at all
new_train_fixed["n_low_csat_clean"] = new_train_fixed.apply(
    lambda r: r["n_low_csat"] if pd.notna(r["n_rated"]) and r["n_rated"] > 0 else np.nan, axis=1
)
new_test_fixed["n_low_csat_clean"] = new_test_fixed.apply(
    lambda r: r["n_low_csat"] if pd.notna(r["n_rated"]) and r["n_rated"] > 0 else np.nan, axis=1
)

feat_csat_clean = ACCEPTED_NEW + ["n_low_csat_clean"]

preds_clean = []
for seed in [1, 7, 42, 99]:
    m_clean = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es = new_train_fixed.sample(frac=0.1, random_state=1)
    m_clean.fit(new_train_fixed[feat_csat_clean], new_train_fixed[Y_COL],
                eval_set=[(_es[feat_csat_clean], _es[Y_COL])], verbose=False)
    preds_clean.append(m_clean.predict_proba(new_test_fixed[feat_csat_clean])[:, 1])

scored_clean = new_test_fixed[["dim_month", Y_COL]].copy()
scored_clean["score"] = np.mean(preds_clean, axis=0)
p_clean = quick_precision_at_k(scored_clean, 40, Y_COL)
r_clean = quick_recall_at_k(scored_clean, 40, Y_COL)
r_clean_1840 = quick_recall_at_k(scored_clean, 1840, Y_COL)

print(f"CSAT, properly encoded (NaN vs 0 distinguished): precision@40={p_clean:.2%}, recall@40={r_clean:.2%}, recall@1840={r_clean_1840:.2%}")
print(f"Baseline: precision@40=30.56%, recall@40=1.48%, recall@1840=32.88%")

In [ ]:
from xgboost import XGBRanker

tr = new_train.sort_values("dim_month").reset_index(drop=True)
te = new_test.reset_index(drop=True)
qid_tr = tr["dim_month"].rank(method="dense").astype(int)

es = tr.sample(frac=0.1, random_state=1).sort_values("dim_month")
qid_es = es["dim_month"].rank(method="dense").astype(int)

preds_rank = []
for seed in [1, 7, 42, 99]:
    m_rank = XGBRanker(
        objective="rank:pairwise", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, eval_metric="ndcg@40",
    )
    m_rank.fit(tr[ACCEPTED_NEW], tr[Y_COL], qid=qid_tr,
               eval_set=[(es[ACCEPTED_NEW], es[Y_COL])], eval_qid=[qid_es], verbose=False)
    preds_rank.append(m_rank.predict(te[ACCEPTED_NEW]))

scored_rank = te[["dim_month", Y_COL]].copy()
scored_rank["score"] = np.mean(preds_rank, axis=0)

print(f"Ranker:   precision@40={quick_precision_at_k(scored_rank, 40, Y_COL):.2%}, "
      f"recall@40={quick_recall_at_k(scored_rank, 40, Y_COL):.2%}, "
      f"recall@1840={quick_recall_at_k(scored_rank, 1840, Y_COL):.2%}")
print("Baseline: precision@40=30.56%, recall@40=1.48%, recall@1840=32.88%")

In [ ]:
print(m_rank.best_iteration)

In [ ]:
# How many trees did the baseline classifier actually use?
n_trees = m_corrected.best_iteration + 1
print(f"Baseline classifier used {n_trees} trees")

preds_rank2 = []
for seed in [1, 7, 42, 99]:
    m_rank2 = XGBRanker(
        objective="rank:ndcg", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=n_trees,
    )
    m_rank2.fit(tr[ACCEPTED_NEW], tr[Y_COL], qid=qid_tr, verbose=False)
    preds_rank2.append(m_rank2.predict(te[ACCEPTED_NEW]))

scored_rank2 = te[["dim_month", Y_COL]].copy()
scored_rank2["score"] = np.mean(preds_rank2, axis=0)

print(f"Ranker (fair): precision@40={quick_precision_at_k(scored_rank2, 40, Y_COL):.2%}, "
      f"recall@40={quick_recall_at_k(scored_rank2, 40, Y_COL):.2%}, "
      f"recall@1840={quick_recall_at_k(scored_rank2, 1840, Y_COL):.2%}")
print("Baseline:      precision@40=30.56%, recall@40=1.48%, recall@1840=32.88%")

In [ ]:
train_end = new_train["dim_month"].max()
months_back = ((train_end.year - new_train["dim_month"].dt.year) * 12
               + (train_end.month - new_train["dim_month"].dt.month))

for half_life in [6, 12]:
    w = 0.5 ** (months_back / half_life)
    preds = []
    for seed in [1, 7, 42, 99]:
        m = XGBClassifier(
            objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
            max_depth=3, learning_rate=0.06, min_child_weight=15,
            subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=seed, n_jobs=-1, missing=np.nan,
            n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
        )
        _es = new_train.sample(frac=0.1, random_state=1)
        m.fit(new_train[ACCEPTED_NEW], new_train[Y_COL], sample_weight=w,
              eval_set=[(_es[ACCEPTED_NEW], _es[Y_COL])], verbose=False)
        preds.append(m.predict_proba(new_test[ACCEPTED_NEW])[:, 1])

    scored = new_test[["dim_month", Y_COL]].copy()
    scored["score"] = np.mean(preds, axis=0)
    print(f"Half-life {half_life}m: precision@40={quick_precision_at_k(scored, 40, Y_COL):.2%}, "
          f"recall@40={quick_recall_at_k(scored, 40, Y_COL):.2%}, "
          f"recall@1840={quick_recall_at_k(scored, 1840, Y_COL):.2%}")

print("Baseline:      precision@40=30.56%, recall@40=1.48%, recall@1840=32.88%")

In [ ]:
enc = pd.read_parquet(RAW_DIR / "churn_v1_is_reportable_request_202401_202605_h3_encoded.parquet")
enc["dim_month"] = pd.to_datetime(enc["dim_month"])

new_train = enc[enc["dim_month"] < "2025-09-01"]
new_test = enc[enc["dim_month"] >= "2025-09-01"]
print(f"train: {len(new_train):,} rows, test: {len(new_test):,} rows, test positive rate: {new_test[Y_COL].mean():.2%}")

In [ ]:
import lightgbm as lgb
from catboost import CatBoostClassifier

X_tr, y_tr, X_te = new_train[ACCEPTED_NEW], new_train[Y_COL], new_test[ACCEPTED_NEW]

def evaluate(name, score):
    s = new_test[["dim_month", Y_COL]].copy()
    s["score"] = score
    print(f"{name:<10} P@40={quick_precision_at_k(s, 40, Y_COL):.2%}  "
          f"R@40={quick_recall_at_k(s, 40, Y_COL):.2%}  "
          f"R@1840={quick_recall_at_k(s, 1840, Y_COL):.2%}")
    return s

# --- LightGBM, same features, settings matched to the XGBoost baseline ---
lgb_preds = []
for seed in [1, 7, 42, 99]:
    m = lgb.LGBMClassifier(
        n_estimators=2000, learning_rate=0.06, num_leaves=8, max_depth=3,
        min_child_samples=50, colsample_bytree=0.8, reg_lambda=1.0,
        scale_pos_weight=4, random_state=seed, n_jobs=-1, verbose=-1,
    )
    m.fit(X_tr, y_tr)
    lgb_preds.append(m.predict_proba(X_te)[:, 1])
s_lgb = evaluate("LightGBM", np.mean(lgb_preds, axis=0))

# --- CatBoost, using the raw categorical columns instead of the one-hot dummies ---
dummy_cols = [f for f in ACCEPTED_NEW if "__" in f]
raw_cats = sorted({f.split("__")[0] for f in dummy_cols})
assert all(c in new_train.columns for c in raw_cats), raw_cats
feats_cb = [f for f in ACCEPTED_NEW if "__" not in f] + raw_cats

def prep(df):
    X = df[feats_cb].copy()
    for c in raw_cats:
        X[c] = X[c].fillna("missing").astype(str)
    return X

Xcb_tr, Xcb_te = prep(new_train), prep(new_test)
cb_preds = []
for seed in [1, 7, 42, 99]:
    m = CatBoostClassifier(
        iterations=2000, learning_rate=0.06, depth=4, scale_pos_weight=4,
        random_seed=seed, verbose=0, thread_count=-1,
    )
    m.fit(Xcb_tr, y_tr, cat_features=raw_cats)
    cb_preds.append(m.predict_proba(Xcb_te)[:, 1])
s_cb = evaluate("CatBoost", np.mean(cb_preds, axis=0))

# --- Mixture: average each model's rank within the month, so scales are comparable ---
ens = new_test[["dim_month"]].copy()
ens["xgb"] = scored_corrected["score"].values
ens["lgb"] = s_lgb["score"].values
ens["cb"] = s_cb["score"].values
for c in ["xgb", "lgb", "cb"]:
    ens[c + "_pct"] = ens.groupby("dim_month")[c].rank(pct=True)
evaluate("Mixture", ens[["xgb_pct", "lgb_pct", "cb_pct"]].mean(axis=1).values)

print("Baseline   P@40=30.56%  R@40=1.48%  R@1840=32.88%")

In [ ]:
def prep(df):
    X = df[feats_cb].copy()
    for c in raw_cats:
        X[c] = X[c].astype("object").fillna("missing").astype(str)
    return X

Xcb_tr, Xcb_te = prep(new_train), prep(new_test)
cb_preds = []
for seed in [1, 7, 42, 99]:
    m = CatBoostClassifier(
        iterations=2000, learning_rate=0.06, depth=4, scale_pos_weight=4,
        random_seed=seed, verbose=0, thread_count=-1,
    )
    m.fit(Xcb_tr, y_tr, cat_features=raw_cats)
    cb_preds.append(m.predict_proba(Xcb_te)[:, 1])
s_cb = evaluate("CatBoost", np.mean(cb_preds, axis=0))

ens = new_test[["dim_month"]].copy()
ens["xgb"] = scored_corrected["score"].values
ens["lgb"] = s_lgb["score"].values
ens["cb"] = s_cb["score"].values
for c in ["xgb", "lgb", "cb"]:
    ens[c + "_pct"] = ens.groupby("dim_month")[c].rank(pct=True)
evaluate("Mixture", ens[["xgb_pct", "lgb_pct", "cb_pct"]].mean(axis=1).values)

print("Baseline   P@40=28.06%  R@40=1.43%  R@1840=33.33%")

In [ ]:
seeds2 = [3, 17, 55, 88]

def run_xgb(seeds):
    out = []
    for s in seeds:
        m = XGBClassifier(
            objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
            max_depth=3, learning_rate=0.06, min_child_weight=15,
            subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
            random_state=s, n_jobs=-1, missing=np.nan,
            n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
        )
        _es = new_train.sample(frac=0.1, random_state=1)
        m.fit(X_tr, y_tr, eval_set=[(_es[ACCEPTED_NEW], _es[Y_COL])], verbose=False)
        out.append(m.predict_proba(X_te)[:, 1])
    return np.mean(out, axis=0)

def run_lgb(seeds):
    out = []
    for s in seeds:
        m = lgb.LGBMClassifier(
            n_estimators=2000, learning_rate=0.06, num_leaves=8, max_depth=3,
            min_child_samples=50, colsample_bytree=0.8, reg_lambda=1.0,
            scale_pos_weight=4, random_state=s, n_jobs=-1, verbose=-1,
        )
        m.fit(X_tr, y_tr)
        out.append(m.predict_proba(X_te)[:, 1])
    return np.mean(out, axis=0)

def run_cb(seeds):
    out = []
    for s in seeds:
        m = CatBoostClassifier(
            iterations=2000, learning_rate=0.06, depth=4, scale_pos_weight=4,
            random_seed=s, verbose=0, thread_count=-1,
        )
        m.fit(Xcb_tr, y_tr, cat_features=raw_cats)
        out.append(m.predict_proba(Xcb_te)[:, 1])
    return np.mean(out, axis=0)

x2, l2, c2 = run_xgb(seeds2), run_lgb(seeds2), run_cb(seeds2)

ens2 = new_test[["dim_month"]].copy()
ens2["xgb"], ens2["lgb"], ens2["cb"] = x2, l2, c2
for c in ["xgb", "lgb", "cb"]:
    ens2[c + "_pct"] = ens2.groupby("dim_month")[c].rank(pct=True)

print("Second seed set:")
evaluate("XGBoost", x2)
evaluate("CatBoost", c2)
evaluate("Mixture", ens2[["xgb_pct", "lgb_pct", "cb_pct"]].mean(axis=1).values)

In [ ]:
import json, re
from pathlib import Path

found = set()
for nb_file in Path("~/Proactive-Churn").expanduser().rglob("*.ipynb"):
    nb = json.load(open(nb_file))
    for c in nb.get("cells", []):
        for o in c.get("outputs", []):
            text = "".join(o.get("text", [])) or "".join(o.get("data", {}).get("text/plain", []))
            hits = set(re.findall(r"^([a-z0-9_]+) {2,}-> ", text, flags=re.M))
            if hits:
                print(f"{nb_file.name}: {len(hits)} features")
                found |= hits

print(f"\nTotal recovered: {len(found)}")

In [ ]:
found = set()
for nb_file in Path("~/Proactive-Churn").expanduser().rglob("*.ipynb"):
    nb = json.load(open(nb_file))
    for c in nb.get("cells", []):
        for o in c.get("outputs", []):
            text = "".join(o.get("text", [])) or "".join(o.get("data", {}).get("text/plain", []))
            found |= set(re.findall(r"^([a-z0-9_]+) +-> ", text, flags=re.M))

print(f"Total recovered: {len(found)}")
print("Longest names:", sorted(found, key=len, reverse=True)[:5])

In [ ]:
nb = json.load(open(Path("~/Proactive-Churn").expanduser().rglob("feature_selection.ipynb").__next__()))

found_exact = set()
for c in nb["cells"]:
    src = "".join(c.get("source", []))
    if "for f in ACCEPTED_NEW" in src and "->" in src:
        for o in c.get("outputs", []):
            text = "".join(o.get("text", [])) or "".join(o.get("data", {}).get("text/plain", []))
            found_exact |= set(re.findall(r"^([a-z0-9_]+) +-> ", text, flags=re.M))

print(f"From the ACCEPTED_NEW loop only: {len(found_exact)}")
print("Extra name(s) picked up elsewhere:", found - found_exact)

In [ ]:
ACCEPTED_NEW = sorted(found_exact)
assert len(ACCEPTED_NEW) == 84
assert all(f in new_train.columns for f in ACCEPTED_NEW)

with open(RAW_DIR / "accepted_features_h3.json", "w") as f:
    json.dump(ACCEPTED_NEW, f)
print("saved")

In [ ]:
preds = []
for seed in [1, 7, 42, 99]:
    m = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es = new_train.sample(frac=0.1, random_state=1)
    m.fit(new_train[ACCEPTED_NEW], new_train[Y_COL],
          eval_set=[(_es[ACCEPTED_NEW], _es[Y_COL])], verbose=False)
    preds.append(m.predict_proba(new_test[ACCEPTED_NEW])[:, 1])

m_corrected = m
scored_corrected = new_test[["dim_month", Y_COL]].copy()
scored_corrected["score"] = np.mean(preds, axis=0)
print(f"precision@40={quick_precision_at_k(scored_corrected, 40, Y_COL):.2%} (expect 30.56%)")

In [ ]:
def quick_precision_at_k(scored_df, k, label_col):
    precisions = []
    for month, g in scored_df.groupby("dim_month"):
        top_k = g.sort_values("score", ascending=False).head(k)
        precisions.append(top_k[label_col].mean())
    return np.mean(precisions)

def quick_recall_at_k(scored_df, k, label_col):
    recalls = []
    for month, g in scored_df.groupby("dim_month"):
        top_k = g.sort_values("score", ascending=False).head(k)
        total = g[label_col].sum()
        if total > 0:
            recalls.append(top_k[label_col].sum() / total)
    return np.mean(recalls)

print(f"precision@40={quick_precision_at_k(scored_corrected, 40, Y_COL):.2%} (expect 30.56%)")
print(f"recall@40={quick_recall_at_k(scored_corrected, 40, Y_COL):.2%} (expect 1.48%)")
print(f"recall@1840={quick_recall_at_k(scored_corrected, 1840, Y_COL):.2%} (expect 32.88%)")

In [ ]:
import xgboost, sklearn
print(f"rows: {len(enc):,} (expect 800,414)")
print(f"positives: {int(enc[Y_COL].sum()):,} (expect 22,491)")
print(f"xgboost {xgboost.__version__}, numpy {np.__version__}, pandas {pd.__version__}")

In [ ]:
import shap

sheet = pd.read_csv(RAW_DIR/"V3.0 Proactive Churn Schools - Sheet1 (2).csv", header=1)
sheet = sheet[sheet["School ID"].notna()].copy()
sheet["School ID"] = sheet["School ID"].astype(int)
print(f"Schools in sheet: {len(sheet)}")

may = enc[(enc["dim_month"] == "2026-05-01") & (enc["school_id"].isin(sheet["School ID"]))].copy()
missing = set(sheet["School ID"]) - set(may["school_id"])
print(f"Found in May data: {len(may)} | missing: {sorted(missing) or 'none'}")

# Sanity check: does our model reproduce the sheet's scores?
may["our_score"] = m_corrected.predict_proba(may[ACCEPTED_NEW])[:, 1]
check = sheet[["School ID", "Churn risk score"]].merge(may[["school_id", "our_score"]],
                                                        left_on="School ID", right_on="school_id")
check["Churn risk score"] = pd.to_numeric(check["Churn risk score"], errors="coerce")
print(f"Score correlation with sheet: {check['Churn risk score'].corr(check['our_score']):.3f}")
print(f"Average difference: {(check['Churn risk score'] - check['our_score']).abs().mean():.3f}")

# Top 3 reasons per school
sv = shap.TreeExplainer(m_corrected).shap_values(may[ACCEPTED_NEW])

def top_reasons(row_shap, n=3):
    pairs = sorted(zip(ACCEPTED_NEW, row_shap), key=lambda x: x[1], reverse=True)
    return "; ".join([LABELS.get(f, f) for f, v in pairs if v > 0][:n])

may["Top reasons for high score"] = [top_reasons(r) for r in sv]
out = sheet[["School ID"]].merge(may[["school_id", "Top reasons for high score"]],
                                 left_on="School ID", right_on="school_id", how="left").drop(columns="school_id")
out.to_csv(RAW_DIR / "v3_top_reasons_for_sheet.csv", index=False)
print(out.head(10).to_string(index=False))

In [ ]:
import shap

models = []
for seed in [1, 7, 42, 99]:
    m = XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr", tree_method="hist",
        max_depth=3, learning_rate=0.06, min_child_weight=15,
        subsample=1.0, colsample_bytree=0.8, reg_lambda=1.0,
        random_state=seed, n_jobs=-1, missing=np.nan,
        n_estimators=2000, early_stopping_rounds=50, scale_pos_weight=4,
    )
    _es = new_train.sample(frac=0.1, random_state=1)
    m.fit(new_train[ACCEPTED_NEW], new_train[Y_COL],
          eval_set=[(_es[ACCEPTED_NEW], _es[Y_COL])], verbose=False)
    models.append(m)

# Ensemble score, then re-check against the sheet
may["our_score"] = np.mean([m.predict_proba(may[ACCEPTED_NEW])[:, 1] for m in models], axis=0)
check = sheet[["School ID", "Churn risk score"]].merge(may[["school_id", "our_score"]],
                                                        left_on="School ID", right_on="school_id")
check["Churn risk score"] = pd.to_numeric(check["Churn risk score"], errors="coerce")
print(f"Score correlation with sheet: {check['Churn risk score'].corr(check['our_score']):.3f}")
print(f"Average difference: {(check['Churn risk score'] - check['our_score']).abs().mean():.3f}")

# Average explanations across the 4 models
sv = np.mean([shap.TreeExplainer(m).shap_values(may[ACCEPTED_NEW]) for m in models], axis=0)

In [ ]:
per_model = [shap.TreeExplainer(m).shap_values(may[ACCEPTED_NEW]) for m in models]

def topk(row, k=3):
    return [f for f, v in sorted(zip(ACCEPTED_NEW, row), key=lambda x: x[1], reverse=True) if v > 0][:k]

tops = [[topk(r) for r in s] for s in per_model]
ens = [topk(r) for r in sv]

same_top1 = np.mean([all(t[i][:1] == ens[i][:1] for t in tops) for i in range(len(ens))])
overlap = np.mean([np.mean([len(set(t[i]) & set(ens[i])) / max(len(ens[i]), 1) for t in tops])
                   for i in range(len(ens))])
print(f"Top reason identical across all 4 models: {same_top1:.0%}")
print(f"Average overlap of top-3 reasons: {overlap:.0%}")

In [ ]:
import json, re
from pathlib import Path

nb = json.load(open(next(Path("~/Proactive-Churn").expanduser().rglob("feature_selection.ipynb"))))
LABELS = {}
for c in nb["cells"]:
    src = "".join(c.get("source", []))
    if "for f in ACCEPTED_NEW" in src and "->" in src:
        for o in c.get("outputs", []):
            text = "".join(o.get("text", [])) or "".join(o.get("data", {}).get("text/plain", []))
            for name, label in re.findall(r"^([a-z0-9_]+) +-> (.+)$", text, flags=re.M):
                LABELS[name] = label.strip()
print(f"Recovered {len(LABELS)} labels")

In [ ]:
may["Top reasons for high score"] = [top_reasons(r) for r in sv]
out = sheet[["School ID"]].merge(may[["school_id", "Top reasons for high score"]],
                                 left_on="School ID", right_on="school_id", how="left").drop(columns="school_id")
out.to_csv(RAW_DIR / "v3_top_reasons_for_sheet.csv", index=False)

In [ ]:
import re

PERIOD = {"vs_t3m": "the last 3 months", "vs_12m_ago": "a year ago"}
NAMES = {
    "product_active_students": "Active students",
    "any_messaging": "Messaging use",
    "messaging_events_per_active_student": "Messaging activity",
    "activity_events_per_active_student": "Daily activity logging",
    "admissions_funnel_events_per_active_student": "Admissions activity",
    "form_events_per_active_student": "Forms use",
    "attendance_events_per_active_student": "Attendance tracking",
    "online_payment_events_per_active_student": "Online payments",
    "billing_net_gpv_local": "Billing volume",
    "num_billing_invoices_created": "Billing invoices sent",
    "num_billing_invoices_due": "Billing invoices due",
    "num_students_with_billing_invoice_created": "Families billed",
    "students_with_payment_method_added": "Families adding a payment method",
    "students_with_online_payment": "Families paying online",
    "any_payroll": "Payroll use",
    "any_activity": "App activity",
}

def plain(f, v):
    # Special cases
    if f == "has_ever_billed":
        return "Has never used Brightwheel billing" if v == 0 else "Billing customer"
    if f == "customer_segment__smb_multisite_2_9":
        return "Multisite school (2 to 9 locations)"
    if f == "months_since_first_processed_dollars_local":
        return f"Billing customer for only {int(v)} months" if pd.notna(v) else "Has never processed a payment"
    if f == "months_since_last_cancel_request":
        return f"Asked to cancel {int(v)} months ago" if pd.notna(v) else "Asked to cancel before"
    if f == "running_saas_cancel_request_count":
        return f"Has asked to cancel before ({int(v)} times)"
    if f == "reason_school_no_longer_using_bw_count":
        return "Previously flagged as no longer using Brightwheel"
    if f == "reason_school_temp_reducing_enrollment_count":
        return "Previously flagged for temporarily reducing enrollment"
    if f == "any_admission_funnel":
        return "Not using admissions" if v == 0 else "Uses admissions"

    # Changes over time: "X down 40% vs a year ago"
    m = re.match(r"delta_(.+)_(vs_t3m|vs_12m_ago)$", f)
    if m:
        base, per = m.groups()
        name = NAMES.get(base, base.replace("_", " ").capitalize())
        if pd.isna(v):
            return f"{name}: no comparison data vs {PERIOD[per]}"
        direction = "down" if v < 0 else "up"
        amount = f" {abs(v):.0%}" if not base.startswith("any_") else ""
        return f"{name} {direction}{amount} vs {PERIOD[per]}"

    # 3-month averages and current levels
    m = re.match(r"trailing_3m_(.+)$", f)
    base = m.group(1) if m else f
    name = NAMES.get(base, base.replace("_", " ").capitalize())
    suffix = " (3-month average)" if m else ""
    if base.startswith("any_"):
        return f"{'Low' if v < 0.5 else 'Some'} {name.lower()}{suffix}"
    return f"{name}{suffix}: {v:,.0f}" if pd.notna(v) else f"{name}{suffix}: none"

def top_reasons(i, n=3):
    row = may.iloc[i]
    pairs = sorted(zip(ACCEPTED_NEW, sv[i]), key=lambda x: x[1], reverse=True)
    return ", ".join([plain(f, row[f]) for f, v in pairs if v > 0][:n])

may["Top reasons for high score"] = [top_reasons(i) for i in range(len(may))]

out = sheet[["School ID"]].merge(may[["school_id", "Top reasons for high score"]],
                                 left_on="School ID", right_on="school_id", how="left").drop(columns="school_id")
out.to_csv(RAW_DIR / "v3_top_reasons_for_sheet.csv", index=False)
print(out.head(15).to_string(index=False))

In [ ]:
for f in ACCEPTED_NEW:
    print(f)

In [ ]:
import re

PERIOD = {"vs_t3m": "the last 3 months", "vs_12m_ago": "a year ago"}

# Plain-English names for every underlying signal
NAMES = {
    "activity_events_per_active_student": "Daily activity logging",
    "admissions_funnel_events_per_active_student": "Admissions activity",
    "attendance_events_per_active_student": "Attendance check-ins",
    "form_events_per_active_student": "Forms use",
    "messaging_events_per_active_student": "Messaging activity",
    "online_payment_events_per_active_student": "Online payments",
    "timecard_events_per_active_student": "Staff timecard use",
    "school_calendar_events_per_active_student": "Calendar activity",
    "billing_net_gpv_local": "Billing volume",
    "billing_net_gpv_local_card": "Card payment volume",
    "num_billing_invoices_due": "Billing invoices due",
    "num_billing_invoices_created": "Billing invoices sent",
    "num_billing_plans_created": "New billing plans",
    "num_students_with_billing_invoice_created": "Families billed",
    "pct_invoices_paid_online_proxy": "Share of invoices paid online",
    "product_active_students": "Active students",
    "school_payer_attach_rate": "Share of families paying through Brightwheel",
    "students_with_online_payment": "Families paying online",
    "students_with_payment_method_added": "Families with a payment method on file",
}
# For on/off "any_..." signals: what the school is or isn't using
ANY_NAMES = {
    "any_admission_funnel": "admissions", "any_calendar": "the calendar", "any_form": "forms",
    "any_attendance": "attendance tracking", "any_expense": "expense tracking",
    "any_messaging": "messaging", "any_payroll": "payroll",
}
PER_STUDENT = {k for k in NAMES if k.endswith("_per_active_student")}
MONEY = {"billing_net_gpv_local", "billing_net_gpv_local_card"}
SHARES = {"pct_invoices_paid_online_proxy", "school_payer_attach_rate"}

def blank(v):
    return v is None or pd.isna(v)

def name_of(base):
    return NAMES.get(base, base.replace("_", " ").capitalize())

def level(base, v):
    """Describe a current value, e.g. 'Active students: 12'."""
    if base in MONEY:
        return f"${v:,.0f}"
    if base in SHARES:
        return f"{v:.0%}"
    if base in PER_STUDENT:
        return f"{v:.1f} per student"
    return f"{v:,.0f}"

def plain(f, v, row):
    # --- School profile ---
    if f.startswith("customer_segment__"):
        return f"School size: {row.get('customer_segment', 'unknown')}"
    if f == "monthly_billing_status_local__5_billing_inactive":
        return "Billing has gone inactive" if v == 1 else \
               f"Billing status: {str(row.get('monthly_billing_status_local', 'unknown')).split('. ', 1)[-1]}"
    if f == "has_ever_billed":
        return "Has never used Brightwheel billing" if v == 0 else "Billing customer"
    if f == "months_since_first_processed_dollars_local":
        return "Has never processed a payment" if blank(v) else f"Billing customer for {int(v)} months"
    if f == "processed_dollars_rank_local":
        return "No billing history" if blank(v) else f"About {int(v)} months of billing history"

    # --- Autopay ---
    if f == "delta_enforces_autopay_binary":
        if blank(v) or v == 0:
            return "Autopay setting unchanged"
        return "Stopped requiring autopay recently" if v < 0 else "Started requiring autopay recently"
    if f == "prev_month_enforces_autopay_binary":
        return "Didn't require autopay last month" if v == 0 else "Required autopay last month"

    # --- Cancellation and payment history ---
    if f == "months_since_last_cancel_request":
        return "No past cancel requests on record" if blank(v) else f"Asked to cancel {int(v)} months ago"
    if f == "running_saas_cancel_request_count":
        return "No past cancel requests on record" if blank(v) or v == 0 else f"Has asked to cancel before ({int(v)} times)"
    if f == "months_since_last_failed_payment_case":
        return "No failed payments on record" if blank(v) else f"Had a failed payment {int(v)} months ago"
    if f == "running_saas_failed_payment_count":
        return "No failed subscription payments" if blank(v) or v == 0 else f"Past failed subscription payments: {int(v)}"
    if f == "negative_reason_count":
        return "No past negative case outcomes" if blank(v) or v == 0 else f"Past cases with a negative outcome: {int(v)}"
    if f == "reason_school_no_longer_using_bw_count":
        return "Previously flagged as no longer using Brightwheel" if not blank(v) and v > 0 else "Not flagged as no longer using Brightwheel"
    if f == "reason_school_temp_paused_count":
        return "Previously paused temporarily" if not blank(v) and v > 0 else "No temporary pauses on record"
    if f == "reason_school_temp_reducing_enrollment_count":
        return "Previously flagged for reducing enrollment" if not blank(v) and v > 0 else "Not flagged for reducing enrollment"

    # --- Changes over time ---
    m = re.match(r"delta_(.+)_(vs_t3m|vs_12m_ago)$", f)
    if m:
        base, per = m.groups()
        if base in ANY_NAMES:
            what = ANY_NAMES[base]
            if blank(v) or v == 0:
                return f"No change in use of {what} vs {PERIOD[per]}"
            return f"Stopped using {what} vs {PERIOD[per]}" if v < 0 else f"Started using {what} vs {PERIOD[per]}"
        name = name_of(base)
        if blank(v):
            return f"{name}: nothing to compare with {PERIOD[per]}"
        if v == 0:
            return f"{name} flat vs {PERIOD[per]}"
        if v <= -1:
            return f"{name} dropped to zero vs {PERIOD[per]}"
        direction = "down" if v < 0 else "up"
        if base in SHARES:
            return f"{name} {direction} vs {PERIOD[per]}"
        return f"{name} {direction} {abs(v):.0%} vs {PERIOD[per]}"

    # --- 3-month averages ---
    m = re.match(r"trailing_3m_(.+)$", f)
    if m:
        base = m.group(1)
        if base in ANY_NAMES:
            what = ANY_NAMES[base]
            if blank(v) or v < 0.5:
                return f"Rarely used {what} in the last 3 months"
            return f"Used {what} in most of the last 3 months"
        name = name_of(base)
        return f"{name} (3-month average): none" if blank(v) else f"{name} (3-month average): {level(base, v)}"

    # --- On/off usage this month ---
    if f in ANY_NAMES:
        return f"Not using {ANY_NAMES[f]}" if blank(v) or v == 0 else f"Uses {ANY_NAMES[f]}"

    # --- Current levels ---
    name = name_of(f)
    return f"{name}: none" if blank(v) else f"{name}: {level(f, v)}"

def safe_plain(f, v, row):
    try:
        return plain(f, v, row)
    except (ValueError, TypeError):
        return name_of(f)

def top_reasons(i, n=3):
    row = may.iloc[i]
    pairs = sorted(zip(ACCEPTED_NEW, sv[i]), key=lambda x: x[1], reverse=True)
    return ", ".join([safe_plain(f, row[f], row) for f, v in pairs if v > 0][:n])

may["Top reasons for high score"] = [top_reasons(i) for i in range(len(may))]

out = sheet[["School ID"]].merge(may[["school_id", "Top reasons for high score"]],
                                 left_on="School ID", right_on="school_id", how="left").drop(columns="school_id")
out.to_csv(RAW_DIR / "v3_top_reasons_for_sheet.csv", index=False)
print(out.head(20).to_string(index=False))

In [ ]:
flag_text = "reason: no longer using Brightwheel"
reasons = may["Top reasons for high score"].fillna("")
print(f"In top 3 reasons: {reasons.str.contains(flag_text).sum()} of {len(may)}")
print(f"The #1 reason: {reasons.str.split(', ').str[0].str.contains(flag_text).sum()} of {len(may)}")

In [ ]:
reasons = may["Top reasons for high score"].fillna("")
match = reasons.str.contains("no longer using Brightwheel", case=False)
top1 = reasons.str.split(", ").str[0].str.contains("no longer using Brightwheel", case=False)
print(f"In top 3 reasons: {match.sum()} of {len(may)}")
print(f"The #1 reason: {top1.sum()} of {len(may)}")
print(reasons[match].head(5).to_string())

In [ ]:
import re, joblib

PERIOD = {"vs_t3m": "the last 3 months", "vs_12m_ago": "a year ago"}
NAMES = {
    "activity_events_per_active_student": "Daily activity logging",
    "admissions_funnel_events_per_active_student": "Admissions activity",
    "attendance_events_per_active_student": "Attendance check-ins",
    "form_events_per_active_student": "Forms use",
    "messaging_events_per_active_student": "Messaging activity",
    "online_payment_events_per_active_student": "Online payment activity",
    "timecard_events_per_active_student": "Staff timecard use",
    "school_calendar_events_per_active_student": "Calendar activity",
    "billing_net_gpv_local": "Billing volume",
    "billing_net_gpv_local_card": "Card payment volume",
    "num_billing_invoices_due": "Invoices due this month",
    "num_billing_invoices_created": "Invoices created this month",
    "num_billing_plans_created": "New billing plans this month",
    "num_students_with_billing_invoice_created": "Students invoiced this month",
    "pct_invoices_paid_online_proxy": "Online payments relative to invoices due",
    "product_active_students": "Active students",
    "school_payer_attach_rate": "Share of active students paying online",
    "students_with_online_payment": "Students who paid online this month",
    "students_with_payment_method_added": "New payment methods added this month",
}
ANY_NAMES = {
    "any_admission_funnel": "admissions", "any_calendar": "the calendar", "any_form": "forms",
    "any_attendance": "attendance tracking", "any_expense": "expense tracking",
    "any_messaging": "messaging", "any_payroll": "payroll",
}
STOP_REASONS = {
    "reason_school_no_longer_using_bw_count": "no longer using Brightwheel",
    "reason_school_temp_paused_count": "temporary pause",
    "reason_school_temp_reducing_enrollment_count": "temporarily reducing enrollment",
    "negative_reason_count": "a negative reason",
}
PER_STUDENT = {k for k in NAMES if k.endswith("_per_active_student")}
MONEY = {"billing_net_gpv_local", "billing_net_gpv_local_card"}
SHARES = {"pct_invoices_paid_online_proxy", "school_payer_attach_rate"}

# How each school compares with all scored schools in May
pop = enc[enc["dim_month"] == "2026-05-01"]
Q = {}
for base in NAMES:
    for col in (base, "trailing_3m_" + base):
        if col in pop.columns and pd.api.types.is_numeric_dtype(pop[col]):
            Q[col] = (pop[col].quantile(0.25), pop[col].quantile(0.75))

def blank(v):
    return v is None or pd.isna(v)

def name_of(base):
    return NAMES.get(base, base.replace("_", " ").capitalize())

def fmt(base, v):
    if base in MONEY:
        return f"${v:,.0f}"
    if base in SHARES:
        return f"{v:.0%}"
    if base in PER_STUDENT:
        return f"{v:.1f} per student"
    return f"{v:,.0f}"

def vs_others(col, v):
    lo, hi = Q.get(col, (None, None))
    if lo is None or blank(v):
        return ""
    if v <= lo:
        return " (low vs other schools)"
    if v >= hi:
        return " (high vs other schools)"
    return ""

def plain(f, v, row):
    if f.startswith("customer_segment__"):
        return f"School size: {row.get('customer_segment', 'unknown')}"
    if f == "monthly_billing_status_local__5_billing_inactive":
        return "No billing payments in the last 2+ months" if v == 1 else \
               f"Billing status: {str(row.get('monthly_billing_status_local', 'unknown')).split('. ', 1)[-1]}"
    if f == "has_ever_billed":
        return "Has never used Brightwheel billing" if v == 0 else "Billing customer"
    if f == "months_since_first_processed_dollars_local":
        return "Has never processed a billing payment" if blank(v) else f"First billing payment {int(v)} months ago"
    if f == "processed_dollars_rank_local":
        return "No billing payments this month" if blank(v) else f"{int(v)} months with billing payments so far"
    if f == "delta_enforces_autopay_binary":
        if blank(v) or v == 0:
            return "Autopay setting unchanged"
        return "Stopped enforcing autopay this month" if v < 0 else "Started enforcing autopay this month"
    if f == "prev_month_enforces_autopay_binary":
        return "Autopay not enforced last month" if v == 0 else "Autopay enforced last month"
    if f in STOP_REASONS:
        reason = STOP_REASONS[f]
        if blank(v) or v == 0:
            return f"No billing plans stopped for {reason} this month"
        return f"Stopped {int(v)} billing plans this month, reason: {reason}"
    if f == "months_since_last_cancel_request":
        return "No past cancel requests on record" if blank(v) else f"Last asked to cancel {int(v)} months ago"
    if f == "running_saas_cancel_request_count":
        return "No past cancel requests on record" if blank(v) or v == 0 else f"Past cancel requests: {int(v)}"
    if f == "months_since_last_failed_payment_case":
        return "No failed subscription payments on record" if blank(v) else f"Failed subscription payment {int(v)} months ago"
    if f == "running_saas_failed_payment_count":
        return "No failed subscription payments on record" if blank(v) or v == 0 else f"Past failed subscription payments: {int(v)}"

    m = re.match(r"delta_(.+)_(vs_t3m|vs_12m_ago)$", f)
    if m:
        base, per = m.groups()
        if base in ANY_NAMES:
            what = ANY_NAMES[base]
            if per == "vs_12m_ago":
                if blank(v):
                    return f"Wasn't using {what} a year ago"
                return f"Stopped using {what} since a year ago" if v < 0 else f"Used {what} now and a year ago"
            if blank(v):
                return f"Didn't use {what} in the previous 3 months"
            if v < 0:
                return f"Stopped using {what} this month after using it recently"
            return f"Used {what} this month" + (" and in each of the last 3 months" if v == 0 else ", more than in recent months")
        name = name_of(base)
        if blank(v):
            return f"{name} was zero {PERIOD[per]}, so no comparison"
        if v == 0:
            return f"{name} flat vs {PERIOD[per]}"
        if v <= -1:
            return f"{name} dropped to zero vs {PERIOD[per]}"
        direction = "down" if v < 0 else "up"
        if base in SHARES:
            return f"{name} {direction} vs {PERIOD[per]}"
        return f"{name} {direction} {abs(v):.0%} vs {PERIOD[per]}"

    m = re.match(r"trailing_3m_(.+)$", f)
    if m:
        base = m.group(1)
        if base in ANY_NAMES:
            months = 0 if blank(v) else int(round(v * 3))
            return f"Used {ANY_NAMES[base]} in {months} of the last 3 months"
        name = name_of(base)
        return f"{name} (3-month average): none" if blank(v) else f"{name} (3-month average): {fmt(base, v)}{vs_others(f, v)}"

    if f in ANY_NAMES:
        return f"Didn't use {ANY_NAMES[f]} this month" if blank(v) or v == 0 else f"Used {ANY_NAMES[f]} this month"

    name = name_of(f)
    if blank(v):
        return f"{name}: none" + (" (no active students)" if f in PER_STUDENT else "")
    return f"{name}: {fmt(f, v)}{vs_others(f, v)}"

def safe_plain(f, v, row):
    try:
        return plain(f, v, row)
    except (ValueError, TypeError):
        return name_of(f)

def top_pairs(i, n=3):
    pairs = sorted(zip(ACCEPTED_NEW, sv[i]), key=lambda x: x[1], reverse=True)
    return [(f, v) for f, v in pairs if v > 0][:n]

may["Top reasons for high score"] = [
    ", ".join(safe_plain(f, may.iloc[i][f], may.iloc[i]) for f, _ in top_pairs(i)) for i in range(len(may))
]
may["Top features (technical)"] = [", ".join(f for f, _ in top_pairs(i)) for i in range(len(may))]

out = sheet[["School ID"]].merge(
    may[["school_id", "Top reasons for high score", "Top features (technical)"]],
    left_on="School ID", right_on="school_id", how="left",
).drop(columns="school_id")
out.to_csv(RAW_DIR / "v3_top_reasons_for_sheet.csv", index=False)
print(out.head(20).to_string(index=False))

joblib.dump(models, RAW_DIR / "v3_models_4seeds.joblib")
print("\nSaved the reasons file and the 4 models.")

In [ ]:
import re, joblib

PERIOD = {"vs_t3m": "the last 3 months", "vs_12m_ago": "a year ago"}
NAMES = {
    "activity_events_per_active_student": "Daily activity logging",
    "admissions_funnel_events_per_active_student": "Admissions activity",
    "attendance_events_per_active_student": "Attendance check-ins",
    "form_events_per_active_student": "Forms use",
    "messaging_events_per_active_student": "Messaging activity",
    "online_payment_events_per_active_student": "Online payment activity",
    "timecard_events_per_active_student": "Staff timecard use",
    "school_calendar_events_per_active_student": "Calendar activity",
    "billing_net_gpv_local": "Billing volume",
    "billing_net_gpv_local_card": "Card payment volume",
    "num_billing_invoices_due": "Invoices due this month",
    "num_billing_invoices_created": "Invoices created this month",
    "num_billing_plans_created": "New billing plans this month",
    "num_students_with_billing_invoice_created": "Students invoiced this month",
    "pct_invoices_paid_online_proxy": "Online payments relative to invoices due",
    "product_active_students": "Active students",
    "school_payer_attach_rate": "Share of active students paying online",
    "students_with_online_payment": "Students who paid online this month",
    "students_with_payment_method_added": "New payment methods added this month",
}
ANY_NAMES = {
    "any_admission_funnel": "admissions", "any_calendar": "the calendar", "any_form": "forms",
    "any_attendance": "attendance tracking", "any_expense": "expense tracking",
    "any_messaging": "messaging", "any_payroll": "payroll",
}
STOP_REASONS = {
    "reason_school_no_longer_using_bw_count": "no longer using Brightwheel",
    "reason_school_temp_paused_count": "temporary pause",
    "reason_school_temp_reducing_enrollment_count": "temporarily reducing enrollment",
    "negative_reason_count": "a negative reason",
}
COUNTS = {"num_billing_invoices_due", "num_billing_invoices_created", "num_billing_plans_created",
          "num_students_with_billing_invoice_created", "product_active_students",
          "students_with_online_payment", "students_with_payment_method_added"}
PER_STUDENT = {k for k in NAMES if k.endswith("_per_active_student")}
MONEY = {"billing_net_gpv_local", "billing_net_gpv_local_card"}
SHARES = {"pct_invoices_paid_online_proxy", "school_payer_attach_rate"}

# Typical ranges across all scored schools in May, used to word reasons as "Few/Many" or "Low/High"
pop = enc[enc["dim_month"] == "2026-05-01"]
Q = {}
for base in NAMES:
    for col in (base, "trailing_3m_" + base):
        if col in pop.columns and pd.api.types.is_numeric_dtype(pop[col]):
            Q[col] = (pop[col].quantile(0.25), pop[col].quantile(0.75))

def blank(v):
    return v is None or pd.isna(v)

def name_of(base):
    return NAMES.get(base, base.replace("_", " ").capitalize())

def fmt(base, v):
    if base in MONEY:
        return f"${v:,.0f}"
    if base in SHARES:
        return f"{v:.0%}"
    if base in PER_STUDENT:
        return f"{v:.1f} per student"
    return f"{v:,.0f}"

def level_text(col, base, v, suffix=""):
    name = name_of(base)
    lower = name[0].lower() + name[1:]
    val = fmt(base, v)
    if v == 0:
        return f"No {lower}{suffix}"
    lo, hi = Q.get(col, (None, None))
    if lo is not None and v <= lo:
        return f"{'Few' if base in COUNTS else 'Low'} {lower}{suffix} ({val})"
    if hi is not None and v >= hi:
        return f"{'Many' if base in COUNTS else 'High'} {lower}{suffix} ({val})"
    return f"{name}{suffix}: {val}"

def plain(f, v, row):
    # School profile
    if f.startswith("customer_segment__"):
        return f"School size: {row.get('customer_segment', 'unknown')}"
    if f == "monthly_billing_status_local__5_billing_inactive":
        return "No billing payments in the last 2+ months" if v == 1 else \
               f"Billing status: {str(row.get('monthly_billing_status_local', 'unknown')).split('. ', 1)[-1]}"
    if f == "has_ever_billed":
        return "Has never used Brightwheel billing" if v == 0 else "Billing customer"
    if f == "months_since_first_processed_dollars_local":
        return "Has never processed a billing payment" if blank(v) else f"First billing payment {int(v)} months ago"
    if f == "processed_dollars_rank_local":
        return "No billing payments this month" if blank(v) else f"{int(v)} months with billing payments so far"

    # Autopay
    if f == "delta_enforces_autopay_binary":
        if blank(v) or v == 0:
            return "Autopay setting unchanged"
        return "Stopped enforcing autopay this month" if v < 0 else "Started enforcing autopay this month"
    if f == "prev_month_enforces_autopay_binary":
        return "Autopay not enforced last month" if v == 0 else "Autopay enforced last month"

    # Billing plans stopped this month, by reason
    if f in STOP_REASONS:
        reason = STOP_REASONS[f]
        if blank(v) or v == 0:
            return f"No billing plans stopped for {reason} this month"
        return f"Stopped {int(v)} billing plans this month, reason: {reason}"

    # Cancellation and payment history
    if f == "months_since_last_cancel_request":
        return "No past cancel requests on record" if blank(v) else f"Last asked to cancel {int(v)} months ago"
    if f == "running_saas_cancel_request_count":
        return "No past cancel requests on record" if blank(v) or v == 0 else f"Past cancel requests: {int(v)}"
    if f == "months_since_last_failed_payment_case":
        return "No failed subscription payments on record" if blank(v) else f"Failed subscription payment {int(v)} months ago"
    if f == "running_saas_failed_payment_count":
        return "No failed subscription payments on record" if blank(v) or v == 0 else f"Past failed subscription payments: {int(v)}"

    # Changes over time
    m = re.match(r"delta_(.+)_(vs_t3m|vs_12m_ago)$", f)
    if m:
        base, per = m.groups()
        if base in ANY_NAMES:
            what = ANY_NAMES[base]
            if per == "vs_12m_ago":
                if blank(v):
                    return f"Wasn't using {what} a year ago"
                return f"Stopped using {what} since a year ago" if v < 0 else f"Used {what} now and a year ago"
            if blank(v):
                return f"Didn't use {what} in the previous 3 months"
            if v < 0:
                return f"Stopped using {what} this month after using it recently"
            return f"Used {what} this month" + (" and in each of the last 3 months" if v == 0 else ", more than in recent months")
        name = name_of(base)
        if blank(v):
            return f"No {name[0].lower() + name[1:]} {PERIOD[per]}, so no comparison"
        if v == 0:
            return f"{name} flat vs {PERIOD[per]}"
        if v <= -1:
            return f"{name} dropped to zero vs {PERIOD[per]}"
        direction = "down" if v < 0 else "up"
        if base in SHARES:
            return f"{name} {direction} vs {PERIOD[per]}"
        return f"{name} {direction} {abs(v):.0%} vs {PERIOD[per]}"

    # 3-month averages
    m = re.match(r"trailing_3m_(.+)$", f)
    if m:
        base = m.group(1)
        if base in ANY_NAMES:
            months = 0 if blank(v) else int(round(v * 3))
            return f"Used {ANY_NAMES[base]} in {months} of the last 3 months"
        name = name_of(base)
        return f"{name} (3-month average): none" if blank(v) else level_text(f, base, v, " (3-month average)")

    # On/off usage this month
    if f in ANY_NAMES:
        return f"Didn't use {ANY_NAMES[f]} this month" if blank(v) or v == 0 else f"Used {ANY_NAMES[f]} this month"

    # Current levels
    if blank(v):
        return f"{name_of(f)}: none" + (" (no active students)" if f in PER_STUDENT else "")
    return level_text(f, f, v)

def safe_plain(f, v, row):
    try:
        return plain(f, v, row)
    except (ValueError, TypeError):
        return name_of(f)

def top_pairs(i, n=3):
    pairs = sorted(zip(ACCEPTED_NEW, sv[i]), key=lambda x: x[1], reverse=True)
    return [(f, v) for f, v in pairs if v > 0][:n]

may["Top reasons for high score"] = [
    ", ".join(safe_plain(f, may.iloc[i][f], may.iloc[i]) for f, _ in top_pairs(i)) for i in range(len(may))
]
may["Top features (technical)"] = [", ".join(f for f, _ in top_pairs(i)) for i in range(len(may))]

out = sheet[["School ID"]].merge(
    may[["school_id", "Top reasons for high score", "Top features (technical)"]],
    left_on="School ID", right_on="school_id", how="left",
).drop(columns="school_id")
out.to_csv(RAW_DIR / "v3_top_reasons_for_sheet.csv", index=False)
print(out.head(20).to_string(index=False))

joblib.dump(models, RAW_DIR / "v3_models_4seeds.joblib")
print("\nSaved v3_top_reasons_for_sheet.csv and the 4 models.")

In [ ]:
def family(f):
    return re.sub(r"_vs_(t3m|12m_ago)$", "", re.sub(r"^(delta_|trailing_3m_)", "", f))

HEALTHY = re.compile(r"^High |^Many new billing|paid online this month up|^Card payment volume|^Students who paid online")

def clean(text):
    text = text.replace(" this month (3-month average)", " (3-month average)")
    m = re.match(r"^(.+?): none$", text)
    if m:
        name = m.group(1)
        text = f"No {name[0].lower() + name[1:]}"
    return text

def top_reasons_clean(i, n=3):
    row = may.iloc[i]
    pairs = sorted(zip(ACCEPTED_NEW, sv[i]), key=lambda x: x[1], reverse=True)
    out, seen = [], set()
    for f, v in pairs:
        if v <= 0 or len(out) == n:
            break
        fam = family(f)
        if fam in seen:
            continue
        text = clean(safe_plain(f, row[f], row))
        if HEALTHY.search(text):
            continue
        seen.add(fam)
        out.append(text)
    return ", ".join(out)

may["Top reasons for high score"] = [top_reasons_clean(i) for i in range(len(may))]
out = sheet[["School ID"]].merge(may[["school_id", "Top reasons for high score"]],
                                 left_on="School ID", right_on="school_id", how="left").drop(columns="school_id")
out.to_csv(RAW_DIR / "v3_top_reasons_for_sheet.csv", index=False)
print(out.head(15).to_string(index=False))

In [ ]:
HEALTHY = re.compile(r"^High |^Many (new billing|students who paid|active students)|paid online this month up| up \d+%|^Card payment volume|^Students who paid online")

_clean_prev = clean
def clean(text):
    text = _clean_prev(text)
    text = text.replace(" the last 3 months, so no comparison", " in the last 3 months, so no comparison")
    text = text.replace("New payment methods added this month down", "New payment methods down")
    return text

In [ ]:
import re, json, joblib, shap

# --- Settings ---
EXCLUDE_PREVIOUS_LIST = True
BILLING_LIST_IDS = []      # Holly's billing list: not available, Sruti will check overlap
SLEEPING_MONTHS = 6
TEST_ACCOUNT_PATTERN = r"sandbox|demo environment|\btest school\b|\[sandbox"

for fn in ["NAMES", "plain", "safe_plain", "clean", "family", "HEALTHY"]:
    assert fn in globals(), f"'{fn}' is missing: run the reasons code block first"

# --- Load everything ---
ACCEPTED_NEW = json.load(open(RAW_DIR / "accepted_features_h3.json"))
models = joblib.load(RAW_DIR / "v3_models_4seeds.joblib")
enc = pd.read_parquet(RAW_DIR / "churn_v1_is_reportable_request_202401_202605_h3_encoded.parquet")
prev = pd.read_csv(RAW_DIR / "V3.0 Proactive Churn Schools - Sheet1 (3).csv", header=1)
prev_ids = set(prev["School ID"].dropna().astype(int)) if EXCLUDE_PREVIOUS_LIST else set()

score_df = pd.read_parquet(RAW_DIR / "newlist_score_raw.parquet")
for name in ["last_active", "cases", "cust", "gpv"]:
    score_df = score_df.merge(pd.read_parquet(RAW_DIR / f"newlist_{name}.parquet"), on="school_id", how="left")
SCORE_MONTH = pd.to_datetime(score_df["dim_month"].iloc[0])

# --- Rebuild category columns exactly as in training, and prove it on training data ---
def slug(v):
    return re.sub(r"[^a-z0-9]+", "_", str(v).lower()).strip("_")

dummy_feats = [f for f in ACCEPTED_NEW if "__" in f]
raw_cats = sorted({f.split("__")[0] for f in dummy_feats})

def add_dummies(df):
    for f in dummy_feats:
        cat, lvl = f.split("__", 1)
        if lvl == "isna":
            df[f] = df[cat].isna().astype("int8")
        else:
            df[f] = (df[cat].map(lambda v: slug(v) if pd.notna(v) else None) == lvl).astype("int8")
    return df

chk = add_dummies(enc[raw_cats].copy())
assert all((chk[f].values == enc[f].values).all() for f in dummy_feats), "dummy encoding mismatch"
score_df = add_dummies(score_df)

num_feats = [f for f in ACCEPTED_NEW if f not in dummy_feats]
missing = [f for f in num_feats if f not in score_df.columns]
assert not missing, f"features missing from the scoring pull: {missing}"
for f in num_feats:
    score_df[f] = pd.to_numeric(score_df[f], errors="coerce")

# --- Score with the saved 4-model V3.0 ensemble ---
score_df["Churn risk score"] = np.mean(
    [m.predict_proba(score_df[ACCEPTED_NEW])[:, 1] for m in models], axis=0
)

# --- Filters ---
for c in ["last_active_month", "last_completed_cancel", "last_cancel_request"]:
    score_df[c] = pd.to_datetime(score_df[c])

def months_between(later, earlier):
    return (later.year - earlier.dt.year) * 12 + (later.month - earlier.dt.month)

score_df["months_inactive"] = months_between(SCORE_MONTH, score_df["last_active_month"])
cutoff_6m = pd.Timestamp.today() - pd.DateOffset(months=6)
as_bool = lambda s: s.fillna(False).astype(bool)
closing = score_df["reason_school_permanently_closing_count"] if "reason_school_permanently_closing_count" in score_df else pd.Series(0, index=score_df.index)

rules = [
    ("Not currently a paying Premium customer",
     ~as_bool(score_df["is_current_saas_subscription_premium"]) | as_bool(score_df["is_current_saas_subscription_canceled"])),
    ("Never used Brightwheel billing", score_df["has_ever_billed"] != 1),
    (f"Sleeping dog: no active students for {SLEEPING_MONTHS}+ months",
     score_df["last_active_month"].isna() | (score_df["months_inactive"] >= SLEEPING_MONTHS)),
    ("Ghost: completed cancellation 2+ months earlier",
     score_df["last_completed_cancel"].notna() & (months_between(SCORE_MONTH, score_df["last_completed_cancel"]) >= 2)),
    ("Cancel request in the last 6 months or open cancel case",
     (score_df["last_cancel_request"] >= cutoff_6m) | as_bool(score_df["has_open_churn_case"])),
    ("Permanently closing", closing.fillna(0) > 0),
    ("Test or sandbox account", score_df["account_name"].fillna("").str.contains(TEST_ACCOUNT_PATTERN, case=False, regex=True)),
    ("On the previous V3.0 list", score_df["school_id"].isin(prev_ids)),
    ("On Holly's billing-churn list", score_df["school_id"].isin(set(BILLING_LIST_IDS))),
]

pool, removed = score_df.copy(), []
print(f"Starting population: {len(pool):,}")
for reason, mask in rules:
    hit = pool[mask.loc[pool.index]]
    removed.append(hit.assign(removed_reason=reason))
    pool = pool.drop(hit.index)
    print(f"  - {reason}: {len(hit):,} removed, {len(pool):,} left")

# --- Top 200 ---
top = pool.sort_values("Churn risk score", ascending=False).head(200).reset_index(drop=True)

# --- Reasons, averaged across the 4 models ---
Q = {}
for base in NAMES:
    for col in (base, "trailing_3m_" + base):
        if col in score_df.columns and pd.api.types.is_numeric_dtype(score_df[col]):
            Q[col] = (score_df[col].quantile(0.25), score_df[col].quantile(0.75))

sv_top = np.mean([shap.TreeExplainer(m).shap_values(top[ACCEPTED_NEW]) for m in models], axis=0)

def reasons_for(i, n=3):
    row = top.iloc[i]
    out, seen = [], set()
    for f, v in sorted(zip(ACCEPTED_NEW, sv_top[i]), key=lambda x: x[1], reverse=True):
        if v <= 0 or len(out) == n:
            break
        if family(f) in seen:
            continue
        text = clean(safe_plain(f, row[f], row))
        if HEALTHY.search(text):
            continue
        seen.add(family(f))
        out.append(text)
    return ", ".join(out)

top["Top reasons for high score"] = [reasons_for(i) for i in range(len(top))]

# --- Final list ---
final = pd.DataFrame({
    "Rank": range(1, len(top) + 1),
    "School ID": top["school_id"],
    "School UUID": top["brightwheel_school_uuid"],
    "Account Name": top["account_name"],
    "Churn risk score": top["Churn risk score"].round(3),
    "Top reasons for high score": top["Top reasons for high score"],
    "Avg monthly GPV (non-zero months)": top["avg_monthly_gpv_nonzero"].round(0),
    "Model Used": "v3.0",
})

stamp = SCORE_MONTH.strftime("%Y_%m")
final.to_csv(RAW_DIR / f"v3_list_{stamp}.csv", index=False)

removed_log = pd.concat(removed)[["school_id", "account_name", "Churn risk score",
                                  "avg_monthly_gpv_nonzero", "removed_reason"]]
removed_log.sort_values("Churn risk score", ascending=False).to_csv(
    RAW_DIR / f"v3_list_{stamp}_removed.csv", index=False)

print(f"\nFinal list: {len(final)} schools")
print(final.head(10).to_string(index=False))

In [ ]:
e = enc.sort_values(["school_id", "dim_month"]).copy()
last = e["dim_month"].where(e["product_active_students"] > 0).groupby(e["school_id"]).ffill()
e["months_inactive"] = (e["dim_month"].dt.year - last.dt.year) * 12 + (e["dim_month"].dt.month - last.dt.month)

def backtest(month, k=200):
    m = e[e["dim_month"] == month].copy()
    m["score"] = np.mean([mdl.predict_proba(m[ACCEPTED_NEW])[:, 1] for mdl in models], axis=0)
    closing = m["reason_school_permanently_closing_count"].fillna(0) if "reason_school_permanently_closing_count" in m else 0
    keep = ((m["has_ever_billed"] == 1)
            & m["months_inactive"].notna() & (m["months_inactive"] < 6)
            & ~(m["months_since_last_cancel_request"] <= 6)
            & ~(closing > 0))
    f = m[keep]
    top_all, top_f = m.nlargest(k, "score"), f.nlargest(k, "score")
    print(f"{month[:7]} | unfiltered top {k}: {top_all[Y_COL].mean():.1%} | "
          f"filtered top {k}: {top_f[Y_COL].mean():.1%} | "
          f"normal rate (filtered schools): {f[Y_COL].mean():.1%} | "
          f"lift: {top_f[Y_COL].mean() / f[Y_COL].mean():.1f}x")

for month in ["2026-03-01", "2026-04-01", "2026-05-01"]:
    backtest(month)

In [ ]:
f = top.copy()
checks = {
    "200 schools": len(final) == 200,
    "No duplicate schools": final["School ID"].is_unique,
    "No missing UUID, name, score, or reasons": final[["School UUID", "Account Name", "Churn risk score",
                                                       "Top reasons for high score"]].notna().all().all(),
    "Ranked by score, highest first": final["Churn risk score"].is_monotonic_decreasing,
    "All currently paying Premium": f["is_current_saas_subscription_premium"].fillna(False).astype(bool).all(),
    "None currently cancelled": not f["is_current_saas_subscription_canceled"].fillna(False).astype(bool).any(),
    "All have used billing": (f["has_ever_billed"] == 1).all(),
    "All active within 6 months": (f["months_inactive"] < 6).all(),
    "No cancel request in last 6 months": not (f["last_cancel_request"] >= cutoff_6m).any(),
    "No open cancel cases": not f["has_open_churn_case"].fillna(False).astype(bool).any(),
    "None from the first list": not f["school_id"].isin(prev_ids).any(),
}
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

In [ ]:
gpv_check = top[["school_id", "account_name", "avg_monthly_gpv_nonzero", "billing_net_gpv_local"]].rename(
    columns={"avg_monthly_gpv_nonzero": "avg_gpv_past_year", "billing_net_gpv_local": "gpv_september"})
gpv_check["share_of_usual"] = gpv_check["gpv_september"] / gpv_check["avg_gpv_past_year"]

high = gpv_check[gpv_check["avg_gpv_past_year"] >= 10000].sort_values("avg_gpv_past_year", ascending=False)
print(f"High-value schools ($10k+/month): {len(high)}")
print(f"  of those, September billing near zero: {(high['share_of_usual'] < 0.1).sum()}")
print(high.head(20).to_string(index=False))

In [ ]:
m = e[e["dim_month"] == "2026-05-01"].copy()
m["score"] = np.mean([mdl.predict_proba(m[ACCEPTED_NEW])[:, 1] for mdl in models], axis=0)
m["score_band"] = pd.cut(m["score"], [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 1.0])
print(m.groupby("score_band", observed=True).agg(
    schools=("score", "size"), avg_score=("score", "mean"), actually_cancelled=(Y_COL, "mean")).round(3).to_string())

In [ ]:
import re, json, joblib, shap

EXCLUDE_PREVIOUS_LIST = True
BILLING_LIST_IDS = []      # Sruti will check overlap with Holly's billing list
SLEEPING_MONTHS = 6
TEST_ACCOUNT_PATTERN = r"sandbox|demo environment|\btest school\b|\[sandbox"

for fn in ["NAMES", "plain", "safe_plain", "clean", "family", "HEALTHY"]:
    assert fn in globals(), f"'{fn}' is missing: run the reasons code block first"

ACCEPTED_NEW = json.load(open(RAW_DIR / "accepted_features_h3.json"))
models = joblib.load(RAW_DIR / "v3_models_4seeds.joblib")
enc = pd.read_parquet(RAW_DIR / "churn_v1_is_reportable_request_202401_202605_h3_encoded.parquet")
prev = pd.read_csv(RAW_DIR / "V3.0 Proactive Churn Schools - Sheet1 (3).csv", header=1)
prev_ids = set(prev["School ID"].dropna().astype(int)) if EXCLUDE_PREVIOUS_LIST else set()

score_df = pd.read_parquet(RAW_DIR / "newlist_score_raw.parquet")
for name in ["last_active", "cases", "cust", "gpv", "logins"]:
    score_df = score_df.merge(pd.read_parquet(RAW_DIR / f"newlist_{name}.parquet"), on="school_id", how="left")
SCORE_MONTH = pd.to_datetime(score_df["dim_month"].iloc[0])
score_df["days_since_login"] = (pd.Timestamp.today()
                                - pd.to_datetime(score_df["last_mobile_or_web_activity_date"])).dt.days
print("Scoring month:", SCORE_MONTH.strftime("%Y-%m"))

# Rebuild category columns exactly as in training
def slug(v):
    return re.sub(r"[^a-z0-9]+", "_", str(v).lower()).strip("_")
dummy_feats = [f for f in ACCEPTED_NEW if "__" in f]
raw_cats = sorted({f.split("__")[0] for f in dummy_feats})
def add_dummies(df):
    for f in dummy_feats:
        cat, lvl = f.split("__", 1)
        if lvl == "isna":
            df[f] = df[cat].isna().astype("int8")
        else:
            df[f] = (df[cat].map(lambda v: slug(v) if pd.notna(v) else None) == lvl).astype("int8")
    return df
chk = add_dummies(enc[raw_cats].copy())
assert all((chk[f].values == enc[f].values).all() for f in dummy_feats), "dummy encoding mismatch"
score_df = add_dummies(score_df)
num_feats = [f for f in ACCEPTED_NEW if f not in dummy_feats]
missing = [f for f in num_feats if f not in score_df.columns]
assert not missing, f"features missing: {missing}"
for f in num_feats:
    score_df[f] = pd.to_numeric(score_df[f], errors="coerce")

# Score
score_df["Churn risk score"] = np.mean([m.predict_proba(score_df[ACCEPTED_NEW])[:, 1] for m in models], axis=0)

# Filters
for c in ["last_active_month", "last_completed_cancel", "last_cancel_request"]:
    score_df[c] = pd.to_datetime(score_df[c])
def months_between(later, earlier):
    return (later.year - earlier.dt.year) * 12 + (later.month - earlier.dt.month)
score_df["months_inactive"] = months_between(SCORE_MONTH, score_df["last_active_month"])
cutoff_6m = pd.Timestamp.today() - pd.DateOffset(months=6)
as_bool = lambda s: s.fillna(False).astype(bool)
closing = score_df["reason_school_permanently_closing_count"] if "reason_school_permanently_closing_count" in score_df else pd.Series(0, index=score_df.index)

rules = [
    ("Not currently a paying Premium customer",
     ~as_bool(score_df["is_current_saas_subscription_premium"]) | as_bool(score_df["is_current_saas_subscription_canceled"])),
    ("Never used Brightwheel billing", score_df["has_ever_billed"] != 1),
    (f"Sleeping dog: no active students for {SLEEPING_MONTHS}+ months",
     score_df["last_active_month"].isna() | (score_df["months_inactive"] >= SLEEPING_MONTHS)),
    ("Sleeping dog: no admin login in 180+ days",
     score_df["days_since_login"].isna() | (score_df["days_since_login"] >= 180)),
    ("Ghost: completed cancellation 2+ months earlier",
     score_df["last_completed_cancel"].notna() & (months_between(SCORE_MONTH, score_df["last_completed_cancel"]) >= 2)),
    ("Cancel request in the last 6 months or open cancel case",
     (score_df["last_cancel_request"] >= cutoff_6m) | as_bool(score_df["has_open_churn_case"])),
    ("Permanently closing", closing.fillna(0) > 0),
    ("Test or sandbox account", score_df["account_name"].fillna("").str.contains(TEST_ACCOUNT_PATTERN, case=False, regex=True)),
    ("On the previous V3.0 list", score_df["school_id"].isin(prev_ids)),
    ("On Holly's billing-churn list", score_df["school_id"].isin(set(BILLING_LIST_IDS))),
]
pool, removed = score_df.copy(), []
print(f"Starting population: {len(pool):,}")
for reason, mask in rules:
    hit = pool[mask.loc[pool.index]]
    removed.append(hit.assign(removed_reason=reason))
    pool = pool.drop(hit.index)
    print(f"  - {reason}: {len(hit):,} removed, {len(pool):,} left")

# Top 200 and reasons
top = pool.sort_values("Churn risk score", ascending=False).head(200).reset_index(drop=True)
Q = {}
for base in NAMES:
    for col in (base, "trailing_3m_" + base):
        if col in score_df.columns and pd.api.types.is_numeric_dtype(score_df[col]):
            Q[col] = (score_df[col].quantile(0.25), score_df[col].quantile(0.75))
sv_top = np.mean([shap.TreeExplainer(m).shap_values(top[ACCEPTED_NEW]) for m in models], axis=0)
def reasons_for(i, n=3):
    row = top.iloc[i]
    out, seen = [], set()
    for f, v in sorted(zip(ACCEPTED_NEW, sv_top[i]), key=lambda x: x[1], reverse=True):
        if v <= 0 or len(out) == n:
            break
        if family(f) in seen:
            continue
        text = clean(safe_plain(f, row[f], row))
        if HEALTHY.search(text):
            continue
        seen.add(family(f))
        out.append(text)
    return ", ".join(out)
top["Top reasons for high score"] = [reasons_for(i) for i in range(len(top))]

final = pd.DataFrame({
    "Rank": range(1, len(top) + 1),
    "School ID": top["school_id"],
    "School UUID": top["brightwheel_school_uuid"],
    "Account Name": top["account_name"],
    "Churn risk score": top["Churn risk score"].round(3),
    "Top reasons for high score": top["Top reasons for high score"],
    "Avg monthly GPV (non-zero months)": top["avg_monthly_gpv_nonzero"].round(0),
    "Model Used": "v3.0",
})
stamp = SCORE_MONTH.strftime("%Y_%m")
final.to_csv(RAW_DIR / f"v3_list_{stamp}.csv", index=False)
pd.concat(removed)[["school_id", "account_name", "Churn risk score", "avg_monthly_gpv_nonzero", "removed_reason"]] \
    .sort_values("Churn risk score", ascending=False).to_csv(RAW_DIR / f"v3_list_{stamp}_removed.csv", index=False)
print(f"\nFinal list: {len(final)} schools, saved as v3_list_{stamp}.csv")
print(final.head(10).to_string(index=False))

# Checks
checks = {
    "200 schools": len(final) == 200,
    "No duplicate schools": final["School ID"].is_unique,
    "No missing UUID, name, score, or reasons": final[["School UUID", "Account Name", "Churn risk score",
                                                       "Top reasons for high score"]].notna().all().all(),
    "Ranked by score": final["Churn risk score"].is_monotonic_decreasing,
    "All currently paying Premium": as_bool(top["is_current_saas_subscription_premium"]).all(),
    "None currently cancelled": not as_bool(top["is_current_saas_subscription_canceled"]).any(),
    "All have used billing": (top["has_ever_billed"] == 1).all(),
    "All active within 6 months": (top["months_inactive"] < 6).all(),
    "No cancel request in last 6 months": not (top["last_cancel_request"] >= cutoff_6m).any(),
    "No open cancel cases": not as_bool(top["has_open_churn_case"]).any(),
    "None from the first list": not top["school_id"].isin(prev_ids).any(),
}
print()
for name, ok in checks.items():
    print(f"{'PASS' if ok else 'FAIL'}  {name}")

In [ ]:
login_removed = next(r for r in removed if (r["removed_reason"] == "Sleeping dog: no admin login in 180+ days").all())
cols = ["school_id", "account_name", "Churn risk score", "days_since_login", "product_active_students",
        "billing_net_gpv_local", "avg_monthly_gpv_nonzero", "months_inactive"]
view = login_removed[cols].rename(columns={
    "product_active_students": "active_students_oct",
    "billing_net_gpv_local": "billing_oct",
    "avg_monthly_gpv_nonzero": "avg_gpv_past_year",
}).sort_values("Churn risk score", ascending=False)

print(f"Removed by the login rule: {len(view)}")
print(f"  never logged in (no date): {view['days_since_login'].isna().sum()}")
print(f"  would have made the top 200: {(view['Churn risk score'] >= final['Churn risk score'].min()).sum()}")
print(f"  still billing in October: {(view['billing_oct'] > 0).sum()}")
print(view.round(1).to_string(index=False))

In [ ]:
cutoff = final["Churn risk score"].min()
print(view[view["Churn risk score"] >= cutoff].round(1).to_string(index=False))

In [ ]:
def backtest_pr(month, k=200):
    m = e[e["dim_month"] == month].copy()
    m["score"] = np.mean([mdl.predict_proba(m[ACCEPTED_NEW])[:, 1] for mdl in models], axis=0)
    closing = m["reason_school_permanently_closing_count"].fillna(0) if "reason_school_permanently_closing_count" in m else 0
    keep = ((m["has_ever_billed"] == 1) & m["months_inactive"].notna() & (m["months_inactive"] < 6)
            & ~(m["months_since_last_cancel_request"] <= 6) & ~(closing > 0))
    top = m[keep].nlargest(k, "score")
    caught, all_cancels = int(top[Y_COL].sum()), int(m[Y_COL].sum())
    print(f"{month[:7]} | precision: {caught}/{k} = {caught / k:.1%} | "
          f"recall: {caught}/{all_cancels} = {caught / all_cancels:.1%} | best possible recall: {k / all_cancels:.1%}")

for month in ["2026-03-01", "2026-04-01", "2026-05-01"]:
    backtest_pr(month)

In [ ]:
m = enc[enc["dim_month"] == "2026-05-01"].copy()
m["score"] = np.mean([mdl.predict_proba(m[ACCEPTED_NEW])[:, 1] for mdl in models], axis=0)
m["rank"] = m["score"].rank(ascending=False, method="first")
n, n_cancel = len(m), int(m[Y_COL].sum())
print(f"Schools scored in May: {n:,} | asked to cancel in June–August: {n_cancel:,}\n")

# 1. How many cancellations fall in the model's top-ranked schools
print("Cancellations caught, by how far down the ranking you go:")
for k in [200, 500, 1000, 2000, 5000, 10000]:
    caught = int(m.loc[m["rank"] <= k, Y_COL].sum())
    print(f"  Top {k:>6,} ({k / n:5.1%} of schools): {caught:>4} of {n_cancel} ({caught / n_cancel:.0%})")

# 2. How cancelling schools were spread across the ranking
m["rank_group"] = pd.cut(m["rank"] / n, [0, 0.1, 0.25, 0.5, 1.0],
                         labels=["Top 10%", "Next 15%", "Next 25%", "Bottom 50%"])
print("\nWhere the cancelling schools ranked:")
print(m[m[Y_COL] == 1]["rank_group"].value_counts().reindex(["Top 10%", "Next 15%", "Next 25%", "Bottom 50%"]).to_string())

# 3. What the missed schools look like vs the caught ones
cut = 2000
cancelled = m[m[Y_COL] == 1].copy()
cancelled["group"] = np.where(cancelled["rank"] <= cut, f"caught (top {cut:,})", "missed")
compare = cancelled.groupby("group").agg(
    schools=("score", "size"),
    avg_score=("score", "mean"),
    months_billing=("months_since_first_processed_dollars_local", "median"),
    active_students=("product_active_students", "median"),
    no_year_ago_history=("delta_product_active_students_vs_12m_ago", lambda s: s.isna().mean()),
    ever_billed=("has_ever_billed", "mean"),
).round(2)
print(f"\nCaught vs missed cancellations (cutoff: top {cut:,}):")
print(compare.to_string())